## Federated Fine-Tuning of Phi-4 Using OpenFL


In this tutorial, we demonstrate how to fine-tune Microsoft's Phi-4 model in a federated learning workflow.

We will fine-tune **Microsoft's [Phi4](https://huggingface.co/microsoft/phi-4)** model using a diverse dataset such as [Math_10k](https://github.com/AGI-Edgerunners/LLM-Adapters/tree/main), an open-source dataset containing mathematical question-answer pairs collected from various smaller math datasets.

# The Workflow Interface

The workflow interface is an innovative approach to designing federated learning experiments with OpenFL. It was developed in response to discussions with researchers and users who had unique use cases that didn’t perfectly align with the traditional horizontal federated learning model. This interface enables more flexible compositions of experiments, allowing for greater customization and adaptability in complex, real-world scenarios

### Installing OpenFL
Lets now install OpenFL and the necessary dependencies for the workflow interface by running the cell below:

In [ ]:
!pip install git+https://github.com/intel/openfl.git --no-cache
!cd PATH_TO_OPENFL_DIR/openfl-tutorials/experimental/workflow && pip install -r workflow_interface_requirements.txt --no-cache

In [ ]:
# Install dependencies 
!pip install torch transformers peft datasets trl==0.12.2 gradio -q

## Import libraries

In [ ]:
import hashlib
import os

import numpy as np
import requests
import torch
import transformers
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers.trainer_callback import PrinterCallback
from trl import SFTTrainer
import gradio as gr

from openfl.experimental.workflow.interface import Aggregator, Collaborator, FLSpec
from openfl.experimental.workflow.placement import aggregator, collaborator
from openfl.experimental.workflow.runtime import LocalRuntime

### Acquiring and preprocessing dataset

We can download the dataset directly from the [LLM-Adapters repository](https://github.com/AGI-Edgerunners/LLM-Adapters)

In [ ]:
def file_checksum(file_path, algorithm="sha256"):
    """
    Calculate the checksum of a file using the specified hashing algorithm.

    Parameters:
    file_path (str): The path to the file for which the checksum is to be calculated.
    algorithm (str): The hashing algorithm to use (default is 'sha256').

    Returns:
    str: The calculated checksum of the file.
    """
    hash_func = hashlib.new(algorithm)
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_func.update(chunk)
    return hash_func.hexdigest()


if not os.path.exists("math_10k.json"):
    r = requests.get(
        "https://raw.githubusercontent.com/AGI-Edgerunners/LLM-Adapters/main/ft-training_set/math_10k.json",
        # "math_10k.json", timeout=10
    )
    with open(
        "math_10k.json",
        "wb",
    ) as f:
        f.write(r.content)

    actual_checksum = file_checksum("math_10k.json")
    if (
        actual_checksum
        != "0342d0d860ad8592b579329337c90e42eefd3d9f2898043140cbd120630418b8"
    ):
        raise ValueError(
            "Checksum verification failed. The file may have been altered."
        )

raw_dataset = load_dataset("json", data_files="math_10k.json")

## Initialize arguments and configurations

In [ ]:
training_config = {
    "bf16": True,
    "use_ipex": False,
    "use_cpu": True,
    "do_eval": False,
    "learning_rate": 5.0e-06,
    "log_level": "info",
    "logging_steps": 20,
    "logging_strategy": "steps",
    "lr_scheduler_type": "cosine",
    "num_train_epochs": 1,
    "max_steps": -1,
    "output_dir": "./checkpoint_dir",
    "overwrite_output_dir": True,
    "per_device_eval_batch_size": 1,
    "per_device_train_batch_size": 1,
    "remove_unused_columns": True,
    "save_steps": 100,
    "save_total_limit": 1,
    "seed": 0,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "gradient_accumulation_steps": 1,
    "warmup_ratio": 0.2,
}

peft_config = {
    "r": 1,
    "lora_alpha": 2,
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
    "target_modules": "all-linear",
    "modules_to_save": None,
}
model_kwargs = dict(
    use_cache=False,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map=None,
)
train_conf = TrainingArguments(**training_config)
peft_conf = LoraConfig(**peft_config)

## Load and initialize model

In [ ]:
checkpoint_path = "NyxKrage/Microsoft_Phi-4"
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_path, return_dict=True, **model_kwargs
)
model = get_peft_model(model, peft_conf)

tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
sequence_max_length = 512
val_set_size = 2000
tokenizer.pad_token_id = 0  # unk. we want this to be different from the eos token
tokenizer.padding_side = "left"  # Allow batched inference

## Preprocess dataset

In [ ]:
def generate_prompt(data_point):
    """
    Generate a prompt based on the given data point.

    Parameters:
    data_point (dict): A dictionary containing the instruction, input, and output.

    Returns:
    str: The generated prompt as a string.
    """
    if data_point["input"]:
        return f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. 

                ### Instruction:
                {data_point["instruction"]}
                
                ### Input:
                {data_point["input"]}
                
                ### Response:
                {data_point["output"]}"""
    else:
        return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.  

                ### Instruction:
                {data_point["instruction"]}
                
                ### Response:
                {data_point["output"]}"""


def tokenize(prompt, add_eos_token=True):
    """
    Tokenize the given prompt.

    Parameters:
    prompt (str): The prompt to be tokenized.
    add_eos_token (bool): Whether to add an end-of-sequence token (default is True).

    Returns:
    dict: A dictionary containing the tokenized input IDs and attention mask.
    """
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=sequence_max_length,
        padding=False,
        return_tensors=None,
    )
    if (
        result["input_ids"][-1] != tokenizer.eos_token_id
        and len(result["input_ids"]) < sequence_max_length
        and add_eos_token
    ):
        result["input_ids"].append(tokenizer.eos_token_id)
        result["attention_mask"].append(1)

    result["labels"] = result["input_ids"].copy()

    return result


def generate_and_tokenize_prompt(data_point):
    """
    Generate and tokenize a prompt based on the given data point.

    Parameters:
    data_point (dict): A dictionary containing the instruction, input, and output.

    Returns:
    dict: A dictionary containing the tokenized input IDs, attention mask, and labels.
    """
    full_prompt = generate_prompt(data_point)
    tokenized_full_prompt = tokenize(full_prompt)
    user_prompt = generate_prompt({**data_point, "output": ""})
    tokenized_user_prompt = tokenize(user_prompt, add_eos_token=False)
    user_prompt_len = len(tokenized_user_prompt["input_ids"])

    tokenized_full_prompt["labels"] = [-100] * user_prompt_len + tokenized_full_prompt[
        "labels"
    ][user_prompt_len:]
    return tokenized_full_prompt


train_val = raw_dataset["train"].train_test_split(
    test_size=val_set_size, shuffle=True, seed=42
)

processed_train_dataset = train_val["train"].shuffle().map(generate_and_tokenize_prompt).select(range(3))
processed_test_dataset = train_val["test"].shuffle().map(generate_and_tokenize_prompt).select(range(3))

Next we import the FLSpec, LocalRuntime, and placement decorators.

 - `FLSpec` – Defines the flow specification. User defined flows are subclasses of this.
 - `Runtime` – Defines where the flow runs, infrastructure for task transitions (how information gets sent). The LocalRuntime runs the flow on a single node.
 - `aggregator/collaborator` - placement decorators that define where the task will be assigned

In [ ]:
def FedAvg(peft_params, model, weights=None):
    """
    Perform Federated Averaging (FedAvg) on the model parameters.

    Parameters:
    peft_params (list): A list of state dictionaries containing the model parameters from different clients.
    model (torch.nn.Module): The model to which the averaged parameters will be applied.
    weights (list, optional): A list of weights for averaging the parameters. If None, equal weights are used.

    Returns:
    torch.nn.Module: The model with the averaged parameters applied.
    """
    state_dicts = peft_params
    state_dict = get_peft_model_state_dict(model)
    for key in peft_params[0]:
        dtype = state_dicts[0][key].dtype
        state_dict[key] = torch.from_numpy(
            np.average(
                [state[key].to(torch.float).numpy() for state in state_dicts], axis=0, weights=weights
            )
        ).to(dtype)
    set_peft_model_state_dict(model, state_dict)
    return model

Now we come to the flow definition. The OpenFL Workflow Interface adopts the conventions set by Metaflow, that every workflow begins with `start` and concludes with the `end` task. The aggregator begins with an optionally passed in model and optimizer. The aggregator begins the flow with the `start` task, where the list of collaborators is extracted from the runtime (`self.collaborators = self.runtime.collaborators`) and is then used as the list of participants to run the task listed in `self.next`, `aggregated_model_validation`. The model, optimizer, and anything that is not explicitly excluded from the next function will be passed from the `start` function on the aggregator to the `aggregated_model_validation` task on the collaborator. Where the tasks run is determined by the placement decorator that precedes each task definition (`@aggregator` or `@collaborator`). Once each of the collaborators (defined in the runtime) complete the `aggregated_model_validation` task, they pass their current state onto the `train` task, from `train` to `local_model_validation`, and then finally to `join` at the aggregator. It is in `join` that an average is taken of the model weights, and the next round can begin.

(data:image/png%3Bbase64%2CiVBORw0KGgoAAAANSUhEUgAAA4sAAAK/CAYAAADAj1hsAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAJcEhZcwAAEnQAABJ0Ad5mH3gAAP%2BlSURBVHhe7J0FYFzXsUDHttgW2jIzO8bEDsdhpjZNGdNfpl9mhl/GtCmklFKaNmnTBpqkYQcccBI7jplJtiVZspjlP2d2r/28khyDtJKsOc7Lrnbf7j64c%2B/QndtvnyKO4ziO4ziO4ziOE6F//NFxHMdxHMdxHMdx9uPGouM4juM4juM4jtMGNxYdx3Ecx3Ecx3GcNrix6DiO4ziO4ziO47TBjUXHcRzHcRzHcRynDW4sOo7jOI7jOI7jOG1wY9FxHMdxHMdxHMdpgxuLjuM4juM4juM4ThvcWHQcx3Ecx3Ecx3Ha4Mai4ziO4ziO4ziO0wY3Fh3HcRzHcRzHcZw2uLHoOI7jOI7jOI7jtMGNRcdxHMdxHMdxHKcNbiw6juM4juM4juM4bXBj0XEcx3Ecx3Ecx2mDG4uO4ziO4ziO4zhOG9xYdBzHcRzHcRzHcdrgxqLjOI7jOI7jOI7TBjcWHcdxHMdxHMdxnDa4seg4juM4juM4juO0wY1Fx3Ecx3Ecx3Ecpw1uLDqO4ziO4ziO4zhtcGPRcRzHcRzHcRzHaYMbi47jOI7jOI7jOE4b3Fh0HMdxHMdxHMdx2uDGouM4juM4juM4jtMGNxYdx3Ecx3Ecx3GcNrix6DiO4ziO4ziO47TBjUXHcRzHcRzHcRynDW4sOo7jOI7jOI7jOG1wY9FxHMdxHMdxHMdpgxuLjuM4juM4juM4ThvcWHQcx3Ecx3Ecx3Ha4Mai4ziO4ziO4ziO0wY3Fh3HcRzHcRzHcZw2uLHoOI7jOI7jOI7jtMGNRcdxHMdxHMdxHKcNbiw6juM4juM4juM4bei3T4k/dxzHcdqhVXvJZv1fi25NLfqo3WboOb0LdZzOpV8/tn6S0r%2Bf6H/2yDaAPxzHcZyk4sai4zjOy1BS3STLd9bISzur5ektVbK5rF6qG1vUeBSp0UfHcTqPtAH9ZMigVJk1YpAM08d5owbJSaMHyfiCjPgejuM4TrJwY9FxHKcDNu2pl4fW75XHNlTIutJa2VvbLCU1TVLT0CJNrbHoIhFHx3E6j/79%2BklGaj/Jy0yVjJT%2BUpCVYoYiRuNZE3JkwdhsGZg2IL634ziO05W4seg4jhOBVNPyumZ5bGOFLN5cKY/r46rdaijqa47jdA8Yh6Py0mTeyEFyxoRcuWhavkwtzLT0VMdxHKfrcGPRcRwnTkNzq2zf2yBPbKqU3z69U17YXi1VDZ5m6jg9BUzD0Xnp8raTh8s1c4bI9KGZkuVRRsdxnC7DjUXHcRyFjnBjaZ38bWmJ/OyxHbKrstFecxyn55GZ2t%2BMxU%2BeN0bmjBxoqauO4zhO5%2BNLZziO4yi1jS1y67IS%2Bf3Tu6S4uskNRcfpwdQ3t8q9q8rl6//dIjsrG61KseM4jtP5uLHoOI6j3Las1JTPbXsbbN6i4zg9F3KimEe8ZFuV/PSxItlT0xR/x3Ecx%2BlM3Fh0HKdP09jcKutL6uTOl/bY8hjMW3Qcp%2BfTqhZjcVWT3P5iqRmNFKZyHMdxOhc3Fh3H6dNUNrTI3av2yHPbq6Ss1qMTjtObIB11w546%2BddLpbJJHx3HcZzOxY1Fx3H6LEQmWHD/j88W2zxFx3F6H6SN/%2B2FEnlpZ62ve%2Bo4jtPJuLHoOE6fZVdVkzyxqULWFNdKXZMvkeE4vRUKVD27rUoNxpr4K47jOE5n4Mai4zh9FgpkrCups3mKvoiQ4/ReCChuKK2TLeX18Vccx3GczsCNRcdx%2BiyV9c2mXLqd6Di9n81l9bK1vCH%2Bl%2BM4jtMZuLHoOE6fpbK%2BRbaUNcg%2BDys6Tq9nV1WjzUF2HMdxOg83Fh3H6bNU1TfLViKLbis6Tq%2BnvqnVqqM6juM4nYcbi47j9FmaWvdJTWOrp6E6znEAhqKvk%2Bo4jtO5uLHoOE6fpalFjUWvguo4xwVkCHiWgOM4TufixqLjOH0WKig2q8HoOI7jOI7jtKXfPq/s4DhOH%2BUPz%2B6W625eHf/LSSa5mSkyKjdNstNTJCOlv6T07yf9%2BhEZ2mcLqzcS9W1skeKqJitccigKB6VKvn5fS6vIhj118Ve7j5yMATJ0UJqk63lxPMylc5LDhxaOkutfNTn%2Bl%2BM4jnOsuLHoOE6fxY3F5KM2oRlRZ03IletOGS5zRg6S0Wo0DlIDC4OxRQ3FivpmKa1pltW7a%2BWOl0rlb0tLpLaxVVrbGa4yU/vL1bMGy8KJubZu5hf/s7lb56BybqeOy5ZrZg%2BRIQNT5Yv3bLYlHZzk4Mai4zhO5%2BJpqI7jOE7SGKwG1KfOHyM/vmayXKVG3tTCDMnJHCADsCIVHok6ji/IkAum5skXLh4nv3j1FBmZkyZpA2L7RPnMBWNtO29yvhqOA%2BKvdh9vmj9Uvn7ZeHnVnEIZmD7AjGPHcRzH6a24seg4juMkjXedNsIMqYmDM6SxuVUeXr9XfvDwdnnv39fK6/6wUt76l9XytXu3yKINe20dzOHZaXL%2BlHz55hUTZFReevxbDjClMFPG6OsD0/tbGmt3Myo3Xc8t01JR%2B%2BsB8c9xHMdxeituLDqO4zhdDvMSpw/NkktnFMhUNfCYh/jPF0vll0/ulJue3SV/fb5Y/r60xFJO/7Rkt/ziiSL5x7JSWV9aJ8OyUy3V9NxJeTI8Jy3%2BjTHCfMeeAtHPtAE%2BtDqO4zjHBz6iOY7jOF1OdsYAuXh6vozLT5cUNaiWFVXLjYt3yr%2BXl8rKXbVS1RBbwoRo46ayejMkf/v0Tnlk/V6hYC2pqedPybP0VMdxHMdxkoMbi47jOE6Xk50%2BwCKDVD%2BtUcNwfUm9PL%2B96pDFaJbuqJZHN1TIFjUeKV5DFJHIHfMaKWyTpwZkqv7dr18/S/mkuAyvkQLK61HCZziO3IwU2y9s/M3rWWn9251jyOf4TrastAGSwffocwxYthz9/CC%2BV5/zHumwbKn6Zbafvs/rjuM4jtPb8GqojuP0WbwaavJgXuFnLxwr184tlCEDU%2BTm54rl4//eKMXVh14WAyOOqqIYe7WNLVKuRuNo/a5XzRki7zxthIzISTNjjjUzq9UILa1pkp2VjfKdB7fJ3Sv3xL9F5IRhWXKNfuaM8bkyWH%2BfQjsBoprb9zbI2uJa%2BfGiHfY8yuf0uC%2BeXmDPMWA3l9XJZfo3xwEcE785bWiWjM5NN6MSY7GuqdWOZ5e%2B99fnS%2BQni7bb/k7X4dVQHcdxOhd3dTqO4zhdTrUaeo9vrJCq%2BmaLAp46Lke%2BcPFYee2JhVKQlRLfqy0YcqSlMnexSI0uDDDgOwamqVGm/3B5shEUHKSvDUzrvz%2ByiFH49lOGy1cuHS9vPXm4nDY%2BR6YUZtl%2BbKyHyFxKlt54tRqy37hsgpyu%2B0TBWB2fny4z1eA8W/ejSA9LfvD60Ow0GV%2BQafMms9RoxajFA8uGAZuZwvEMkPSU2PE4juM4Tm9iwFeU%2BHPHcZw%2BxbKiGvnX8tL4X05XguFUUdcsM0cMlGFqoA3LSZMJBRkyNi9dxuRnyIxhA81oG5uXIdkZKVJR3yJNTFZsB8wujMbtFQ0W3SOVlAX8uZ8Ux3lqc5U8t71KKtUwnTIkUz51wRg5e1KepYU%2Bt61abn%2BxVBap4frUlkp5dluV7KlpsuMZq8fBnMit5Q2yYU%2B9RSrhkukFenxZatSmSqYaoqSi3rasVO5fWy4vbK%2BWl3bWykPr9trcSwzVYWpAcuQYuLcuK5GH9b1n9Xf5XqdrwQlx2YxYFNhxHMc5dtxYdBynz%2BLGYvJgsX3mHaq9ZsZgwcBUi8yx9MXZk3LljAk5ctLoQXLC8CxLWSVSh3HGPEDmJGIMBjAUMbye3FQpF0/LVyMvXSrVsGMZji/fs9kMQNI/h6hRSiTxHaeOsMjeog0VVlTnl08WyRP6WTabE6nflZeZKnNHDlRjcIDsrGyQTWosbounowZjccigVGlUA5bf/cp9W%2BSOl/bI0xicW6vsNzE%2BqfQ6d%2BQg%2Bxzt69sPbJP71pS7oZgk3Fh0HMfpXDwN1XEcx0kaLI/x40e3yz%2BXlagBFYveNTS3WoEY0kMvmJovHzhrlNz29pny97edYAvcX3FCgb1P6umRwEL%2B80YNkh0VDTYP8ebni%2BXfL7V1DixRQ%2B%2BGx3eoEbrP0lkxMgt1aw%2BikCz1wSNguJbVxp47juM4zvGGG4uO4zhOUsE4%2B/p/t8grf7tC3vW3tVbshtd2VjSq4Xgg9XRodqpcqMbjFy4eJze9cZpMGHxky2as3F0r33lwq1zx6%2BVy0S9elLtWHCh4k0hz6z4pqW6UppbWWJrsoAMFcKJU1rfIf9eU26PjOI7jHO%2B4seg4juMkFaJxxdVNsqa4Vh5cVy4/fWyHfO7uTfK%2B29bKe29da1VD71tdZtFA1mQkLZVlN95/5sj9KZ6HAxHLEv2dzWX1ViSHYjks8H/WxFy5dEaBVVP95Hlj5JtXTJAvXDTW0kz5PbYBCUtvQG0jx91o0dBWLyTuOI7j9AHcWHQcx3G6hVg0r0le2FFtRWDuXLFH/vZCsfz%2B6V3yyyd3yq90u39NuVTVt1hV06tmDrY5gUcCVVFZzuLqWYPl/WeNlI%2BcM9qMzveeMULecdpwue6UYfKm%2BcPk4mkFto5jfyuf0z5Nra1mMDqO4zhOX8GNRcdxHKfLYc4hEb3zJufJrBEDJb%2BD5TKIOobCQz9/vEj%2BvKRYVhXXWnXRcfkZVvn0cGH9xYmDM%2BWVs4fIx84dLd%2B4fIJ84rwxZnSeOSHXqrFS2ZRo4fKdNR4xdBzHcZwE3Fh0HMdxuhwMs9vfPlP%2B%2B7458pVLxsmCMdnxdzqmsqFZVuyqkSc2VdjChWkp/Wwdw8OFZTVePXeIfOeqiXKWGocYj7urYobhA2v3yt0ry%2BS3T%2B%2BUT/x7g3zgtnWyp7bJop2O4ziO48RwY9FxHMfpcqhkmk6ap9p649RwxHjsaqYNzZJrZg%2BRDDUSWbbj%2Bw9tk1f8ZoW8/o%2Br5OP/Wi%2Bfv3uT/OKJnbb0BfMUWaCfJTscx3Ecx4nhxqLjOI7T5TS2tMq2igZpbN5nC9%2BzHt6ckQPj73bMyNy0WBRSbTiWq6htOvwqpFlqJDLXEfuPNRNfLKqR5btqbMmOospG2VXVaN/JPkQeWVC/vxuLjuM4jrMfNxYdx3GcLqe8tln%2BsmS3pYHmZqTImRNy5D1njJQ3zR9qC/KzQD8M6NdPcvR9Fui/ds4QiwzOHjHQys5QCGenGnlRSBul5AyRy8SoIAmlYQoiUU2K3STagtOHZskr9DeY10iK67GaimSxtuiPsiRkKt93rF/oOI7jON2IG4uO4zhOl1Na0yS/fXqXLN5cac8nDcmU/zl1uHz%2BonHyntNHWrVSit%2BcPzVPLp9RINedMlw%2Bdf5Yed2JQyU7fYAtrM8cw/WldfFvjMFyGI3NrTYfcWxeukUsT1RDk2ghayESUWxRC47lN86cmCsXTMm3fdgW6t9vOGmovHXBMP0724rrHOuUxdrGFvseDEWW6Thx1CCZr8fD/EnHcRzH6W0M%2BIoSf%2B44jtOnCFU3na4HI6xGDamyumbJSU8xQ2qQPhYOSrW1E4nuvU0NxLeo4fbquYVyytgcGZGTpp/sJ1vLG%2BSfL5bI75/ZJVv0eRSijhieY/PTrfIpi/ifMynX9qOKKvBdVD09VR%2BphHrp9HyLWr7jtBG6b57kZqbInppmM2Jz1DAt12N8fnu1PLO1yj5/yfQCmTEsy%2BY%2BbtpTL39bWmKvtwfHMEWPZ8LgDBmpBuIZ43PkHDWCOf%2BntlTG93K6CpwAl80oiP/lOI7jHCtuLDqO02dxYzH5sEzF4xsrZMWuWkmj4E3/WIILKaTRNFLmE64trpPblpXKN%2B/fKnev3GPGXGLkb%2BXuWhmkhuCo3AxLb83LSrXlNR7dUGFVVHdUxOYm8h4GI/MS8zJTzUDcWxszCn/91E752eNFMlQN14lq5LHv%2BpI6eWBtuf3GkRiLW8oabF7lJDUa%2BY1sNYjz9XFdaZ38d03s%2B5yuw41Fx3GczqXfPiX%2B3HEcp0/xh2d3y3U3r47/5SQT5igSEWS9RRbDTxtw8LIYRCFrGltlZ0WDbCqrt9TOjpg2NNMieqSr8h2knS7ZVi0b99TZ9w7PSbP3MSLTmbcY/x1%2Bg7mURC5LahrVIBxoqawYrev1sy%2BoIQmktZLGmjogtvQGxu6hoCjPzOED7feYg8kwu760Xp7bHotUOl3HhxaOkutfNTn%2Bl%2BM4jnOsuLHoOE6fxY1Fxzm%2BcGPRcRync/ECN47jOI7jOI7jOE4b3Fh0HMdxHMdxHMdx2uDGouM4juM4juM4jtMGNxYdx3Ecx3Ecx3GcNrix6DiO4ziO4ziO47TBjUXHcRzHcRzHcRynDW4sOo7jOI7jOI7jOG1wY9FxHMdxHMdxHMdpgxuLjuM4juM4juM4ThvcWHQcx3Ecx3Ecx3Ha4Mai4ziO4ziO4ziO0wY3Fh3HcRzHcRzHcZw2uLHoOI7jOI7jOI7jtMGNRcdxHMdxHMdxHKcNbiw6juM4juM4juM4bXBj0XEcx3Ecx3Ecx2mDG4uO4ziO4ziO4zhOG9xYdBzHcRzHcRzHcdrgxqLjOI7jOI7jOI7TBjcWHcdxHMdxHMdxnDa4seg4juM4juM4juO0wY1Fx3Ecx3Ecx3Ecpw1uLDqO4ziO4ziO4zhtcGPRcRzH6RSGZafJ604slN%2B9YZp87%2BqJMiYvPf7O8UteZoqcOzlP/vim6XLOpDzJ1b8dx3Ec53jBjUXHcRynU5g5PEsunzFYXjlriLzt5OFyxoRcGTIwNf7u8Ulman%2BZPCRT3rxgmD5m2N%2BO4ziOc7zQb58Sf%2B44jtOn%2BMOzu%2BW6m1fH/3KOlY%2BeM1peNWeI5GakyOTCTPnLc7vl14t3yjNbq%2BJ7xBibny7Dc9IkbUDMsKqoa5b0lP5S29QiW8sapLqxRYYMSpWRuk%2BOfhejVHPrPmlobtXn%2B2RPbbOU1jTJhIIMSR3QTzLUQOO7eH9HRYOUVDfpZ/mNVH0/9hvNLftk4546Kdb3Anx%2BsBqzfL5Fv79Wf5fjqNFH9uN7YN6oQTIofYA9D3DMHMPI3HR57dxC%2BeQFY%2BQ7D26Ve1eVy8pdNbK3vlkK9RxG6ft8JzS1tNp37qhotGMdmp0m4/VaNDTvk6y02D67KhtlU1m9PXeOnA8tHCXXv2py/C/HcRznWHFj0XGcPosbi51HSv9%2B8p2rJsoZ43NkhRpLZ07IlRo1/r55/1b554ul8b1Ejbp%2B8qVLxsvbTxku%2BVmxlM1n1ZgcV5Auy4tq5Mv3bpZlO2rkDfOHynvPGCEnj82RRjWsMOR2VjVKfVOr/OulPXLrCyXyg1dOkiEDU2Ti4Ewp0O/atrdBfvzodvm3vv%2B%2BM0bKdZHf4POfvnOj/HHJbjM8Od7vXT1Jrpo5WA2%2BNHt/bUmdjM1Llxf1OGgbf19aLGlq6D3%2BoXkye8RAUXtT%2BvUTSdXPPrm5Ut8vkdFqDH7s3NFmtDbpDg%2BtK5ffPb1LHt9UIW88aah88KxRZhhDmRq5/1peKt97aJtsKa%2BXt548XP7v8vGyXQ3cyUOypFWH4988tVM%2Be9cm2985ctxYdBzH6Vw8X8ZxHMc5Zt6yYJicOGqQvLizRm54osiMqWGD0ixyF%2BWLl4yTy2cUyH1rymTBD56zjYjb4KzYfpmpAyyS9%2B7TR1g08PN3b5Krf/OS3Lu6zFJaCxK%2BjxTQ57ZXyftvWyeX37hc/vp8sXz2grGycFKu/OX53ft/AwMS4%2Bxj54y2uZWfOn%2BMHccDa8vlVb9bIW/%2Bc8xpMCg9RQ1EtQiVKYVZ8uD75khVQ4t88B/r7Xuu%2Be0KWbShQqYWZsoINQIXbdgr3/jvFsHr%2BnV9/Px/NsuGPTFD8OPnjtHzLJfX/mGlvO6PK%2BXulXtsTufr1IjEwAUiokQf/%2Bevq2Xh9UvV2N1hrzuO4zhOT8CNRcdxHOeYOWVcthlZm8vqZU1xrfzyySKprG%2B2SOPpugVOGZsjZXVN8uj6vbJyd61tj6jBRXQNKBhz7dwhkq%2BPD%2Bk%2B/1hWYmmsNzxeJJvUCCPKGKVVrbSn1DD9z6oy2VbeYAbl/DGDLFr5x2d37/%2BNX6gBu6e2yeZRXjwtXy6ami8VenyL9bNEAZ/cVClfu2%2BLFFc3Sj/9Bzy/ftEO%2BeYDW83Q43uWqzH89NZKMw6JMGJIFlU22v5Feg6kweZlDrBzJp32DjVSMS4fXV9hz4kunj8lT43FDPtMgxrKW8oaZHVxnW61srsq9l2O4ziO0xNwY9FxHMc5aki/JI1z2tAs2alG00tqTNU2tpqxtkWNt%2BnDstR4y96/39BBqWYcrVLDK4AxuH1vzFhk7t4C3b%2BxZZ/tw/w95hCyD2mmdU0HG4vVaqxt1d9hrt/A9AFy1sRcixyOzkuXC9Qo%2B/DZo2w7c0KORTpH6THw/UT2iioa9bP1UlXfYobjPWpwMqeQ%2BYtQrobd35aWyP1rymVcQYa86/QRct0pw2RqYZZkxOchtkdeZqqMykmz41pbUit765pt43yYz0iqa4i4MneRuZQ8Oo7jOE5Pw41Fx3Ec56gZlDZAFk7MMwOIuYAD9W8MNjbm4A1VA226GpLM7ZukBhrFXjCcynULFFc1mUEI/fv1s2IyjS2tloYapV4NKn4jCnMYw2tZqf0thZXf4PHVcwsP2jgeooMYnBil1Q3NbYy0uqYWaQrfp%2BeCoXuGGpr/c%2Bpwm3941cwh9t2HqnpKxJH0UqKOrZGv55wwIAfoOTJnEjBMMUqDgeo4juM4PQk3Fh3HcZyjhgIyFHIhfZRKqL9/4zS5/31zbLtgar5VPiUt9OLp%2BVbVNBAzlWJgN4XUT5F9luJpRHc6DPgcv0G08VsPbJWzf7ZUFv704O2tf1ktNz9XbPv2U6PtwO8GYq9QyGbS4Ay55a0z5L73zJFLphfIP14skat/u1x%2Bsmj7QVVVE7Hj0K29w%2Bdcec9xHMdxegNuLDqO4zhHDctOzBk50Ob8vfOWtbLgB88ftP15yW4rXkM0bumOaosgDs9O218hFMblZ5ixCS2tsfTPzJTYchhRiGImvhaFaqTb9tbb3EnWPBybF5sXmEiLWpSV9S32m5nxJSsC2en6G/p55j5SAZWI6Xcf2iZv%2BOMq%2BeUTRfG9Dg0RxPrmFqvQGj1civfMGD7Qooik2TqO4zhOT8eNRcdxHOeooBIpC/AzR5Co232ry2zZjOhG2idG3LTCrNg6iuUNMqUwU04Zmx3/FpGLpuXLJDXugLmDd67YY%2BsjUgjmtHE5ZnS9ef4wmTEsSwYmGHdRWPvwrpVlNify3Ml5cumM/Pg7Iqfq93zh4nHyRd0wbjFcKTKDQcjxU9n0S5eMk1FqHBJbJE2UNFSWzsCso9gN0UQMX46FY%2BoI5k9SsIZ5nBwH1U6ZQ3nu5FyZpcbiC/rbYY6m4ziO4/RkBnxFiT93HMfpUywrqrF175yjY/7obFsyIzsjRX70yHZbMiKR/MxUmaBG2VQ1nDbq%2B3tqmswgo8AMhhdGHPMbR%2BdlWAGbB9butSI5kwszZMqQLHucpI9XzSyQYTnpFrVjeY6Vu2otNZSqqVQz5TPMXWShfOYEYqjxeSKE/MbVswZblLC0plkN2BpZX1pvhhsL51O85qQxg%2BSyGfob2alW%2BGZpUbVsLq%2BXuWpY5mQMMINv5rCB%2Bvcgm%2B84Vj/D4vpr1SisbWq1ZTiIFg7oJzZXcXdVk4zXfcbkp9vi/1SLpUIq7/3l%2BWIzVjG2F%2Bq5cywsNUK00zk2uNfcR8dxHKdzcGPRcZw%2BC8sg3LFiz0Fz6ZzDh/RRjLKXdtXI/WvLrXBNInWNrTKgfz8Zmp1mKaR/fm63NDbv089lyoKx2TJzxEDZpYZVjhqcROSIKq4vrTOjCkOOaOKEgkxpUEOMAjUN%2BlmilUTnKJjD8hQsf4EhGuA9itxEf6NADVO%2Bm3UYWfSf76CQzlg9h9lqEI5SQ5RoIBFAlsJYvKVSnt1abWmxRCA5T76H7/3mA9sslZYiPVQ4peLpeD1Gqr1yrhh/j22sMMN1ln7mhOFZdq2qG1rlt8/skvtXl0uJvjcyJ00K9Xu4frEUXf0x55hwY9FxHKdz6bdPiT93HMfpU2A4vOfva80wcZIDaZ0UfiEKGCqAvuGkofLBs0aaAfduvR%2BkgFIVNboPhWFuu26mLTnxtxeK5eeHOX%2BwPSheE%2BY%2Bht/gNX73qY%2BcZOtE3rh4pzyyfq/t4/QePrRwlFz/qsnxvxzHcZxjxecsOo7jOEnjD2%2BYJje/ZYa8dl5h/BWx9EzWSGTBetI9v3/1JHn4A3Pl2jlD7P2wJAYRPqJ1LPx/LBCR/OfbZ9rGAv3AvMVPnjfG0lBZI5LNcRzHcfo6Hll0HKfPcvuLpfKRf623uXLeEyaHr142Xi6dVmBFY1YXxxbmZx3Gh9aVy63LSmR9SZ0tmv%2BNKybYGorcG9YtpKBMKH7z0Lq9srOy0T57NFAF9bzJefKVS8db6iwL/1M4hzmMNz2zS25fXipr9Tic3gVR5w%2BcNVK%2BqvfVcRzH6Rx8zqLjOH0W5qa9WFRjFTrdVkwOlWrwkfZb09RiC%2BKzrdhJoaE98vz2aisSQ%2BSQVFXmKbLMBQVk9tQ2y10ryuTxjRXHZCgCi/tjqDJnsS5%2BDBwTBXp%2B9eROMx6d3gdFk6g%2BG6206ziO4xwbHll0HKfPgqH468U75ZdPFtncNcdxei9Ux33PGSPkmtmx9GXHcRzn2PE5i47j9FmYn7Zg7CArpuI4Tu8FCZ4%2BNNOWI3Ecx3E6DzcWHcfpszDHiXXzWPIgVMd0HKd3gaEYW2ZloIzJS4%2B96DiO43QKrh05jtNnYakEqmBeO3eIDBmYEn/VcZzeRL9%2B/eTksdkytTBTstMHxF91HMdxOgM3Fh3H6dNQGfO184bKjOEDreCJ4zi9BzICiCZePmOwTBqSIQNYkNNxHMfpNNxYdBynT5OZ2t%2BqJ146Pd/W3/P5i47TO8AuLBiYIhdNy5cLdRuR4ymojuM4nY0bi47jOMp7zxgpF07Ns/X2HMfp%2BaSn9LfU0w8tHCVj89IldYA7ehzHcTob14ocx3GULDUS33naCPnQ2aPUYPR0VMfpyWSk9pfzp%2BTLJ88fYwYjhqPjOI7T%2Bfg6i47jOHFYnH1tSZ3ctWKP/PqpnVJU0WivOY7TMyBLfHxBhlw6vUCunDlYTh2XLYOzUuPvOo7jOJ2NG4uO4zgRWlr3yc7KRvnrC8Xy2IYKeWlXjf1d3%2BRGo%2BN0FxSuyctIkZkjsuTsSXlyxQkFMnvEQM8CcBzH6WLcWHQcx%2BmAB9eWy3/XlMuSbVWyrbxBqhtbpEa3OjUcm1q860wKDFH7WtSKbxZp1a2l6cBjc6O%2Bp0Z8WpZIRk7sMcDrTfUi9VX6WKf7NvCiSP8UtTxSRVLS489143H/c4wPn/vWnVC4BuOQSqdpA/pJjhqJgwemyNShWfKq2UPkzIm5Ujgw1ecoOo7jJAE3Fh3HcV6GbXsb5PGNFfLCjmp5fnu1rNpdI6XVaqw4x4QNPjoE7X/ELuSv6GsYhQ3VIjVluu05sFWXiJRvN0Ow/5h50n/qOdJ/5Cxp5TOtrdKqhuK%2B3WtENjwpUrxWpGyLfp8anRm5IjkjRPJHiWQViAwcrBuP8efp2dIPg7Ffv5jJaI/6jz8Oes3pKph/yJI2w3PSrMLpuZNyZaFu09VY9OVtHMdxkosbi47jOC8DUcQQUaxtapGGpn3S3Opd57HQ1NQkVVVVUlJSoltx/LFUdu/eLVu2bJaamlopK9sje/fulfra2lg0sVWNvegjhuS%2BVjn7govl1W98m5x8xkJZuXKlLFnynDz%2B2CJZt%2Bolqa9UI5OoYgtRSL1nGIJEFtn66XP%2Btsgir6dIRmaW5OcXyJDCIVI4ZIgMGVIYf14ohYXh%2BRDJy8uXlBT9nNPpYJgTWUxlG9DPUk0HqpGYkdLPl7ZxHMdJMm4sOo7jOJ1Ka2ur1NfXm6HHVlFRsf85xmBZWZm9VllZKeXl5VKrxiBbXV2dGok19h7GJN/R0tIiaWlpapzl2Zabm2uPBQUFMnz4cDUsiPr1kwEDBtj%2BK1askF27dklxcbEdx%2BjRo%2BXUU0%2B179uyZYusXbtWtm/fLo2NajzG4bOpqalqS%2B6zz6Snp0tmZqZtWVlZ7T4OGjRIcnJyDjqucGzhkY19%2BH7HcRzH6Y24seg4juMcFgwXGFkYdhh1YeNvDEGMPQw2XiNqiDGI4Rc2/saIw0BkPwwzNgywgQMH2haeY2hhkLFlZ2fvN8bYiOg1NDRIc3PzfgNw/fr1sm3bNvvuKVOmyKRJk%2BxxxowZMn/%2BfNt369atsmHDBtm0aZNt7M9WXV1txiLHwn75%2BfkyZMgQM/QwDMM5Rs%2BX/TjWcEzsy5b4nC3x3KLPo49uVDqO4zg9DTcWHcdx%2BjgMAxhKRPGIwGEQRh8x7DDO2DACS0tLD9r27NljhlgwCjHY2BcDjKhg9BHjC2OPRwwpDEEMM7bBgwfvfz527FgZOnSo7UOkj%2BPjt/h%2BDMQXXnjBtsWLF5uhyvfxeT5z6aWXWjRx%2BvTpMmzYsPhZHoBzWLJkiTz77LPy1FNPmaEZjh1DkN/G0Bw1apR9J4YhG%2BeEYcn5si/Xpb3rFX3k2nIOfE/0/MLz6OucA9cpXKv2rh%2BP/fv7moKO4zhOcnBj0XEcpw8Quvr2HjESMaDYiPzt3Llz/0ZK58aNG2Xz5s32HvuQ9gnRx/AcMAaJCI4YMUJGjhxpj2ykjWKEBQMJww5jsSM4NjYMNaKSt9xyizzwwAPy/PPPWyopv0kEb/z48XLOOefI1VdfLSeddJJ95%2BFG6YiGrlmzxr73wQcflGeeecYMQX6X78DQu%2BCCC%2BTCCy%2BUM844Q6ZOnWq/i6GI0cj1iV6r8Biesw9GI4RrDuF59DVSa6PXKvoYnrNhVAaDMXoP2nt0HMdxnGPBjUXHcZzjHKJlzBOMFZGJbUTpSMHkOQYgxg2PIb2TyBhbiKjxnOheRkZGrNDLkCH2GJ5jsIX5ehiCGGxEwjAcQ5SM50QJeQzzBA8VJSOCSATw8ccfl6efflp27NhhhhyGEMbT2WefbRHEE044weYm8tvBkDpcY4lzChFTooYYxRijRByJXBIxDamwRCmJVp522mn2u5wz6aPhOkWvWXjEGA3XPhqNjd4HNvZhOA7XK/G6RR%2B5zlzzcA%2Bij9HXo0al4ziO4xwNbiw6juP0UohuYeSE4jFsGFNEAMMcQh7Zh7l2/I3hGDZe55HIF48YcBhGoThLMP5IxwzGB%2B8TAeMRQ4nXeGQ/DMFQHAaj5kjBuCISh2G4evVqq2xKimhRUZGltmKEMv9w3rx5%2B%2BckEmnjePjdzoDrwPXDMMWY5jgomsOxYFBzjYiWck2IMjInEmN15syZFuVMjGhijHKfwjUP9yDxkftD9DTcw8TH6HN%2BI1z3jh7ZMHDDPQxb9L6Gv7l2icftOI7jOODGouM4Tg8EwykYGdHCKhh4GAz8zXMiUtHqojwSqeIRQ4RH0kwxCEJxlVBQBWOBx/AeBmAwIthCkRaMIww19iOyyGNnRKwYfjBUMYKIIlJ0BuPwySeflHXr1tl5cXwTJ06UadOmyYQJE%2BTkk082Ay0cT1fCsVG9FYORjXTVUEiHDcOM%2BY1EG9kwHseNG2dRTqKQR2KAYVQS2YwWBQrPo6%2BxRdtDaB/tPTK/EYMxseBOeB4e2Se0iUM9%2BnxJx3Gcvocbi47jOEkEowBDEEOEVMXwSCokG%2B/zGgo/RlRIUwwb0S3m6/EcY5LP0I2HlMVQCCUYdjzHGCD6Fk1TZAsRQ4xE/sYoSEaEiXPk/ML5UM100aJFVnAGI4z3w1y9WbNmWbrpWWedZcd6NBHLzoK00RdffNHmNbKRshqK3WDQzZkzRxYsWCCnnHKKzJ071443GGOdZdhyr/k9rh2/HX0MG3%2BzcY1pS6GNRdtb9DnthAhjtG1gjLNFX%2BM8MJCj7Sw8Rp%2BTLutzJh3HcY4P3Fh0HMfpRKJdangefSTahyIfiqKEjcgaxgfRJf7GUMQQTCxYElXCUd7ZUOqjRVB4ZD4dqZo8Z5%2BekGbI%2BbNhxDAv8J///KcZiRiLXBeMFs6FKOJHPvIRM7j4GyO2J8I5hMI4Dz300P7z4x5x3BTFYaP4DhFHXg9bMiB1NxTaYQvPE1/jfhB9DnAO0UfgOU6F0NZCOwsbf4fXQmprOM/o%2Bbb3muM4jtNzcWPRcRynEwgpo6SGRouX8EgqI8Ygz4kKYRiF6E4ohhIiQETV%2BJsIDWmCRKfCRnSHlFA20gJJdSRyFaI9IaoTojwh0nMkBV%2B6As4npJree%2B%2B98txzz1k6J9cF45jzPPHEE82oOv300y2Nk/PlvDBye2rqI/cxpAVjdFEUJxTGIUKKkY6BxX0idZbCOGykrCbDAA5LoYQ2Fn0efQyRyrBF2270ddo4HKr4Do%2B0TdaqDG02bOHv8IhRSRt1HMdxei5uLDqO47wMKNQYBhg8bCjXPBK5weDhPZ6zYfxgNIaN94jc8Bn%2BpsvFCEJRZsPY45GIDQo00bUwnyzMF4tuGB9sKOb8jbLdU4uTcM7M8yMCR/omxiLLcDAXkWMm2nbxxReb8UsUlDmJRKY4r94E9xRDn0I8FMYhQsx5UxSHjTbC/SLtl3MlchoK41CkB%2BOpO%2BHYaaehvUbbbvR5MIyDDIQtKhM8YoRiNHIfaeuh7SY%2Bpx2HeZNBDqKPYcPodqPScRyne3Bj0XGcPgtRPFI9MeZQiMMjBh%2BKL0ovSjJ/B0U5urEPhgARF95nXyJ4GHkouSjDGH8oxhgEKLy8F4zDqELMnEJeY38UaBRqFO7eBEY1148o27Jly8x4eumll%2Bw5xWs4f4ylyZMnW9EaDCXWMOR6cI162/l2BO2BdhEK47BhJBNtZL4p9xjDOBTGYSOaOmbMGGsHPdUwCu2ctt9e8Z3wnH2iTpMgV9HnPCJ/ob0HOQjPEx%2BRpajzJDxPdKgcL23IcRynp%2BDGouM4xyV0bSijRE0w%2BqKPKL0hmoJyi2IfTbdj6QTW10OpJQqGEowhRHQIRT48sqHsEiVDScXgQbElQkiVTBad529S8jAIoootnzse4DpzLTGcMRKJHi5dulRuvvlmu3a8x3lzPUg1ZWNxeyKJx8s1eDmYG8jyG9HCOKGCLW0Uw5miOGxEHImu0m7YaFe9bX4fckV7QJaixXfC8/DI%2BbNvkM1EOQ2vAW0IuQoR%2BPCY%2BBr7Jcopj4mveVVXx3Gcw8ONRcdxei3R7ivxOUo4EY5QyIMoVyjoQUEVojworSEKEiWqnPMc5RLFnahYKOjBRjSI6BjRDwxDlFUMwb5AuN5EZrm2d999t9x1112Wcsp15bqRPkihncsvv1yuu%2B46W1rCEZvX%2BPDDD1txHCrAEmXjerJhVFP9NRTHoV2FaFlvMxpfDpwxtBVkMshmkNfo3xjWOHiiMg7Rv3mO0wY5TSy%2Bw5b4GkYlJMp64Hi71o7jOEeLG4uO4/QqMAJRrolMUIAjWoSDqBbzBolYBEMwRCdCQQ82DBweUQiJbrERmUgswMGcOpRMIoMhYhE2FHiMyLDmIH%2Bz9YWIBUo%2BBuJjjz0md955535DnGvPdSKKyuL55513nhk/XMvOXDi/t0P7DanNXLvFixebAUnUkXbMNcTQ5ppREZaiP0QdicoeTyDLoQhPooxG/w6Ryo4K74TXQup4opwmPkduub5RWU98HjbuhUchHcfpy7ix6DhOjwDFESOOCEKY%2B4Tyx8bzoCDyHCUbgyW6hZRSPs/3sIVIQ2LhDKpToiwSESQayHyokB4a3Xif9zByUDT7Mhg4RGOZf7hkyRJL02UeHhv3jqgrC%2BazUcAFIxtDEWW7pxbg6QnQXjEY2ZjXGS2MEyqqcm2Zz8hcz5kzZ1qqKs8xbvoCGJTIMzIelffE59G5xaHfSHzORv%2BC6hMtuMMW/o6%2BjvzTb4S%2BI7pF%2BxU%2B40al4zjHI24sOo6TFEKEAKUOwyM8orjxiNJMuhnKXjAWo1uIIKAQYpwEZQ5jJDxHaQuRwhCdSVTwMAAxFkPBDPbpK3PnjgSGBu4ZinYwYCjUsnz5cqtsyj0g8oIRg3HIHESiX1T4JBLrHDkYRERouc6rVq2ya878RoriMI8WowlDkcgtj6T1kgrNxjXv65Fbrg/9SjAIo/1H9G%2BeB4dT6IuiW/Q1DMBQZCfahyT%2BzRbti6LPo38T1XTnieM4vQk3Fh3HOSZQ0EgXSyxSgeLLY0gzQznDGAxGX9hYaoCoIUYgf2M0okyh%2BIaUMR75OzzHKExMF2MjksV7KG6k8KGgOUcG1597gZEYloFgPuLjjz9u9w/jmmgsBgtpkUQSzzrrLLvWrgR3LszZI5JLeipzQZlrG4weIt2091NPPdWMdO4Hc/GC8cL7Pu%2BufXB00B/R34RiO4nPw9/IQmLfFp5H/6ZvwmCnHwoFd8IW/Zt7g9FIPxa20K9F/0aWPFLpOE5PwI1Fx3Felmg3kfgcZQpjL8xbCxvpibyG0cHfKGcYlYcqIsHfRPyYK0hhlFCYgjQ8KkbyiDHI664Idy7cSzYMEubPPfTQQ1awhvuGwY%2BxznU/99xz5XWve53NSUT5dZID0S4qqlIUh40IL8Z7uG%2BkpVJlNhTGiS7B4bJydHBdMc7pvzoqvhMekROMUD4TiD4H/sZQRG5CwZ3EwjvhNQxPMh7a6yMDfl8dx0kGbiw6jtMhRAeJYmAMhjmDPDKXasOGDabAoiShUPG8vSIyIbKI4oOhxzyrsKE0sVA5FR%2BJWPE3j8z/CZHFUJQi/I3HnUenc%2BDeUJiGlMfbb7/d5iMSUeS%2Bcu9JdZw3b54ZIkSxuFcosii9fX0eZzLBECHqG1IoifhyrzDs2XDaIGPID9ErIr7cr9NOO81Sg/1eHR2h/0osvpP4GvckGpUMW%2Bg3w0ZqK0ZetG8LW/Rv5CtkULRXeCe8TiTZ0%2Bgdx%2BlK3Fh0nD4Iyg3KJcYAkb9QACKkYaGU8ncwAsP8nbDxOgYG30OqFAYcEcGwriDKS0i94m%2BUHpTYxHk8GIUhLYsoSHh0j3nXQ%2BXY9evX75%2BDyLw45srRFrhfGBikN86aNcvmJTIvjogHiqzT/SC/rA%2BKYY/zhjmOL730kj0SHeZe4Yjh3hGV5z5yTzH%2BfU5p50NfSN/YURGe8HdwwoS%2BNzxPfI30VvrVxP4y8Tn9Ln0o/WzYkN/o32z0vzjcHMdxjhQ3Fh3nOCJ4ulE0gpEXfUQR4RFFE4MvUVEJcwpDRDEoLMHIC48oHmwYdiieKCukvQVPN0oJfwdjkQ0D0ufgdB%2B0De4phgVGIgZiKKSC4cj9JM2XQjUUT5k9e7ZFpygGFNIZnZ4JUUfuIfcyGI3c51AYh2E%2BFMahmiopq6EwDk4djzomD/rWkI0R7XsT/2Yf%2BumosZnouOPvkCKeaDCyhdfCYyjqFQzOaJ8entNPu7w7jhPFjUXH6SUgqigG7RVYIBLIe8EIZMPoi6ZAoTSy3AGvsR/fF1KeUA7CI4pHSPVEgQhFG6Ib82owLHifv4Ph6PQsMCJQKFE8iUJR1ZQ1/e677z4zJDASiEJgEFKs5qSTTrJ0U4wJT23rneAU4H4//fTTVhjnueeek3Xr1u03RpDrCRMmWNSYDeMRJw8Ghctxz4E%2BHecefXdi4Z3oa%2ByD4RgdE8K4EP0biEYm9uVsOAxoAzzi5KMthLEhuoVxImyME47jHP%2B4seg4PYj2xDG8xoCPYkAxhWgxGdLQQpVEDEIUCJQH6Kg4AkoDSgEGX7SwAoVlUCRRGkORBU9d6p3QbnAKYCw8%2Buijcu%2B991o0kVQ44B5TxfTKK6%2BU888/39ITPf33%2BIM%2Bg2hjKIxDkRwyCAJEkmkHFMU588wzrcIq7cDbQu8Ao5L7yZgQLcKTWJAH4zLI/qHGGRxIRCDDmNBRAR42opWB9tqLtyHHOT5wY9FxuhmMQJT6aAEZHokEkVpGZAhlgDmCPI96i8NGahORRaIKeHyDpxivcXgkeoQxSKoR7zPQR73EbHiO2fAYozSw%2BYDfu8BRgHHw2GOPycMPP2zPQ8SZe0864oIFC6yqKRFE2gbKoUcSj0/oFzAoiDaykYLMMhxEHtnoU7j3OA%2BYczxnzpz9hXGY60j2gNNzIXuAexzGgmjRnehr9AtElhlbwpZYfIcN5wLjSRgTGAOiY0R4DSci/UkYXxK36OtEKh3H6b24seg4XQjiFYrFMAiH1CEGY1JCeZ1BHCUOgxDFjb/ZeM77DPZ4hHnEkMPIixYwIPqHIUi0EIWPv3k/zEMJGwM7%2B/MdPi/l%2BIJ2smnTJoswE0nkkaVLmLdGW8M4JN0QQ5EF9IkOBMeBp5L1LUIGAhkJpCIzv5FKuGxEp%2BgjKIzDNmXKlP2FcdjoR9x51DsJUxjC%2BBLdooV4GHdCynLixhgWnuOAwghNHGeiRXjChjMqjFftFd8Jr9MfeftynJ6HG4uOcxQgNhhvYXANAy0bAyoGIht/YwiGATZqLKLI8xrfw/dR/IWBlQEzPDKA4sllAGbAxRiMDq7hEWMxRAfYzw3B45vQ/og8s2EcovTzSHVTFEOix1TCxDg8/fTTTeknxZD0Y8cB2hFL4DCXFWORgkcsyYERieFIn0J6cih4RIo6RXEwJOlznOMP%2Bg6MxzBmdWQ48jpjWxj3EsfBsPFdtLNQZCdxi77O8zD%2BhS3xbzbapRdLc5zk4cai4yTAYMmG1zSxSABGXtgYKDH%2BElN5iBgGDyzRQvYlbQcDLprKE6J7GIIMkihf0dQdUsJQ7jH%2BQrSQzzh9FxwQKGW0LyJDixYtsnRTDETSlDEEcRhgIFKshojiwoULTeHyCKJzKEhnpA2FwjhsGI7BMKBPDMupsFEQKarg%2B9zmvgXtAWOQMS86DgaHaNj4m/YTHUcTx9WwYQDi5GL8C4/RLbzGWMi4GcbSxLE1bIy7blQ6zrHjxqLT54mKAAoTRiAbg18oJBMeSe1DgWI%2BIfskik9iCg0DFcp7KAwQLShDihfGIINfiBp6Co7TEbQ1qlpSzfShhx6Su%2B66y5QwnBoo6rSl17/%2B9XLeeedZFIh25ThHC22LaPX9999vhXGWLl1qjgraIco5BU4uuugiK4zD/EbSmr3/chLBqKTdtFd8J4yr4W%2BcFYm0p6LS/nCmhrGUrb1CPPSB7TkxvJ06zpHhxqLTp8ATyoCExxNjMHhDidIwjwcDMBiCRHESPZ94RIkUYlQyCDEYJRaSQWknEojXHU8oRmB7Xk8%2Bz6AXvJ8e%2BXESoc2hSFHJdMmSJVbJkhRB5gsRXSRFcO7cubbcBQo76YG0N7zutCvHOVpQ8ukDMRrZQmGcp556yqrq0l/S1ogs4gRjTmwojBPmwzoOKibjJX1Ze8V3ohtGZTQqmbiF8ZrxmTGzvXE1upGJw1jMuHyojX08a8dxOsaNRee4IKSFhvTP8MjAgnHIvAleY5/25laE1xnMUJIw3FB2MPjC3EA2DEIGFuZNMMi0N5%2BC/Rh4MAb5m0HLcY4EUplRzkkvRUHHSMTzjpFIG5w/f74ZimEeIkYinnRva05XQR%2BJ4wLHGvNkQ1Ecqu2Sbo%2BDIhTGIQ06FMVhviPveTqg83Iw/kbH5WjhnejGWB3G%2BENt7EufSPtLHKfDxnuM4zg9omN94sZ7ZAC5Uen0RdxYdHo8wcPNAMEAwCPGIY8oKXgkMQgZPBIHEDyRvBYm7DMYEc1LNPLCvJvgieRvtjBIhIEkeCEZYHjP01mczoB2iWODiqakZmEkUmyEwiMYjrRLjMLx48dbiilzxjAWSb2iPTtOMqE/JsKNocgWqu/yGht9Jes3Yiiy8Tw4NXB2OM6xQH8ZHeejG1Hw6HMcbInGZuKGDkE/GozFMPa3txFNjxqZUT0ibLzn/bJzPOHGotNthPWhEie7h1RP3uNvOnMMPtKe8GhjAOLlJl2U6AvvYyzyfSH9JGwh1ZNIIX/jGaQISDR1lFRRKvzxnPd5ZBDwtFCnK6G94vBAmaEtMyeM%2BWEsfYHCTXulLZLiR6SGhdMxEknxc%2B%2B205Mg6h3WbiRNlegjfXLI2KDQUkhRxdmB0o1CTn9LO3ecrgD9AYcyjrjEwjvRDf0iGI1BBwnPoxvqMoZgSF9N3EIBHnQM2njQQ6Ib%2Bkh4zlQBny7g9AbcWHSSRmJTI9pHRx6d6M5GJAUvNYoGf6N0YDxGSYzoBSMvFJDhkagLi0pjCNJx8zoKinfOTneDLKBEYxg%2B%2Buijcs8999g8MBQW2jbtmcIhV111lc1HJIroUWynN0BfTSVViuI8%2BOCD8uSTT9rrtHkUZfpm2jbFcc4880z729u2051gHGJMkmYd1UVC4Z2wYVjSRyeSqNvgaKYPTyy6w99BR%2BE5FdDbW8rI5cHpabix6HQZKA2kKBERDBPTeY4xyCNpIlFvXtiiRWT4m46X1A%2BMQbx24ZEOl3XkeM77GIREBKNePCKLPIbIIo/eETvdBe2d6AvLXbzwwgsmHzhFaO%2BkL2EUXnLJJTYXkTXtQpTbI4lObyE4QkgBxBlIlJw2z0Z6NX0/Tjs20lIpjEMF39e%2B9rXxb3Cc5EKbJbU1URdJ3Ei/Rm%2BJRiXb24hUsm9UF2lvw3lC/05EMug10Y256RiVjtPduLHodAmkkLLo/Gc%2B8xnrPFGSUSB4DCkfGIMowWGeQNhCKgfv8TeGIF66MB8gzBPgNRQOnofUDk8ddXoypJz%2B6Ec/kr/85S8mF0BaHsYhc7swFpnfheJAupO3Z6c3Qx9PBglpqWxkjDAXFycJxXEYC%2Bi/Sa/%2Bwhe%2BsL8YjuP0RGjPOPbQXzoqvsOGroMTEMOSNh7mULLhRAmvMR7gHA%2B6TeJ25ZVXymWXXWaFohynO3Fj8TBp0cu0t6FZtlbXy86aRmlo0UFQt/rmVmlq9UuYCMbizq2b5Qcfea/UVVeJDEgVSUsXSc8QyRykzzMkNVMNP4sI5qnBeMBYLCiIGosxYzAl5fic15I%2BoL9kpw6Q3PQUyUsbIBNzMu35gF4Q/aTVN6kMbK1ukB01DSYfjczDa0ImWsXFoi0NDfVy629%2BKY8/eL8MGJQreSNGyay5J5qSPG78OMnPb5uS1NcYoE0/S2UiLy1FcnUbMTBNxgzS/qJ/78gIaNaGX9bQJFurGmRnbYOND3U6TtSrrPBeX6a2tkaNxqL9hXF2bVovdSW7ZOjgArnmfR%2BR6dOnu7HYAZkp8bFCZSJPx4jJuZkySP/u3wvGCpp9o7b/Lao/MVZUNrbY37XN%2BqhvHm9aaGtrrChf1Eg8sFGAJ2YsVqnhWKcGozTW62DaoI9s%2Bty2Bjnpostl4ZWvlBlz5sW/2UmE5j8wZYDkIBeqQw3NSpMJ2ZmSqgNJ7xgxegduLL4MGIll9c2yq7ZRVpXXyKKivfJcaZV1duWqEKAgowg47VCrRuLjd6v21KhWkSoAqhxLdp5IwbDYY8ZA1Qz79vxBOrgR2rmNy86QsYPS5aIxBXJCwUAZqa8VZPRMA5mBn0G%2BuK5JdurAj0w8XVwpm6vqpUrlorS%2B0QxGZMdph61rRSrLYnLAlpYh4ssK7AejcIi2/XE64CMT8wuzZeHIPBmtz4epXKT1QKORlt6q7b1EZYKxYvmeanl8Z4WNFbVNLWo8NkuFbhiMToSyYpFdW2LK8byF8Red9sBAHJmVbmPF%2BOx0uXTsYJmSl2XjB%2B/1RJCJam3/jBXbqxvkkR3l8kxJpezQ5%2BhQpfVNNpb0WR9Kq/YHGInVFSI1ulVXxp5X79W/9fmUuSJjpqg1lBP/gJMIw0FhBg7FdBmrsjFb9acLRhfYeDFcZQMni3PsuLF4COjAqpqa5c9rd8vf1xfLcyVVqgS3xN91nK7hbFWM33nCSHnLVDUkeiA4R55V4/CmNbvkzs2l5jDp6xETp2sZmpkmb1B5%2BNjcMWZA9jRwjOAouXFlkfxjY4m8pMZirTsRnS7mkrEF8t6Zo%2BSVE4bEX%2BlZ4Ch5dGeF/FHHinu27jEZwYB0nK4EA/ENU4bJh%2BeMljmDB8VfdY4FNxY7AG/YirIa%2BeHSbfKCDvy7ahqkRgd/7%2BicroZUI9LuTh%2Beo53dGJmUmylZPcQ7tq26Qf66brfcrgrx%2Boo62dvYLC2kEcXfd5yuIKV/PxmcniozCwbKW6cNl0tVSSbK2BOoUBlYUlxlY8WqvTUWRanry9ESJ2mQmTIhO0POHZUnH5g9WsYNSpe0Ad0/VtD2mbLzu1U75T9b9simqnqLrHu2iZMMiDYW6HgxPT9LXj1pqG6FMmpgz3My9iYGfEWJP3fiMPgv3l0pP12%2BXR7esVcNxUZLH/JuzkkGzOHACGNux3bdRg6MpaQyv7G7YN4V6XX7IydlNVLZ5AqxkxxoZzVqgO0m9bm2QfapMjBelWTmbHUn5aoAP7SjXH61osjSsUvrm20%2Bu4uFkwxoa0yHKVIdhfGCFFVSUlO7Ma2dY9qqxuENy3fIHZv3yJq9teZ8d5lwkgVtjcyOknqmyjRaNtQUnO69ZI5vT8SNxQRQip8rqZZb1hXLbaoUV5M2EX/PcZJFUI6J3on0s3kpIwemd0tHRzR9j3a6/9xUaulEq8przaB1nGQTnBbMASTaPqtgoBWD6o7xn2N5YleF/FXHiru37DHlxKXCSTYt2uhw3DFWEFEZPShDCjNSu2WsIHK4o7rRdCfGik2V9SYnjtMdMD2GebG7dcxIHdBfpuVlSYY%2B%2BvJpR47P/EygWBvV/dvKbPCn0ql3c053QcYOHrG/rN0l/9m6x5Tk7qC6qdVSsm9cUaSDf53PT3S6FRwVS4orLcq9dm9dtxWNobjT3Zv3yIM7yr1wjdOt4NCramyW36zaKQ9uL7eISndA0ZrnS6vkNyqbO3W88rRTp7vBWbF6b638eNk2WVZabfqMc%2BS4sZjAI0Xl8uiOvVKkioDj9ASIWNy/rdwcGN3B5qo6%2BefGElldXuOVf50eAQoAUYsfqgKwu657nCj3bi2zyCJea8fpbjDLKMBH0bFHVIfpDhgj7txUKusq6mxpDMfpCeDgplL1r1YWyfqK2virzpHgaahxcICR9vfzl4rkcVUAukMpZt2kt00fIT9dOFXefcJIMxKIdHJcztEzb8gg%2BdLJ4%2BWLC8ZLfkaqrYFGasLRcNX4IfKFBePk/bNG2xo%2BeE%2BTcX8oDjAobYCcOSLP8u6TlUTB/JMndlbIz1fssPlZ3eUnvuHsqfIlvX%2BUix%2Bo58%2BcSQ9wHhss0/LZ%2BePk3TNH2rzYp3dXHdX9ZY7UKyYMkW%2BfNkmuHj9YBvTvZ5HorgaDcUtVvZw/Ot9KpCdrnhbREpaG%2Be7SrfJ8SbXJSHdwoZ7398%2BYLJ%2BYN9auAcoQ18M5NqieeOcVc%2BRqbdP07ywJdLS8fspQ%2BeW50%2BRilbWnd1da5K2rYcoA1YNPHpotWSnJm9NLJtZ/t5XLH9bsSsp5tgfz%2Bj84e5R8Xvs1Cv6ckD9Qj6ks/q5zLHzyxLHytVMmyCnDcqxo0dFCAb/rVcf91EljJVPb57PFVfF3uhbyBAkCLRiaIxNyMru1BkRvxK9WnIbWVuvM8YyxFEB3MKtgkA0qJxRkyXzt6C8fWyDT8rPi7zpHC0UwpuZmyUmF2TJ2UMYxDaCs/8YANFcVChTUNFYQTwIYpCiCrN9G9dFksbGyThXiKlsXqzuMM6r9Ucns3JF5MleN/gUqF2eNzLV5as6xka9G3vS8LFWOB5pcHO0VZV1ElNPZ%2Bj2sETo0Mznrg%2BItRjFeVlplFUiTBcU6Ht%2B5V9bGC3d0B1xv%2BrNzR%2BXLDB0jzlGZmJqXaY4U59hgvKCfma19PDJyLAyL3yeq%2BDJXKhnQJjdW1spK1WWSCYVslqosdtd0CWBcOE9l4mQ1CE7U8WKhygVLUfWUauK9GSq0MwZPzc2Mv3J04ExErz1xSHZSK5Siv%2BDwXlFWLdur3al2pLgExWHOyZOkFKnS0V159hNzMiy6iIeYDh/BnKCvOQ6Uart4QQ23ZC7fsqGizqJE3VWkICdtgFwzsdCUthDZnJE3MGaoU83B6fO8uKcmqQoqa8U9UrTX2mN3jRWUhCcClj6gnxU2GZgywJyNjCGOQwXIFSoXyWSVGqfMIe6usYK19SblxnQo0nFx%2Bo/MSpcrxg3u9qrJTs9hTXmdbK3yaWZHihuLcZpa9sWWA2jsnqhiYWaqjBmULhna4W2opDHX22usoXSs3k3n%2BID02RXltaqgxl9IAjtU6dhS3T0dK2vrsVYSSjEKyPq9tTY/DDlZOCLPoo6Os1plAkdKsmCKAoUSunPRfWSCaBXp9Hdt2SOsAXzK0GyZX5gd38PpyxBpX2OVtJPHFlXAWb6ju2CsuGTMYNObVmqfwJaV2t8ii4wVvmSCA%2BjXXpPkyHFjMQ4pTRu1c%2B2u%2BYEsHHqWKsAs1fHM7kp5rqRKmAqzUDs6UlMdx4zFsuqkRhZL6hplVzd1rMFQZH0kqvs9WrTX5ALnydtnDLfoouOwEH5JffIii/UtLZZy150VUImgsJzO8j018uVnNlkRKiIqpNs7DkWfaKPJZFdtg40X3cUwlYfXTC40w/CB7bGK9kQXcaqMHpQumT5HzVEwFndoW3WOjH77lPjzPg0esQvvWGoGY3esIffDMyfLNROHyLLSGvnw4%2BtMEfjxWVNsbsqf1%2B6SL6lC0BFvnTZczh%2BVL%2BNzMmxeBCmspA7i%2BaZwRa52nm97cJVUxKOmefo3Ka5vnT5cJuVk2hy%2B6qZmG1z%2BuHq3vHfWSItwUhaetZIOBSker5081BT7jz6xTt48dbicPjxHFfpUaWptlW3VDfLzl3ZYxPa0YTlyme7PPI5mbXZ4dzAA/r6%2B2OYehcvOOUxSxef9s0aZAkQqYj/9hyHP9/16ZZEsLa22VJMo4ftPGZqjx5MiTfHfYOI9E5p5/5criuT3q3faPLzAVeMHy4WjC2xQCdGqvWqYPaVG%2Bz1by2Txrgp7Dd4%2BfYR8YPYom%2BP1/aVb5eZ1u2V7kiJvOEZJwXz61fOTllbzhac3yk9e3N4tc7MWFGbLB%2BeMljdNGSY/Xb5dbt1QLNP1/L99%2BkR7/%2Br/LLd5xu3Bgu2XjC2Qy8cOliGZqTYfD4OTlL31e%2Bvkcr3nK1VGqPIa/Y7Ltf1QxGhaXqZdY9oYlf34LG3qpCHZ8q5H1siLe6rjn2gL8nax/vab9bhXqUzRBpnjSltmUj3t%2BLniKqtauKGy3opZnaptc3BGqlUQ3FpdL39Zu9uOK7EQ06smFsqVeuwHFqTvZwraEm3P96hy9HRxpRXrClBMgO/mnJhPyLGF3yDdHa97QUaK/GtjqXzsifX70yqZS3LWiFz73NjsdJXJAdovtloEj3NnEfoQRcCT//rJw%2BTj88ZYH4N8/mz5DnsvGTBP6efnxIqCJQP6HsYKzjXZy8jQjrmfX1gw3lJRuW%2BfWrxebjp/hvVh3P9vPb/FHI6J8NmpeVmW1r1Q722h9sPU0KRP/femUuvDr5ow2PrGHy3bvt/7TrvEYYlsjFOlm/XK6M%2BRC6L9FIyo1Tb9d5XPB7aV22c64jWThsp7Z46UF/Qa7lEDf1r%2BQMueSdPvZOmHxXrct%2Bp4kKuy9hYd107Qc0zvr%2B%2BpHBLNvWfrHjUEDv4NjATkgmyD0TreZeh4xtizp75ZHtPx5Q6Vs0TDCTmgKNBrJhWaIYFckl7MveW%2BfvWUCSojDfJRHYtv1z4CuH6D4239vFF59rukwje27DN5opDKnTpmMpYFPqz917dOn2Tp/K%2B6Z7lds2TAWHG2Xo9HXnli/JWu572PrlFdZXebcTlZnKlt%2Bl%2BXzba2%2BXHty9B/aGu0W4ruoIfQhhJhnu9s7Ruv07GdPpq/KVrFWPHIjnK5UNs%2B/SgyEtWH6IORi8vGFWh/GUv/xmB%2BaMde2VXTKB%2BeO9pkKegrHcHvfVfbCM5P9qNfvVbbc4HKHY5hUuzRk9BdeP3iMfmmH6XoTa7QNkt/zHElFhXD0UoRLAIOYW4gcsr3L9Lvow9PBF3wI3rcjHNDs1IP%2Bg3knPntTIW5%2BM5l8U/EoF9BX8OxS8Ezio3xW7R7jo3rvlfvC/D%2B7XqfGN%2BvV93i86pjJBMizJ%2BbP06%2BrjLuHD5eDTUO1bt%2BsWKHDRTJHP5JtRuXnWkG34isdHlGB/u/6WDJfBgUuSmqtNLpPauvM2BGjw3D5jo1%2BFCmmdDNYI8RlqePFDuYUZBlxQ8GZ6TJb1fttPQpBv5zRuZbxbBz9ZHvYGBB8aSjRNiZII6BRmdJafhDcfrwXFVeh8hJhbFiAKepoThElRDmD2AU0onQ6WGose9wPUeKwozQzmuCnjfGMArXcu3oUFQ5Poo2fHD2aOuIGdSZEJ2mnQ9K/wxVLjCKg%2BEYDHsM1DdOHWZVGSerYcirfIaF7OlYOZZBqSlWeYsOmSp3nDPH/ja99lyTEapo8HVcDyZzY0hzLgx%2B27SD5T0mZVMNjONijivHnczKb1xLqleiXCUDjGWqAydZJzZO1MGEe8Ng%2BjM1FqnWma3XfZ7eAxTMIh2QuY9R5QxYqP1aVUrfNHW4zBkyyGSMQW%2BIygEKNrKBvCBjpJ5TxAdwerzZ2kKetVMUBn4PRwPfiaGKwYVCRNvrCNoVCj1KOceJUoFSrIdhc8vG6d/MRaYdMyifoUoOhZM4TxRQFg6mai9pXdFKjO/Twfgt2sYZtHNVxpEXqsmhCHBOtFkUpVJVkjl2ZJviGh%2BbO8b6Er6bY6DtIOtj1Ajks8gQqZz3qbLLbR6rx4eRTfT2DJVZjgtnDVV4SQGeodeAc8TYJILB%2B8yXY19%2BFwXpmSRVuAOO%2BYpxQ6woSTLAiPq5Klr0tclGb59cN2OEnKPGCvPSbt3Akja1dm/pa3kkO%2BXxnQf32zjgJmo7xtFFHzlR7z9GNmdg7VH/Jo2V/rWqqdnW6iMqQ9ulKui71BCnj6cPpaYXbYuxhfGCNsYi10Q5X64KLkYWx0//yW/hFMURka3jDbLJ79Emkc9ThmXbMdLukBeOmX2fVIMSA41jx%2BmJA%2B91KrsUHsPJyeu0Sdo4fT99JuMFxl8Ao/V/9HOMSRim9G%2BMX8g658V1REm%2BT/s/ri99CP3He2eOMgNzliri/AZjF9eCz4UKizgZA4x7GBv0NYzrZfqYLMaqrL5dr3WyYCF%2BnADdMVZQqIt2jQOACsX3al%2B2Kn7fqKCNscd4jU6TCH0jlVMvVCMMxxfGBP00DmGKdtG%2BaQ98H/oQ79NH4syg3WGU8TuADofeME9lhTEA3YHfjba9RNCV%2BH0KKtH%2BkSn0IwqH8bvjVU8ap8eCLsM5Iq/0%2BznaXpEZZGOvtlX6YtoZn0OHQ6e8QvUbshDovzlEnB38xqScLNO9kNeQIcH5vuOEEfIG1SeRJeQOZQi9kr9xGjIeMkXqTzoGBmjj79T%2BgWPjWFFNuEYEDBiX6Hc4Lhyu6KDI6Ov1NzgfHKIPqUGeTGieOMtoK87h43H5ODQgBqBkd3QM4njw6QDwIJFnzzEQycEbiiFC0YLTdFALHRLEBv8MGygZuNZV1Jr3DKMQTw6LkLJPoQ5wURB6hPoc7Uzwht62odg8X3zm0aIKM4RQVI8UOq8LRhdYh4oyzXcSMYSLVCjpkBlYb9HXfrNyp9yyrtg8XAzM5%2Bn7dHCAMo737Mpxg83Te9fmPXZOLDbM93GN%2BK5X6z5R5RDDEi8a3/OvTSW2PxFEPL0o3Biakctn1xJP3lumDTMllzS2W9fr5/S4f63HR8QJxZfvpBpnXy2mgkGS7OgJ4DRA0SMyvkmNOTyUOHKI4uJlRTlF8UQhTARF85XaxhlEWQvvD6t3ye%2B0LfxD7ymDIxHkgTpgBWgLOBO4z6cPyzXj808qD7RTPovxg13AsRxpK8C4wluL8n0T36ntklRaHDoXaZslSsgxEUlEZm7fWGoDLW08FLfi8wzIb1HjFwMaQxV5pZ3ymftVMWJOJzKAAlMYl1%2BURSKDKKtcO9Ky%2BAzXgvaNM2WQDv6JMO%2BNPoJoKNV3/6qyyu/8Xo8dJYwIK8o2RjXKVV8DcSA62w22ojmyuD85qsit21snS9Qo5zAeU%2BMQRRinA8ptInjzMfq4rxTFIWLCPaU9YHAyVmBsJoLj4rV6r1mGgd%2BindIf03%2BjBI/SNoYyeSQQKaRtMsY9tH2vykXs%2BxiziPJxnIxtD%2Bt79OHIIQYARhkKONFGFE4MTuT1DVOGqoKdZUban9fE5Ai5/c/WPXY9OGdkDaORbhwD8%2BoJg21Jpe06Bt2k%2ByKXKMCMuSjRiZCNc/HYfDMOUOQxin6/KiaDHCPjHkoxcoHsYmz0NRgnumOsANou/RG/T4SXPhJdgYgY89xppxhxjCtRMGYuHVsgF6g%2BRAYSYwRtgTaJYThaDahYpOxAz89z9IJXaN8alhwKffHf1u%2B2vpa2daRQsdUqSetPkbGEnN2scoH%2BQpslMIBhyTrg9MV/0/dwdmKgnqU6TEhBxxlIpJ2sL4xk%2BoYg68gFn2FsxcA7V8dQrgmODwxj9EnOGVm6Sdv1b1fFzmmb6oo4bugnojC%2BkcGDzPK7RPF/p8fG76GvUfCI%2B4LjFIer03txY7GbQfhR8hBYOrZoyiPRxA1qBOJdukA7p6jBwsBHuiXGVXlDkxmKpAwS1v/281vkx8u2WRSNtWWinFg4yISXlJ8btfP47gtb5YdLt%2BlntsqX9PMMgkeTRkInvb2m3lI9vqO/z3fGUplikQcKpeBZ5T1SCkmVsoWD9ZRi3rzY96AI4TVHkbhFOykW3ea7OB%2BOlQ6MFA9SK0gvQbFmIOdaoPCjwHz%2B6U2WIvoD/dzXlmyyz1jEOHIpUDxmaedI6hIRAjq2Lz%2B7yX6PKNbXl2y2whFcc4wSvN3JiuY5MacGaSrcs0VFFXr/Ym2SuVms8UT7wJPPfYmul8TgSKoMEXkcKMgEbY628H/PbdZ2sc3S06JRIYwxlNDZBRTSabUB71OLN1g7pc3xWdYPOxo9CGMbOaYd/kxl80favlBii%2BsazVijEAXtLfzO9foc5QBlM5R7H5yRYpEdDGOWT0Fh/8oz%2Bn26L99Hm8WpgiGBAkMEhn6F/a9QY5RvoX0jE2zI00efWC%2BLVIkg3ToKn8MwJXpJ1PZ7elzfem6L/Q7H%2BF3tJ%2BijUMJJ%2ByKi6CQH%2BqLRAzP2G1pr9taYEgc4HFjKA6cBEQCMrqiCy/1687RhZtg9qEYYbYb7SRv6tLZ1Ui0TK8oiFyjhOChwoAQZ4pEx5rNPbTRlkPTkI4E%2BO13bGe2Y4/ixjhM46XDs4CDEmGUc%2Bpr2waQz0/Zu31QimyrrTXFHcUZpJQKDwxCn31qVdb7re9q%2Bw2e%2B/MxmGwOJcGDsEgHit3E0kiGCDP5VlXL6CMaXb%2BjvIfOMIYkwBhOJJKL0QmmV3KD7fVXHC67fd1QmfqNKNUo4MkdWDOOzkxxoCxiD43IyrJ3%2BW/vvsN4oznacc9x3%2BjQicVGYvoM%2BxNJpfI428H2VC%2B4p4wZTBYi84ZwMIFevnFBomSE4MX78Ykwe0L2%2BoLrH79UgO5r1TjkPsklwtofv4xhwNDJW4ByhH2daCBu6FrpNZVOznT9po/QRROtJQUbvWrRzr/xI5RXZoG2j19yossYUDFK2mTaEYyZE8zGo6VPY96vPbpafqGx%2BS68Fv4cjJUxTCBDEQIfC%2BYrcIEvokuzPb/5Cj5FoOpFOxthowMPpXbj2283QQZD%2BQwfEYBPNbWfOIOlcpITOVyMvKmgIN%2BkVdA7Mf2IgjYIhxnwNUpWiYJSSbkOn9I8NsUhdoFIVEDxWR7MGDfnp/9lSdtAEdxKFUCRQshfp8aCEB8q0UydtIjGFk4Efzy5ecpSIaKdLWhQd6DI1qnNSUyxtAoUaY45BnE5pjSovKAGhT%2BP7Ub7XlB/c0ZGKcY12%2BBgXeIVJNeL7Aww6RJS4JxzPO2aM1GvuBVWSBeuMkiJGm7xjU%2Bn%2BuXuksRCRXq/KIQow3kqi8gEGPKIh9S37dECslSJt3yFVmc/SnpCNioYD7Q75eqUqkMgG0Q3S6aKgbNy/vcw%2BmzBWvizbVf4YZEnZDB9FoWcj3e%2BBbWUHtTsiVhjERLUDFlnUa0GqHvNmyAaIQvsl7ZD51jhB8JQTaeH5SJUnDFJkk0hUlKd3V5gHPso0/RyfVbEyA%2BR%2BVVRwtASQre%2B%2BsM0MSSKPpCY6yWG49vmfPHGMpRsTEUu8d1QtRjZIWfvEvDFm4AQYZzBkqrWftkJROmZEQSaYxxRlZkGWGWaknbI8CVH9UAGW9svYgoJIWzgSGBMoXY8iy/ECzhvkgD56e3WjtdVoejnZBYkK%2BOkjcs0ILKtvlrt0bCGdHBkHZJ50YQqdkEFDn4BcoPCQRo0D5nkda29WxTzKC6r8J74GKN4Ym4zAzKlMXJT8nxtK9s9tZH0/nC5OcqD/J1tkcHqqULyEyvYB%2Bl0caThXcG7hJIiCw3G0yhVpohg4pEsGmF/6uac22hrDUd0Bw3NcTrrqNfss2v6w9pFRWGuSMetoYP3Wm1QHC3A8OPaRGY6D/p/2DOhu/9xQaucWQOavnVSofUWaOeoX76q0%2BexRiEwStUSfxBHENCWcQlwf5Bq9MaoXMiYhXy9pHxD9LSCCOkU/ixzj/GGqVAD5ZX4xOi3GJPfpSLMQnJ6D92jdCEYIKSsovXiNPjJntBUvCdtTur1jxghTFplDRWpBqADJYMQcOzquJ1XpW6cKaRS6NgylaOeH1wgjkw4B4Y70qQYD7cM7ylXID3zmyEj4wgh4v6LHcijCJOxELxbwCt/DPkBqBF5cjG2qhbZXjQ0lBCMg%2Bn1URiPlA4%2BhFeiYMky%2Bf8bkgzbmsJFXT0GHyXlEa3ytpmTBPWHgw0v607OnyEOvmLdfLv556SyLPCIDRBbOVEMqQESRzxIxw0hLrBxLqhGLVYf2A7QdIjHMpSXtkshBIrU6CDLnIjZTqnNA/pj0fzhz31BSy1UuGcRDoYAoGJ7IDMcXXErIBvN9OW4ipolgHCQ6mcbmxJbq4bPMP/nO6ZMOkomvnTJB3jRtmCnbzPWi33KSAwoXyh0OMqLu3zh14kHjxedOGiczVS6YT4qjJaRCMr4wZxaDkZS8arIs7J0DrFCZwOkQBaMUucBAxIFSF3FgAG2N9PCos%2BNI6KjZoxgnttf2dqWdm1yorDPdIHHZK44PI5eiTGFf/k%2BUpkGFb4%2B%2BnlhAiqwajN9i3Vp0nAww55m2zhhC5fLvRWSCjb9Jd6UvYVxnfyc54CCkQA0GPXLx78tn75eJe6%2BaI19YMM5kAN1pfHb6QVFfK56n9wq5oG9NTKPlfu/R9h2cd8gWTgOc1eghtL3EtrlZ%2B9REA%2B1w4bvaa%2BscVoP%2BL7EXb288Cm0do3JnO1U/MfiCoz7IIEEHxlPGIow7HDBR2I1xJFHW6WPoV3C0MnWIojFRufjaqRPMqY8OxSNZP07vxI3FbgTDjVRKJhIT%2BkcJRPCiG8JJJ0YEjAqLsQp2MUWQ9Ds6BaIkweMbBcGPdn4YO3yGDobOL7EQLoo1HWB7iuWxwsB7uPMZ6KCjkZVE%2BJ6ooYuHjOvA63jBEiGNJBrZAf2IDuyx5k8nRu4%2BHrnohiFPTj570ZHiYXe6HiaeE7WiPRJBoS2ghAWZADzIeDHxVmIgBlCkua84RKIGYYDWwetRI5LWw/fieGEgTYx2A023vbZ1TOgxcJyRQzkk7MsxtCdHvJYoM%2BirNFk%2B095PkF2QmIbKdUCe6CsojICDKioTzD25ZEy88JT%2BgKcVJQeu9%2BTcLCvkQV%2B/W/sz2nGQCTYcBiiBtBP2JZJIdBl5wNCkfdMWEp2EgJGUKC/he%2BmPqZad6HgBDEh%2BrzPhZ45E0vh9FNlEueAv5DmxujlNlnNhnEs8ds6V8YJrfHAfERtncDJSxC0qE2HDcUX/g9MJ2XOSwwkFWZZZhO5CFhH3N7Rd%2BiiKG2E0cUuYtkOafQCjnteRi476SdpIaCbsj8FIW%2BD19to%2B7Yd06s6EX7H22N4BdgDOf%2BpwJEIbb9JjRD8M79JeuW78BMfeng5I/xCK4QToU/gcEUPGbcaHqEwwLYK599wLUs%2B5dk7vxI3FboSiK%2BRxkzbB/LjYJOS223%2B3lpkwMy%2BPSAteoDCoIex4fyl/nwhzPygEEMALhlKA8oDxGc3DBzpA0jO7O4KG55eqWwcf3QHoeDAegPMnWkSfzet4FxPh%2B5jrEv0%2BurzQ0TOQkE5CykR0oxIoxR%2BYD0pVvERvm9M1nKdGOmljpNmR2kKBi/bkgkgI95x9UQJoEjVNOkDqQMfreDsT21CsKmqqtvUDcsGQWRc3IIkKIGOJIBuxttVRq%2Bx6cFhwDJxbIsynRWaix8f5MPjzGQb1RPC0Jxa4weBEiUBxIs2Q%2BTKJcsFGMZC/rN1lc8KcrocMB5RcoiLMxSXlsT2ZIMJGuibzuJmLhbMLJS%2BkdKKsYcwkQjQeeYmCIYjCuT9aFu9zD0DFw5R222MyQS6mqmEcnbsMmAEcd5j7GwNH4z47F8Y5PpsIl4fF3KPjIyOFOWR0I8rankzcodf%2BjyoXzCM73Cwa5%2BihOdK%2BmbuKEU9WSHsywfjBEjM43SkCE81EadK2wG1GLmL9ZPyNCLSfECnG8UAqPm2B16P6VQDZI%2Buru0Hnac84Q38kQEGhqXC62JQq6nb%2BZPS0pwMi54l1G8IYQ1ZXezLBNAaKBqFD3as6bmIqudN7aNvSnaSAx5eOjqIspL0wcZmJ%2BYkbefSsX4WhQmEDqocSXcRjSgljhBUlAk9yFDoJBkrSxQKkqeF5pkPAwCQNk44jQLorxXZQHLoLPF4cEteGSdtRBYABHqUJJRcFxhRb7bQxKvAeokiROsW1DWeFZ5HzZG4bzwN4jhk8uH7MSfvB0q3yrodXH7R9dvEG%2BaG%2BziRtqvUxSDhdB/edAZv5E7lpA2zwp4BEbGsrG8y9IgpISurV44fY/d2kMkEUmUgMRqRF3%2BO3HYMJJwlFWQaqMhjA%2B8p8J5wvtJPESnYolJYWq/IUaUJJgzZK1AjlhPkh0TmawPEhE8g0ijCKKoo%2Big3nRHVU5tsmKg4UzWKZmSjIRKV%2BnnmKi3dXyIceWyvvfuSATLz/0TVWXOcny7bZOl2P7Tx4npvTNYzLjq17SVug3WOQtCcTf9DXmfeEA4QqhUw94F6ipGE0MiagDCa2BVLrqIYYhTl/jBnIEO9np6aYDAEPGJ3IanuOymSA7JNdQAEbqv5i7EaPD2cpc/2RW4xe5AKjL6SrIi/0EVHoH6gGmWgc40DhOrLhwGL9xehY8eHH1snXnt0k17%2B4zcbs6HxLp2ugv8chwrxDsrKQi/Zk4gbd6KtWldeYg57CdowPtBUyt%2Bgn6T/p%2B6PGEMYREUuci8HBggxRT4G2x3eMzEq3xygUGItGL5MJDnBSqzkndCic/4myPkyPmUI/OEQ4n%2BZWIqHNlq5Nm2ceMNcj6Ev83%2Ba/q%2B6FcygKwQccrRsr66040Pt0fAgy8R4dN/5Xxw%2BKpCEXFDhkqTGnd%2BLGYjfBOi%2BkeCGIVEEN1R7bgzmEGDR0BLE1DbPNUAzLO1AMhIpWUdgHoaeschSUQQpeoHCSJhAtgsBA%2BaapsfVvuotdapBxfKRQMYczqhjjsfvfOaNk7pCBNgcSI5HOjnmWpPFS0p9iQShVYaBnTsJ4HUxY04v5WwGKNVBiG2PzvJH5bZQGeN2UobZ47P1Xz5O3TR9%2B0LVyOh/WUCP9lIGMtk5KUWL6cBSKwzAvkftCW2fw5zUKIKHgEm2kzD2RZcCoYvA/Y3iOKb4BFIab1xZLSW2T/vZAW4omCql/zAGjLR5oQckDx8ZTuyrMQYRRjNxGoW3Tn2AYoAjfu3WPLZaOQkORg6F6fSiJTuGNKGePzFXZOLjfYDF3KksyF4ViEPQf4foBSjRFVu67aq7ccvFMW0rA6XpMmVUlF6MHxyIFudqD%2BUZ48RkXKI%2BfHzfkavVvinVlqoKIgp24JiV9Jm08Cm2ByrcYqBiSKJ8hSkebQFE%2BWccZ2kR3wLER2abfZ1xk6YoQKbTjU9m/aHSBHTdz1qm2Sl9CRgLOV873vbNG2f4BCklReTgRsldIfSfd7oLReW2WJ8HQ%2BMKC8fLwK06Uv108y37T6VrIkOK%2B0y9TEOmxovbXhMaBhqHIfUdfoJ8kZZK2TK0HHCmMCx9WfSNqDNGevjB/vPWBUUcz8kAfCchR4np9zJ%2B8bOzg%2BF/JBf3uHxtKZGdNg7b/NG3jOVZhPgrV5qmFge7DAvs4NqhqjJOJ4MGFKjOk9lJQDbjOjC8UEUIHi4KTFYfStLxM%2BfqpE6zIUADDmyDGHy%2BYYXLx8XljvMBNL8aNxW6C%2BYp0WiiCrH/GhPqOwMNLGiTeIgQWjyieNASdwRIP2EfmjJFbL5klPzxzsvzmvOnyrdMmxlJcE3LPGWCpbEX6xAdmjZbrF06Vn509VX517jS56fwZpqh3Z1rREzsrrQoskUPy3793xiQ7vp8snCI/1UdKPeNJpqLXPaoUAwoA1R5Zo5IF27968nj5afy8fq/n9PG5o83LHIWUXJQhUiTg4/PGyj8unWWf%2B75ewz9feIItwIyShmFJtbtD3SPn2GFwokgEEQLmKlIo4FBQKRhjEccAxj6GJuk1KMzPl1ZZ1OwvF86wtk1buPGcafLt0yeaUyTiN7AoHBXslpRUmry8QQ2gOy6fLT/QdsBnKV5x1fjBbeYEJguKNv1g2XarTopizGLLN18009rp9SoXyDrVXDEkcCDhWUYmcDBdv3ybKUgoNF8%2BeYL8/JyYXGDsYTRwLaLwOZYJuXfbHpmgCtRfLz5BZWi6ysUU%2B60fnzVFDcRhVtyGBZVZY8zpWoic07ZR2ohqhOkH7cHrtIOw%2BPjlqrSyJhwFPH724g4bS/j7/06daG2bNnSbjhuv13tK24rCd/E9VEol4v997YsZW2g/v9B2dJO2C5x6UUU6maDkU9WVYyRy/t3TJ8kNemwU1qCd0m6p5ooCzX4sLcLceaKyVGLluK/Q68O4yfhykyq1FHAiKlTfQvGPA1eZ3/rSM5vM6GAZAAo/8f38FrLxTZVBCnzQR%2BC8PJrlp5wjgz6cyBn3kWVk6L87gjtJRd0d1Y3mXHzNpEIbb0jbprItt/qiMQXy2/NmWNtmQ/c4Z2TufgdJgHvMeoX0xzjgvrRgvMkFutfNF51gRifLWHQXFG9iCk1xbZP18d9QI%2B5nqtcwnv1B2/hH544x/RNn%2By9XFOm1q7P%2BgaI8rK84SK/LJ1Qf%2BqWOl8jRr89j3JxkBiEGZhQq3P9zY6kFOy4dM9j0JutXVC6u18/%2BUq8jTlvGJIqpoXc5vRM3FrsBPJ6k75AeiSKI0YLx1xF4fpYUV1r1NtLzLPqin8ejw7ppVDXE%2BMP7g9LIUhJAWkF108EKLtXrME4xfvC4MTCioKNAUG4Z4/Noq9t1BniqSLsllYe0IVICGYSZKE1ECIOZRZCZM8VaQQHmVrFQ%2BeNqDJOmy4K0fIb0qRpVnighHe3oeF5US2e5Q%2B7aUqoGQ4tVVWMx92v1GtLJEp1CUbrhpR2yTa81v%2B10HUSzKCpEWg9VDF%2BuohxRNJRfBiCiG2ZopqfagEfaMOsb4nW%2BSNs2bYgIPBGHdaScapsIxNI8W6ycOLLBfUYuiPwjF0SncRQQqeuOFsDxkEnAGpEMzqjmC1WJoZ2y3hcOnm36PkowCyITjYVS7VuYN/K1Zzeb3KPYIxdsRFCfVJng9UQwnIle3betzPoook6k%2BRHVPGVYthnXf11bLP/aVNKmCrPT%2BdBu52vfhJLKcj44GA8F/R1LQDToI/0fDkZkBSOG%2B0YfS9EJ7iltiAqFpNi31xaIxtDX3q9tAblEPq/S9nPWiDxLTcOpk1iFNFlgND%2B%2Ba6/8WMeKh7WfxsnEunlE3i8bWyCTc7KssvGvVhbJndrH4zRBfom2M4b8ee0uS%2B8m%2BsJYwWcxDMj0ofIw42OAqD7OkZ/qby0rrTYnIkajFfFQGaTIFteCsYvvPdR47hw79Ev0z%2BhSpAZzT%2BmfDwV1CV4qqzYHGVkTpCezNBOOeHQH7jFLAV05bsj%2B6Tj0gUxxiUJKMm3hNtWhcEpyLPSpHA86BGs2ssxEd8F8c5Z3%2BZ22ccZRoqaMjRwfbTxXzx%2Bd8YtPb7S2jlwwzx99ijEG4xkDnLmd6EMU/wMcJolTcRhrWIuSuaFMcZqqckABNOSCNHiW8QlzSVnGpr2CQE7vYMBXlPjzPg0dDYuVJnYMXQGDLJOPy7WTw7ihs0IB6wg6J95n0vGGynor/U8lQwYnvKooEXiGWGcLpZL15R7SwROHLymYGEYIKx5nFM8KNcJYJB9vj0Vw9DN4TFHO%2BTxpbSjQRGie0OM7FHic%2BU4UjSdU%2BcR7FM4lpV9/O1fSd0gRCd7uAEo415sO7SmMOf1NzoU0VNY4QqHHIMYDxnfgyWM9ub%2BqIYBCG13TB8OSa8D8T9J2d6khyLUiuvKo/jbzr0g94nyIRjHAcF2IGjKw765tMuOR68E58FusIXavDhYMGCydAHa%2BLXq%2Bejx8F/uitCQL5k%2B8e%2BbINhPNuwrOHeMrGWCck77FvcdoY3B/OQOd68C93KjtllLhDEy0GZwwpdYOVC5UVrhfKHkMjszNJTpPu6d9oDwDa23RDzBZn4GRdodcPKZtDlmZqnKBwoxBxnsdwUDLudCWSf3jPJjTEWAOCW2G88SJwXGwLxAlRc5pi8gMczADyA/tnPVEWXMryAULQ6PMc81YzDxcMx6obIfCj0yS4r1drwXf84wa0hR9oL1zrsg%2BihSfxBjBQYUCxm9xXdiH30IJYp4iEXn6B64958t94Bwo/sF3JS7J0dWQjpaYWtlVcF3oT5MFbZUUa679f1XhYs3Y0F7aA2cC7zO%2B0C8v1TZP1gXKcLHeV/rXXdrf0X64T8/o/SINjag8zhXuNXKP05D%2BmX6V9rDTPlNvbZL%2Bkf6eDBCcjEQWaFO8figGpaaYwojc0Y44JqANUV6fMQoZJRLIcQQyVCZom8g2/RHjFO2cYyQDYY/KLA4d3qe9kpaIs4iMk3tYY1SvQxT6ffoGxhBkib4DOVmkso4izfqsnAvyyZjCb3NNN%2Bt%2BGMcoyaGPQC44HxwzTG1A3oNjEsOE44ydb0VS9IsA9/LtM0bE/%2Bp6yGpIXKO2q6Ad4fzC2MdwYyyOtpf2wDBs1tvCHLuVe2P3lv6Lvp37SL9HX8yGPDy5s1LIXrpcjUfaJ%2B0u6EPcR2QC2cBIQp%2Bijyfba4XqX7Rxotoc0yPanmiTHUEhJrKlaIN8P7IaBR0KZzbtiveRkQD9AjpBGGc4D97lmGi39PkEDXiOrKM7PqW60H9U56QPj46vnD9r9dLnYxTSzyFnyMHDeg7PFVdb%2Bi390OKITsB1QJ64rptML63X826wa0j2DzLBWpfoY8Axx9b6jp0vcpdscHolpg87h6bfvsT1E/ooCNmCW5e08Zz0VFBGETg6CjpBFrkPAy%2BgkH5s3hh5/6xR5iU98/bnbR8iMBS9oTOj84h6T/m%2B96ghwpwNBmE8U0wOd3oOpFsuec2CNpPWu4qPPbFefrRsW/yv3gEpNqRJotShCIYWziDF3F7SKU8blmtROiLLS3ZXmfOG4gc4Ahj8UCICzIN6zaSh8q3TJpmMnP/vF5JmQDuHxy/OmSbv1b4rGaCUnXzbkvhfvQPaMCl7gOKYmD1C5OHd2u%2BfMizHIgVfeXazKYqML1R2ZGxBISZqASpKNl0BWWJ%2BFgrlD5ZuM8XQ6RmQabTompPif3U9b35gpRX/6U2gD9H34xyn%2BjNGWbCfaN/M%2B73rijnmgCHbibGQcYTIPPP7cEwn6owsbo8ssXYzU36%2B88JWyx5zeg6fnz/O1ql1Dh9PQ%2B2l4G3GqKP4CnP0Eis4MgGffHo6vM1VddIa7wEpYPPPy2abcpVocBBxYeAnjZPo29KS6vg7jtN7YCL93TrAf/LEsQdVNERhplpubP5XPxvAn95VaYblayYPlVsvmSk/OnPy/jTuACnfKNFUfSSKkMxIsuN0BhSaoN9nowpuIowfpKxiRJLpQnVEePWkQnnoFSfavN2RWQcK2ZDOTeGPmfkDLXpGxOL%2B7W4oOr2LN08bJn9iXrvqQ7R/nPABqqx%2Bfv54K8pCMIFINLBmKXNjkSUK3yXCXHlSOKky%2BljRXtmlRqjj9HY8shint0UW8YadPixXfnv%2BNEuRIPUBA4%2BJ9XR4VAwlTYO0nF%2B%2BVGQeP5Tc/5kxQj44e7QVr2A%2BB5ObWXCZ%2BWKkreAVIy3jJ8u227yOaLTS6X48svjyfOXkCZaqSzELUuOQAVLIiJIwf4v0IFLNiIRQIRGHyrT8LFMY8CSTlkqKJZFJZIm1uZgXCETa/7Jul6XoOD0HjyweGvoNCtPgQEQeiATyCMz7WlCYbelupIa%2B99G1tnwK0wIopkREPSOln6WUkfJp44X2PxiKyBKVWZnCQXq203PwyOLLwzy%2Bd54wwnQp1i9lSgz9Ps7EWOVslqsRW0KLpR9IUaXtYywy/49xhbTpMJUBRzvVUNGlSPd/9yNrrBJviMg7PQOPLB45PmcxTjLnLHYGDOQ2YX9fP5vYzxqMTC5mviHV84iWrFIDknlJzGXCY4xXgLx0Pkf6BYoxBiWfQVlm/R3mszDoM8mZdCWnZ3E8z1nsLIIEs04nbXyKGntUZOM5y3Og7FMMBuWXOUsM4zhFcLowj5aBnoIVQZZIT0WxJm311g0l5mFG/pyew/E8Z7EzsIJe2oaZc0uaNpFGihwhF0RQmJZAkbDfrdpl83SDassjCjGfYT5xkAscKIwhzJW6Zd1ueXBHuc2ndXoOx/Ocxc6ipqnV%2BnIcJdbvxx2DOFdIv0YvvGn1LrlL9SEMRabt0PMzVYEK6yOy0qxY4bS4LBFVTNHXmaN6/YvbrQBOdI6h0zPwOYtHjkcW4/S2yCKY92tQhlVlw1DEQExhMolCdJDoCJP8EwvLYFheMLrAOkXWzSFVj04QxZkCBqzTw/Vweh4eWXx58PzO0LZNRVvkgwqHzDNhMj%2BFb5ALig4whzcKKapURiRaQlVV5AsogET0hGVnVqgi7T1mz8Mjiy8PrfmycYMtDZtoYlgSAGV2Z22DFZug%2BnMUou5EWC4dG1uvEKWaMYbxgqI2FBqj6NjLFRdxko9HFg8PjEQcTfNULliTlvbNHEaqCuMkRB8K2SlRqBJN9dPowvfsUxLXvW5X49npmXhk8chxYzFObzQWnb6HG4uO0xY3Fh3nYNxYdJz2cWPxyPECN47jOI7jOI7jOE4b3Fh0HMdxHMdxHMdx2uDGouM4juM4juM4jtMGNxYdx3Ecx3Ecx3GcNrix2EdhmYCCjFQrmELp58LM1Pg7scVoKQNNxbzDgTUaJ%2BpnZg8eaCXWWfy8K2C5CNbJo9Ilx%2B44XUF%2Beoq1ZWQjtGWWHKB6MAs3Z2l7Pxz4zCSVC9orshSrrdr5DMtKsyU%2BpujxdtVvOH0b2hVVH%2Bl/6eMDtGvkgmVpqKx9OCA/yBGfQ0a6AtZHDcdLRWQqJDtOZ2N9vLb9WdrO6Idpd4wZyAVjCHrR4bY9PoPexWe6iuy0ASYXjBfIBZXwHedwcGOxj0IHxoK0/7h0ltx2yUy5bvqB9Zh%2BfvZUefDqefLJE8fGXzk0LDXw83OmylPXzpdPnzTW1uHqCkZrZ/rwK06U/1w5V66ZMCT%2BquN0LqzZt/hV8%2BVfl882gxHOHZUnd10xR559zXxt74fXvi8aky83njfNPvfJE8fIgC4amN91wkj579Vz5a8XneCDv9MlsPTMD8%2BaIo%2B%2B8kTr4wGH4yfmjZG7tX3/5tzp2t4L7PWXAyX62dcsMLnoqrXOUIh/pMe7SI/366dOkPmFyVmD0%2BlbMC787vzp8qSOF%2B%2BbOcqWJWP90k%2Br7vT0tQtML5qn%2BtHhgCw9oHrXL/QzXcWZw3NVjifL/TpefOO0CbZGt%2BMcDm4s9mHQK1Eu2aI65oD46zweDqozmOKQ2r%2B/PR7mx9olW41YFkz99umT5MsnT7AoZ4DvZe07tn78qON0AXSKtDHW2wrtDGU5Rds3bfxwW7h9Rje%2BC7k4Vt4wZah8V%2BUi6tgBvjvIseN0FbG2HOvjA6Ht4Qg53CbObrF%2BvP8xKyCTczPlvbNGyV8ummmRkigkBcTGsWMbkxynI6J9vP5nyhAPsfEi9t7htj3aaWeMFXweh/2nTxpnS0Sw3nCA8Swqxy4XzuHixqLThhte2iGfWbxBbt2Q/EVlRw9Kl3O1c3vFhCEWsRyUeqCJltY3yaf0uL7w9CZ5cldF/FXH6XqWlVbLl5/eKB96bK1srKyLv5pcrhg3RC4dO1gm5hysFN%2BzZY987qmN8u3nt0iLr5rrJAkWLr9tQ7F86ZlN8uNl2%2BT5kqr4O8ljSm6WXDuxUC4anS%2BD0g6k%2B7Fe8i9XFMknn9wgf1izS9ZXdI/MOn2PbdX1qjsVy0ceXyc/Xb5dNiR5vEgb0E8uHpMvV48fLAuG5hw0ZWdlWY38SuXis0%2BpXKzeJZWNLfF3HOfQuLHotOHfm0rlt6t2ymNFe%2BOvJA86tgmqDI9RozFDO72o76uisVl%2BvbJIblq90zo9x0kWGIgonTcs3yG7ahvjryYX5sYwLyZ1wMHd9jPFlfI7lddb1hWbAu84yYCW9tjOClM6/7mxpFsMMtLoSBXPTR8g0UyY8oZmuXvzHrlRx4v7t5XJjpqG%2BDuO07WU1DXJItWdfv7SDtOldtYkd7wgYsjcR1KxydSKsqWqXu7essf0u/%2BqXNQ2u7HoHB5uLPYwMlQRJLrGxHxSMHPaKQCQEyYp6z4UG2BSNR0E%2B/I6gyefZ24IE68psIGiyecOJ8UBY400hsQCN4O048GI4/v4XjaK4wzNTIulYCTAS6RVDFcFl3QhJlVzTGwcO58dgfIb/zCKMPMS89NTtWH2044uRabkZcq4eKdHgRu%2Bg8IIBbpPFOZgci049%2Bhv8LvR4j1AgQWuMefIezznWMI58RvtdbRO94FckGbGvaE9JDZjmhD3dYreb9oAbTIwJO6AoN2GtoF8sB/t%2BXDuM7IV2la0wA3yRJED2gvtKXw3x8H3djRPMSslJue0u6iscn58lmOmvSPbyA/v8RlSmwr1Pdo214H3OVfOjc9Ff41UKOQiFNmJtm/6A%2BSZfaKM1e/kmMboteYYoscX5ILvTPyc0z1QTIN2xz2iMFN78Dr70AaQI%2BCR/pY2w32lXYS2yz2n7z%2BcQmUjBqbJdN2fohzRYjXIBX0032/tR7cgu7kdHCfk6Xu0wegYxsax0/ai/T7yg/wyfvCPKCP7IQ/IBb%2BFnLBPYpERjEwbkyK/wbjC%2BINcRKEoCOfHNeI95DE6zvCbjD2Jv%2BF0H7Rt7hH6A313Yn9Ft8x4glzQ1yEP7EGb514iA9G2QdulvfC9L0cocEN7YdyhAGCA3w3fT/tmH9oPbT4zsl8U2jcyTJ/NvkFWeeQckYswJvHIcbI/qaa0SeZRcvxp%2BjdjEnLB9yAXjCcBxhvadrTP55HfQE9DpgJcTn6D3%2BIch6nMcRyhL%2BFzfA/nStqr0/vpt0%2BJP%2B/TbKtukAW3LrH0le5k7uBB8qE5o%2BQ1k4bK4t2V8rPlO%2BSuzaXxd2NcMW6wvGfmSDljeK6lO/x42XbZrcd91og8e32aDpoIfXpKP2ls2SflDU12ft98bot5gisbm60Dff2UofKxuWMsGkHU5DvPb7Xv/8%2BVc2TekGyLVHzsiXX2GlwwOl/eNHWYXDCqQIZmpUpTa6tsrqyXnbWNNoieMjTHPLlEOZ4rqbKOiOP4wKxRcu6o/NhgnxFTFEh/IEJz79Yy%2BdGybfq8Qb64YLwe0zCZqh0QnXurtswWPbY79fz53nV762TxtfOlprlFvrFks/1OgLz8104eKhePKdCON2bk8ht4u7lGP1y6zV4Drtv/zBghl40tsN9mQvrJQ7NNIUDJIYL5bHGV/OKlHXLP1j3xT/UM6LiXvGZBG4Wmq/jYE%2BvtGnU3VE/87Enj5PzRefJo0V55ywOrpKGlNf5ubJCkkMBPF05R5S7Foh3/99xme%2B9/54zWez3YBrBh2m6Bz%2B6ubZInVB5uXrdbHthevj8q97Zpw%2BXGc6fJVpWZ1/53hbygbZm06O%2BfMdkG2HP/9YI8pbIJKMgXqly8dfpwa0MoofXNrdbuXtxTLScVZtug%2Bq9NpXYtm2nUyinDcuTdJ4yQU4flmtKSobLKe9VNLbJdf/ePKo94gFEU3jx1uHxU5dQUfR13OU4iJ4/t3CufWbzR2v17Thgpu7QPOOMfz6lcxn6DNoIcf%2BPUCTpwH1Dmg1x8%2BZlN2s4rpUp/M/DXi2fatSal8Bk9x3drfxKOr7y%2BWZ7W/b//wjZZWlp10Oe6m1%2BcM03eq8eaDJZo33DybUvif3UvtPl7rpyrilw/%2BfDj6%2BVP2m4SoT1/6/RJ1m4uuXOZrCirkVmqCL56YqFcNX6IGV20Ffrr%2BuZ9lkZ3//Yy%2BaPK0Aul1fYd9Iu3Xzbb%2Btk/r90lH1i01l777hmTTDa2VtVbJOW2%2BNQF5OCV%2Bvr7tO9HyaXtVmibZfrARh0zPjZvjI1JH3t8nfxjY%2Bwz9PmvnVyo31coCwoHmdGXETdYi1VWn9c2d0c86wV%2BrTLK/F2bL6l/0%2B4ZU365InYc3zl9oizUMTFEUoj2BD44e5S8wcaaLMlJj/WlyB39wN903HtoR7m9BpfreEvxkZN1fLvmnuVynso74/M4HWfoMorrmmyM/uv64m7JxumIhSNyZdE1J8X/6nre/MBK%2Bcva3fG/upcvnzze%2Bk36068%2Bu0ke1Psa7a8w4H593nQrUMN76D6rymtMnt6uugHtHN0FBzuUad%2B3Wdv4fdvK5CvabwauURmibZyk/ez3lm6VP%2Bn5M3XmbdNGyLu0f0ff%2BrrqKowzwDj1PZUZ6jLgmKB/L1P9bPGuShmj%2BhHOFaY8XHjHUtsf2O%2BC0QXaF4%2Bw8QdnOqmmDSqrtPe1FbWm2yFbp6tu8/vzp5uuhfFH%2B0Tu9%2BhvnK5jA/rDu3WsYPzhvJnWszueKYP%2B80499yvGDzaDEqOX3yAqf9eWUvn9ql2yUq8RIM%2BMOx%2BaPVpK6hvt%2BBn/2HLTUqVRdUPk%2B5Ede%2B36M4WoJ8Fczm%2BcOjH%2Bl3M4vLzr0EkqKHEIJQYRAxkenURGaYeBYO9Vo%2BbGFUWq1NbboP%2BZk8bKadoJYGCtUKF%2BWpU98uXpGGebETraOqmXI0zYjjqWGTCpuni5Kt0DtTMkDXRVea15m1CI%2Bf7EKAoKw3dUSaFT4Tzo2DgmNtIfUFKu1I4ppggPkA167i/tqdHOqdEU551qQC5ThXu5vkaHxren6W%2BwBU8hRi/K0BfU0ERpoROls31md5V1UHR675wxUq5XIyJ43zhMFCNSXjFOGVT5PjrCddrxEsE5c3iOKiLDzTB3uh/kgntDuwhyQTsIMKhfrfefAf4lbTNL1KjBC/x/p02Ud%2BgASGQDxSG0v52qfGLEUbEUx4HaQoeE9kH6JwNwaHtEOVBGP6cDz1k6SKuNaO139d5aOw7mF%2BJ5TQQlmmp5l48bIjmpKbp/TFbX6OdQeKflZ5msIVcYdrxOqmm1ygyGKIMwRt5Sbec1ek4o7ana7pHZAArGW1RZun7h5P3f85LKLLJEv8F84B%2BcOVmunVR4UEQI2cf7fJbKBMoOrCivtutPcQSq6aGE4413up%2BaplZzzGEu4czjvkUhgoACjKywX522nwWq0P2PGlkYWsgRyiBtiQ1nKcbd6ycPkwtVQaUdHYoDY8UBuYC3ap%2BMImnjlBqJ9MnF2h/PVaX6jVOHxfc6AGMFhuL/6meQJZTclTq%2B0I8zziB3yBr9Nc4MlFWMTvqEWpUB9l%2Bucv9cSaXJB0dCH8%2BxIR/hyPj7J2dNkQ/Mih0bSjS/wbmz35Xa3yPPUbnQj%2Bjn%2Btv48ZWTJ9gYWKey%2BFxJtY2vRENfpfu/Vg3Iydo3Od0P9xQdhcg31a0Tnavc00m5GdLU0irrtH/dXlNvugJGJm2AiBwpm3wPbRedisgz8wBxNHDPO4LWZn2ythnkI7Q9Im30uXz/SJU7ZA2HIo4/9DLGqMTlZ/idV04cYtW0GUsq4v04bQ%2BnDs7PmPN7uOkxBAJwbuLoRNYxRGP7V0lDy75YgRs9eaKMPIZjY87vN3WsxOnJ2EUQgDEJ%2BRqckWKOlS%2BePM4c7AGuId8xd3C2VdbHGYvuxu%2BVqawT0UQnQ%2B/ECHZ6N24s9jAwEjdpJ4WXEwWYED/CGxiRlW7KMoYOgyL75qWlWuh/nCpwvPb9pVut2MV3XthqEbjv6iODG%2BXDJ2iHeDScNypPTlElgygi34sn61u6/WDZNrlva5kN3qHjATpnjhNFlY7zHt2HSA%2BfZfvi05vM04uyQidHp/PkrkrLo8doa9bOmQ6P%2BYnk/W%2Bpan/OCYrtq3Wgnjt4oHm3b7KIEue%2BRb767GbznnEdL9EBHgVgaCQlFcUBT93928otesb5cK3wjtbrIIJCTSqG0/3gXMCzW6Ttj9ThS3TQylcZCKDI4Q1GqcTBsFQ3nCQomMgMEQM8vKH94R2%2BX1%2BjbRLNZ7COGk2HA0o1bY9Bkt8kovGNuFwQtUMJPzAcHwCFEnnFeOWY2J9j%2BsaSLZYlgFOFaN6pw3IsuoJXmnZZpN9HuyTix37IBYpGe%2BA5xnhGISYKSIaCyaz2C9e/uF2eU2MT5ejaiUPlHJXtKHiUkQ36FvqRbz%2B/1WTplvW7zVg/URX%2BkwoHtUnvdpIPWRD/2bLHoiYz80mlPLi/ml%2BYI/O13%2Bb9/2wpM2WS6AJjAQrjH9bstkgx95j28RNtG89q%2B6L/PmtkrvWBR4KKkzlg2GgfODW%2BqLJAu2Ms%2BvemknbnSZHqifyiUKNA//ylovgYtsVk5PaNJbJHFVBSXlFAcRL%2BZ%2BseixriREGZ/52OFbTzxR0UP8MgRV7JPiEV9hEdf2jX39bf4Nx/r59nbvIc7fMxdhMNAs4NRy1RR2T227p9XWX2cZVP%2BhFkAmet0/28oH0e7ShV7wvOkmgKJe2SiOBIHRfQZzAKkQXGACLuONNob7QJ2h/6xE9VLjAax2dn2hIxwfF8uNCWGGcwFDP0s39Zt1u%2BYm0vNh4RscSpgh4UhbROItpkZGHwUUjK%2BmQ9JiKZf1T55ROnDMuVabovY8RNa3ZZlBQ5Qye6Y3Op/ETHC76/PXCqnqYG59k6VuLwIQrPeMNx8UjBQ5yYZK4xnnLdouSq7DbquERmARHab6qex2eeUDlEd0Wuh0Z0WKd34sZiD4QUOcL6KIYM7CijAQZvvMXsg9KI8sbEfjqHv60vlt%2BroJNWhyLJBH/Scf66rtgmXWNYJXrYXg46L/Lp6ejwttFh/WZlkfxTB31S6/6snRVponjJQhof0IFx/BwDqVFsHAt/s5Fmx/Hr4csQ7bzpIxmoMRR3aofH4L%2B3oUkeK6qwjp9U2kRQGLg2VPyiM3tox177Hc79Lv2NW/Q6cC1IGxulndVrVFGIzjmgciTKNp3pX3U/zucfeox0lkU1jeblG5zhnVxPgcEfRRZjBoWPyCDQpqkQSvoyHs11FXUWicbxgkOCe4siyGNofzx/UpU87j%2BK4xyVq6xI5d3Dgcj4QlWokcHbN5Za20PJoB39RtsQSiQpOongrb1LldybdB%2BOi/05JoqEsG2qrJd9%2Bg%2BDcXBmikUvMAg4Vtr5RlUAcKqgvHRUoIDoCxuG529WUvwm1iew8fzX1sYbzJA4PUHBxVDEs4yMcp1I/fu79i0UCiHVNV2v/xTtlwYnzBt2kg/G30Pby%2B1eMjUgMRJINASnwI7qBnlY98NoRDYW7dyrSqW2P20HpIHSh8faR4kpecgO7Q9n2pGAyxDZnJE30ByXtJ8/x/tk2hCOOJxziRAxJ4J%2B6wYdw1bv2j%2BG0Y/TJ9P2GB8Yg1D%2B6QNo/8tKa2yqRavKC7KOc5AoeHswB/d1k4fJyIFpFn3h2EKfwG/9SccyHEiNKmM4MHEURh1IzNhBdrlWt%2Bi5MG5QDZa0eFJRCzPTbI6c0/3Q79EOyC5CJkjLDHNKccBdM3GIOShwZiwvq7H%2BliyNf2s//gftk2mn9MW0P9oGRhmRNozO0SoX9JFHQnD%2BYTyt1XZOm472rRTuQ/eh7UVhbGHco23iCGdfjoe29/f1JdZHVzY1W9tmPGSMwLnI%2BWPA8TdR80MVs0G3w6lPyvdzqivxW7fpuYcxid9FV0tXg5r9TtIxIwp6Htk0yC19CeMZ53bPljLT65C3IzWunZ6HG4s9EISaDgEFAGMRYyhAdI9oAR4jFEiMxu14k1RQP/HkevPoZA6ITcJnoKdIznhVpOmCDu6GDg8ihqfqb47QDpeOh1RRHoNdyLESBV1dXmvGVwClhI6YY8KzTIdFVI8JzygvpFSgpGMUHi0YfqRdMaATeWSeVaKiQMeJB5oUWSasRyebk4KCAkLHSsoG0PFhENMB85lET5/TfWD0o5gxgKLIESlg0KaYAF5h5iouI41Z2yOgFH/h6Y3y4cfXWRonaaPIBe2PqCAVFFEyjxbmpqAg4ohBwUY5jvKUvobXOhFk%2B%2BNPrDfPLYZuKLJEFgHzbVFcjv6oYuCAwTm0fm%2BdKT7ROSPI78362lp9D3ngHBJh8EehieouyEbwTnPcRHGd7oV7wpwl7hfRLdoRiiOQfTJKFTXuEu%2BzH%2BMFUXYyO1iCiOg3bYB%2BmfbHo93yY2iAjDko4syNRNmOwjjB/D6cHtFyCfTBRMvf9%2BhaM8Y4L5R6ZJXxjmUxjlUmsrTN4lykXyeS8%2BiOg%2BcXMha8UFJtYyvF1Ui5RtENIAso02GOF3AKO6rrpZrxQoeKaATL6V7oj0mJRM84b1S%2BpX4SOSPKRzvAmMLBjN5AH46j4qPaL//0xR1m0NH%2B0KEwNGnPx6IJ4HA/sXCQZUwxt5AaEwHGM/QUnCVVjQdH/xi3frB0m3zmqQ1mhKGPMN4hF2RlcUztZa8cCThfpuVl6jVoNJ0SvZNrA8gp7Z1oOjI6UcdN5iVGKdMxgawfHLUhYICznfMBW0/V1ahej/dsPRAMF1JktqoBhEGEIAeo%2BEZnh3AyeRphjsJ8QrxmzBf56ikT5JfnTLWJ3KTvYPgdKXSupPuhlNMZtFcAiFSomLHY8XBORJSJ48yrZII3iygzZwaFvzNgoKezSwTPOwNBVWNsXovTu8EJwXxDFMg5BaRCptncCOZroawRXcChkQhOl1dNGmLFbr6v7e8PF8ywif5UCj1aUM7ZmIuVKIewSOWTKGFHYHBRHAc5oLjTL86ZZnNrSWU70gyA9iDNtai24yUDiNbXtfScIjXO0UG/trq8xvphFLm3TBtmKZMUsCG6vDc%2B7zWRgSn9VYHNtjlR9Ms/O3uqtr%2BpNsf3WPplUvqYH0XaKFsUIpak/pXohuLcHkRhmPbwzhNGyA/PnCy/0fHr8/PH29ysY4VfpFgJii/XKxHGC1Ldnd4PGSZkJqG7UBsBA4t%2BlUjjhOxMu8/0kYmgX5H6yVz2r586UX6qMvH3i2fa30cLx4D%2B1aLjBBHxPdoGEylVPQVHXnsQ1cNwpfjTx%2BeNUR1qsvzu/OnyNz2u6DSlY4HoOI51ZDQKcrq0pNrmPzp9FzcWeyh4aEhLwECjY2PwZK4QkQcGYDzF0bRPCtz8URXg%2B6%2BaJ5%2BdP07eNn24FShAqW7VDgolGw/WsdLeN5A6UdF0INoYQOH49mmTrFof1SXfP2uUTYSmI85Wo7ZBjWLmUnYGh4rG4ME%2B9liN0xPYVFlnFWppykQTMfaILBMRwTBjIGbQA9ofxuFtl86U36rCSTVVir6cMSLXSn1j4LWnLBw57bctnBftff/5o/JNCb7zitlWgOeDKhcUUiJlMH3AAFNmO0dWD93q7b1j/xmnm%2BEWkjpGaiWpaCwHQLSB1GocfcxPjVapZBoDCue/LpstvzhnqhV0efWkoZaOhjKNwnosGR8xYm2vvW9h3KpvbTUZDiDDpK8yhv3pwhlWGI3qtlSrJFqKkk0hm86AUzvU6XmB%2BOMDonekcOJYZ64p01Zo%2BzjoMIBIsySKDGRJEEHEYXLbpbOsqvb7Zo20Ktp8Bqc50e6jJRpYs3bfThtrUJlIdDpiuFKB/u%2BXzDKD9QsqqxSbYeoAkU8yoKJ64LHS0Xdx5i4WfRs3FnswT%2B%2BuEiq%2BMVjiUaKqKPnypFo%2BX1wV3ytWIhsjjEIVDLrMd2SyMRO02X62fLtFBDvy5B4KOo/qxpihiSKRWK0Lwlo%2B0VQDjpliAkQ56WxJZSCt7ZvPx46J7eGi8v3pn8fK8Kz0g%2BaXBKhsSodLzjzedqd3Q9rcIzvKLV2HVKILx%2BTbXBBS6V4qq7Y5gpTtJgWTCr0Uqjh9WK45NCiyZAUznt0s33hus/x3W3mbyMeRQNo18kG7a6%2BaKsUSEr2%2BGLUoxTh3iKCQqseyMMgDBUB%2BrrKKcsM8rGOFdbsONa%2BQa%2BSppMcHyMWW6gYzzoii40ShUAvNktdJLQuQkkflTpwT26vrrV%2BmKNrXVC4YK0j1PhYnSrPKH6lnKOdsiZBKR7uMpviTQYNxSIYAURjGsF%2BuKJIvP7PZZIMUwY7mIh4J/CIFzzCq25tHxXjRWdkuTvdCKiV6D1VRSQ9maS%2BMP4o72VxfHUeCXJBuzDJBOO1wJBKVJFWf4kroLBRoejiynMqRgu7FGITROTGXtNa2coGsJGaUXDS6wCoAU8GYz5NCToEyxi%2BO60eHKFxzpFCoBkd%2BYio12TOTcmKpuE7fxTWFHsyy0ipLL2Iy/9lqEF6iinHGgH6mYDIpO0DlQyqxUaaZ9d9%2Bq8rnDct3WHVGJkSzVhae5qO52Xi6SGGiQ2J9H%2BYaYgjS6QHKJrnzNhckYo3xGh48vHXMHWBCNgYsx8Vkbjo91rmLKgxHCgYDCgTRSX4HJSmxQiPVAZnzybHtrGmUBj0fp/fCEhAM5BR94V4TpWNtqdg83z37jb88bauUIsdgJCXugW1l8kttf1RLRAlFLohSRtvskYKnuaqpWQoz0izNOlo8CVDKE5fOQIlnviXzyjgPikVxTMgG846ZQ8MxHcNhGSj7RIeYr3zKsOyDlBCUZBQnlr8wZ1AnRWyc7oV1aJk/zpxBHHW0NebvJhpZ9NUYiqRi3rhy5/5%2BGacFc5bo84%2Bll0RBZw1e2j7z/qIw9x1lnXlkUdnD4XJRvMokUzAYuzgmjFcqYjMGHY2zMwrGA2nZGKMo31QkjoJMMpeZ8Q3Zpo9hqQKn91Ki4wFzYClec/qIHKvSO1zbIDJB/0vGFXDvrxg3xPSWZ0uqrC%2Bm/gPtj/WcKbLH2HO0kGJK0TD0HaKCZIgFRx26FG0OHQbHdhTa6Uk6ttBHU/TpVzp23aDHxCPnRfT0WOUC/QxdjCkdGNNci6CXIaPIJtk4RDJL9ffam4fvHP%2B4sdiDoXBNMNRQANhQhpl8HfUUkwuPJ5cBjsXt6UDodChYQLVDW9dnYNpRRREoiEDBF46DQZY5MXjfUATo3FDIqZBFumt0nUU6GSIbaL3MjSKySHQyFLe5RBUDDNxEzzO/R7SRqnqcF%2Bsk0Xm1VzgAZQcvOCX%2B2YfKlCjoHFf4Hda043iDJ5G0Xqd3w8C/qKhCDZ2Y4kslVGSFwZT5qUBlN8qNkzuDIcn8FOZihIIZKKyk3R3LfA/STFm3kEgFqUEXaNtD5mh/KOOUIk9cjzBfj4ljI/ppBaNUgQiyyrkgX8xRbm9%2BMfKDfo0HGOWCCEhHzpaNqgiRYsX3snYdMspxsVHQ4H/njDL5wLhAaXJ6P6wtSgEW%2BsL3zhplKagUyKD/jkKbQUENa2/SgpAD2gNzHHHyEXU%2BGjAzn9VjQAHFKcI8Lyrn0u4wxC7Vfp/F0mNTAw5AM6aPp43TbpFj5ILP4QRkviIylQjODpRlnKEU/kAu2sswgQqVtwe2ldvSCET3WRuRc6b4CL9DyquV%2BVelGIX4Ue1jouOs0/tA77hlXbHdx8na75OiTUozBWWiU2CQB9Zopn/dUlWn7bfeXmO8YDmXN00Zbu3jaN0oOB4obMNvsswFThRb7ky/nzUIr1JDlWUyEiOLyCp6VbW2WdYRJe0UuWYMYymLD88Z3aa9c4ToUSSnMI7QHyAXHY0VVHmlyA9BCaY7YcxyzsgF1X2ZQ3/uyHwb517U8Y55oE7fw43FHg5Vpei48MSykaKGkRQFRRmDjLS2d5wwwooU/PisKVYYgLlanzhxrBpvpM0dXUcHTHxGsWBR5a%2BdMkFu0N%2BgIAdzTD45b4wt%2BhwFTxieNAZzUos%2BP3%2Bc/PTsqfKThVPkLxedID/S46PDpCJplI0V9bYYM4YpxXyYW/MmVS4YzNuD36G6JAoAi6B/67SJcoP%2BDnPCbr5opilNVGHFk8gaRWGOgtN7oVgRJcdxnKBQ4iihGFS0dVeqYkhbqtP2xSL3H5s7Rn6l7ZUFuX8fLwxw1YTB1s6OFpRw1ufECCXC%2BRWVC37j5%2BdMlQeunmcDb%2BLgT9VeFGEGeBY4Rj4pavNL/dyfLzzB2i1p3e0dFjKOHGOUfuPUiWagdpQy98iOvbb%2BKWl1r5k81PqDG/S4YkVMptgSAqQVEeF/cFtZ/FNOb2ZFeY2tL4ojgmIYRNKoQs2ySlGQH4wyMlJuv3SWtQmKeNBf/vq8aaZYHq1c4MAhKoMTDwPusnEF8k/mgOlvMFbQblE%2BcWxG5wYSzSQimaPy/Ekdr/5w4Qxts5NNLv5z5Rx545Shkt%2BOEYgjCAcgzskvzh9vYxPjTXuQqktlSYqxoUC/Uw3Zv2o/gMwxZnzrtEm2gDvXhyUA6EOc3g8aBv0uznLGC6KNd2zec1BGBbISc2iLfHDWaKuxQLugDTK3F6MMx8LRTg8o1u%2BmOvxiNcwoLPXxeWPtN5gbSaGa754xyZwhpHBHIVDANCAciX%2B64AQbvzguCrQx1uDYSdTt0LnQvajQO1mNveumD5f/U7nraKwgw4axjDn/OBfR0YIO9TPtFzhOphLhAGK/9RVti2U5xz9uLPZwHt%2B510om4yliu3VDiTytnU4U1rP589rd5rlK6ddfFhRmm4cUY4sBmHXVSG8jkkFaQ0eD6aFA%2Bfzp8u22rg%2BpG7Y%2B2/AcU2xJc6VEeii3DMy1/O2qIrl1fbFVvyPCcpYah2cMy9UOO9Xmnd2pn1mhA3dOaorNLWM%2BF54zqlmi4NC5kS43f%2Bgge689qGjHWnXMa7lXDVq8Z3ih8QKOy063Tpp1E9%2B3aI150o%2Buq3d6EhiAKMVEu4kYEwWwojdoqnGo3PaYyg4LGK/YE29LqqTSNpj3u1UHPtYfvXPLHjOoeJ12eCTUNrXach5vfXCVrc%2BWrqM21SVJe2Z%2BCUpzYlRnnR4zxUaYJ8bacqSvstAxXmXGe9ZmZIF0FFsiMVNVhgPIGanURNuR71MT0kujbK2ulz%2BorH70iQ2ysqxWhmelyql6XCjqRGC4dp96coPJxhbd1%2Bn90H5wJL6k7R2IAvB3RI80WA%2BUNLbS%2BkaLUOC0m184yBRpooJvfWCVRZtJ8ybKdqSguMbmzG/WfrxWClX2iFYwHm2viS0SjsMEgzWAwvqBRWvl6d0VFg0h6n/m8DxTUmtVTq5fvsMW3ceG5XtCpgkKLnPJSAdnjjByzDm1B79G9V%2BW0vnVyiIbl5hrj1OHzyG/rOf4f3rcpId79cfjAxwRd28plW06ThBdpu9jig/TEwK0d5b5oi3hgMAwpI9lDh/Rxr9vKLY1dGlnVFMlEt9e9kdH8EukiX9m8QZL96a949Ch3dHP4%2BghCwCDLAop2OxPZDIvfYDtz3JhLImzZm%2BNjW/IWJYeC/06y0M16Xmx3ihTlXCiMBWDz3R0vBwb6zB%2B/umNNibiLJmp4xFywfiEYwfnyZee2WRO2gNXzelL9Nvnpb8MhHTBrUvMuOppMPhdMrbAnrNGVWKHApbGpgI%2BUjsgyizTwZHOyfpqeMxCERrWxGHOIxFKFjFHuaUJ0OEExZbUTfLXSXfFQxzAG0vHRtoEBh8DNx0LRUWsgykYaBVcUU65jnROpPsxb4pICoYcigQevW1VsYX3WQ%2BOYgd0lig5GH8YhnSKw1TBJRWEZTFI98PL9vopQ63zJ3WC8wjQgZM6QVpHmIiN8UrEkfXkSLMI0KnOLMjSASHLlHIWZ4%2BuQwfXTCy086Xq7GORa9DdkJK15DULOjQSOpuPPbHeomc9EeafjM%2BOLcS/RO8vUeNoZ4YnmTZP2uWQDNKw%2B5nXFeUTxwnzuViInwXESd2mrROtREElUohTBCMUxZZ2j1FHeh4LcaNoAu0TT/Fp2l5pgxiewGeJxKMQMIDThjFe0Y%2BJdNNWWc6GVG1kFUcQzgzaIcYvRXuANh7a7ol6Htx/DF9kh7aLccl3oRCQnvu39cV2jkB0CGWfOZ0U9EAZBrzjyMVi1vvS88CwDVw%2BbrAdM5F4FI4ovI7yhHwyh4cITaLcdBdkOVAgJRngIDj5tiXxv3oW3CP69HGDMqwd0afTTqLQL5NeRnVtDERAwURJpl0z7/3ycQXa1lOsDdNOaKPM6YopqbXmPKQ1LYxXJEaJXran2hTiANkgpD/TjyKLpP8hZxhhZMEA65PiHESppU9jKsHIrHRL1aYdo9AjSyj3jGE4QhnXMDiRYY6f9EJ%2Bh99AsX2xVI%2BjotbS9DheDErOI3odYmvLZdnng%2BHJGIaThXEvOsYytjKOYSA8X1JtdQQYR4HrgkMHpZo0PtY15Zx6AhS%2BW3TNSfG/up43P7DyoKq7PQX6aMYBHOXUXaAvTuzbAP2EaSzWj%2Bu4QGScSB/6CsYkS8Iw5YH7jKMbHYdUzZNUjnCs0E%2BjL5BlRboradg7ahpNVwnpzDHjbaCNJ8gFWTHM77WoprZ3CgjWaPvGMA3Qv9P30%2B/zeeQChym/j%2BzgaESPQs/ht0h1RVYYQ9AdOV7k4m9qCDIO4FQZoTJGWyelFLkHdDRSY5nuRMop1wOHDuMKzn90tDCNh%2BOYqn3IPD13xhn6BH47CjJ%2BteqSXD%2BmAPWkrC4y3chycA4fNxbj9GRj0XECbiw6TlvcWHScg3Fj0XHax43FI%2Bfw4%2BiO4ziO4ziO4zhOn8GNRcdxHMdxHMdxHKcNbiw6juM4juM4juM4bXBj0XEcx3Ecx3Ecx2mDG4uO4ziO4ziO4zhOG9xYdBzHcRzHcRzHcdrgxqLjOI7jOI7jOI7TBjcWHcdxHMdxHMdxnDb4ovxxumtR/rz0FPnArFFSkJEqA/r1s9eaWltlQ0Wd3LVlj2zX4zpcJuRkyMVjCmSwftct64olK6W/nDYsR38jVX7%2B0g6pbW6J73lo3jh1mBTqd7xUViMPbi%2BPv5pcMvXYTxySLeeNypOndld2%2BXFcPWGI5KelyB/W7LK/3zx1uLTKPnmhpEpWldfaa4GTCrPlFeOHyC3ri2VzVZ3UNbfG3%2Bl6%2Bsqi/OfqfT9rRJ4M0XYYqGlqkaeLK%2BWOTaXxVw6PeUMGyUL9rrKGJrl1fYmMGpQur51UKOtUxh7fWXFYMj8lL1MuGK2ypbL0hzU7j0guO5tTVaavGDdYHtpRLs9r%2B6xsPDy5PlKGZaXJO2aMkGV7qlUOqiVb29zCkbn23r/1HpTUNVn/8vopw%2Bw1aGxplcUqr7dvLIm/khz6wqL8%2BTpWXDA63/rFgRH5ZwD/wQtbZXvNkbXJsdkZ8tG5Y%2BRh7VufUbkap39fqN9Pf/eQvra3sTm%2BZ/swWtEXnj0yT8fPerltQ3LveRTGTtpquo4bz%2Bn9eXJXRfydzuccPV/6g921TXL3llK7BtyTLVX18t9tZXYsr5w4RGYXDLLxvVVVrCrtu25YviOp%2BkVfWZR/WGaa9kuMF7G%2BKbCuolYeK6qQF7X/OhKumVgo41UW1uytlQdUDt6o/Ruyt0T72seK9sb3OjRTcjPlPTNHWR/9rMoWfWV3MHfwIDlTr0tmygD57aoi2dtwaJk%2BWrg%2BQQ6QCfoQdJW3TR8u31%2B6Tcrq257/tToGT8rJtHE42eOFL8p/5HhksZtA2Z%2BtgkxHdOnYApmgnVOBChzbmEEZ8j868NFpTVRhOlyGZ6XLK9Toecu04TJyYJp99ko1at44ZahkDIgZoofDRaoUv2byUJmvwt9dpPXvr9dnoLxVz6Wrj2OsXu9L1Mg%2Bd1R%2B/BWxwf6UoTmSqscRhWPh3rCN089xnE7ncr7eh9dOGipnq4EXZIINheBNU4fJJSovwbFyOEzJzZJX68B0kd7jlP79ZKQaQW%2BaOtyUXJS5w2HMwAx5pcrWW3XwK1TlpDs5IX%2BgKcYMzFmqBHQVGOrvnzXK5IPuY5IqQK/R%2B3KyysVA/V2Ug1fr38gODiruETJEn8bj4d8h5%2BVgTLhK%2B/K3aX%2BIUzDIBErttdpX4eDjfmDQHy7DtR3/7%2BzRpkxy/2YWxPpbjKHDcUYhgtPys%2BS1OlZE%2B87uQMXaHH7X6pg5S8%2BjKzljeK5cMW6ITNdz768XASPlqvGDZY6O51w3nDlvnz5C78cgu0eMw29QmbhG79OYQenxb3GOFcYAdAT0pEu1D8LwCHJBG0Be3qC6D3KRMeDwx2nGBdr06cMZ//vJ5eMG298YXocL8koQIDjruwv67Ffp2Md1OJK%2B4UhhrECPRQZHDky3/uQ0vX6MBYl9CWMw1/JN%2Bh56FP2N0/NxTbeboGN7qyqsn5s/Th4t2iufeHKDvPXBVbZ9/In1ktKvv7znhJEWQYgKG95glEW2GTpYTdDvIQp3uOABmqwdSPgONgY9BDyRnLSUg/YboUp2eqTT5fko/Vx0n/GqyGSnxY6XToHjRbmZmpdl70/RR84HJR1PdvSzbAymRFnpbIbr76XpbwxVpYYBl8%2BggBZmprY5Bz5Dx84AwnFzTnSU/C4ertH6vSgU7UGnxncSeUIB4noy4OAtxyvJ93Ks0/V73j1zpLzrhBHxTzqdCdcZ5fczJ401R8q/NpXslwk2vNYTsjPla6dMsDYV2iLtifYRbQ/tteeOoM1wvxPbFBvRtURFY7R%2BN22K93lELjj2AG2WtkcbZB/klPMKe9A%2Baef8Hp8P7%2Bfq6/wWf0ePgXbH/nwvMoEjCNni2GjXnD/XAjmZEfkcxxCNzOKB5zfDteK3OzKW%2BX6%2Bl88TqSHzgmgWx040s2WfmGFBBJho/NseXGn36LYNxfa9nzpxrCnSzrFDu7h4TL58ZO4Y7b9T5SvPbt4vEx9%2BfJ3cuXmPfEiNPhwI9HcB7h3jTGgPbLSBaB/%2BcjBeRD/PRtSEthcFRZQ2HPahndGHRynQY0%2BUU9osDjfaCg4IPsfGbyA/fGeatkX2C2NI2PibfhuZoa3TPtmGxuUiQF8Q/RzHEMZUrgV9SXSM4prhhImI9EEw1pD9Q%2BSJK8nnkAfGD/qC9%2BgYQTbDX9cV2z363NMbZXNlvXxs3lhztDjHDn02coFRTkZBo96Pjz%2B5fr9cfOXZTVJU02DOrPfPGrm/n6Ofph1G2wPtjfZ1uNBWTR%2BI9NP0u7TZ0K4MbT%2B0o/HZsfZHv85vRccKniOnYTxhM/nS4wkOUY6X19D1%2BA5%2BN7R7HHnR42Dju5AZ5G%2Bo7odcsS9OU/pvIOuMYwljDd8RlRmuF9/N78bGqIH2d0cG5yD9XvYpqWuUmuYW%2B%2B55Q7JlZ22jtLQenLzIsbx9xnCTESKeTu/A01DjJDsN9dOqTBGleGZ3pRmK5TrQBJlCmE4dlivfO2OSdXi/WLFD7ttaZu/9%2B/LZlioEDTpCrSirkY8%2Bsc5So04fnitfXDDOOpV3PbzaOhk8N2O1E7jwjqU6mDWroTNSPjh7lAlzoLa5Vf6xoUTe%2B%2Bga%2B/v358%2BwNJsU7azw3AV%2Bu2qnpdIwSAKe6I%2BrAvO6KUPtb3haz%2BcnL263NLUROkjfeM407UgGmLLLQLqzplE%2BsGitDeivmzzU0kyj3L1ljzyyY691dB%2BfN0ZyUlOkWZso53fDS9st1Ylr92r97CQd4AOfenKjGRcNLa36nflyw9lTzdjL1U4vXZWRR3fulfc8ssZSGRPhepBiSNrt/y3ZLHO1k/v2aRPlHxtL5GfLt1sn%2Bis9DzyUpPIiMSgZ73p4jSzeXSEVL5Ou1ZnQcR%2BvaagMwDeeO83SRH%2B0dJv8UY2QpshAw8BKm/m/UyfKs2qwfEGVsNXltZZ6jQFJewzj8M1ri%2BVdj6y250TDuMebqurl/Y%2BulRN1kCJtkRQh0rNL65t0n0L53zmjTTmN8jNt739eu0sKM9LkU2rEIlukWtI%2Bc7Vdl9U3azsplh8v2y6b9fuBAfaWi2eZ44SIPnL6nB7v5Xcts/N51cRCec3kQotEZGv75ntItf69yhdt8PFXnXSQQlHV2CKP76qQzz61Qd4/c5RF%2B1K1/fFdi4r2yveXbrUUOKKuH54zRpWM2Oe4Njev2633b7ulwpF2Q3SjWmWAa8X3fnrxBrvOiaCsXKTX9YdnTpLL7nrRjp/v/4h%2B/6f0M016Th%2BdN1oqte0jV43x%2B0Sa5Lu1j8EpdLL2py1JGl6O5zRUohrv0n48V/vzc/71vPVxQSzQJ%2BmLfnPudJmlfTX9%2BNe1D6P5cK%2BIOKIQAp%2Bhb/6LtmfSxMicWHztfGs/tAEiYvSt9%2BpY8714WivjxY/PmmyfD2yparC%2BFvkjBRlDlWkLG6vq5Ewdg2CTGke/XrlTf%2B9A3/E%2BbbfX6ZgXjfyRovaH1btMyTxR%2B9efa78NKOSDM1LkxT011r74nEX19XcCxXVNcuPKIvmPGst/vugEM/LStfHvUZm8b1uZvOWBlbbfr7RPebNeB6A5riivkS8/s8nOEzn4Wfw3J6q8orivLKuV/31snaVftzd142cLp5pDFLlaqd/1%2B/Onm9H4Tx0vNuh5//nCE/Ta1cu3n99q6bwYq/w%2Bxj7y9qd25K0rOJ7TUDEUMbwZL57Ytdd0E/rCIBcYWuhJtDlS50%2B77TlLd8S5TfYVulVgo94znFxffXaz/f2js6ZYRPCB7WV2D9GH0F84N/QBxt5bLp6pv5%2B9fxzmd9Eh0cXQ1ciOufequfJIUbm1S5x89Nc7VKau0P40jBXoFsg3skp7B/b5%2B/pia197dGziHD6iYxPtnfaKo4Lx5q4tpTruTbTINsZfgFTTs25/Xq6ZUGgyg7z103/0G2/Se0TK52kqp7/Ra8fvM9YwlYYU6jfeH5MZHIFcJwzAHG3rXG%2BuBzL9H9XREjlFr9f3z5gsa1QeGFOHqJFK9g4633VqvGM0Bhg7n9J%2BB8dOqZ4T4/BHHl8Xfzc5eBrqkTPgK0r8eZ%2BGOQUMojx2tX6DZ5P00Py0VBM%2BBu7m0MspzJVjwKMTvF8HmzXldWb00EHhWcaT/w1VCBAy0ltIP8WriXeTkD773Lm51Lw2DMAIOsrAG3TAQnHerQPzB9Vg%2B7N2fk/uqjRP6RxVNBDeF0qrLMWGdMt67VzoIL/53BbtfPvba%2BxLJBRP1OdOGidjstOt8/k/3WdJcbUZuWP1NQzQXdpB4Nk7QTurp/V3frB0ux07nSZKCIrOTfr3d57fYscyNjvmTcb4%2BpP%2BzTXBgLhFFd6vL9lixt97teMk1Yi5BCgr92jHjFF6xvAcVfqbZZ0aspNysywfvkaPge/l%2B%2B/XjpA5JuEqL9COns6COTsXjy2IefYKsuSV%2Bt18/wx9jjKfrx34cyXVpjj/VBUtvPjA/jxHoaITThYoM0Q3iTIlg4d1sCOyhMHe1eAF/aAqnuuZK6Jte%2B3euvg7MRARBp37t5XbwIbSSgrSu/R64MHFeKedY7jQ7jHuue/TdcBjMGMOFs4IBjDkD%2BNxQ0W9pdJ9%2BeTxNifl58uL5Hq9z48UVcipOqBiPGJ00ZaIoqF4M5j/WpXUv60r1qPqJ1eqvFSrUomjaZa2mS%2BdPMGUki%2BqQkqboY2coQP6mSPy5MXSaosM8hzD8wk1Ar%2Bt7ZMoBNGaT5w4Rgf7Fvnys5vMMbNKr8U43Q8Dd2mJGpK6P04fBvE/rN5pyjKt71qVs8vHDpY7VO5RwDkXBuzTVR45b64b6VWk0GEcv%2BOh1WpIFtt%2BUQfK61Vx%2BYQaDBgApPsiWygWDPykY9F3YVRwvijk/9pYKiWR%2BSikxqJEoazdpEZA17eaGNxPZDoZ0H/SLzbrySXj/Ejznav3H4cAc9ETf5N%2BEicDfSFOBzIj3qn9Kwrf89p3oVhiyCAjpJziLNiqii1RBvZjfh%2BGEYojiud6Vap57To1UHEuLNU2%2B2E1nuhLa3WQmZibYc4EjGYyRxgXMIgwEHHGsN9k7R9RuDFkme/6IVV2STtbr/L2UVUMaXuMU8glbYU5k6MHZZgjJUUNPubrf/mZzXZOGFq03ad0DMHY4rOMLcgmsoRCjbEwW69RucrG31XxR6lnHPmdKvr01f9U4/ZLT29SJbzE0hMn5mbaGFXR2GK/SR9w75Yy%2BeEyNV61D8Ehyvtca/rcS8YMlhvPmy5v12uCQTw1L9McnZwTRjdywbUgivIjNZDv1rGBfoOxgevKdWYcvmfrnjZz4LsKxk6ON1nQt67Uc0uGg4g%2B/FWThphc0N7QB9BXAhzBHtWJntldpW2/VNZW1MpI1ZPerDKBwxGn9pfVOGQc4b5wT8mmwKAkpZp7trEyNqf9lWp0sc/yPTXWpr6megPROIyi76gxeZ%2BOR0TTTtK%2BD8c0uhhtGvlhjvv12o9/S/UjZIy584xNW1SXIeLHsbxjxkjT5X7y4g47j0yVmUu0L6/QtryLMUX1J/oAvhNj9DeqM9KuaLfnjc5T2Vwvv1yBU3O3jhd1crW2b/ptjn2rjpFETZF5dD4M6gtHF5jegxMfZ/CNK4q0P29Vec1VWc7RPqTCxhfGiqFZqfLwjr1mSHMdV6tcRGs0fFZ1QMbO102OOaUY085XI/1yHROJHDIOkiVE38C4yfd%2BfsE4G5MG6PXmnDCccdwkE/oTjtM5fJKjcfYCuBB4iYimdTUYUyiG5Y1NVrAjGj0B/qxqarZBmsghgz9eHpSh%2B7btMePpMe0IntCNfZgbMT3v5edp4AX9kxrEv9TOgc%2BzMSjzOoYlERGEF4iGMKjRARHpu3HlDpswvkB/61WqQNCBYtjh%2BcWwe1T3IZq4em%2BNDZzk%2Bwc4v2W6Hx6pxaqEoIz8Vc%2BBTuq29cX7j4WOiF%2BnE92hygyddZ12wnQmGLF0eBepgPOIosJnMKZ/uHSrKUykap0X6QAwHBkUUKA4l%2BggVqyGI0VzUKJq9fsYCFAAH9COP1PbwVr9LMowgw0DBMoIHRrn212T1bk3R5JC1hkQ5c6IeC27EgYPvI4YNh1NxOfao1giF3j9Z%2BtAyoDEIPjYTrYK%2BZ0aKeyDgc%2BAGjy27YGCQTEpDLPf6iCMt5bvoN1s1vaXp0oCKTYBZPNWbbM4SP6j7YEIC99BVH%2BBKo0YSxzTraqwPri9zL4LJRT5oO2iiKBYcy8x0jA6UTZIdyZ6R%2BT8e9qekRU%2Bi4xzrqQD4jBi8F%2BnCgHRTTzitGvOn4GZARhjmTZMu%2Baa4ADCcz1YPwsowBgdKBLI4m691lHW6rV4QNs98tBPj/FpVRxwPG3U15ETFGCMQD6P8o5yFYXoDHM6t6oydHCv1nUgEyh6yYKoF%2BlYIYLb1WDUIRsY%2BR1Bf0V6MPcWxRbDnuuP4nevGigoxTeocouih6KEkfZyLNOx5U9rdsvPVTZoi2wYkWSH4ETjd8J4QZolBlLYb4nKH/cEJZxdmH/MdeMYY/vstUwR2h/Hw1ynAOeA/JkRq/3%2BIt0Xp%2BrvV%2B%2B0z9KuiZizXyzCj0OvyvoM%2BmqOhSgMjiQMVr4LuQi/yzGQWcO4GUCentUxhT4%2BVjSqWWU91oLr6CO0L/iXyhSvkwWEk/XvKqsYAKRAYgT%2BWa8VYw1jE8Y7Mkmk9nPzx9tzvpu%2BJhlw7ZM9VmDkENlNBuhEpB5jNOGk474kQntAt6Gfo92jP%2BHo4n5i6BEpu0f7Zu4Z7Zg56S%2BXsYMzgn79B9pH08fSppAtdAeclPTttElA31ivOhPZY%2ByHk5NsJRx/8wbjVMuxcQMHAxkBtCEc0DhIAIOLuY%2BAPoaxxbEiF7Qxjv9bz22134%2B17Qpru4wDOEgIfCAj/I3OxJjIeXId%2BF6cnQ%2BrkcrnkF10SeYdnjsy3yL4RBwZbxmj0JUYhxIL1eBsfGRHhelqHCM63R91fED34vojd/foWIaDjegsUV6c8Di9kiULieBoT%2BZ4cbzgxmIcBjI8ndH0r66CwY20ATowBsswKHUEKQzn6oBar/s/XlSxX0Hj8wgiHQCeMNLZDgXKHZ0aAzleI6KSl2nnQC76vgTVDuFGYVypHQSd3mL9LAMgCutlYwebB5pPLNfBkk6EyNPO2gbtrKptf74zwODKe3jcwn4Yi0RB%2BA48x2x4ig91/RFwvKUYm6HyHwo3nSvXZHJulkVSAgwUpPG2B8os14IoEl52IjykHP5OjcJivSecB0Y5Hbj9hl7nQylryYCB2OZtJrGfIxqRrJTX/tLPjFMMmsOJ1mJA4XTh/mC8YMgBFWxprygUl6rSnB8fvNsDg3OVtpOfLt9uEXPaF22RghWcd7Q54pAgXQ5PK55SBk5khAgl8xgxEjGWaP8MhqG9MJCilKC8oaCiULAPgylGZFB0iGbglKDa66yCQWbkkVKNd9omwHQAxiLtgsGXAR8vN9VacWzQ/hcUHqieyW/R9jsCZQMHERXt%2BCzX9TeqLKCYoDAvVXknlRGFPVEeSEHFCEFhR24S%2B5Sugn7vSAp4HSs4FMmwSJbCQZ/Iv2jk5FDg3MHxRyYGbQ9lkYq5OP1w%2BKEoMg/25UAeSJmk38dZyXhxkiq6OQn9AXJBX4tzLYBDAyWeyB9XiUf6/aAgIqscDymhyDBtJ8A%2ByBewH2l9pILTpkk5Jx2aNt3RfFugb2Bs47dRkl9SWQD73aJyG0u5DuxHKyWTp0iPj7abCP0LzpzrtY%2Bgb9mo8k566a9VMX5Iz4Eo512qbGPMIi8B%2Bh%2BOkfu3Qn%2BfPm1EVrpFeroa%2BtEwPy1Z0F8yXiQDdJ5BqSnmiKYvPRwwoMhe2aQygYGE/oROgsMNhzQRvJczsGnT9O0YQZkDBsiVOk5gZBIxY55gFOQCIwsjD2gnOMf5hUm5GcIUH9oD7Ym%2BmvGA78cBgfzgjGcuPeA0R5ZJ/wf6cAxLDDmMU5xDjFvIEd1ER1eE6DdjXHVTs9yqBipGJCCHyCNOsFOGZVtWCroa%2BiltuqO%2Bh8wdUmZxtHDsjF8/1vEBfQq9CqMcucBoJevmVDWQyTRg/OgupzvnmKx2ejzhxmIcBn7m9kVzv7uK0MHR4TGgdKRy0HEx0OAJwUjDI0bqU4DBhxRJUmnwBKE0HgoGRibyE9342NzR8vVTJsgXF4y31NXEtEYUYYQ/SqN2VAy2wZvLeYS5SgE6BeZCRaNRdMqJ0VOMSdIWSJn46cKp8gU9DgxQDPaO4EoRdazUDo4BPAoeP64p1zNA2ml7gz%2BgPHMt6JBJM8IQQ%2Blkribnx3VnXuXLXdNkwjFjlHTcYjqfgowUm0yfDGgh3ENkEedNe6CoowgBbYj2QlvAsIkaJ7Q52gTH/3LZAsgZ7YD2%2BN6Zo2zuBemwpK9G27ENnjrARdsyx4sBxneg9NJ28LAS6eEYgDaIoYWnl0glfQyKROJ8KOZBhaIDzI9kHibFMojSJ%2BghB8Fv0ya4Dno4%2B6nS30VBR2boa4A%2BA6XkYGk8AEYQmQHjdUPGAScOjhjeS9cD4fiihVK4H1y/L6kM81kiKERZosfSlWB8M/czWXDeOAWSVQk51ooYL%2BxJu3APgvHKvaav4P4nKnkoaLTZcO8OBXLP3FMif99TmeD%2BMgcSuYhCm0IRpu8P0OYxyEIBiyw9HuShUpXUKBixHCNtNIDhFiIYnBFyhVzgwGG8IlLHRgpiRyDzyCLjE7IWoI%2Bw7BD9/lDEA5DF6H5RuK6MnSHNDvnlb%2BaIknLNWIFDl0hX1BBEDj771EZ558Or5atLNpth8bbpw2XOkK6t1grILqmayYT2wjVPBtyp5tZWGyf41xG8bzqW7sIjf%2BOMjEIaM44VZOblpAK5IUKG4XndjOH72%2BMn5o1p45QkYri09EA0jjaGU5s2OSQjzdp1qgo1choNGCAnW6vqzPgMMs2Yw/x4ZBeQd9ocbZBUVo6DaTWkntM2Q3%2BfCEYg7zNWEPUMYxnHuMWWAWuxe8j3Y5iicx6KEQPTrD2Tls2epKgjJ%2BhNXG%2BOH%2BMUI5WIKvLxqxU7TIfrLnAYcA2cIyM5o10vgE6AaBtFYboaJkLvqWu2wWpm/qD9HUIiTIxGKessiFT86YIT5OaLTrC5hZ9cvEHO%2B/dSy3cnxSyZ/PycqVao5WPayTKfZOHtz1sEg047GeDhuvmimfLMqxdYWgaFQTieOy%2BfYwMBy49QqOCbp/WcSdC0hVP0WJMYRJHJ2gZnHEaKc2fAgImSScpkR1XXMFjw4nZkTB4NKHlPvmq%2B3X/awg%2BWbZMr737RvMJdtYZhe1AU4a4r5sizehxEKb/7wlZ5zX0rbA5VUBK6mreoMXDLxSdYgSfmMVPB8b6r5sln5481pxLRpWdePV8%2BMne0Ra%2BAapDI0Tg1oIjQ3qBbMpk/hDlzyVOMaZtci1D1uatBgUOnO1SkiD4rFoHuPD4we7Tcf9VcuemCGdb%2B3nD/CpuTSJQ5WSDnn9Qx4m6Vix%2BfNcX6PgrRsR3peqtHC8WtmOv/1LULTDHn3jPv6t4r59rcLyJSFEy54/LZ8qE5o%2BKfOgCK%2BdLSKouWYiCw5FJXE1LTkwn9MjpLMiBtmAwgMgow1jsCWZ0z%2BMiWzjgU9Hk/WThFHnvVSVaUhqgbbfGcf70g25Oku8AMbXOfPmmsjVtfOXm8OSff8fBqm4tO1lVHjo/O5ssLJliRIdalnqlG4n%2BumGvjKM5WUsAZF%2B5QnYoCgYytS/dUyfKyGrt/3QXyylJ1zpHhxmIc8u2Z45SMFBHm/zDhmlD4KyYO7tDLSwXHH5w12daIw/OD9zUlsitRDyqekarAPK9QYasjrFqkdqxUn3vNfS9ZChzRw8ToIJBSEiILAaJv9Mt8hvQNopGJx05nalHQBO9dgO%2BkaiHKHRO1X//fFfLt57aY1wwv16G6OMoNMJcTr1CiMYFXE6ObFLzDgTmQX3pm0/45jV99dpO865E1lh7L5HOq4V1y5zIrPtJTIIJy7uh8m7%2BULChOlCylgwgDKc1EtfCatgee1K%2BcMsEM%2Bdg8rkZT2vFeRj3MWan9LWJcUntwJDCRE3SAe7MaQBSFZrK/FX5Zu9s8qokGGoqrzR%2BLXH48uDh9OHbm2ZK%2BjVySYhRSXYh%2B4m0lqrinITZ/JBGUO%2BSTdT0//9QmebseB6ltRGcOdfxQb8bEPrsOURsaOSRyQkTzcI1N5tRQNIeUQhbXv%2BTOpXLpXcvkP1vKLFJywR1LrToqc8hIv2Vpn1%2BcM1W26vNPPbnB0m1f7ng7m7NUeef%2BJwuuK0WNEvvHroKpAKSDEbVtD275ZXofqPr5MTVeuNdkd%2BRpu0ssTU8kALk5VMSANkREcYZupIR%2B9PH1VlkUhZR2ntiWiCBEMzqAPpr2HzJAaPNEbpCBKOzHZ2mj7cGx0AcxvlFo7f2L1lqExiKBh2jTtEHGKcasqDFBH4HDjajbnrqm/Wmxh4JCKcztv/o/L5qjl6ImF92xTP738XV2XH9cs9vGjrc/uNrS3/992WyrtojMA0eJSHDFOZKojHYVZM1QVCuZMF%2BbInHJAIPo5y8VaVsUK9rEPLhEcGK984QRcv3Cyeb8xdFI241GsQEnLAWPcAy%2BXNeVre13dsEgm4f4/kVrbK47bRF9J/GzjBdUxA5z3tGp6OPRmYjEM16QrcX4EXV%2BDlR5oEBUg35hR0Yfxg5TFegb3nD/SiswSCoo8nmorn5XTaOOmc12HhxLyChD70XmkVHm%2B3akvyVCQSj0OKYj8fzae5fbPGdSv3mk8ivVUPeqDOGgf4salQ%2B/Yp48%2BsoTrfYFFe1J4yWAkQzdG6gtMC0hO8J5edq3UvogDKCkoZLaglLelWDQoBTvqm0wZeu66SMOSndkAP36qROsEyRljAgHiiMDH0pKSL9hML5i/GCpbWnRzrO%2BTXpFIuT4M3CSmoaBhMFHZTiKECQafRwPkQWUTToyJj4zH4lSx39bXxxbykN7JSaMs/G9pKsxaRvFm7kx7cF%2BpPKwJAZ59xhr27Tj5zswSlDwOwIFgPmKDMJ0woDyzkDAHBwKMjCJ/HAgRZGOlk6RuTDMUaTDI7JFWgoFE8irT1blupcDRZFrhOc2OrB0NUTaSTujollX/y4OEeYFodu9ZvJQS1GOgjPH1pLTgZ2J%2BQy2DLoM/pRJD0Ycnn/mNjFXlraKQdkRpC9ScIr2H4rMoIQiE3ggo2npqXr%2BKJl8N9FIogSkiE7NzbJ5MM%2BXVln75DCofsj7MC0v0wrtkIJHcQLmciSC55tj4edof7RFlF2OjYjjoVKPcRIxfxDPPs6jkM5Ke8HoZu7Z4UZIkQUME%2BSD548VVZhCzb3BIEQm2Jg/w3FRXZBCOCgr9FOJqetdCcoXKZL0UxhGyYKKh/Q/KMcU8%2Blq6JsoJMG9/MGZkw8aK0gre4cqxKRoonQyttCnmdNF%2B2MMPvpUxhSUM/rJtdqnEek6FFxbFEeUa%2BZcUVCD8YUKw4mLk9MvcC2IugVQRJEVMkXQXWk7pO/RRgEZpwANFSQpSEOxkPag1Q9S%2BcaNSJorc3Jx0CBfh1okHXlA9rGJmccVCq7RR/A3j5v12Kjq%2B3IQBaH909%2BQcs78TGSA4juMgdQA4G8UZPYZqeeJg5c%2BAkjrQyEep68TXVyfUOW5syElnLE6mQ4UoC1yfxPTlLsC2jh9LfO8ubdkTUUNRq4BfS5tjL5zW1WDPLmzwqYr8B5jC20cA4X%2Bg2gvfT/G1qFI0XaDo5o2wTxY%2BkGuM8t8IWNR0HWoYko9CUBu0fUYX6wglbYf2jTjB%2BMNDm/2pegNayIyPxbDrT2QTZxVDa0UZqq070PeKRRFQTfkqz34PY6Zc%2BdYwvxjdC%2BqX2NIPqpjK%2BPq4cDv4kBhvHhUx4pFuqFfoieuKKsWCkohF79fvUu%2B9fxWm%2B/%2Br02ltlGgjSlUyMR/VW8jg6Ir4X4wJqI/k4rqHBm%2BdEYcZAuDkTmBeAsZ3A7luTxW8BjhOULgKcmPd5VOhw6PFFHmTqEkMLGfalcIL%2BWQ58ULZDAAMXAz8GEgUQGRQRtFmUjGne0snYE3idxxlFjmdPBbdC6xtD4G2GarBkkBGzxtKaoUscYOn3nTtGGq0A6w8ti3biiWEt13pH4PSikGDL/JXEgUbJRuIhGU%2BKdqKoo4Xi9eZ6Cns52pHTTHy/OTCnPMGOW8OFbm1aDgooQyj5HjMI%2BZXreaplbr7FGu2RfhZy1JDE4iQgzcGFacF8UHKF7SkaLMMbN%2BF4oVHRoRIQwSjFKuOVGiRPgMHSvXngnmyVg6g7ZB6XYUELylHYwDXUIwEGmvTLzn2nSVVBDhZmI/k/5pm5PVyGLeBs/ZKM/N3DkiXD9ets3mPdDGmVtK2iYDJ%2B2PZSRw/OBgofovC/mjPFHoI3HpDArUAPeTuVJ8jnT0hSpHtE3uN0oESuWpqpQwZ5TfpO3xW1TfZRCiWACT%2BnE24MFGFrhmDODnjS6wuU20F6qdolSwYDFygVziHWf%2BG%2BtL4gjBAYLiT2ohRjrtGSWC4jNkJKDgnDMy3xR5IkQosrRw5JCIAn0D1SeZI8J7N63eqYN6nZ0T7%2BMAeWJXZYf38czheabsY/hxrVHGkDMUF0q7AxFZ1v%2B6SM8NeX1BlXj6gnCvcAjh6KJSpF6GTsecU3ourDnJvU3MNOhKEAmyGLi/KEpF2jcnRts6E9oUcoeCg3OPvpD7yHXGofg/qvTRPmnbD26PFR7ieGg/tCX6e/ZlTUQifChqFEqjjbW3dAbOAYxDCsDQJokG0qb4DtrjOJUnDGbu%2BcDU/vY6YxdOCto0f9PmWW6G36JADjJDO2YOMfuwli39Nu0XQxHnI/0/SjzLL9DWUUCJBJJNgKLN%2BMV4hLJH/04bQ2lHjjHWWCuXyCmjOW2SIiZksAQZ4rNz9LMX6/jG2EoUHAOYawp8R2J13wDVXyk2xXjKfjhoqEDOMks3rdlthgvjADoEYxzRC1oEhjr9CcYM/RUVw3HUHm4GzJGCc4vlnzgnrk8y4bw5Z5R%2B5oXSBrtOKmI6G7oBehF9Ae2V6037u1Tv8RkjcFyL/HX9bqHg1vbqWNYHskN/T5uk72DpF%2BSLpdNoq9xnZAGjijTT6NIZa7RtEoFDLjhf2iXGHUtdYCzTZjl35OxtqlugGdAu0JXO0j6ZQjQUR7pT9SzaLf0IYzptH2MRZzvthb6FQmPIJX0xBZ1om4wxtFlkDblmCQ9%2Bi4wuloKgjyeIwHVHhyFiT7%2BMAU8/QjEd9uezHA86KPMNkQnG1n9vKpF/bCy168l1pK%2BhgE1HcF3OHplr14NiWCEAga7IGtTIMjBnHx2Lth827gHONp6TwdVehltnwb0arNeXZccYG%2BnPnCPDjcUE6HBI4SEylljtrzPBkGJApAQ3axGijOElZ2MwpNMi7YZUMKCDQBlgfbbzVFFkPwZBBnrWlKJToaNEQUD5pKNAqeF8MHoZtNkXRZPISPgtFEC8TXRy/AZVP%2BepEFMtC%2B8dxihrAzFv43eqdP5ixQ5T/ugUWKOOTg4DC6UF5fmp4grtdIutk6UzRknme/GM4wWkQ%2BDcMAh4D8P0NFVGiWBhYBJ90f7JBnMUFtL56EDp4PEEU6E09r051imxxhVVyUiBo0pluj5nbSC8zo/EB386x/agwyV9i2UKuH4cAwo2gwIdG8ZFInjkMJ6HqxJCqh7R4a7s5DDW6OypAMgAiJKcbBgQiGpwjSgM0dXGMcoYCiWDMOcc2mpu%2BgCbp8T6hbRxQEkjEocsoECerbKBZxUj52tLNltbZS1QlFm8yjgh8tK0fRQOsggLEUqcGChWKMoonCinKA/MFcQQxXFU3dRqBQxwPPDTKLvILLeDtE2cMywpUKQbHmfaNGlgKB5EGlirkjRX5Iz2g9wgX8glx4jySEoSCi3neurQXHukn6AUOXJAtcaYDKmxoG2CNCTuzTP63UTogbUWcTbhAMKRQSU6qgUz6OPkwCON/JGG3VGrZRDHoCWKg/cbpYK/WdoGwxmu0muN4o1iwkYfEe4TGwoVMhzSyzsT%2BjUKRHC/PnniWDOIglMjmeAIoOARSiX3tSshysv9oG/DQGetNK4z9wqF%2BbvPbzVDhBR6%2BltkFcMFpwT7YrzQ333vhW1WwRA5xqnBWoFheRbaEoV7cCawzmldS4spw9F7%2B6C%2BTv/I60TscQigePGbe1U2Xj85tiYi7%2BG8u0UVTe4%2BMsYtQgFlsXC%2Bk9/7wdJtVl0yVeU9X/8%2BUeWSY6eNotwCBjmOI9oh381YQxog4zPtmrlr9P0o22TCoPSyhhxjGdFunIthzEMmyUChxD9jFAYz15DfQhGPFumJwjmeo7JMz8d%2B6AiMtfTNVIAMkRiUcyI99DesW8fvYowgI8xB5vp1hV5B62ds4JjIviA1u6NaCF0JU2JwLnAPOU%2BMnq6CMQCZ2Kn9nOkhcZlg4zoQEWdayW/0/tBO0ANoSzicGSvov4mIo8tgEP1F2yHFyYh2Y1QSNUNvYT8MX86JfpZIGEYZuge/xW8zjtDP0Y/TT%2BJoZvx6StsKugh6Dn0ieuVnntpo4xayg1yzxiHORaKLyAdOFsYU9D36fyKPjCFkQoXoJ%2BdBBgjZHbRpnIfIM2sP4yDHaYBMcCwYSlSJR2fi8zj8yG5h/VwCFSyVgROH/Vm7mnEJ5woGJRFCxtKOwHlKmjhjF8vZcLvP0e%2BkfyGIwbXtiPmq/zFdhDENWewq6Hfoa5Dzj88dYzpxd4wXvZ1%2B%2B8gZdA4CQaThf2bxhvgrXQdtFmPnoLard4Q4Gh1X4rIaRNRQAgx9i04MY4X9EAAiMXwXwgsMIPzN3Cde4X0G5gCf4yfYh0cMAfLY%2BZtfiQoVXtyo4sdhMMcqOihRAYx9OC5eRennK3gtOnDwG0Q3w9fz2xi14cjCOfF59uPow3dwDlyD6LFxfuE3eY9ITZMq1ex/4FcPho8zVzV8L9/H9UIhoNKavtQGPsPxsF%2B4Px19/7HC8RAxIcUFzzQDYHfBAIIC9%2B5HVtsA3dVwfWlbXO9AaCOJk%2BMZDNMGHGgPifvRPrln3Cfz/sf3R0S4z7xOpITP21foC7xKm%2BA4yDbgPkfbqz6z5/xWYjvTj1i7tdTR%2BD58nkEewvFAeA1C%2B0uUJ86F4%2BOR34FwvkHe%2BBdrlypB8Y/zm1G5C3LD37zeERwD1wh5Yr/Ev8M%2BHGe45onw27HqyZ0vH0TacWL97%2BzRcqEq4%2B0fQdfDeaE4oth9Jwlzm/VyW7%2B2v51GoF3TFqLXur17FN2P17mWTfqa9b36N59BLniNtkKb2j/eKLzOj/M6n0F94DfYg6YR9k1se9DR8bAPr/C6ySWf08%2BHj9rr%2BrnocYTP8FXsx/eE7%2BcfbZXXwMaQyGeDzLAPL/M%2B0F55rT34HWSLPfks58fxcFzIMMcQhe9EZvicwTFqP5F4jzoLzg%2BHDnMlX6HGOI6M7oJUXaq0v/2hVR0a350J15l7H20fwD1K7OvYg2sV5upB4n68R5oy9wo9xPpy/Zt92GiPTAki8wmQAcYIPgPso/%2BZrhbamO2rrzGu0F70ZYOXQ78d/zo7HmufurFbaNeh3cY/aufNsYXP8Z18FvjNoJ9EzzfIG5%2B1MYQrov/xsajM8BmOi28Lr7UHv4OzikOIyhswLupPdcj%2B66z7JI7pnQnHg2H6KdWjGC8S56w6h4cbi%2B2A5x4lAM8OCxInI9XQcaLQWeO9Z4I%2BaTCkGKLYdRd0%2BqQXPbSj3Aok4bFvL/LqOF0JHnNSiN84ZahFehILpiQbxgU840T1WFOMKHdUOXWcrgZjgiWfSOt9xfhCYW5oUNi7A4wOIq/36xjx21VFNlewowJGjtOVECFmjXLm1rPuKuNF1FnlHD6ehtoOeF2I6JDSQvojYz8Tcb3Dc7oaBnnSFJl7yhwjm5uT3b2GItC/4gmkchxpnaRvMQeHVCNXjZ2uBMcJgzyLOjNv99qJQ22uUU%2BYd8KxkQbLWEHqFt58xgoqJDpOV8JYQW0BUhFJ/6VYHn1zdxqKYJE3Ha%2BYG07KHynBpPl25bQexwnQ/pguRHosc6CZ5kXhN6ZghWiwc%2BR4ZPEwYM7Pg9vLbK4GRR/IR6cYRleGzp2%2BAY4JDEEUTuZ7YCgyt4z5C8x5wCjrid0bc0GZp8d8J%2BbrIBPMbwjzjBznaKG9Ey2hoAmpdcyXZb4PpdcpKEQRos5aN62zoViLzRUqrzGZYE440UZSvxznWMAxwVjB/CvW1GReJPOwmLtLml13R9k7goIvZGkxfy8mE7Gxwp3vTmfBeIGByHiBUYjjDociczKZV%2Bypp8eOG4uHCYUtKGTAxPpnSiqtiIUrxh1TVVklO4p22PMxY8bKwIHdN9%2BuJ4MXuDAj1To3KkzSwTE3kcnqPR0UYJRhKtBSEIjqiMiI0zFbtmyRuro6yc3NlREjRsRfdaLgGWatMdbbZKkHik5RqIUsj94AhSsojoFMUAWaub4eaeyY8vJy2b17tzDXcPz48ZKe0bVLV/VWcJBQdIQpCTPzB9pYQfopxmNPh3RtDEXTn1QmKDLE0llO%2B7To9dq8ebM0NTXK4ILBUji0MP6OkwjORQq3kX5NMSAKCi1Q2WDeLkak0zm4sXiYcJGYLMxkXCYa89wvXMfcfvvt8ta3vtWe33HHHXLeeefZc6ct9GcoSkz2ZpI%2BynJv6eLoPZijEjYPoByaCy64QJ555hl585vfLL/4xS/irzqJ0P6RA2SDRyLwvWXc3z9WqDAwXvhYcWhuvPFG%2BfjHPy6pqamyaNEimTVrVvwdJwrNX8Vgv1zEZKL3jBWMDQfGCR8rDkVFRYUsXLhQNm3aJB/96Efla1/7Wvwdpz1isnFALtClkA2n83Bj0ekSbr31Vnnta19rz%2B%2B//3658MIL7bnj9GXOOOMMWbx4sbz97W%2BX3/3ud/FXHafvcsMNN8gHP/hBMxaXLFkic%2BbMib/jOH2TvXv3yoIFC2TDhg3y6U9/Wr797W/H33Gc7qFnTvxwHMdxHMdxHMdxuhU3Fh3HcRzH%2BX/2zgLOrur445Ns1t2zsY27EMPdvUhpS0vtX2%2Bp679K3aAOVChSoPIHCoUWL%2B4JkBB3zyZZdw//%2Bc57N7y83Q27SfbtS5gfvE/2yb333HPHZ84ch8PhcDi6wJ1Fh8PhcDgcDofD4XB0gTuLDofD4XA4HA6Hw%2BHoAncWHQ6Hw%2BFwOBwOh8PRBe4sOhwOh8PhcDgcDoejC9xZdDgcDofD4XA4HA5HF7iz6HA4HA6Hw%2BFwOByOLnBn0eFwOBwOh8PhcDgcXeDOosPhcDgcDofD4XA4usCdRYfD4XA4HA6Hw%2BFwdIE7iw6Hw%2BFwOBwOh8Ph6AJ3Fh0Oh8PhcDgcDofD0QXuLDocDofD4XA4HA6HowvcWXQ4HA6Hw%2BFwOBwORxe4s%2BhwOBwOh8PhcDgcji5wZ9HhcDgcDofD4XA4HF3gzqKjX5CamiolJSX2Sk5ODn/qcDgcDofD4XA4DhW4s%2BjoF8ycOVOuueYae02YMCH8qcPhcDgcDofD4ThUMOh1Rfhvxz7Q1rlbtja0ygs762RxZYPUt3dITWun1LZ1SEvH7vCvHAFa21qlrq7O/s7OzpakxCT727E3MpMSpDg1SUZkpMiI9CQ5eXiu/p0siYMHhX8Rv9itkqO5o1NeU35YXNko62ubpbK1XSpb2qVJP1eWcUTh5ccektrKCikZM1amzDsm/KkjwBCl%2B7QhgyV1SIKMVD6Ympsu84oyZVpeevgX8Y9WJfwtqisWV8AXDcYPvOraOu07x97Yum61rH5lgQwaPFjmn362ZGTnhr9xGFQVGE8kJEhBSqKMzU6VWfnpcszQbMlITAj/KL7RqWZmQ3tniCf0tamhJcwXHaZD0CWON9DR3iYLHn1AmhsapHTyVBk3Y3b4G0ckEgJ9kTBYhqUny6TcNJlbmGkvx8GFO4v7ABODcn%2BlvF5fDWYUr6xulC2NreYgNukLQdfuks6xn0hRIZeVNETykodIrr4mqXGMoDt2aJbMiVOBB70TOHlpV50sUsW/prZJNtW3SHlzuzSqQYBR0LZ7txsA3aG5QS2nDhGCJ8lp4Q8dARIGDZKkBH2p4wA/YACMzkyRiTlpMrswQ04syZEc/TzeAK2jK17cWSevVtTL8qpGWVfXYnzS0N4hDaonmlVfdDhTdEV7q05es/6BV5QhMvjQcIBiiSTVE0lqGOMcFqYmWSBlXFaqzCzIkKOLs2R6nAZT2pTeN9Q1y4Jd9eok1sva2mbVFa3mKMIT6AsC8c4VUcAsR1fs7tSHnxJ6ObpgMPpC%2BQKdkZOUKEPTkqQ0M1kmqL44oiBTji/JlhL9zHHgcGexB2AQV6hAe2FHrTywuUqeKauV9Sr0PDLs6G9MUYfxzJG5csHoAhV4GZKtziQZl3hAi9L/quomeWxbtdy7sUJeLq%2BXpvbdFjl2OPoLBFSOU8V/%2BfgiOVb/HaFOZLIa0PEAHEB0xUI1iO9cv0ueVV2xWZ1EjGCHoz8xS/XDeaX5pitm5qdLSkKCGtDhLwcQaAMC6kurGuWRLVXyn82Vsqi8QfWHZxEd/Q%2BCK8cNzZa3jyuUk4fnyNisVAvMO/Yf7ix2AwzfssY2eXxbjfzo5Y2ysb7FjGSHI1agtAKn8etzS%2BVYFXqFqYmWdRkoICQ6Vcsvr26U21bvlH%2BuL5d1tWQDHI7YYUxWinxoyjAzAsZlp8qQAeQJgK7Y2dQmT6mD%2BKvFW8w4JlvicMQKeSmJcrzqiK/MGSUz8jIs%2BziQDiO6gkDJMuWFG1aUyf2bKq3yxOGINVjS887xRfL%2BSSVqT6VZ2Wp8hN0PPbiz2A1Q/vdsqJCrFmywiDFGsk%2BSI5ZAoLFusSA1SX52zDg5a1SerVcZKJBp36F88eXn1lpWkbUmu110OGIMyo4oQ32XGgDfmjfayo4GElWt7XLH2nL5%2BaubbR2W6wpHrEG8hKzJ8PRk%2BfMpk2VuUaakDxm4Ul6qrwiwf%2B6ZNfLSzjqpaSOb6FzhiD0ImmQmDpHTR%2BbKr46fYPpioAOMhyoSrlKE/3YoKJ34z6ZKuWXVDllV0%2BRrTBwDBkivWRXvhroWW6fCuq2BKEeFB7aoIfzr17bIf7dWy66mdi87dQwIoLrWztdtzVO1OmpnjMwbsEgxaxDv3lAhf12z07IovnbdMVAgSNGo9LipoVXGZ6faOt%2BBADywUu2mX7%2B2VZ7aXiNVrR5UdAwcoDxoskbpcHN9i5w4LEfSBjCQcijDi3ijQPe6x9QgpnGHO4qOgQTUBw0uUZp8cHOVrYkaCJBphyf%2BvbFSyvTvDlf%2BjgEExifBi0eVJlkPVdfWEf4mtli4q04eUr5cVF7va9kdAwokMqWfz%2B%2BolUeVJwh0DwTgSwKKD2yutICOBxUdAw30xa7mNqNL9AV/O/oOdxbDQKQRgXh8W7W8qEYAW2I4HPEAOso9v1ONABV2ZDNiqX5R9mvrmq0sm9IiN4od8QB4gnVQN68ss46jsQzsBbqCxmcLVFeQPXE4BhrQJZ2oMYppshTrPgvwIBl2KrPgSc%2B0O%2BIF0Oau5nb5x5pdsrqm2fSHo29wZzGMIPrwtApZ2js7HPEEmsmQydjW2CqxXGaM8bGiulGe3F5jpU4OR7yAfQsxTFfUNEl9DJvKmOHR1CaPb6%2B20iaHI56wqLLB9oOmCiSWgAeXVDbK83pthyPeQOb9v9uqbQu8mtb28KeO3sKdxTDI2Dy8ucqi1Z49ccQjKAdlH7fOGPpsBE5eUwOADo/uKjriCWS969VhpInG9sbW8Kf9j/q2DnlCHUX40SPUjngDfRfW1jbJq%2BWxXbZAp2xezTEM3DgcvQWSGjuGJWbr6zzI11e4sxgGUQeIqNZLihxxCsrdKPOJZcMAMporqpvcUXTEJaBL6LO8OXaRYjIoT26vNUfV4Yg3wBM7m9pjvm5xpfIh%2BsJ1hSNeAW1Co7EMLh4ucGcxjHY1wFfHuJzJ4egLaNOPsxjLzOL2xjbZ4qV2jjgGZdLlLbEruWvq2C2LKxqsGsXhiEewpIa1WbEEJdnoC4cjnrGurtmW8zj6BncWw2A91vamNi9BdcQtWD%2B4uaElpmsWa9razUl1OOIVW1Xxs34xVmjfvduMjTb91%2BGIR9CgL9YGMXrCGwM64h3sF13V4nTaV7izGAbmd2NHp28L4Ihb0Fgj1t1Q2dMu1l31HI6%2BgDVasey8aGsl29k/LvyBwxFngB9a1J6JJdATHmx3xDugUQJ%2Bjr7BncUwUPxkbmLZgt3hiHcgWDHGHQ5HCJSBs17R95BzON4AgUx3Fh2OwxPuLEbAdb/D0RXOFg5HBJQhnCccjq5wvnA4Dk%2B4s%2BhwOBwOh8PhcDgcji5wZ9HhcDgcDofD4XA4HF3gzuJhhiGDB0lpZoqMzUo9oNfQtCRJGxJ78shMTJDh6ckyWu%2BBMSQlOIk6HA6Hw%2BFwOBwDgUGvx7IPfxxjS0OrzLtjoe1PdChjVEaKPHbREZKXnBj%2BZP/w4OZKuWllmTyypTr8SWzwrglF8tGpw2RiTpo8t6NWvrtgo%2B0t6Ahhks7LwsvmSYY61bHAF55dK79cvCX8zuGIT1x/0iT5%2BLRh4Xf9i4W76mX%2BnQvD7xyO%2BMQJJdny1MVzwu/6H1c8ulxuX70z/M7hiF98Y26p/OCoseF3jt7AncUwDhdnkYwczkR%2ByoE5i/dtrJDrl22XBzZVhj%2BJDd4/eahcOWOETMlNk6e218jXnl8vr1U2hL91uLO4f3jPxGK5cEyB5CYNCX/Sd7AZ%2B5KqBvmj8gXyIlbI0jGfMjxH3j6uSFISBlsg588rysLfOoA7i30HFRwf0zmbmZ9xQFUkNW0dqi8q5dZVO8KfxA5vU57%2B1PThtuXVp55aLRvqWsLfONxZ3D/ML8qU80cXyDHFWeFP9g%2BNqi%2Bgye0x3u8yfUiCfOfI0WYrrKxuknvVlnu2rDb8rQO4s9h3uLMYxuHiLGarYfkBdbhSe1D%2BZ43Ml5PV8Gzq6FRnrFYWVdR3u5HumtpmebW8QdbXNYc/iQ3cWdw33FncP3xz3milq%2BFSnJoU/qTvYLuEJ8tq5MvPrTUlHCsUpCRaxv2LR4xSoz5BblxZJv/7/Lrwtw7gzmLfMSEnVa45drycOCzH9Mb%2BYqfqzOuXbpfvLtgQ/iR2gKd/e8JEadu92/T3kkqvQgngzuL%2B4exReSpLhlsg4kCAXQVNrlVbKpYguHj3OTPkSHV6X9xZJ9cu3SZ3ry8Pf%2BsA7iz2He4shnG4OItvhh8dPVb%2Bd06pVLa0y9WLtshfVbhvboifaOxJaricU5pvUe/l1Y02vk31Hi0O4M7i/uHScYVy9sh8yU7uOm%2BZiUPk2KFZkq5zyl6rZCfW1HZ1BtlHbIXS5K2rdsq2GEaL3Vl8c7iz2HeUpCXJ%2ByYPlam56d0GF8dnp8nYrBSTNWTVnyurlbr2roHF2rZOeXhzldyxblf4k9jBncWe4c7i/uGIggw5c2SezFNnKxrI31EZybZMJjlhsKyqaeoxmA3PfE3l9I6m2NqU7iy%2BOdxZ7DvcWQzDnUXHoQB3Fg8%2BJuic3n3OdGvstK62WW5YsV1%2BvXhr%2BNuBR27yELlgdIF8eOowM%2BrvWLtLfvbq5vC3DuDO4sHHp2eMUJorMcN4Y32LXPzAkphm1HuD900aKlfNHyPt6ixeeP8SM94dIbizePAxQh3Fd08sli8fMcqCeD9ftFm%2B8lx8Be5oEvj7kydZefmiigYrD394S1X4WwdwZ7Hv8FaTDofDEcdgTdj/qYN44f2vyRn3LpLfLYkfR9bhGEjAFwR5j7nr5ZiX%2Bzkc8QiqYz7%2B5Co58e5X5JP67xPbYtuk0HF4wp1Fxz5RmJoovztxotx37kz53pFj5IOTS%2BQnx4yTe86ZLg9eMMvKHYjQHFWcZdt2gGl56fLx6cPlT6dMkjvOmiYPnD9LHtbfPqj/ksH548mT5DMzR8h0/V00zhiZZ9e799wZ8sOjx1q2J0BJepJdj3O9Y3yRTM5Nk2/PGy23njZV/nPeTPuccd54ymT5rJ5/RHpy%2BEiH49AFtR8tnbulprXDXpQ3ORyOEF9Utbbrq0M6vUjK4RC4gPX11coT9eo4tu12vnAcOLwMNQwvQ%2B0elF3goLGuhXVc1N%2BPz06VrMQhMkh9Q9zDZ8NbXFC7f86ofFsgTkcxOrMO5kcKiCxRnUm6OdapINva2CKPb6uRf22osEY2AfbV4GacOo6UYOYkD7Ea/A31LXY93nPuJH0l6PUwILine9ZXyD16/sOpQY6XoR587E8Z6pFFWRYgyUsZYmscK5SfWGfLC1qEBuEX1ozwPbRZot8dOzTb9g/l%2BUWuE2ts3208uaGuWR7ZWiWR%2Bp11MrMK0uWEkhw992BZWF4nD21%2Bo6zotBG5NhaOeaasxvYmDY0lydavgFYdT3lzu60Dfrm83oyJwwlehnrw0dcyVGh6VkGG0amSu2X6yhrb5EilzZwklVf6WV1rp6ysabJu2wDanpSTKrMLMyU3OdF%2Bl8DBik4laLIkZNbhI0pMWTcciWOGZlmJNo7itUu27VkfRpAT3TAmK0WqWjpsLKy3nJWfYZ3CWW8GWjo7jS%2BeK1M%2BrW%2B26x0u8DLUg4/9LUOliRR2UVvn63LLqjKzpwiWZ6vtggWO7H9sW/UeGkeHcMw4tbW4TqrSa8AXnKNeaXm78ha2Dc0I21S%2BB8DGumLSUAuWb1E76AXlnWD7MewHdAXnhe6xv6blpdnn2FEBXxCUhOeX6nHLD8Oty7wMte9IuEoR/vstDRyYPy7fLo0dh5cRFQ0MS5Q5Aum5HXXWEKC7bqgBMDbfNaFYlW%2BSCRO65nHsf7dWyyIVVJtVoLCdwBPq2BXpbz4/a6S8bUyhKeR1avi%2BoNd4taJBhVqjvm8xhU1jBfaDHJmZbEYyQjIAi8sxLlD2NLZ5VK9Dtz3A3pEfVYMwRY1sxjNKndF2NShovEBt/nI1ZLgXjPGxWWlqKKTKdjUeaEpyuETXUBzMAQ5BLPDQlipTNoczoFUayGCsEo19paLejNN94bzSfPnItBK5UA3VYqW3qar4z1Rj4KyReXK0OoRHKw1Dm6tV%2BZMJnKPGMEYtW3icpUbsiWrI4TiiuDEK5hdnyoz8dDVGUsxgLVfjAYcTwHM0XPjMzJFyvPJutdI4/BfgfZNKrNEH54PO4W8ME7L0nJvPj9HXDDWUC/Ve4d/V3TTwOZRBq/vuGlL0BzDS/qS64nAHtAndwh84bP9Yu8uCIj0hT38HbX9u1gg5bmiODFfDemx2itH8CUqH0OCk3FR17FSubK4yvoE2Lx1XZPx3utIrDs7RxSF65Tt0AWuvguZTO1Wes01GgIvGFMiPjh6nPJdljsouNYDBSOWjzx8xQt41vlgm5qaZTpielyHvVF124rDQuY/Tax2j15qmn6NT4H0ylPDt4YBS1Y8fnFISftf/%2BOf68sO%2BwRD2EHL0OKUfAh3sBd2bvajfOb5IvnvkGKPnHU3tctHYQrlkbJE19ENXEJBZXNEom9S5g99OUTvtfZOKzZY6Vf%2BGXgO%2BIEAyryjLuhmjG3A0IzOIacorOEKXjSs0nbZe7a4gyMP136%2BO5HsnDrUGVs2qYy7Sa1w0tmCPrmBMswszLHjKljrwHHbVYcIWBu6VeXX0Hl6G6ug1iBw3qlC6fc1OufLp1XLlU/p6eo1ct3S7ZS7OVwMaZ4/f4aDRhOPDT6yST%2BrvPqO//9RTq%2BTLz62TBbvqTPGXpCVb5JdsYF%2BBkGxQAXaDGm0fDV/jY/rvt1/aIE9sq7GGB5yb8USWsjocBxOJ6rSTQT9dFQ97OOIcEu0tb2mzYAlBKCLIX1AD%2BntHjjbjG3LfqEbBwvJ6e8Er8A/0yl6Q15440SLPRJf7Avju1OG55sgSAd/R3GZZRAIpdcorfMZejV%2BYNdKcxqBs3OE42EhKGCSzVfbizDV3dO7JUBAAxOEcpExwYkmOfGX2KNvrkUxHh1qjS6uajCfY0olMH7yCUUww5ENTS8wJ7CtoEHW8OoaXqIFO1mVtbYsFMMnI4Bhy/k/PHCGX6veuKxz9CeiPYAqBGBww5DOBdIIc8AWZPQKD16sOwJkkyFGln5MZhC8WK93ivOHE4Tx%2BduZIy3SOVNneFyD64QsypPAGvEfG/xW9xtraJkkaPNiCKf8zuUTeq85lpuo2x1sb7iw6eg0ECkr2F4u2SEtHKOtBOURNa0jIYQhTKoFB8MyOWvn3xkppjyiP4BjK/Nh6gI1qMYbp3FWUmmjCqy9g/8d/bqiwbDCGdgDG98OXN1kkfPfragioETIzv%2BvaSIfjYIDyUpwwyte///JGOfvfi%2BWSB5bK2x9cJtct2WaKnYzhKerEUZKN48Y%2Bjaf/a5GV9fE65Z5X5aevbLYSJM6H4ifD0lfDmOwIjmmZXvMbL663a1yk5z/vP4vlWy9tsGvzm0m5afIlNRLI8Dsc/QGWKZB5WabOH7wAT5yjvPHBx1bK7at3mLxn3Tm80dTeKY9uqZYL7n9tD0%2Bc/5/X9Lil8ovFW6xcjvLrOQWZcsm4wvAVeg/2Vs1OSjRjO/Ialz%2B8XH6wcJMFUoYob55dmmdZFYejvwDdD1O5%2B68N5fKxJ1bK25QOoUkC3jhqbHdxxaRiKVCbCL6gmdm7lE7hB2gWWU65719X77IquMSEQRaoZNlBX0BlUlFaouQkJ9iWNx9%2BfKVcwDWU597%2B0DK1rcrNeSX7z36T6ao3HG9tOAU4eg2EU7U6hmzoH1QkUBFEeQIlPFcv2iwf%2BO8K%2BcBjK%2BSPy0JOXGTlAn%2BTUaS0lNIJQDnpXBWQKOu%2BgDWmq6pD9f2R18ChRflXqIFBWQals5RHORz9BTLjtCZ/tbzB%2BICsIhlGIsc4ZpS/wQvQJcof%2Bud3QcMaSt9Q2HR2ZL84zkeJNbTbF0D7RKh/9MomeXBT1Z5r8O/9myrl%2BR21xheU9JFx51%2BHoz%2BAOIcPMIDhhQr9Gzrc0Rj6l6AG9D1k0GDLrNyyaofJc34X0Czlc39To5isJGV2ZM4JLPYVlL0%2Bsb1afqJ8wTUYF9cg4PjczlpZsKveSr6Hqi7an/M7HL0FAextygPPhJfOmPxXWwhaxDaiYopsYmiZUK08vb1GVtU07iXLlyg/PK90S2kpBjzLc4K1hr0F1hb64s515VY%2BzFg4d7WOhfW96Av0GRlGW2aRm25jc7x14c6io9eoVWFS0dL9%2BkYWWLNeASH4rL4QOJRc0AiEOvjzR%2BdbJ9WPTx9mTROGhSNhCDlq9ClL6gs2qnClbKI74Dy2qiDU/63ULlbr%2BxxvXTy9vdaa0wACKBif0B%2BZxQc3V8r3F26S7yzYKI%2Bpo4jDFo2tDa2ypiZUtgrIVvZVOeNosublVTXQg3W%2BAWj8QdMnOkeS0R%2BaRiMe5wtH/wE6J1se8AJgvSFreDFMb1pZJt9buMECKKw7j/wdIMACzdKkg%2BUPqUMSbA1WX0Fwk3Gw9pprwJ%2BA88OflI1jnGckJXgAxdGvgL7Lmlpt3TMBEAA5Qos0aSJw8tvXtsp3Xtog1y3bZo4htBkJeIHqLZY5YDcRdNkfG4dGOYGjGD2WF3fU2fIIzDLsOJrAZdKkyvGWhVsLjl4DpVvf1nMzHIATyCJq1k1dNq5I/mdKiXxIX5%2BYPty2y/j8zJH2InNyIKBxDQLT4RhIECkmIoziJkMYDYImdJz7rRrEv1q8xUpEUe40n6ADHY0KWBtCKRGNPAJbmfUkrEvpC1D%2B6/V6GMTdAaODF2EZOrFSEuVw9BeQ0Rii0SCwiOymE/c1i7bIXWqwVra2W/Mu1u2y9RLNilhLRZMmmp2Fgn6D9suZIwBT2dJhRnY0GAs8iRNLFqWv64Qdjr6AOii6jDa0d29HEbj4/bLtVn7NMh6CKjQOZF0tfMFax5OH51j5dpbyAtRKxr2vdBvoLa7HNaKxtbF1z84AnBoexGl0vHXhT99xUMB6LAzcc0blybfmjZYbT50iN5822ZzFy%2BlAV5JjjUDo1LWtqbVHg7a3QPFT1udwDCRokEHWbl/ddikrTR%2BSYOVFBEmOH5ptneroVMg2NnSu%2B%2B0JE%2BWT04ebwby/wPClnInIsMMx0MAhpJSuJ8AXZM8JMBI4oTnTO8cXW2Dxa7NHyY%2BPHifXnThRTh%2BRZ9Un%2Bwv0xJsFOR2OWGC3imZKoWv2sXURjh/0TnUJ5Z9njMyVD0wKBdy/PX%2B0XHPcePma6o0D6eaJvmIpkKsKR2/hzqLjoCBHlf5nZoywxhln0BlSHUfLuLDX3K46%2Bc%2BmSmtG88Vn18rZ9y22tSwOx1sBrJklc3jDKZPlqYtmy62nT5XvzB9jHR7PK82zMm0yjTQrOBBHjzImGjuxFsXhiGeQB6GJBwGTW0%2BfIg9fMEv%2BeMok2/%2BM5QpshUIWhWZP8ETkPnJ9BQGdduUNh%2BNQAAHFr88ZJfeeO0PuP3%2Bmdcf%2B7MwRxhenDc%2BVmXkZZl9FNg/sK1ARreq5vr6nlsXh2DfcWXQcFKQnDpbzx%2BSroEuW2rZOuXPdLrnkwaXy4cdXyZfUQfzhyxvNWWTTcDb1p3TC4TjcAa2/Z0KRfGNeqTmFRItZB0I3YNqgP7uj1srxPv/sGrn61S3WxGB/1TfHuU3siHfYmtn0ZPnGnFJrzT8vvJcjZdRkI5dVNVjDqF8u3iIfenylPL6txvhif2EGsfOF4xAA2cLvHTlG3jG%2BWCbnpFu3eKqo1itfsJ0MSxronI2ueDRif2qHo7/hzqLjoIA1JaUZKZKakGCNPtjrkI3DMYbZS4jmN2yb0dix27YROJCyIofjUMFcNYTJkrAeMUGdRDYO/9HLm%2BTHqvBZl/LrxVttg/e711fIyppGM6T7tvrE4Ti0QJOaC0fny5mj8mwtFs0%2B/rJqh3xv4Ub56atqCC/aItcu3Sa36mf3baz0ElLHWwJkFE8elmP7LA5NTzJ98AfVDTRH%2B1lYX7Ce8bY1O2yLsHba/DocMYI7i46DBDVySZno/3RljC6nwwhmbcpxQ7PktBG5kpfsbZgdhz/YbHxybpolNtju5cYVZdbs5uaVZXLXunK5d2OFreuiIGiE8od3nHMc7shWGieDQkt%2BSqYXlNfJ75Zss66oOIhsIfPQ5ipZXt1k39Ncg7XuDsfhjDHqLE7MSbN162TZ71hbLtcqT/x%2B2Ta5bfUO0xePb6u2LTMsOJ91YE0CHY6%2BwJ1Fx0EB3bXYg7FdHUWaFRxrm4on22axOIkIQTqkfmV2qW2nQVc7j4s5DnfQxIMXRu/2plZrKhAdSMlS45lN%2BI8rybGOjA7H4QyCinRWhNJpdMa2MezFGIkkNYYL1WhmP8aRGSl97gzscBxqSFEaD/ZLxI56ZGuV7ckYCbbIoCqLrsFUcjkcsYJLYMdBAdtqsJFreXOoBfq7JxTLUxfPlnvOnSF3nzPDFmtffex4mVOYaXvAdWc0OxyHG2iugaOIcUxjAkqN0obsnSV5z8Sh8rmZI%2BW0ETnhTxyOwxcEFhvbQ3sqsqE4Jdo0s4kE%2B7p9bNow%2BdPJk2RcVqoFXByOwxkNbZ1mRwH0xednjbTtMgIMUR6YlpsuV80fbbxRlOpLeRyxgzuLjoMCGhBcv3S73Lxyh7y0q94cwRHpKdb6eWJOqgk/Ghdct3SrfPjxlfLk9hqpV%2BGIwMNQoKzC4TjcQNdfykzpVEpX1O8fOUZuPnWK/PmUyXLzaVPknnOmy6dnDLdS1dbO12VXc7tn3B2HNQgo3rKqzLZ5YckC8v/HR4%2B1LsFsuXT7GVPl18dPkPdNGirDM5JlV8sbG5g7HIcrllU3ygK1nTY3tEri4MHWQfvqY8fJbWG%2BuOX0KfKbEybIxWMLrTfE1qiso8PRn3Bn8S2GR7ZUyTdfXG9NNp5Sh41W%2B/sCe1Rdt3SbfHfBBvn1a1vl6bKa8Dd7gwzKqpomW2/yi0Vb5PsLN8pVesyPXt4oP3mFhh6bbIH2rat2ypPbauSG5WXyA/2OTZn/u6V6z2LtV8sb1OnkehvVoNhhe9gFqGptt2MYP2PvCTiu1y3Z%2BsaY9/Fbh6M/wabH/1xfLg9sJuveJlPUKTxrVJ5cpAr//NJ8NZSzrRTvznXlto4RHsJbHJqWbJ3wHI7DDWyp9NLOOrlhRZk8W1ZrHXzJLl44ukAuGlMgZ4zIsy7CVKCge9AZOJYEHAkuss0MZaoOx%2BEE7JZH1T67ccV2WVPbZJlEluxcqDxBQyj2IWWv6lfURqKzPF20mzt2G/9QsYVucTj6CwlXKcJ/v6VR19ZpDNgYLgM4XLGxvkUdvlp5XpU1DTferBSUbMeicIt/upqyvmRfQMEvr26UF/X8XIcXBgHZRhoW8D1XXKfKn9%2B8oK8V%2BjmZl%2BB4On2RjVlW1bjX88Cofn5H6LxbG3seB/fEOfaMeR%2B/PdRAie9Hpw2ztQuxwEOqvHhGhzNYJzJMnbNtSic4d9DM6prm8LfdgywhpXFlTW2yROmU9v7V3bT3xzAmk0LQg9I7%2BI/XZn3RHZhrPba1Wv6%2BZpcs1L/hA1ql85vnlX7X17VIghrGmUlDzHjYoO/hG8YZgEYhcM9qPR/8gWENr0SDtS6cq6wxNGau%2B2bBokMFdJydV5QZfte/oHsnHWwPd7APIi4Z97tE6e2xrTX7zPBRHcKWSAmDBxv9Ive7208XWoU%2Bof8mNXar29r1962qj0K8gf5YqPqC7TNuUUcxtJ5xkPESfLm4stHoFl7JT0mS1CGDZXlVk9y3qVJqw/SMYwm9w5N04oa34Ito0HgtI3GI8fKeMavuOByAU80%2BlrECQTHm%2BnAGGb9MpRd6VkNPT2%2BvVXp8c3qhSRM6GxvoGaWx1eoMQs/RoLIEOoRHsM/IMrKdDNdaoXzx0q46a3xDYzRsKPZbhI9Cv20xXoWWR2YmS0VLu223gS2EbgNZqkfQdzim8BGBe3iwOxSmJhl/YIc9o7porfJed2M%2BFHHisBxrsuXoPQa9rgj//ZYGzDbvjoXKrJ7ad8QvaB608LJ5Mdun8gvPrrX9zhyOeMb1J02Sj08bFn7Xv8CRmX/nwvA7hyM%2BQdOspy6eE37X/7ji0eW2NZDDEe/4xtxS%2BcFRY8PvHL2Bl6E6HA6Hw%2BFwOBwOh6ML3Fl0OBwOh8PhcDgcDkcXuLPocDgcDofD4XA4HI4ucGfR4XA4HA6Hw%2BFwOBxd4M6iw%2BFwOBwOh8PhcDi6wJ1Fh8PhcDgcDofD4XB0gTuLDofD4XA4HA6Hw%2BHoAncWHQ6Hw%2BFwOBwOh8PRBe4sDjBShwyW00fkyoWjC%2BSiMaHXBfr3MUOzJTtpSPhXvUNecqIelyWn6fnyUxJlaFqSHFWcJScPz5HEwYPCv3pzzCnMlBOG5ciE7NTwJ7EH4x2dmWJzE4txTMtLl3lFmeF3InN1DpiH4tSk8CdvYHh6spw1Mk%2BK9Lu%2BzKvD4XA4HA6Hw3EowZ3FAcKQQYMkS53B6eqk/PK48fK7EyfKb04IvX51/Hj5wVFjZL46L%2BlDEsJHvDkm5abJt%2BaNtnNN0b%2BPLMqS/51TKr84drxkJvb%2BPJ%2BeMUJ%2BfPRYuXhsYfiT2CNN7/usUXlyrd5Lf48Dh/1DU0rk8zNHhj8R%2BcIRI%2BUT04fLlLy08Cch5CQPMYf%2BhlMmy5yCTBun4%2BADei1JS5KRGcl7XjjpzH9fkabPF8c%2BLyVRcO2TEwZbIIVzDemls88xhamJNqakAQ4QpOvcDNO5YI4SVI70FwiEME8Erfg7ReegQOeQVzBvjCXyGY3Yz2fkeHPwrHN1buGDyDnntT80maTPk2Nzws%2BXZwl98/x6S1fIPwJqjGsgwWihS/g8Q%2B%2BjPwHfca0gmMsc8B59DhgL8wGPMr8j9MXfvZU1jr4B2mW%2BI/mBF/Ia3d5XQP/QNM8XNiDwzvu%2B0BX6gufOOQbyuXP/zEOx8nV/6grOnZmkskCvg54A/IuswtbtDjyzQA874h/uLA4QSpSJcFDuOGu6vFLRIJc/vEzm3bHQXm9/aJlMzU1Xh3GsnD86P3yEo79w6vBcFVrJUtHaHv5EzOhYV9ssz%2B2oC38SwmdnjpAfqSPt6F%2B8e2Kx3H3uDFl42bw9r0cuPEI%2Bo/PfV5w2Ik/%2BePIk%2Be780ZKiynNWfob8/cxp8qUjRklpRkr4V/sGgZvfnjBR7jpnhkxS3hxIUHnwwPkzbY4wZPoLo7NS5c6zp8unZgyXCdlpVqXwi%2BPH22tUeN7epmOJfEZPXzJHvnjEG0EXx8FDSXqSfGX2KOODyDl/6e3zLFDYV0xTOl6gx35aeWqsPmuqW%2B7S5/1lfX5FamC%2BGbCBzx6VJzedNlm%2BOrs0/OnAAIP8VydMMD7v7%2BDi5cp38ACyCCP53NI8%2BeXxE%2BTDqs8BY%2BE5/ee8mfZ8Xrx0rvErlTKOg4/SzFSjv0ie4PX7kyaZbu8rvqh64abTpliwODUhQX56zDh7/87xReFfvDnQF8%2BqLPz49GF7ZOVAgPu/Xufh1tOnmiPXX8Dpe/%2BkoXKbXoeqOHC06ov/vu0Ik1vd4Sv6zP6mepj5dsQ/3FkcIFAqeoE6gsuqGuW6pdtkcWWD7GpusxdOyg9f3mQRM8owETwBvn/kGFNCvO5Rw/VXqqQm5fTeUDivNF/%2BdPLkPefg9U81ED4/q6uBd3xJ9l6/u3LGCBkZIfhQfl9WRo/8zTXHjZdjw8ICQxaHl88QVnz/FxW684uy5IyRefJDdboij%2BX1bTXoLxxTIJeowv/o1GEWmfrg5BI11CfYXDAn71WhRGYv8ri36TFBtPekYTlm9KAsuC6Owdfnlu6JeEWDqG9ywiBpbO%2B0aPsUNaIwApo6OqWtc7cUqOGEg4hjf86ofGnQ3zn6D1fNH2PPf3FFg3zwsZV7Xs/uqJVzdf6vPnZcnzIpPHeimJlEivU/aIiSbSLFCb08T9LgwRYBhaYHOkOQqvcDrZPR6M%2BhcB0qH5ZUNsqWxhabQwyfjXUt0qp88T9qHOOwPr%2BjTv4n/Iye3l5jBgp8PMDTdFgBAwy%2BOFfl9y8Wbd7DE994cb1srG9RnhhvcnJYWnL4iDcHfFAYzphA02QhoO%2BMxN5nFgm%2B5MJLSQNfYUGGFB6FbvsTZM%2Bp%2BClrajUan1WQIVl6/227d5vR/PFpw%2BR8dbz/s6nSntEPVJe/rsfx/AJD2nHggH4vn1Ak3ztytOrwJPnugo17%2BOJ3S7ZZdo/gLkF5aKO3gB%2BQdemJgy2zSMaY96l9qCJCXxSlJBmd9FbH9AeYA3giT19q4vQbuMY8tet2v/66dOqLpUNHDc2Sjt1Q/t5AH79H9QbLo7Ah%2B1L15hg4JFylCP/9lkZdW6f8cfl2aVQHob9BScPbxxWak/fXNbvk4S1V6pjsDn8r0qkMtqWhVapaO2RVTZOsV%2BOMb3HoTlWHic9fVUO6Xp2Wi8YWiPKmlLe0m2DAUUJ537exwoTb7MJMc6D%2BsmqHHKPO37tUuJYqg2J4b21sVQX3uq3Xm6BjqWptlw1qeJC5mKwOE0xdqeflWmQWgrWDvB%2BeniLvnzzUHEqU5KvlDaYQjyzOMmFboce1qEH5qekjbB0lQmRldZOsUUeYc759XJFeI83%2Bfk2NUcYyX49lHDhoS9WJHqfXo5Rjpc7B49tqVDm3ydF6rveps9ipF2McNW0dtraQ6OJOdbR36m9m5mfKF3SusnUemMdFFfU6h812fY4DnPvyCcWm1M8YkScT9br5KtzHZaXIWeqQzCnMUAGYKO06P9sb2%2BSDqnCYG5z7Nj0J47pvY6WNG8M5VsBJ%2BKgaIzi1scBDSpsv7Nw7u9pfQCHjfMAbq5VObl%2B9Ux7dWm00w6ta6RNj7BR1RqD9bTr38A00c/n4IqNbvuOFM7hCnzeAvqFLaAUDjlI7njvPc8GuemlWnoeG3qHnwBgPzsErJSHBggMY1ccprcNbGNKcj4DH0cV8NkTKm0P0DnhGH5oyTM4uzbOg0Inh9b%2BLlc4hP8ZzutIcx3MN/oYeTa%2BqQicAA58HYyD4Mkp5tkzpkM%2Bv0Dkiu4lRDB1UqzzAEDiuJEfepzyJs8Zx0/MyLAgCD4Az9XoEVeYUZdpccd5GnT/4KhrwMNUN71QeuW7pVtmgMugI5YkT9Br3qmyp0zkh0o6R8JvXtspDm6vsGcFzM/MzLFP1l1U77X5jAZ5n5Jrj/gTy4E%2BqK2IF5hPa5HndpnN6x7pylWkNNt8b61pV1rbJycNyZbjKpGql8RXVjVYKeYLS69vGFFp1SkBLiWrEwjvoOQJxH546TJ5TXUCwEpmGPF%2Br5%2BUznjFLGT6m8iY4ntfsgkzJ0ueOTJ2uYyPjTAkaBjJZPX5zhP4GIzmgPUCm4W2qryL5dMigwVKj9AvnQOM4vDhU3OtpSq/IZZzh45XumANoODiWsSUq/SN/P6T3cbp%2BV6i8l6q0y/0jqwGBJ2R9cNxU5b9a1ffoHrIt79HvWKMOP3L%2Baco3O5QnCBZ2Y%2BvKhaMLzUl%2BdGuN3l%2BLnZtx8Exq9LxkajGAb1P59W/VEfDnOOWHk9Q4XrCrbs%2B4%2BhvoefRWrPDP9eUWWIoFsE2QMdhEyOQny2rlH2t37eELdAPPYJbSIXLsAZVPtcobOH7QLzo0oAd%2Bg7zDzgJnq/6HF6DvZ/S8FykPYUNxby/p88POImCNzCGAzTlO1hc6pLKlw%2BhqjI7tvZOKZYuOA9mPXXaS8ii/wZ4JdAXnhY7RewFtzy1UftLxIJexP%2BAb7DbsmvOUl%2BkngbznHPAbMj0YBy/4B/sJeUjgHV3FHOEww4/oKniNYB/3ik5BrpMB5DjAONFd8BSyBd4gUYBthm0XDYKIjIXvnimrMb3AOJlX5BUyJwDl7t%2BcN1r1dqrZnzyvB/X5xBLoZO7b0Xu4sxhGLJ3F6fnpJhgIWxEBg8EidRJiBObCGUKx4ODgJP3smPFmuP9%2B2XYzpHF%2BYFAyYZtVaSGo9uUszizIsCzZCzvq5OeLNstT22tkuRoWY1WYIVgG63gwzjGYceTIIPxSDcHbVu%2BQZhUS/AYBjbAjO4iziFL9g46H32CcI3Qmq4JvUSN0iY4dQcv6gfs3VcmvFm%2BRx7ZVWxR7hhoZOMIoVIQ8Y8HAmKGKGmH25xVlFuk%2BQsd8lyqh3y7Zaor72/PGmBL8%2B5qd8ks938vl9TqNg6wcCmzX8SAUEZJkaG9aWaavHXqfbziKAGGJUY0xQkSMCFhV2HhgDjvUuUUYb1dhj4JnvRvnwbElE4kwdWfx4ALj9afHjjNegL4f2FwZ%2BiKMzaroMLzy1XEj%2BABv4MTBAyhv6L1Inx9OHc93W0ObPR%2BMhZ6cRZ5ntiq2j08broo2x8aAw4PzRkCDjAGOEjR2wrBs%2Bx6lSZky/2Jkwn%2Bb9FzwH5kZ1tp%2B/oiRxnfQDTTN9eFRaBvlT2SVcY/X62TrNZqVOAlA81uO5V4IViArUGpk%2BBaqY4ujeaLyGHSKIYHsIMgzLjvNDA6UIAYRgahZBelWXo2cwCkkwv4RNaqh%2BdD5h9h33F8AxgMfYnCg7MmacG8EuDDg%2BY7ngCOAgwD//W3NrvDRoXKkSTnpKmeSjIcj5Vp/4nB2FnH2qAghMPKJp1bvpaOQy/ABBhjyG/mHkQud8Kx5jiPSU8yRxFmBbgiqYYhCw/tyFkuzUoymLlM%2BgP45lsAfdJCbMkRldq3REroAuoHW0D3B7%2BAFaKdCaX6G0vH71RHE8ITfoJPTR%2BYaf2BcI2dx4qicgVfgY7Iy3JMoFb1zfLE5v2l6nxw7V581ugYdUakygb/hEb5HHqOXXthZa8bzh6eWqEObbUa%2BXVf5qVX5jWtyfbLgJ%2BnvMOoLUkKyBbmAvCBLgmMyUuXJycoP6EWam8GbtXpdDGz0JfPP7zkv971A%2BeJ5ncNdeu/IAfiI4M1/Vb8GBnl/43B2Fnlup%2BhzvHxikTy8pVruUocEWguAnsCm4/nxbB5UPYYsglYJOiB7cfhHZyLvMuz5rq5ptmNwkHpyFlfXNhmP4KTiIEHLo5VPCJicorQN3RFAQC9hHxFYzAjrgUnqHOF0oZOgCzKj0PEH9Hc4khwzRsczuzDD%2BGmNXh8HF9nGkgn4IzNxiOkLbBN%2BTyk0PMff8B364lLlVwJGI3R80B2Bfmi/ZfdueVF1OWOi5Pzj04fben5kNeMgSLJZ55DqNsaJjoK2x%2Br9UZVD5RX3hvwLgB2E3CXQb3aT6iQcQPhknn6XqWPbGtZ7yC0Cq8fq/CHTCC41q0wjGOTOYvzDy1AHACgsDLU6FQRrVPgg0PYFmPXdE4otg4dzhSIHMB9Rfs6F4Yrw2Rf%2Brkbd559Zq87mNitLwPDl2g3tKLm9HZ6dTe3yojpJj6lywyj9mzpnZPeGqnLECLl0XKEpcpQfDiYCAucPxyIxYZAKtjdKY8l6rlXBh5AMfvel59bKj17eFBZqoSYAjIEsT09I0j8Uoz8AAPbhSURBVPMihF6rrDeDBqAUfrFoiwlynASiyAFYCxqpQCJBieMXnl0rlz64VBbomO9WRffxJ1fJRx5facqc%2B/rWi%2BvlGj03mVz%2B7elcjoMDggjDlTdQ0hi03QHlzTOiTJso6cWqyMk04zS98%2BFlcvEDS%2BTXr20154ZyVRb378uvzlJDAmeS4M0janRc%2BfQaO8cnlRYYBxmN8WoQAxbqc16U249f2SSf0N/cvaHCjHCMD2gTg5aoab0aHdA45%2BK3CYMGy3UnTjJlTnaHElKyfjivH31ilXxTaQ1ljEP3iBo2H9Nzcyyf49ARbSaLc8daAifbLLBxozpjP391s1UWvEsNIJxQgk/v%2B%2B9y%2Bcrz69SJaLLIMdlBDFiQoteEZ96udP%2BeR5YbnUeCaPL/zi21uaOcDsMCQ%2BXPp0yWj6lDTbkX63BwRm9dtcOuFwmCGTjYtRHrfx0HBvQFwSHorjugPaD5zz6zxoJoGI1Xzhhuht4zqis%2B8sRK%2BdBjK%2B07DF1oYqwapW8GgiVkYW5ZVWa0yOt61R3oLYIk8A2OFIB3jR/Dv3t5V70ZnDhq0BDrv3B4qfLge%2BQuuuxINTTfrrSL0R2Ayo17lK8uf2SZ0X8oq5Esf1%2B7y469RI%2BF/8ke4ZTOUCP600%2BvNocXx/kvK3fIL1VeQ4u/PmGC5KsMuHrRZjsWGYGBiiHMeIAOz87/b%2BVF5op5RC8FOpGxwdc0jrv5tCnmpFB58uXZI42nqS5g/ScyBP7%2BlDr031uw0YIwPAsqH3CScSQw/h0HjpzkBDlOnz1VWDhAPK9ovKb08JNXNsu7Vc4RGBivThMZunNK8%2BSWlWXy3kdXmM5/Xo8nAPcJlXdvVq6Kg0i1FE7RD1/eaDT1wcdWmE2FfXK%2B0hSOI8A%2Bwun707Lt9ruvPr/eAopWjgzNKN0ity9S3XHX%2Bl2qB1bKZ55ZbXYWNH%2BZ2ljQJUBnJOsL2rpc7wf7hd%2BQubtSaZ/z8/rOSxvMDmQ9IDbSX5XHXqtqkHWqy9CbZEZxPj81Y4Q6ca2mezjuWpXj8AvLmrhHdB3nJnByp9pG7310uXx/4UarxIkEfMQxX59TasEWkgBUdX1u5kgLVA1T%2Bv%2BcvqfhIktHcF5ZEsRyhciqA0f8w53FQwBEwDD2KIeLrAEnuvTktlqLzMKUYzJ7t5AaQ%2B%2BqI8fIc5fOkYVvn2dZleiunkSXECaRQHlimKIAGQ8lCdFOJllHjNHI8%2BHURv8uACWgjIEXDQIC4dgdBkuogyzOW506uJFgvDi1CLgAW9S4qlLh/GYg8sgckklFwNs19LjI0glH/wOjkvI2nkV91PPtCTgvZLHvXFe%2BJ%2BiCgrxZDUYivl9URUVQpCeQlfj3xgo5%2Bq6X5Ro1KFcr/QKCGtADPBaAzDTGHhltnFYyk/eoIiULgZIk44IBrTaDBUIC/sGI/fOK7TZWDFTKbwDRVoIQQaSW6Op5/3lNPv/sWtkUzvaRlSF48fo%2BcnQoYjJHKPJ/rN1pBsnLahRwvnLlC8p681NCRhD3yz32dLZbV%2B2Ud6lBzRg43/%2BpgX6BjulrL6yXh7dUqnNbJUfesdCqBCgfigbRbzK3q/S7nkfs6AvIAhMHxEnrDZCBZNKQxS%2BpIYxRRibx6le3GK0SVDh5RE741z3j6lc3m3P1q8Vbw5%2BIOUDdBc3Iev85IttKNQZZCrIaOGMT9V8qPchEA3iV8z%2BpRuNcdbxYWxZgqRq3gfHP77794gYbR5DNxYB9QB27DWoA9wR4jY7WOLM4j2S9AOejWoJADdUDBiVU5odrRus8AJ/esW6XHHXny3KdGtVPK%2B//Qun/pHtelU88udqCn19Xp/Zt9y%2BR36rTHoCs7D/OnCZ/O2OqTM5NteDOa2rkOw4ctiZQ5QwVGej93gCHH8cdWv2VPieWNSDjcTbhEZYDpCXu2ySGRj7w3xUW7EAHgF1N7ZbZjO5lwLjuVzqlogRArwT3oEmC6cepvmC5D44fx2%2Bub7XAIKXLyNZQkCRk0xHQhy%2BwucCLeg8EI0/516t70Sz6iiBKd2sFAZlL7D/Ox1iw1cCzei/oUBITF48tkJGZIZ1JgIplBj0FOb67YKPNBeci64iumHfnQrlG%2BQPHlCAVgUkc8imqG5hjHN8/LS/bo2sdhwbcWRwAoJhYp4FTQhkPRnJ3oIyENRgsZMJYgP0jRQC2McIBucA5iO7uC0S9iAJdd9IkK6m5QRn2M8%2BskXs2lO9lFAOUamB8B%2BAtn3AtHDf%2B3vsXYkKK30XeUtRpDDS%2B%2Bfd5M219B%2BW2RMwond2pinlfgGC53%2BhzBrIxcgYw7tW9Db/bG5Se/OaECXLvuTPsb0qk7tG/bzp1sho4IYH5y%2BMmWFc7R%2BwAXXX3fLsD60IovyFwgOIJaIC1RmQaKSnCYSTY0hMwPKF9gg0EB8jsQZc3Kh0QIY4MPnTsDilisnoEP6B1DITyljb7Hdcapk4b56MELgg2YGxC4/yWoA4lTe16Lr6vVyUc8BnHMW5kw5eU7jA02TqGNSmhcrzuQYkQCphy1KZ25EGoFIh7wlCh3JDvQZNek897ml7mDmedLCcgIk1pI%2BfkGJxPeJSxRxskbPdDafdLasjQtOv13jxEx5si9OSVL%2BzfNwdZAYJ5PEcczECWswaRkjBKiKHBNwPPmHPAYzRFu/ucGfLd%2BWPkaDVyIwHdYnSz7CAAZXYEfILrUP2CwQkvBOD8ZNv0cCmJaMyzrbFNdkSUunEc45iam2adKeFPuvRSYtoTkpXnkQ8Y7BxPNhFAss%2BpYcz4qBIgwAqVwofQfrTOA9A%2Bx8M3VM206O%2B4D%2BRLbkqCfQff8op0GKgQ%2BPZLG8yh/P7CTaZTWFNMuazjwIFMDMmY3skZSiApx4QeeE7QAstNkOnIOZbo9GSLBYCekLPQI4G/606aKP9Su%2BGW06ZYED0S6Iunt9cqjYdonusSROEcNEnCvqOJDnSDvIX2%2BK5Sz79G9RnrFoPMPeXN2I2MF%2BA0oid40SgGWwa%2B%2BJnyR2FKz9tkDM%2BgXDXRKl8YS2D3MQdLKhvsOmQW0WfoOGg8mKvuEJIvBLSY15BuoW8E4yOI%2B7zqQcbNe7qQw8N/XLZddUqr3avj0IE7iwMAFCsRGxiSsi/KF7oDZUCsTyMSDMMi6CJ/SrkBWQXOQ2ODSEXcHSgL4FwoO9bfUZZ674YKWVXTbOUckcDApp49ElwPGWTCtqPDnNPo0leyCkTnKCXqDmQcKddh3QhHEnn73ZKtVk5LNA2F3RM4I2t00nVc0Q4ApW/MTU8ZzGggBHkGCGzKZBdX1Fvkm2u06Ngpz0PZB%2BWujv4Hjj1lyNBzoCSjQRaYNaPQDiWk/K5dn1fISHuD5nBkUITQCQ7ovoAz9e15o%2BW7R44xAwCSplECRkGkQwRlsfYvkrT5E1pHObPGiqYzkGB1S8eeY6FJlDoOHDwMz/AVxkQkyMqRDWEcrLnFMKCUisY2EbfWBVybO2QeIn%2BGgmZeEsPfAwySffEI0e5PTR9uTTvoVMfaDrbCoLybUjvKD9mChHWZgXFEmWSoU2eBZUFZb7rcsqGOgwHohv/21eUTZwu%2BAawvhMZ4/oFxGYB1j3zU095nkUBGEyz7kj5/6IDMCJkPgiWRQHdQPRKZ%2BQyCKUEAk395z1KKSHBvjDFSj2BkBs4dH1NOC1%2BwToylBmRLeRGk6QnwvPGinlsvEYHXTd4zDziUBGwBhuvev3sDGO00IfmO0jjrssjMMKavzRll6znJYr5D/6XcjmxQALKwrMVHt5G1YW0Z%2BpcAsePAAM1Abzjv%2B3Lw0A/BEh1okF9GOynQGsECftPzmUJA5l0xqVhYakBpNXwHLZKtbA3TbAD4jKAE9hbgumTyoEmqBYJ1xvBp5IjgE%2B7N5Hp4QBbU1PNzToAORFchd9kXm3JXnLLlVU3Ge3trgjfAdU1nKh8yFs4L0JVcE52EzkQ%2BME3R8iMaF6ueonyXZlTo0S8cMcr0F%2Bt6ydx/duZIqxqgcRW6g3W9D22u3MPfjkMH7iwOAIi%2B0AwGhmGhMU5cZKQXJ%2B0cNcYuVQON73CCWAOC8iMqiaAI/Q7HK9e%2B39bQakpwX2BvR/BkWY2V0rFekpI4oqvRWUkymkRmiQgjsFiPNSYrxTIXRGYpnUCY0fCGZiIIPTpQUpNObT1txbsD94aDTJTvEXXIGAfdYCkV5IXB3RMQoqw9wMHDQAUIPoQQhi3Csred5liPhrOMM8gxf12z0xr1IPi5Bs05vrdw457yJUf/g%2BdLhJNni2HVHShTJkNOEwKMYpQcBkPIUH6DhqFnaAMFu7cq3hu24H5olhnFZNsJIFy/dLutAYNHg8gr4Oysu4jkFP7GAcQAIDoMT%2BtbyUt5YzNmvreotr7nfN1FVKH7Y9SQRPGfNjzXIsuU3TEWOhfv6x5Q6HzLPESOjXVubIOAgb5vlf8GWONJAwKaFjB%2BDA86OdLEh/kkE0TJLTKI9/A%2BBjNr5JAnNNJ6fNve6yAdBwb0BXIXY6w78MzZXgkHhs6pZFsIHPD8o51C5C8fvZkRyDo91lLRIIW1SI9srbLSy1tX0yxsbxlLpjL6OtA89ANPA2ie9%2BiGSMATfN4dTwD0Cg1J/keNTUpaKZdDLvMioNMToHgCi%2BgoPX0EBtk8Mg/IhkgHtyeQlSdTOzM/XR3F9FAWSMdLcw9KzwnIkJUicEvmhGZrfAfPA3iIEj30M84pssFxYCBg/YLKRegOuyFYkx0JbCqqhujaTCAOmjQ5uTdBWFaM5wINvpmc5FqfmDZc3qcOI%2Bf514Zyo8WfvbrZlsdEApagCRidcgG/D21hEdqWCzrardeETyNHxO9YjtE10PEGJqrdRkCP/hGMnyUHbKlDQ7%2BQExj%2BYRSoLAnpzMFmrwUVK5wDnh%2BitI4OC/j2zUBjIJrj4KxSdTJR/2XZA/PNfM4sSDe%2B4LP5xZlms/J7HF1KXgnyY7thEzIGR/zCn84AgZT/K%2BX15oR9Y26pdTsNHCa6pt106hTbu4pOoVcv2mLODOUKrIOgkxS/o1sdXdxwfHi9GWpa1ZlU2VCcGroOr09OV%2BOU7KYKCZR2ILTobIcThjKkLIH26XSpw7G6QR0pHC3WU504LNsWYsPwNE84Qo0VMpw4t90BGYSCRkYFTXaIQF2mAh0DFaFvxoe%2BiP4itIjsouD1I1tfgINK10iOxYglyoeDgcCkUU1vgZNNl7JqnVcyjWREiRwS/erJeHH0H3jerG3guY5XZRhdKgcd0Ezii%2BrYfWf%2BaIuKYkhnqUMEH0EfAMeRTcVRRGSLMQp7AmupaNaEAqUpzGeeXmNdWHHecIYCfgAoeZxYAiiBMYyRQWCFkjvWMOLschx8w3gBtIvxSKdGSuygr2hAvzRg4J4wPhjHE9tqzAEMztMTiIrjEMI7RKsZJ%2BPjnDhzRLejMzo9gcYirK0igEOzqkseWCqXPbRUHtpSaV0HL3pgib1u0t%2BQLb1kTIE9C57bt17cYJkUx8EFuoJyYowreCPS2IUGee601Ke0nw6M7UpDFSrraVgRMgBD2Read5BlQaby7HoCshkZi0xfXd0kX39hg1z51GpbG4WxR3YiEtA7PMErAHKUzo0Y9UhSOodyfYIRAeBvaBQjBL7pDtzpWB0HzhZNar6qPMpn3DMbpvcEeB7eZ67g2SAIydTRGAX5gOyAZ98MVBg8qPRPQyieA%2Bvd4IEvPrvWxk2jnk8%2BuVo%2B9sQqaz7HHsBkHdFNAIeGeyXQBB9S6uc4MFBtwbr0xvbdlvVmyUCkswFN4ijS7Rf7iv2RoQUcd%2BiU54GjBG9QEooNAj3goO0LVDXBG3SU/5I%2Bf8Zg6yeVnuCxSHB%2BHKWgaQ7XZC0x44CXCExSqYL%2BYCkBY%2BE7aHtidmh7l8hgZSTgYxw0Kk/Y45Z1hZyPa2A/9QT0D80LLVuuc8Y8AeQKNhVVbqxJDlXqvDm47k0r6BLfIL9estWaUrHunfWK2KzoD5rosEyE8lhsRJZW8CJLz3X5lwxpTwFiR3zAncUBAsxFdykW/6JUbj99qiy8bJ69/nnOdFsn9OXn1spd60Jt6WmCcdq/Flk06qdHj7Pf3XHWNCv3%2BvErm7t0NewO312w0SKz/6MGRXAtFlojZDAuEEBBpGlDfbNFX9lE/4VL58qHpwyTf6kh%2BbUX1lnZBOP/7oINtj3Bt9RZe%2B6SObbOiy6WdAb718YKO080atraTcBgCBMZYwyPvW22XDl9uGVQOC8CBAeV5iGsJaAL4%2B1nTDWHmnugO9clYwvs2AcvmGmR3P99Yb1lNfqCQFEsrWoyAYmg5bpkdZhnR2zBWgeMQUra6Kb22Vkjwt%2BEwF5qdOrsUPq7/OFl1mwAowEj4e1jC/coawIqGM2UgZEN6ckQDQDN4/QFx5NpJxJKphGaCIAip2sv/EM3RDLv7HNKNoduvGTcGRO2IB3faF0OyLazpxU8TPdT6Ksn4ORRphaUWVNJ8P5JoRbsPYEM6HY1PuYXhvbjwtgg0EMpEIbIHSpDetPoKQAZEYyJwGBgPBjbzZ178wRt2/9H5cJ/t1WrwbKiVwErR9/BGlDkJYGtl94%2B18ogA5Bpv%2Ba48Vahgux8VB36BjUyX1RZyhZGlE%2BOTE%2BxLR7o3kl2DEO3t9lfDOvSrDfWE1J%2BTHlqNAg4UsodgEwnFSBkm/EWaQKDQQ5dAuiZ8RDspOkNmfx9AT4MSkaR26y931fre3iN5lEY23RLvnhcgX3OdbkHHLa%2BLDHIUPpnPz6CiEHghXPBJzTywSkGGP%2BUYtP9FGcb0Hzqi0eMsK6ti8obrGTRceAgO76gvM466hLMZr10ACqyyLohA5/YXi13qgzEqcdpYUsHSiORc2ydgQOHHYANhfP5ZkAUE/QIZDTbwtx2xjTjsUjgRJLVHJsdogP0BWvRoUmy89hi6Av0A2X%2B2F/YgheMzrds/qNbqyxI0hPgg0i9xf1/TnXmCB0H33WHjWrXraxptEAO1TRB9pvy7rePLzSdScCwL51KaRYFj6%2BtabYlGQRrCXoShA9wldptx9/9isy7Y%2BGe1z/0mbD0CP1ER1b0vyN%2BMeh170JggDkgYKJLsQSGLu35iToGIKlFpgKlEx0BRvkH0VmiYBiBdL0jWwfjk53AycNwQFhhTCDUyMih6DA0iEgFCGroMYSJOC9Uxc3aJLIgDCmylTTXIVobAEWJAYBzF4AyNRwvsqBErdjbkcgqn0Ua7bSNJqrMbyBAxsb3nJOZwADmnsjIcL8Yrji1GAGMn3UiQbQYcH9ck3umJAgBTlSc0tSeonPME4oCIxdBhZDj3FyLiHN3DiORde6XbmFE/Hk%2BscxC8vxwkoP1Sf0NthehVDiWQNHTaIbIMPQUAFrGwMOwRNFCt5RGs30GLbz5LU8C2mDPLIISZMco56ZMkmdKBmC2nvf6kybZnp90TqVT6M%2BOHW80AK1Y1lBpFkP3nNJ8uXnFDttfi02WUeb8Dr6BZ6AhyuFuWbXD6AH6wHD%2BxtzRVopNSQ8KnbH%2BYGGo9fhZI3PlA5NL9HP97b8WqfH5uh13hn7%2ByenDjb6gW6LgXIsI82dmDreyVJpSYYT/8KixppAp5SYTypgpBw3mDPpg/Scd%2B4h%2Bw1vfO3KM7SVJafXPXt1ix3cH5uq0EXmyrKpBvvPSRrtvNlJHHtGaHWDcfEp/R6aXroKUpUeerUyv95vXtsnaulC5fX%2BD58lWH7EAxv78OxeG38UGlP1iDPJsyEwHZWLIQLLqrBNl/TeddylxQ45dOX3EnvVxlGUSBGHt3N3rK%2Bx3BCKev3SuZewItJGd%2BaoakFRoUJJPWSuGK6V0QcdejG06KlJK9vul26RJaRSDGJ5jvVSQmYH%2ByC6QgWadI50QPzK1xLbiQL4i4ylVY0smxo7DNU91259OnmSbeN%2BgtE6gkEAO20YR/KEhTdBZMnSPacan/91WJV9%2Bbp386Oixth4eeiOw%2BbNXN5luxTGghJVjuS7ZVq5734ZK%2B/uPek1A8JY57A4EiT6gTudZo/Kt8yOBTTp5X3PcONvKAxkAv3Lf6J7vHznW5ANNfJARBCFt7jeE9iLc19r8gwW6fz518Zzwu/7HFY8ut2cZK%2BAkkblDRpOdwp4I9AU2AzxCIAIaxJ7AgSJYgeyiOotN%2BHlG0CyyG5pfp5/RRIm1pThrbL1BlRe6h3uj7PTdE4bKB6YMNWeI0lOOx0lif0Sqquh8vb62RR66cFZo7bv%2BDhsBGU8gjy03CP5ALxxzxcShJrdpOsMY9X%2BTMTevLFO902y68GPq%2BBJopNwVOwcdgUz%2BxrxS4y/OhQ2ZotdgeySqCj799Bqzu96j5//E9GFW7fWdBRus6yqVCAT74DvGRknqrpY2o21%2BR8D1Pcp39MIgodETcHA5N13A2UIEXUXWELuO8dPtuCf89oSJlhUmcPW5Z9aEP40NyDZTieHoPdxZDGOgnEWHoy94KziLgA3hiRhH3icKGSM32qDDMMZBw5hU%2B6HL7zBKOReOPQqXEmvWTBBlxXnDaXqbOpwYG0RJMTJwBl%2BpqLfgDFmRSjUCJqlhEpTKBE2WUPB0fMMADDJxGAQY0GRlMEbIYqC02YYCwwLFinHAWhX2KgwMbKLbGOyMF4MBrFGnlyg0hsFK/ZeW6TiGrGuk9AkDGMcZB4JuczjawbGUOmGwL9WxcQUyQnRsxbjgmJ4EPyW0OKScm2wpGX3eU0XwdFlI%2BbPOk4xqdAfAAGzfQXfjoBNef%2BNwdxYBmUXWqPMsgvIxAPncZ%2B32m/bQICTAcyRrEQQX8S95JgRc%2BB0GJ4ZosCk/7/falF%2BvQ7AuyJABgg8Y6ThEVLVgHGKwU/5McI9jAEY78xRslQEoe5uj44mkGZw6DGpomuZo0Dn0/kp5gwUhuA/4lSBHsOUM98FyCJxYSlEJhPxnU4XdLzRJwBGjn2wSoJsw/AY4FgP5KaVjsiiUiOMQA0qou9sOBhCIhbcwwtlsn/tjGwacFHgYAz4A88M1KVek2yVApvCMcIBjVbVyuDuLAZD/OIEETSJB1ckipWv6EAQgcAe9Uo0UIPp3ZJ6RxTiTzyidoBvYCB8ZTxCewAF0SrAPsISFIAqfI1PJHtN8CScNuQ/N8FucMpbusLQoyEQzHngJ2iWoAOi0%2BvwOldtK4ziB8A3ZUK7xnH4eBBrgNfQMvIPMRxfhvCJv4TH0H/KXdbTcE2CfXvgL55c1yVa2qscSYEHPwY%2BAgCpr0wmeord6AvdG4IilIGQIAbTPml4qIvZlT%2BPkl6qzSRXbQ5tjuym/O4t9hzuLYWBYHXXXy5YN8BlxxCveKs6iw9EXvBWcRYejL3irOIsOR1/hzmLf8UZ48i0OMhIY4JS1OBzxCCgz1uRp1wz96XA4FPBgkL11OBwhuK5wOA5fuLMYBhPBIvaeFgY7HAMNaJM1SrGkUEpcgoX8Docj1BCJLUmCxhIOhyO0NzN7VzocjsMPztlhDFZDvCgt0YxjhyMeQTMkmvfE0kZlDV/k%2BiiHI97AFjzBep9YgDU%2BrIvzuKIjXgE/xHorAvRE9D6aDke8gXWarLF29A3O2WHQRZOFxllORI44Bcqf5ijB9iaxQEFqkjWYcDjiFcPTU6wVfKxAQJHmJh5EccQrMIjpoBtLoCeCZkoOR7yC5j90f3X0Da7twkDxn1iSY50MHY54BN066UIZy/K3kWpwsD2FwxGvGJWZLDnJsQvyUQpO90W2F3I44hE5agzTnTOWYGuWWF/T4egrhqUlu52/H3BnMQyixScNy7X24bS7dzjiCWyzQAt79g2LZRMm9pPECHA44hGIarYoiGXJXVriYJlXmBHT0leHo7dAOxSkDIm53CaoSHDR4YhXYDoRXCz2DHif4douDIwOuqEeq8Y4G/g6HPEE9idjo3TKfGK5ZnF8VpptRO9wxBvIsNNo5uThOXv294sFKHk9eXhuTEtfHY7egkYzyOxjS7LDn8QGLJEY57aTI06B3YSNf8zQLJnkNk2f4c5iBCCmt43ON%2BODmn%2BHIx7AhtCnqHF6qtJlrDswkkWZnp9um9h7p2BHPIFN2c8YmWt7j9JwJlaADyhjgifGZYU2inc44gXT8zJkXlGmlKQlhT%2BJDTKUH%2BcUZshpqqscjngDfUmozJqm/IFN5egb3FmMwvjsNDMCThvhAs8xsMAvZC3tScNy5MRh2TJqAMpBcU4nqzF%2B%2BYRiK9FOcofREQegLLs0I0WumDjU6DKWgQyuxFKFS8cWytHFWZLlgUVHHAC6JGhyyvAcpcvsmHd2R1ccUZApl44rtCYiHlx0xAugxaLURHm7yuzx2am%2B1Gw/4M5iFJISBskxQ7PNOA4MgVhncxwOZBmKf35xplw2vkhmFWQMmIBD8Z86IlfOH50vRfp3LLuxOhzRQB6Pyki2rOLpSpdZSQPTaGZuUaacNSpPDeQMN4wdAwqoD%2BfwuJJs44lJOQOT8R6pfEmg/Rzlixy1nZwtHAMN7JXi1CQ5Y0Se8UahN7fZLyRcpQj/7QiD/exGZKTYVhpra5qlvr1T2nbvlt2vh3/gcPQT8MMol8hOHiLT8tLlS0eMNOVbpMJuIEGGc3ZhpiypbJSdzW3S1vm6ODs4Yg2cMvYaPX90gXxmxgjrEDxoAIMXlu3Xyy9Wvmjq6JTXnSkcMQbGMPvGjclKlW/PGyNHFWcN6D5yrAubUZAhC8vrpaq1Q9rVcHK2cAwECCyyD%2B9Jw3PkG3NHm7z2rOL%2BwZ3FHpCoBntJepJcoEYJ2NHUJuUt7fa3w9FfQMnPVEX7nolDzVGcX5RlyncgDWKAQcI4ZhVkSqdaxJsbWqSurTP8rcMRG4zOTJVPq5P44akl1tRmoLPcZHMoh52UmyZLqxqlrr1TOjyq6Igh8lISrfLjF8eNt4AejZcGki0S1BinImt%2BYZbUt3fItsZWaVC%2BcDhiDbZyef/kofKFWSOt/JRgo7uK%2B4dBryvCfzt6wPq6ZllZ3STLqhtlaWWjLKlq2JNxdDgOBKy9YrH1iPRkGZ%2BVKrMKM2RWfoZMVuOTTZXjbePv1s7dsrG%2BRZ7eXiP3bKiQBzdXmfPocPQXQutNkuTEkmw5tzTflgkQIY6X9bM4hzVtHbJMncW71pXL/ZsqZZ3qDIejP0EF1Dx1Dk%2BnvG5krswuyLTgRbwkTlpUV6ytbZbHt1XLv9ZXyH/1X4ejv0E2MTspQU4YliPnjMqX41Vv4CjGeg3v4QZ3FnsJBF95c7tsVCNgY0OLbG9ok%2BZOdxZ7wvLly%2BWOO%2B60v9/73vfK2LFj7G/H3sAQJmNXkJJom8WOzkqx%2BvqBLCPqDXY2tVmZ0cv6WlLZIOvUKNiun9W0dphD6XDsLzB2KcWm4yjrZek4yrpAWp7PyA91sovXdeQLd4V4YlFFvayobpJN9S1S2dIuTR27PajiOCBQPpeZNMTWXA1LTzJeOLIoS%2BaqwzhW9QaBx3jEloZWeXlXnfJFgwXaN6gNVaa6oratw5YzOBwHAjRBEHTHdsKGmqm8cUxxlvV6INDoa8oPHO4sOvoFd9xxh7zjHe%2Bwvx955BE5/fTT7W/H4YdnymrkFTUEiCLvaG6T5o5O8UUq3ePZ556V6uoaGTlihMyaNSv8qSMSlJaSNRmekWwbi8/Iy5AZBemSF8ON9w8UZN%2BfK6u10lTK8OrUMPby1O6xcdMmWbp0qZXan3DCCZKVmRn%2BxhGJZOUJAig0kaEK5Ug1huOx%2BqQnQP9Pbq%2BxQMr6uhbZpbqipcMDi92hvaNDnn76aWlqapJx48bJlMmTw984ooHcQF%2Bwfn206gv2%2B5xTlGlrFb055cGDO4uOfoE7iw5HVxx77LHy/PPPywc/%2BEG58cYbw586HG9dXHvttXLllVdKYmKiLFy4UGbOnBn%2BxuF4a6KmpkbmzZsn69atk69%2B9avyk5/8JPyNwzEw8CJeh8PhcDgcDofD4XB0gTuLDofD4XA4HA6Hw%2BHoAncWHQ6Hw%2BFwOBwOh8PRBe4sOhwOh8PhcDgcDoejC9xZdDgcDofD4XA4HA5HF7iz6HA4HA6Hw%2BFwOByOLnBn0eFwOBwOh8PhcDgcXeDOosPhcDgcDofD4XA4usCdRYfD4XA4HA6Hw%2BFwdIE7iw6Hw%2BFwOBwOh8Ph6AJ3Fh0Oh8PhcDgcDofD0QXuLDocDofD4XA4HA6HowvcWXQ4HA6Hw%2BFwOBwORxe4s%2BhwOBwOh8PhcDgcji5wZ9HhcDgcDofD4XA4HF3gzqLD4XA4HA6Hw%2BFwOLrAnUWHw%2BFwOBwOh8PhcHTBoNcV4b8d%2B0DH7telqqVd1tQ2y%2BaGFmnu2C1NHZ362i1t%2Bp1jbyx7%2BSX5%2Bx%2Butb8/%2BPmvyNgp0%2Bxvx95ITRgs2UlDJD9liOQlJ8rUvHT9d4gkDB4U/kX8AsnRtnu3rFWe2FDfIpXKHy3KDw3tndLauVt2h3/neAN//O7XZcva1TLnhFPk4o9%2BKvypIxIJSvqZiQmSl5Io%2BcoTozJTZFxWiiQprxwKaFd9UN7cZnyxSXVFW%2Bfr0qi6Ap3Bd4698dKjD8p9t9wgCQkJ8onv/UyKR5WGv3FEIm3IYMkxXaF8oa9pqiuykhKUX%2BJfV0D2LZ2dsqamWTaqrqhuVV2hOgJdAX%2B4rtgbLY2Ncv23vyJVu3bKCeddJGe%2B64rwN45ooBUywvoC22l4erJMzk0zfRH/nHHowJ3FfYCJ6VQpt72xVZV%2BqyyvapQXdtTKa/pvQ3uH1LR2Sl1bhwk9RxQa60R2bQ39PVSVf2p66G/HXsAoLkpNkuEZyTI8LUlOHJYj0/MzZKwax8NU6MUjOlVk1LZ1yhY1hDfVtcjzyhOvVjTY%2B0Y1iHEaMYz5nSMKa18TaaoXyS0SGTkh/KEjEkMGDzIn0XhCeWCGGsXHDM2WcdmpMkb5IiUOnUYonYDiFtUTm9UYXlLZIC/uqrN/4YVq1RP1yjMEURxRqCwT2bZe/1DTbuIskRTXFd2BoGKx6gh4YqS%2BTh6eo0ZxuozNTpFi1SHxCHRAVUuH6YYNqiueVV2xuKJeyprazFGsau1Q/ug0Z9IRgc4OkTWLRdpaRAqHi5SMDn/hiAZxdQLt2EuBo3hCSY7qixTVF6lmYzkOHO4s9gBmpXX3btmpQu3va3fJvzZUmJAjk%2Bhw9CeOVsP4vROL5fIJxRYxw3iOlwhZoPxf3Fknt6/ZKQ9vqVLHscOCKg5Hf6EgJVEuGVcon5w%2BXCao04jDODhOMipB1mRHY5vcsmqH/HtTpayobjQn0eHoT5wyPFc%2BMHmoXDimQDKGJFhFSjzpiormdnlie438Q22o/26tUgdxt/KL6wpH/yJlyGC5ZGyhfGzqMJlTmCmp%2Bv5QyMDHM9xZ7AFEgRdVNsj3F2yUpdUNZiBTcuez5ehvJKshXJSaKPOLsuQbc0fLpNxUSVdDIB5ACdGNK8rkznW7ZHNDqxnErvwd/Q0UPWV449VR/IgaAG8bUyjD0uMjm1Ld2mHZ9e8v3Cjr61qkpq1D2jtVV4S/dzj6CxjFJVSklOTIV%2BeUWkUK%2BmOgQQBlfV2zXLtkq/xnU6Vsa2yzgIrHFB2xAG5hemKCjM5MkXdNKLbgO8sZHPuPhKsU4b8dYaD8iYZdvWiLLNhVZ%2B99rYkjViAi29S%2BW3Y2t8mq2iYZkZ4sBeo8DmT5HfS/VZ3DX7%2B21bLs69VpZH2ic4UjFoDOWBteo7J4c2Or0uNuGUeJUdKQ0A8GCBUt7fLg5ir5rRrFlGLXtXdYOarDEQtAa5Rz7mxul9U1TVZ2l5s8ZEDX97IsZ11ts/zs1c3y0JZqK0HlM%2BcKRyyBvmC5zDbVFwTwJuWkmQMZLxUphxrcWYwCBPbCzjr565qd8oAaATSx8WIiR6wBzTWrgt1S32r/Dk1LlpEZKQNSSoHziuN6%2B%2Bqdcse6cosYe/DEMRDoUFpkTWxlS4dQcDenINOU/0Do/zbly8e31cjf1u6yf90gdgwEEMU0UKLSAxocmZmi%2BiJpQIxiliPQ3%2BE21RV3qq7AUIdnHY6BALYLyZ5dze1m20/PS7cKlUHuMPYZ8dcpYICxtaFFHtlSJf/dWm3NCFzMOQYK6FiUP5m8BzZXyqb6lvA3sUUdJdnlDXLrqp3WvMMzJ46BBIEKGsfctnqHZfNokDEQoCT7fuXLZ8pqvHGNY0CBSCbD%2BI%2B1O%2BVRtV%2B2qpM2EKBhzQLWsytv7mpuM2Pd4RhIYK%2BQ6Wb5DL0WatSecfQdnlmMAHLtHjXM71lfLitrmsKfHjzQtSkjKUEyE4dI0uDB5ojGszANrRNKsC5slLXE%2B3h7CyJLWXpPqQkJ0rL7wIy8TH2eWfo8aULT2tk/c4Owa%2Bl43WhnflFmTKNi3BHNOm5V5f/4tmqLzh1sJOrcUR7CPCYqX3C/8U5lNB6Chhjv4eAowOuUdNI5bgiySfl8f8k5SZ%2BnnUvplewGs3OwyYbzMe%2BVapyytjc3JTGmjT0Qg1Sf3Lex0tYp9hcoKbTSKb27eM/mIwNzkhJtvAz1cMgoQcvpKpfgd3idZ7C/d8UyAngijeepfNEfz5NzNquuKExNlJn56THXFS%2BX18tfV%2B%2BUp8tqDxrPG10lh%2BgKmXQo2yA8DWgJ%2BYgNCA4Hmwo%2BYeuKFLUXUYcHck80o8HmxPZkW5WDMTucA9uFwDdbztA1NYascVjAM4thINiIzD26tVqWVR98RxHkqsD7/pFj5ZELZ8mvjh8vx5dkh7%2BJT4zMSJaPTRsmz1wyR246dbKcOiI3/M2hjfdNLpGHLpgld58z3TrIHQh4no9dNFt%2BeNTY8Cf9g2XqsD2rCphyiljqFgzylcoPNCnoL%2BOPbmXM41MXzza%2BoElDvOPLs0fJkzbeCebsHuqgWczVx44zWv7JMWPlyOKs8Dd9x1FDs%2BUPJ0%2BSBW%2BfJ5%2BbNdKa0vQHWI9y38YK28%2BwSWV3rIAhVN/eYVnFdXXN4U8PPiCr28%2BYKg9feIR8bPqw8KfxiwnZaXL3uTPk%2BUvnWnfOwwHQ8s%2BOUb542xHyM%2BUPmsnsDwjGnFOaL7ecNkX%2Bdc4M%2BfSM4eFvDj5eraiXl3bWSUVLR/iT2IA17K9VNsh/t1WHPzk4oHTwHp2zF5Suzh6VF/700AQNib4%2Bt1QeOH%2BmXH/SRDlleE74m0MbdHB/SeX9fcr/xwzdf90BLh1bpPMzS/6rco9y6oMFlg1g369Qe4a15Y6%2BwZ3FMDCKny6rUcOjyZzG/gDKn83XiWoUpibZhuzxDIzg7OQh5jSyvxMZucMBWYkJ9gzYl%2BdA13WwOTINaPi3P4Ggo7To1fL6mEYi19Q0yUK9Jo08%2BuuydO%2BDL0ZkpBhfkNmKd5DxGZHOeMloHfrOIi33aaJkzZSUlg%2BkoyIZFPZ9o/sc89RfzjRdeBtVVmMc72hqC3/a/yA6zTKFjXUt/byV0iBbq4ysItIe70hKGCTDVE%2BgL8ieHA6Alo0v9J7gC/hkf5Gu%2BhPjl8BMjvJFfwFbhhLppeq4xRLLqhpkUUWDlaIeTFDVxJyNUv0QL13B9xfoCjJw2B7sr5x6iN9PAPgEvue%2ByAweCJAdPG/22SWrfLCA%2BUIPkqVKpyyncfQNh4f1fxCAgKWxTXlzuxkhDke8oVJpk%2B1cYuksUmJHGaqvU3TEK5ZWNloDpliBYCJldjROcF3hiEeUNbbJcpXbscTKmmbL8ruucMQzViudbmkYmDW9hzLcWQyDWv/lVY1SH8NyJoejLyBiu6yq6aCtBekNtje2umB1xDUoK2Lz71iBhjqLB7CxjsPxZqC5zCo1imMJsjXb1Ul1OOIZLB1wOu073FkMg2jYBhV2pKkHCpR%2BUW5ESSMlL7woWWBRPAuIoxPyvKdEI0u/53fBMRzPeaJLycjoUzPPd3nJe/%2BecjEadrC24kAS/zTICJ0rwUoT%2BDu4nzz9m%2B85f/S98i/XDxZ9R4JSBBY7c67oMVOy0NN4OYZyny7XiJqXAJyHa9n8hI/hxd981t0ziCWqW9tlZU1jTLMZlJ/ujGGJXzSgWcqfI5%2BjPROlX559d2UqVBZDX9HPnmPgpWga5xQ0T4i%2BBn9DY3x3IOUwVs6t9MM5oT3oEl4IrsM14BWuQQlPwDPB9bnP7viSEmr4iXMF4%2BZf7oPzMQ/R4BqcLzgmuEaIL3u%2BR86Xo/cQXIfXvp5BLLGmVp1F5Y1YwfaRU4Ojvxpa9QbMOSV5PLvgeQTPEnrtCXwX0FdwHO%2BhyejnCN3yfKN/z9/QAsccyKM3HtXzwBucK1Lucg3e8xtoP/pe%2BRte4btoIKcjz8WLv5H9nK%2B7IXNMiJfeOIZrQPc9GUnwX3c6JpjP7sYWK5Sr3GZJTSxBdr%2BiJb6M8CH6DHjuPNfIZxTIyO6WoRhvhek%2BoIXgGJ5rJDgaPuEa/D64Bi9ojmOM5g6AFBgn5w7GzLUi7wVaZwwBv0Z%2BxzHdlYXqT21cHBs9Zs7PuaLBR5H6Kfi92acJPd8gcxwtdyLH1s2l%2BhUsHyhr8gB4XzHoddreOWw/oFP/tUg29OMecqxv%2BsVx4%2BXMkXmyYFe9XL90mzUOCXDx2EJ5z8RimZCdagwLD1EW%2B9KuOvnn%2BnJ5envNXt0oMaJPKMmRS8cVytzCTGNAvqb5wqqaJrl55Q65P%2BL8MOl5o/PlvNJ8GZOZar%2BHUZs7dkuVGltE6NlIly0aKMtlHFdMGipfOmKkLK5skGsWbZG71pWHz9Y9rtH7Yyxc/5nttfI/U0pkeHqSCstBqsDa5MUddfLNlzbIOaPy5KIxhTK7IEMFRoItOGa9w62rdth6oEhMyU2Ts/T3F4wu0HGnmPBhzDA8pcM/WLix27VDX9RxnzEiT0ZmJlvn0%2BAarBuZlZ9h5WTz7lgotW2hNRYIrsk5afKVOaUyUe%2Bd9Zo8A7LNNHm5Tp/XK%2BX1e61pvfX0qTafD26ulHc/sjz8af%2BAsUzSuaBxCEohFvjmi%2BvlV69ttbVh/YUTh%2BXIR6aWyNuUHp5SGv/yc%2Bus9BVAo/DEmfocx2SH1qwQ2MGJfWxbte3lxTOJBAqQZkycc7TSS4o%2Be0p32Z/vlYp6%2Bdkrm2W7OsCsAwWsHfn4tGEytyjT%2BIJAB1wGD5SpXHhE6RE%2BgnYC/OaECfL%2BSSW2zvmSB5YqX/a8du3YodnymZkj5KiiLPn4U6uM9uD1oFkG/HbDijLLVsE78NwI5RnoHAPsBeWZ3yzZKjsadczh66DMGfeXjhglRw3NlEKdJ9Z64shwvluU96FJSiUjQbOId4wvktN1PpFHgGAA8o8GXEcWZ8rDW6rkD8u278WHPIPLxhWFN8IP0R7rBPnN39bslGVVjXu61p2h8u0bc0vtuf5y8RaVQ2WypLL/SuKQYdedOMmaccUC0MFp975qaxf7s%2BSO57/wsnm2DugPy7bJd1RuBoCGaJhCww9oHBDo3NrQKk8qD/3w5U32WSSYp2/OG206g3WQwRp0gqQPKH2zPU/kczquJFvOHZVvDStK9PcYqoE8pNyQ67BFAjoKzFJZfudZ023t0iefWm2t6nsCY0Gef3ByienbZ3fUyJHKH5NVvsG/OOJU%2BkA7dCa/VGnvAr1f1g4y4xtVT/997S6l8aouWwoFvD8lJ31PcAg%2BWKS8zz6x8Gx9RPt85vkEvdeLxxbovWbbetvgGswNOmem3ht7aX7h2bV71jvxXOCL0/V6rKdDf6Cfdylf/HNDuW15RMt%2BwBgun1BkcgCZdse6XfK159fbd/0BntMJyn9PXjQ79EEM8LEnV8ltqr8P9jpemm399YypNscf%2BO8K60LcW6Avvzp7lEzNTTfHhqAAfAL9/p/Sz2P6TKmeiQQN19DpZ4wMPVfAPpbsefzI1iqzgwKgh49Q2sDOQTbS2wE%2BQS7UqV2xTWnhjypL0WvYGTibvzh%2BvJyv58emuFZtirvVttsXLp9QbLIN2c78njo8V%2BbrnLAGlrnmPFcv2mw6DtmOfUnHfSofCGqxXzi2ZiSQ/ScpfVymv2duCJQwZjavp3z5N6rzF6qNGgnm723Ks9Ax69FxNmuVr55XGwz74HOzRlgV0ieUDuBLAG8V6bU%2BNm24nDYix2yvoIM4vIFdC5%2BgzwN8XH/7zXmldn7ss4O9ZZgOSb6hNt73%2B7kp4eEG3zojDBQ/DFXbj12SiK7g9IxTR4Q0OMy4JqxMPqtKBIEzQ42AhEGDzXlDYRKNmagODEoUgUXEEAEB052rAuezs0bKPBVutOPGieGVrUYfBkRJWrK1rceQA5%2BYPsKU21Q1GFGGdfpbmByBhxJmM1%2BMToQPhjVKDSWJsYvR%2BrwarTiU%2B8K79fyzCzPUYEiRKXlp5vQi5GB8jOMxKlBH6wthjFGBcsVYGJOVYgKITX3p%2BFmmQhacooLxA2pQXKLG9Sh1%2BuiAiLGCwKH5wySdGxTCmppmGzMgisi8vFMFId%2BRMSHzwDF07But12JcPPM/Lt9ugotnc7waURjfJw7L1vkcYnPDnCPgmX%2BUwU4dW/DMAOPi%2BaB87noToX8wwDP5qCqOnrKjBxuhfeRq9zgC/YFSfe44SZNVaaEYHtlSbcpjpvICxhUKEPqAVsiuduhgoJ3x%2BiwxeunSyjoEQLTyfepsfXz6cHWMMkyBoqT5F2U3SY%2BZqEYkBj/0MiM/XT6hyolrYPxBi/yea2G0EmgoVVpuVJ57dketXQNgqB9RkCmbG1rkH2p07Gsd6QSlD3iV%2B8HBm6b/Qn/QFq3GuRf4DwMABwtFz5YuNMAqUacRXoIGV9Q0Gs2SjTm6OFuumj/aDOMM/X3A%2B0TFxyqdTlV65dw43fAfYI7hT%2B6VOYePCLqwJQtj5P4ZF8Y5LfA3hLeE%2BKTO5XsnDjW%2BZkxlTe02TzT%2B4LhROnZ4gutjcCDfuA%2BuQTCHuYan%2BwvM/HlqxMxTZz8WwLEmcISs6k8QYIPXabPP83hCDVucAHQIuuL80fnmxOEI8SyRsTxDZB68tKSqwegF8Czo2HyhzhP0BM9wHHZ9iJdSLQuzsa7V6B/nCUeOaxUozRKAhAe4ZXQSvEdAjbJ4SoChMXQStAXfEARlD8yewL1hlJ%2Bm9Esr%2B2FhWd7U2Wl6DxqHjqBlHDh%2Bi%2BMHveKUoSvQbzXmBL5xHXTFJ5WfoYVEnQ%2BM3xY9H8bwhJxUM%2BzZhgj5wtxwrik6V184YqSOJc9kBHPJHDCX6Csab2Fg0jTmoS1V9h1jvkJ56f2Th9o4mU/2b0MO4GAQdIT/uD66DKMZWXOU8ji8jEFOZ8b%2BBHz5QaWTWIHAHc/8YHMFjU4IiENXbG22JGzPvBlwmr6mjuLJakOkJQ42fY6Mgk/Q5dP1efBcsLWgB2RnKHA5zOwT6Blbi2eIA4iugR6gkeVK981K86cqzXxWnaQT1XZAJ%2BOIQj8QDDIdmyVf5SS8RgAdGwSewmbgmiQNsLf2Bej/vNICpcck4214nuqiTp1p9ElpZqrxMPYLMr5deZpx2Hc6XngpsKmQ29D7u5RPPzp1uNExNEsgvV3/xXZCFoxT/oanGXMA5gX%2Bwn4ElTpvBOGxpwhConvhKXgfewjHnDn4ypxRFhhijFyf5TQET5BJzAM8GKlb5xWF9CDPA/sMfjvYIJByuHT3jxXcWQwDJ%2BT3y7b3awalO2dxixoeKOZPqEEGE2Oo/WXVDjMMECRE/Yn4oywx5DAa%2BIxoEBEYtt9YrsImFBmukud31lp0GAMXZoTh2YiUrCKGB8IBw45rPL29Vl7Ua5DR4Hdk%2BVDCRIUwgvfHWbxInSccq2QVigiov69hz6Wa8GL7QaYsMfwRGE/q9e/Xa9FumyoGBDHZEaLjZDIBAuqCMfmmeB9T5Uokii0kXtExI/inqUOA0CfLwQsjf6o6qf87t1SFe4qNGWOe%2BUSRqTy36B/zwRgCZ5H5xbnEqMch/%2Bf6CnOUOJ75wsDivjCWuE7gzMbaWQyeY6ycRWjhuQhB3h%2BATqOdRYwvDEkikijtB9VIu29jpTyxvcaePRFMjEucFRQQzxbexYkjM8ezgg/YuP0ppTOy8xiWXItMNQqa5zhJHccPTysxo/T/1paroquQZ/SZw3vwBTREcAMnaKkaKUEEtC/OInRNFJdroxgXltfJvWr04IS/Ut5gBgDGd5HeJ4YL2XWUJ7yGwsTIzFDeJ%2BNHFod74x4xzMuVL3lGbCFBhJfzkRnFIWbeMHSCrB%2B8yTEYO0TnOe4ZvQ5zgeHDM8DAhlcDOUO26NMzRsgUlRsr9fPbVu%2B01vjMD3MxVA2SI/Q4%2BILtXTAWYu0sgvNj6Cwiu9EV/Y1unUWVkwRQzh6Vb92JyezeqXKH7DPzDC8gEwlMwCPQOMGAs9RwZhsTni/ZkbtVvpHtWKB8gTMF/WPsIdcIMLx9XJE54BAOfIcByDWg3cqWDqMt6BBZub6%2B2Z5vX51F5Db6ADqB/u5VGob3oWPeY7Ti4BWkJKmeqpN/6zieUV2ySPmarCHHbdZ7xOlSNjGj%2BkPqHJGJ4t7/vKJM5YXqOD0f79G/XI/5gMZx/ghgvndSiVw0psDkA1UE7LMM/8EXOKToRORR4CyS3Se7gyM2Wg31f%2Bm4ud8nlC%2B4Z%2BaQKgWcC/Qmen4gnEXmJ5bOIvov0NsHE/vjLLLHJBm5C8YUml5Abj1stFUrq6tDgcXZKr/Rp1SZYIPgRBIYw0aDj3g%2B9yifIKdpLAfNoqNwwuBFOuOepXz4NqUdAhzIbY6BfuFXOokT/Oc5EHijKdb%2BOIvIZRxegjQ4r/AvspuKGmQR38O/0BgygOfAGHAYsXUIOCKX6WpOsIXtOj44ZZjxH7/7t9Iu435Jx4JziE5Et8JT2GKM8zh4S20xgq3Q7l9W7TRdzLxwXXiVpEOks4hTSwXN%2B9XBxL7j/gkoUCFHhQ/8iP5mjPAMGV7mPRbOIud3Z7FviI3F6egR7POHUEN4bFCli3P1q8VbzJm7aWWZ/EGZhTQ9TH6yMjlOIszFXnQ4cQ0qECgNvX7pdrlBf0v5GaVft4edNBgQRwtHk%2BzbQypkblNGv3bJNrsGAu63S7bK39futGsQ4WVd44GAdYcoSQxSytkoh%2BVfypwQLAh9BDfjpcyIcdywvMwi9myhgNGBoCASxSb0OMsIxuv0HjEA%2BP3vl26TP%2BnxT%2Bk9shXGBaPzVTCnW7YDowHjG0MJQ5754BhKucgILFTDA6EUCZxFhChG/9/X7LI5oXzEnoEex/FEB3HsKTty9C/IAFOWRkkYjRp47jwP6BWaoRQIg45yzDmq9PkttHWuKmKyaiirG1Zsl9%2B%2Bts2eIcdDb5SuYSTjbKOsUEQ4lSg4njlOQHCNXyjdYExTzoPBQlai%2BxVPvccWdS4pO4IO4QuuCV9C79VqhPMdvAIfQ3eUv7YqrWLUBOtIcD4xOAJDnnmBfzjm93rMn/VvSviQKRj9ZNYt8KMOH9kGAlI/fXWznZ9SQUqhuCY8G8kXlGFfrMYWzjUNM3BKKFNnfgK%2BwIEl6k5pErKD7JSjfwDNEgSknJkSNBzx6/W58QyRT9AUf/M5cjAIZI3PSjPnkiABjgu88DulO47h91TUoEMsG6DXgAIwsNmSBPkZnJffQ2foCvgCupqtchMePRBAc5SkwdPQFbxH8HObGqosUYD2CBIyTnjmWh07DgOkFmylAO9jtOMkIPfvWLfL%2BJdz8cLo/IeOu0oNX%2BYPZzpYt3vpuALLzmMwcw1%2BC41D6%2Bis6PVNGP/oCwx0lq1wDPqUsTFPXBejneUXGPJkVxyxxUnDci3T1qKymyDE717bavIfGfkn1Qs84/X67AjUYWPg2CNjKdHG6XlOHZvr9JkiT6F7nu%2Bf9ThK%2B1fh3CntUQFSrfT0pDqOlLQGfALtYJv8fNFmC2Zge2HzHAhwtsjGoQ%2B4Bjx84wrkfciZQpfh6P1J%2BTMYA9%2Bz32aQ/UN/IBemK%2B3jDGJv3aK8wVgD2v314q2mgxraO%2BTo4izTMzjRyBL0wCbVX9h12FTMJfNyo84LAfxoEIghAE8ZKg43OiaQO4FdyPYuOJpk6JlPR/zCncUBBsp5lhq7CKjHttZY5CUSRKSIhhGlIiJ5wrBsy1KQOSCDsLm%2B1SKnZDgiQc35/zy20hQe666IBn3thXXy4cdXGrOS4UDAUNpDmQLruojmhoq6DhwILozYoASOLCBCjdItIlVEYYNyTgwTjPVg7QvACCDLh2KmXJBIHdmhyDVClIJgODTpNUIRqmTb5wunGsH4cnmdRYMjgVB7tbzBau0jgYNK2RGGCYKYKDNCnlenXhIjBmMKhcKYHP2LWQXpMr8408rfyPZhTBIpBdAAz4k1U/AHkf8zRuWaQsUwQ7kR4cfpj8z4kRn87oIN8s6Hl5nhgLFA1pR1MO94aJm9hy/gRZ47mRauFR1YOBBgyG9Sno1Em46RcZLR%2BZuOOWhgRDkTPAEtRgJ%2BJdPNHHCP3FckLLKs94XbRhYfoxqjlYwfPE6GkKxhcF8EiYhQ4zRHdvhEeRPRZi7IsrJFRcATvCjJxmgnKkz2lZJdZIqjf0BmAYdomNI4c09WInq9LnIS4w0SIhuMccj6JAw92hPcqU5UZCYDCuCYb7y4Xj72RGjNGcdihL7v0RXyg5c3WYADvqCUE6MXx8xK7Q4SqPBYXdtk9BzId/ihdfduGzNyf4fqsAD8Yoc6cOiUANA6VTM4xBjIGKKRwDB%2BtqzOymYJzBBcIutDoIMlG8gZgk9kmCIBT0RX03AsDgZVLMgZ%2BBM9Dk/g3JL9ISBJcBGnlJJGR2xBQA0HEP3wc7WBKJkMwPOi2oQsaJsqd%2BiGtb8z8zMtC8f3ZMyis6RkvT%2BqPPLhJ1ZaMI7GWjidVzy6XL6uthU8gd0S8AkvqpYibZYDAefCpgr4lww3lSXwCusjCVBEVgER0GGckZivtEsFGTpmcUWjZfsi7S70Aw4dthX2FEFbnFSqCLJUHxBsInkRCeQHn3GbkXdKACaoxsLJZnyR%2BoOeAyQ0OP8MfQb7apLjGHi4sxgnQEBhwEUbhgBnijKawPBF%2BQ/PSDImZA0XteZ9ARm790wcKl%2BcNVJ%2BfPRYufG0yfLt%2BaPN0cIQ6E8wUgR0X%2BQnkeLuFjk3d3banOH4BQIZQwDDBqDkcZSjgRKnfCoSGA04GQixH%2BqcXHPcuC4vjG1KmMi4OGIDFDDlZ5S%2BRQOFxvOPRLoqKMITOEB9BUbg5eOL5cuzR1kDG5oqYJyTyY8FGDPrv3rDGu2q1DepTMCA6A7MW6RBT2AJ%2BsbAxnCOvgZOIoZyEwtewiCRCa3DU1Q0fGhqSReeoBwW5xqpQYaXdTWO/gflYTzHaBBQ2dnUHgrMhfUCjgzPiHcYmsHa7t6A4BiVGu%2BaUGwNIX5/8iT5kf4LX8QCjBk6xjHrDSiDI0Pe3T0SuNxY37ynsRWgtBC%2BQCe0dCMzcMaXRDkN8ASlwSy1IEhCM6dInvjZMePk2JIc%2Bx0OPvrWEXsYLygt8IpeX4yzBA9hWwGcfEqeCTKzlryv%2BoPjWEP8qRnD5aojx8i1J02U28%2BYZhlogvz9DQJ/lEr3FvAUyxOi%2BYp5oaSc5RBkZQMgQ%2BCfypY2qYrSxegUbDR0VxDoBOgN%2BATb7L2qJyJ5JHiREMBJxKbtb9vTcWBwZzFOAJPB7IGCjwSftL/%2BhvAiEkO51%2Bv6H8f1tqHtFeogXq9CDIX/GRVqMDCLwCfnpEuGOlgYnr0704Ghb2I4lEnqLrvDbXc3Z0GpIE5pdwKUiHWQ8QyAnGJBdmFKkpytc8KC8ujXyPRQR0AXarEDjx1l1B1fYPRFP199hAZ4ozeg4y/NpW47fYoFCT4za4SVVJ5Ykm3RaXCwIsNvBrivu/vsDvyqncxLD/fJkCPPBc1iGJuc6eZ%2BmEcy/pFyhqkMgr0YUnRQjeaJY4qzbA0PoHwdA8HR/%2BjQZ88rGsFzD%2BmF0Gc8kUBmkY14Mxrjl5Rc/%2B%2BcUtMXNMaBRy4ZW2DN1Ar1eUcahf0N7qm3lwvou7t7hFfQI5HfcK%2BU09p8hT7aCwQVox1PSBwdnKb0Pj03zbr/RvIEBvAcnT8qAPgt6/AdsQd0Ay3w6u7Z8nlg8fCc0P8hWgjRUW/AUqCfHztO/n7WNPna7FJrPEUTKfikICXkKMWCV/p6BcZk9l43Y4N3cK4jv2FeeE9zuWjeYr6QK2Ql%2BTsAvIWdSrUJgdhIHgleM/IyLBsLP/F7R/zCpVicgMgW0f/uolCsAaRLVcBMGM80rsAA4DgYLRpHqxBDcNGxjbV8rLOjfvycUflWnkGEmXI11i9StnT/5iqLNsVArvUZlHPk6/1Hg2gemUDmLDCGQoIrdBOUlna3VsBKcMPZxwAcwiL1Hc1t1hTlphVlXV6/fm2r/PSVzfIvnTdHbIBzDv12V95IyRA8EwkUIJRAI4FoQAs0mKCVOmsw2H6A93RPpVEBxgJlNqw/oYyNtRyUWUaWu8UL4Hl4oidDlHmjc2kAsiY4F2RRMGKjJQbzRTOPyH1O4SIzrvUPmpt0xxOURbHG%2Bkcvb7JmK5ElTY7%2BA%2Bu4c7qRiWQAKINDxmEAA4y7IANNlrw7XqIM78oZw83RQZSyrdE7JxRZyRqZFtYc/XXNLivHv2dDeZfS53gBdI9O6O4eoXHWKNKYJAAmccfru02HoE%2BiwbpDum5HInBAkAsLyutt65hovmBNFmswWc/4%2BLb%2BbWTj6B5krIKSx%2BgAL29ZZkB2GBBYJqMGryAj4aNosLbuXcoTbBFBMy2aC7LdyqVji2RuYZawNdijW6ptTR/rZFkzSPa/u0D3QAM%2BoQokOrjHO%2BQH/BP5HcFE%2BIMKhe54K5RFfMMOAzikOJHIDxpHRfMIL9Z5slQKHUI1hCN%2B0b2l4YgZYCgW3SeqYKPdNp08IwFzsg6LdXLIHMruaIhBWSYKi6g%2BTlHAwLAqgoCGL18%2BYpRlD2nlfZEKNVrfk1F7ZEuVXKPMyR5631u40RSbrXHaW24MKBBOlImgkFkgPTEn1bb1iBwjEW6MfYQbv6Nsgn/ZB4t5Zd0bTQhwAgLQ1GAEi9mjmjJQsoKwYn5Z6P19nRfW8fD69ksbzFGk4yQNPfbV5c9xcECpC2UtPNvzSwtssXygvKABaJwSHwIflBJtb2gz54YmMTx7eCJYrxeAcrB3jCuSHxw11vgDhU%2BDDtascD2aeeD0fPPFDfL9lzdag4OtDSFaiieggDH%2BuXe2L6DULRLICtYPMkcNbSFHd6saLdwjUVzKqZmfwDjG8cRwmqPyIdIQQL6wJoYMJt3rkBMBT/C6asEGawABT7D%2BhDUy7iz2H8ik0zCjSekdQ4%2BsOM5PJFhKQCdOgoiUZLZ27DaZSCkesftjirONbiIBn9Bg4ouqL1hfx%2B8ILrK/IJ22Mei%2Bt2Cj6Qv4AzlIFjrewBIOjFXoGyc3EjiDOH4Y/BRNM35KT6Fx/iWQRCVBdMl5sMYrEruaQlstYASz/p1mUZF88a2X1lsjLbpWsq6atVqO2IIGLTxbaJ1OnsHSFEBAjMAYtACfQDc0F2TNOPtvonPo8oytEAAbgnXfdIb%2B%2BtxS472Th%2BXYlhkELFmjSuDsuws32H6ov1i81YIq8F48AZ2KHqAKhDW7zA96IgA6lm7CfE52kbWQBFQoP0UNsuUI6wsjQQnpxOw0m6/IZYcBb7G0gYZVNPyJ5BPeE2h5VPkEZ5K1vo74hTuLAwycNxYl4zCyp%2BB7Jgw1w40oDUKIBfI0bIFJWZ9Iy2YWBZMZxLEZmRFq6DItN9TwhmMRgjPzMqyzKGv2aIRAswqMQzYMJirMej4EKEqS9vgIVBzPsP044Aia3tAEh7lgDzpaHZNh5T3jPlKNoitVeBMFfE2NWdZ1EskjuofgmVcYakTA/pHBMXRIo2V2pCIAZY1tFllknt85vtgMDiKSHIcBQeSd8t3bzpgqH54am82/38pgrdBzZXWmyCiVhg/ouogihwZoaHTysFxzAGk8RJc4osL8jcKnDIzMOr%2BFL3j27A91pj5HVBINmAhIsCYLpUZzDRpFYQTye7J2dJKEr2iaEU%2BgLI6GIHnK32RF2XMxoFXu872TiuV45We2RMDYR8kzn3RBhb8pMSSDhEHAcTiOBKrYKysyG0kAhTlhWyH2ybp8YtEe2cRzIIjFWra/nzlN/nn2dGt5HmmUOQ4uMLz%2BpQYoG8WTWSRQRgv4yGfP/mEfnjLMAhw0%2BKKTJ7JtkTo18MfbxxVakIXfcgzP8wNKQ0fkZ%2B5ZGw94jko%2B1vmRbo%2BsacXh4vds8YS%2BiCcoC9u2HtzvfL2/j6qMZqzwMi8abbAVD//i5KFzWaPL3pHoQ%2B6X7TOQ88F8EoRBxwT7ygWgS%2Byryk/wAGs3kUHMZ%2BhaQ1SmJMl35o%2BWO86aLj86epw9J8fBAU4O897TK1RlRL%2BCRmFDevY5/NqcURb44NnwjAiknWMNbQgCvG7OPF3iWcOH00jAjEDicSVvyFVsCIKKBJ9xfmhURmM99reGnmggSBAZe45jSpUmqOIiEBNPMpGAHi90HvIe3co9BbRLFp0tduATEhLwCL9FflSow0jZLduxBLwF3bM8gf27o0GlGnZmTnKCXQc%2BCOQO1%2BL9t%2BaF%2BOSa48Z3CXw54guu2QcYRH/pPkWEEmHHPlX/d%2BZ0%2BdXxE%2BS3J0y0xfJsoIphe9/GSosQA4xiSoIQTjAqdfP8nuNuOW2KdU2l6yJlMpSf0U2yUYXcEYWhjc6vPWGC/EIZ9ObTJsu1J0608opYrVnsC9jmYF1tixozGfJdFo7rWH913AS58dTJ8r0jx5pzh6N96%2Bqde/bSe7mizhxqskrsu3XDyZNtXv6sx1yt93z00CzLzkSCbl20kcagohzrD%2BoY/uHk0Hz%2BVucK5U/0jHmM7pjnOPggE/ayGmUofTY1/p4%2Be9ZPoVR4JrcqjdPamzVF7B%2BFI4TBCE9QDokx/SE1mm84ZbL8Wn9/s/4excTWESvUKCDoQskp7fkxLogaY9j97sQQ38FD/3fWNNuLqrsy74EEjX0oHycgcooawPDF73VuuE/4gs2WS9VIWVxZbyVRAVfT7fHZHXVq3KbIN%2BeV2u%2BvU37imKuPG6dGTcjQCkAGhVI69rYjuHLl9BHmFCI3eA5/VB75tPIKa7foGrlFjYugY62jf4Dj/%2BiWKuOPY1SOfV%2BfPc/wlyoToVnWGULjNJygq/Nr%2BlwwZG9fs8OMYRz8b84tNX4I0csU2z%2BNoAl7yrJFBaBDKcHFy1T3/PXMaXLN8ePldyoH79Ln/3W9RrBONV6A3L55xQ51AusseMg%2BqHedPUN%2Bo7zM62rVj8gDskpkMjbWhRqmUYXy7Rc3WDdxZADVOH86ZZLN57/PmynvnFDcJXOPEc0ec5TmjlM%2BY95vO32q/Ob4iSY/kBvsw4hxRZAGeeQ4OGBZDbKnu9ePjxkrn1PbBtmPrUQAsb2Tffsy5Y%2BqB9Dp0Dz2w1fVgSQ7TyA90Odk7tkOYrXqEpwiKlCwN6CFG/RYbAnojPNiSxGEofKCLOT7Jw01uYhuwm6Ar9A3VIZxTDwBXQnt4vBhCzLuXyvtIkduP2OqbQ8G/ru1xragYS0iXV9DvBXaogYHj7m8Veme5ojwDrZo5K0S6P/%2Bgo1WCYezePWx4%2BUmtTe53rXKJzTKYr0vTuXjyiORDXUc8Qd3FgcYRICJGFO3faMaumQPMVCJcLKRMiVl7DPHVhgwLKWZgAjq3Rsq5PdLt9v2EESH%2BD3lQ5RYIAQxnP%2BjQhNmpMSO/YbYaoNGLWRYWGA8PS/Drk8LZSJjRJEor6DVejyA7OK1S7favwgqsqgYAnToS08cbIbsJ59abRkQOpghmBHilEux1xBbArCnEPN5rBpXCCSyStHbF5Cl5Rq21QgZW50j9lw6d1S%2BKQ6AQ0mdva9B6X9Ah7TAp0z6n%2BvK7bmSKaZM7lSlAYxVtpRhXyj2Jg3WZGHI3aLPne1hyJQcVZxpe0URxSTCC51/5fl1FvGEbx5WwxGHinKhWcp38BA0Bj8ReLhjbbntBYUxTWQ5HvxGsqAYLJ9/do0Zo8VpiTZm7hMHgkDIX9To%2BZXOzSJ1uAMFTqkU65NvX73TjCiy9WeMzLVMEfcK/SMLAhCgIouJw8j6Zsq0aaUOT/AcpuVmWLCLfbkoK2Lucdgd/Qem95/rKyzASAUFWRCeIZliysdw8Ngv8KOPrzIHHr7gxZYRX3purZVFsiYLfkAmHq38gZHHOu1bVpXtKeFH37BcgfI75OZ5%2BsxZo8VemgTl%2BB46G55Gtr/ruvCBAOv42X%2BXvUvZXxgddqbODQ3LqBKgpPxXr22xcmoy84CAC91Ov6tGLaVwzCfGMpknHGuWZ7DnXiRYg0Z2hhJsNjSn9JXSX/gP/UuJIryAvmb/yqDjpuPAcURBpsme7l5k8k5SOUjwCnl%2Bt/IJpfLIepaesOUY%2B43yrLAVblIZyRYxZAkB8o5yetar40yRBYN%2B4C02qseWoGzyen3uyEIC0jxfgpXwQGAvUOmBrmEPaAIwyFBoERkdD2AbJO6b9bR0PaUnALKAyi3272YLKUqrb1eZgHxHf6AXkDnQNPoHXYgNiSNOJU%2BoOu6NDswAumf5wg/VFntBdTVLIIJGNziPbOuzWL/HprpNdRJ2qiN%2BMej13rbSPMxB%2Bc28OxYaI/QXUEQINZQJTh9CCsclAGsjUPiU2wUb4yP02CuI0hciNJHMiCIfr0pwbmGGrUvk/HzNMUTH2P8mcp9BMjGUXrCfYLCnDcKAqCrGIiV7k3JSrakHhjrMy5gobcKBekYNjTdrbMAm4AhGMnAY4ZGgRp4mOyj1O9R5RaFGgsgdrZYxgMkoBmC9GqVANCShVAGDHYXNlhnr65vlfnWIu2utTrSLa1ImhPDGAaFMFQOBEgpK7n63ZJsZS4D5pOSUZieU8CLcqOGndh%2BBv0LnFAMMQyQAmWBK%2BJhv1qn0N3B8F142zyKWscAXnl1rG/D2JyjVYW0Q2zNQBvTApqo9fAhNs4k2a3kp74JuedTQ%2BKrqRqXxhi57jPKs6fCLcQu9YEDz7KE76BKnKOAjsgZcl5IjMvs0v4C2WKvB%2Bjv20uJclGrCq%2BwnxbGUZbLGD/4iyxfJl9GgpJlMA/fJPp/wJZmfAJdPKLZuo%2ByJFb3PKqVC0CP4qypv%2BBCQ7aScB6XLeaEHjFbuE1rFSCGTFJ3pI0M4We8X3kBZMzcodc6LDORzHAYcAqoXAuBcIAsoyQ1oj6Y5VDpQEg%2B/BjNAGTwl9YwLowt%2Bjrzf/sD1J02Sj0%2BLTXk4Gen5dy4Mv%2Bs/IKEps0dWvbiz3jJiAXBkoA2MtqB8iyYdNWr8rdbnRjAhGpzvvNH5%2BnzSJFflLLINA5nSaxwisvjBuiFoC%2BMXfcH1Ac%2B7TmmF4AlylBJkPoOfyM4gp989cajk6O/vUT6BznsCY2H5A0E/6Ik9PNkAPwByDsO/VPUaRvkryueBTIDO0aPIhBXqBBMEjQQldPA%2BQVPK2NEX3BfO4sv67Mh4BDI/APdIUJD5xElgbtCJ3BfjI6CCEc298jkgM4PxC%2B8im5LDsomSXeg94CEcEz2dzSXXQKZxXkof%2BxM4vU9dPCf8rv/BfoMEog420N8sDWG%2BmceegOyj0oS9QoOgF3IbJx7atK6begL4hIwgTh4yPXieAbClcKB45pT6A54h8pHAQiRdI9vRT/wb0FogF7HboCsCj%2BiTIIsJD6JTGAM2IHJ6X0A3UkKNffI7HDy9xwDouk9NZynOIJO1kXYTwEGGDqB5aBe9CaBpAih0bR2arvamjh3aRV%2BgCx7bWt3FPgPIHGgYO4lj0JXoX%2Bab%2BwSsacYeAswHjjTl8vSdQMcSrALMOyX13H/kfpDoGvQHPIiDGv18DgbY7obMsaP3cGcxjFg4iw7HgeJwdBYdjgPF4egsOhwHgsPFWXQ4DjbcWew7vAzV4XA4HA6Hw%2BFwOBxd4M6iw%2BFwOBwOh8PhcDi6wJ1Fh8PhcDgcDofD4XB0gTuLDofD4XA4HA6Hw%2BHoAncWHQ6Hw%2BFwOBwOh8PRBe4sOhwOh8PhcDgcDoejC9xZdDgcDofD4XA4HA5HF7iz6HA4HA6Hw%2BFwOByOLnBn0eFwOBwOh8PhcDgcXeDO4lsUQwYPkuHpyXLM0Gx9ZcnozJTwNyKzCzPlpGE5MiknLfzJvpGTPETm6DGnjsi1Y7KSEsLfHFykDhksJ%2Bu4ji/JtrE7HP2BkrQkOXV4rhyrvJGZGKLlwtQko7vTlMazk4bYZ2%2BGYj1mrvIFx8EXg8KfH2yMzUqVE5UvjirOkkH9dRHHWxqQ1ayCDDl5eI5MDusFPpuof0Pf85TOi5VvegP453TlI44rUh7pD6QNSZAjwuOdnpcuuaqjHI6DjcLURJlXlCmnqL4Yp3IYustU%2Bwd5j67ALsI%2B6g3gJeQ4x/QXClISjY%2B5DnyRlOAugKN3cEp5iwKhdt7ofLnp1Mny51MmyzsnFIW/Efnp0WPlzrOny6dmDA9/sm9MU6Hz02PGyYPnz5LPzBwhE7J752R2BwyQRHVkM9RIT9dXQoT1OywtWe4%2BZ4b8/cxpck5pfvhTh%2BPg4syRefLA%2BTPlltOmyPiwYYxhe9sZ0%2BSRC48weu8NTlJD9RfHjZe/njHVeClB6fpAkKKKHb7g30i8d9JQ%2Bb%2BzpskfTp4kQ9xbdPQDBitdff/IsfKvc2bKp1XGB599avpwuV3p%2BxfHT7AAY28wJTdNHlY%2Bul356QTlqwMB%2BoEgYpY6oJG6YmRGsnz/qDE63hnytTmlMjM/I/yNw3HwcHxJjvxaaf/%2B82bK%2BycPleFKd9g/V84YoTpklvxM7SKcst4AXrpL7S6OOVCgB7DxeGFPBZhfnKV8PEbuPGu6/O/cUsnzIIqjl3Bn0RFXyFSlf/LwXPn5sePkeyrUxma9kfF0ON7KeM/EYrlGnc8PTSkJf%2BJwvLUxISdVjewRFiwZlenVJg4HQZOp6qB%2BXZ3Bb88fbVlPh%2BNA4c6iowt%2B%2BPIm%2BfDjK%2BWmFTvCn8QOparwiVCT3aGsI2XIGyWtO5vb5IOPrZCPP7laHttaHf7U4eh/vLSzTj715Cp550PLZFVNU/jT2OLsUXlyYkmODI0qwb5z3S756OOr5CvPrZPO118Pf%2Bpw9C92K63dvLJM%2BWK1fOelDfL8jtrwN7EDWZy3jSmQ%2BUVZlkUJsL2pVX76ymb5wGMr5bdLtsry6sbwNw5H/2JDXbPcpHzx7keWyfcXbpSV1bHVF0kJg%2BTc0nzTF1Nz06xCK8Ci8nr52aub5SNPKF%2B8tlVqWjvC3zgc%2B4Y7i44ueHJ7jfxrQ4W8WlEf/iR2oJxoeHqSrX9B6EUW1TW0d8o9Oq5/b6yQ9SqQHY5YYVtjq/xnU6XcoY5ZZUt7%2BNPYYkRGiuSlJNp640gsq2qUe5UnHt5SpQZ8%2BEOHo58Bqb1a0SD/Vr54fFu1bGloDX0RQ7D%2BsTQzRTISB0skW9S3dcozZbVy9/pyeXFnnZQ3DwzPOt56qFYH7BV1yu5cV262VEWM9QWZxTFZKbb2PjKAAsqa2owvsO9eUL5o6dwd/sbh2DcSrlKE/35Lo06Vyx%2BXb5fGjs7wJwMD1iSNz06T6fnpkquGYYdaf00dezM0i5RZDM0i6rQhg6VZv0dx54c/p%2BEF5xiXnSqjM1Nt/UZxWrK0qWBo0/NhUCYnDLZr0OCGYxdXNsizKkTA7IIMOw%2B/iTSMMVTH6Lm5Lmu5aIozVAUS15pTmCGj9D1CEgMCoYTuZn3VaBVc4/Q343U8vBjfKDV8h6lTmKrCrFGdQKLUfE6TjiP1xXXq9fNdquSZg45wxuQ4HS/306Fv%2BT4ADQy4DmML5oBr0ECBmv3atr0jaDij84tCi89pYoLBwbHMGX/T0ITxI0zjyQDn2X902rCYLUx/SB0QlMpAA76YlJsmk3PTzUAsb2kzug2AgiTQQFMLnh8OVRA1HaX0MjYrzbIQlK1BG9AuTZI4hufbEsFjnOOC0QVKM53qHJbLDqVlGhnMLsjU49OkqqVDWsNKlusUpCgP6HmNL4znUoy%2BoCXOxXhX1jTZXAa0xGdkzuGjCfo7fjsGXtVj4Snup333bnvO3M8Reu1LxhYavVYoT2CA8DdGMU7kjLwMGar8REYlAOfI1t%2BwxpJxcz34imtwPy3KRMiDyHmckZ9hY8nR8cH/0/S8zBufjdTrcE3uvT3quIHG%2Bfq8aDQRC2xvbJM/qa6IByC7jlaZOEafKc%2BFgFo0SpQuaJqBPKxqbbdnDj9BN8hKni20MVb/RV%2BUKF9gZAb6ArA%2B8V0Tio22X1Ndcb86iMhHjp%2BsfImc5aeB/kTmDlWdQzmc8VxAP0pXrOviXOjchzZXyYpw5oXzQfsYutAcvMTY4NcR6SmqD0PrEgO5P0VlwRkjc%2BXIcFZxkeqd9ET9jYrGts7X7do04%2BFe0SGRhjE6BH4NrsM1hul981t0UXPEb5G5XAvZU6fXRm7wd6BnRjBfXEOPiyfjG7nxwRiWrP9THfMllfGRweW5TNVnxLNqUznaovQQKa%2BgI%2BwfniM8hI2FTEMfwEvQrMnnMG1Au9A49gzOYADogkZoJUrrz%2B2oVd5olHalAeTsLJWlOUo70ERAF1wXPcb5ObfJY6XF7OREW34DTe1UffOXVW9UdSGHi5QvWOvLWLCl4At4lfvD7kN/cQ30CusSqcyCV5EJ1Wr7cO%2B7mtvs%2B4l6PNehEQ86LqhGYe0vx/Ad%2Biy4BnzBfbfq/DBHAHkAr8JjRapL4D/ojWOZt9Kws0pWs06vHz4sbkCDHxoyOnqPQa8rwn%2B/pUFUdN4dC42hBhIzVICx0JmmFS/tqpdfL95i2bRIUHbzCf3N0SoUECq/W7JVals75ZQROfKhKcNkvDJrgTIwQgbmrldmJTPy01c3W/kmhiZC8V0TiuQLs0aacrxFz0PZDnj4glnWEfVva3bKZ55eY58lqfI/a1S%2BrZuiTBTHEWN2S32rlYem6rUwSHC4b1xRJi%2Br08gxGAmM9Xhz8kLGJmhSowJHkGzIr/QeN9S1yDfnlZoRMVGPQRgxLoQMWRPOu7amWRZeNs8MkqsWbJQb9DMMDO4Txr9sXKF1sUS4QdQ4oRvqm%2BXu9RXyS70G7wNif9f4Irnh1MnmIK%2BpbbZyDZyBtMTBZlisUgPmZp2Tu9RZYO7iBRg4zAFGTSzwhWfX2twNNGgS8M15o804JDL6roeXWZAkAEoJo/G3J0wwerhxZZnS8yalywT5zMzhqjzzTUlj%2BAFoFz54YUed0fl9Gyv3KM33K%2B/98eRJslllwjv0Oq8qLV%2Bsjtovj59gjufx/3zFDAMlUclXJU/JzxXKF/AMxjBKe1N9ixnVNNagOzA8zFxCWziAxw3Nkg9PHaYOTpYMU6XKmKF3%2BAJn5K86JrIi4IqJQ%2BULR4y03wB%2BhyP8VFmNfO359XK58swnpg8zp3a%2ByjB4Hv4hgELg5Ts6b6z9ZT0wwKHYqOOjROo5nUsCQgFf3HHWdHNwCR4t2FUnH5xcYgYJhgQBl5dVJv1q8VZ5Vu8/OgAzkLj%2BpEny8WnDwu/6Fwt1DubfuTD8bmDBs6KxGPQPfVEWGhh0AOMUh%2BEnR481w/HM%2BxZbGTWy7rLxhXKOyvTAqKMpBob1dqUj9MRNKsfRQdAb56G5GLL/ttU75FNPrbbPaOB00dgCpfdW00P/t3aXXReauVTl8UeUxjFAqRLBOVyws05lcovpBHTuF55ZI3cpnauqMPp%2Bt9IygRruCyOYz5DxlerkvqbOILrg98u2S6fe4x9Pmaz0OdRond/AWztUF/1%2B6TbL6lx93Dgr24a3ce7J8gBkJw2nLhtXZMY37zm%2BrKnVfsM9PLyl2pxlZvL80fny1dmlxkvvfGipGZoXjSlUZzvZjqtS/nlAnV549r86b/FSCk7zoKcunhN%2B1/%2B44tHlcvvqneF3A4tvqcz7gNIGNP89lXP3b6oypyUA9P6X06bYsyRg8WPVFWuULwi8IP%2BPC3fqDWgDWQe9Yq9ctWCDyXgeM3rhSyqb5yg//XzRZrlV7x/nE7n5MZVHTyk98Xt0FvwCTf/kmLHWMA0Hk89qlLYXljdYUAdHlaDH6fcusnHq1zJWHbazRuXJ/ygfw6vYUImDse1Uh6kNtaauWX788ia7FvqEpoU4bvAOY4R/cXCPumuhOcef0HEdXZwtj2ytki%2BqzEBvYKvxHdc4Wx1NAjocDw9g3z2oc4RsWaxjI4CE84j9%2BNmZI/Xc7Za9x/4j0YBtCS8iR54O3z86LZ6WSHxjbqn84Kix4XeO3sDLUOMMq9Uhul6VIcYgERqis9HAGUIY4NhSd46jdaE6kP87Z7Rly5o7O62EFOGxorpRElVREwH7zIwR1qFxf0Dk/qOq%2BBFaCBYM6EUq4FRmmGKfqa/obo8FKmyvPna8vEOVMoJ3U0OLjYkXwovI1HlqaH/piFGSosYoRgznxUhH2GxVAfOSGqyv6HUQNt2BzAnGEAYxRgaSHcGFIY2TNzw9xYzy69SYJPobiSQVuLbmRY9jTlfo9V/aWS9N7bstQobgxBB3DDxWq0O/vKrRHEQMPDrjRmZXs/TZEkRBGZPdfkGdGXjnp8eOkw/p8ycqi1EX0B/GbUbiEGtvTsMYZZE%2BA0cR4/lb80otQ9%2BkThhBEhwtxnG2GuJkDaNxiRrXdKLDUCfIwu8ZE/82Ku0RccbIRvkS%2BV2q9833GDvN6kzC70%2Br8QGtdpdJAkSrMZauVeeZVukVLR3m5CzQF3ID5/vXx49Xh6HIeCgS8OKxahjD7zgPzCdjwDfHmKL5lDcTiQ/w/J/bUWcOIgGzE9Q5igRZ5SPCnUD5HYE2giofnlpiBm2h0in0RNDkRZW1GMSlGSnyTqWL00fkGR3tDzg3%2Biakp0LG5FY993QdCwHHaMAvyFqMT7Zywsg3HaZyfJHyBS4jBjy/IWMDja5WeU0JNnOgt68Gd70F/7ifnkDG87oTJ1pTHCphqICBtxgf5zhXeZKgVHd8gTP9vSPHGt/CiwSMlql%2BxfG4WHmaOUNvOAYePBvsBqo%2BLigt6BJcRXNAm9AO61k317dIqeoVmurRJR5HCfqC/qAN9A5y%2BQL9jo7vueHAW19AsJLOqdg8hTquLWoPYd/sUP6AtpHJQTA9QG5SolwyrlC%2BOmeUOZIEOKFz7m%2Bt6kSC9iQN0GHoolq1qxgvPEDgsaKlzSq9CO5FVs9E40y16%2Bho/z51lPNTE60SBr5YqnMTSiwUy1XzQ3MTjel5GXLxmELTsev1uuhA%2BArHFpvsc7NGKl901YOOQwvuLMYZUJKsryAzka4KkYwAUdoAZOfILpHB2KwGb7kKD4xiPiOTwGLqb764Qb6lLyI6X31%2BnWUfalS5UTIwMuJcfQH7Vc0qSLeM25f1nN94cb1886X18o0XNsjf1%2ByyyHCkvZ2phniozCfVSjz%2BvnaXjmOdjYnXlU%2Btlv9srLAxUz6GIn56e63cv7nKDFOMHwT4H9RxpoEH89EdUAKU51EyQSbnN%2Bo8M7arXtpgkXayM4yLzCYRQwRYAK6Jw4ogvWbRFr2X9fId5uyFdeqcNNm8j8v2bqzxACKcBBswODEsLxyTr87aG4qVQACRdMpCcbooByL6icOVpbRIpoEoakB/n392jdFGa%2BfrqsTTLWve173YKO1kyxmMTjISNA74X6Whbyr9feLJVRaoCcqnI0E5ICWnZDI%2B%2B8wa%2Bz1jgk%2B/u3CjZV5wRDECoEGis0THuXfKqZ7fWWtZU8q%2BcIC7A6VR56mRlKvnQekTOec639LXDxZusjVmGAHwzlkj88JHhYADS5k3juVXn1sn31Je%2BtJzay07Q5UCvEZDEXjHMbAgqHifylGcF0ouo40yZCsvgg7/3lgpda2dVgKHE4nx%2BCuVl8g8Xsg/mptBG4FzBv/0BegBgjanqL7AoSPT9umnV5s%2B%2BorKf6pOyroJ/KEvoNnhGUlKr7Xyk1c2h3SY0h467C%2BrykwHkKUkY0k53T0byi1jT8UIWYvrl26TX6gcf1oN/O5AwJIACoEdxnmfZfvXGO%2BhM3712hZ5RR1UgrQfUF0RLQ/I8rA28h%2Bqy7iXb%2BvY0GmUl1O2irHPPTgGHmTnlqgeIAgIrSdHRAOxAa4I2wLbG1vNzsDhP214rtHXFn1PxdbXlR%2BgP2jjBy9vtEAEJdonD8u15TN9ATprdn6G8QVBid8prX5e9RE0xPn/unqHZfjINEZiptpcBOP5/ImwHvi28gXH0VTq18q/rcrHLFNgOdCWxharwsIBJugNz9y1fpfpi8jy2UigYwggwevw0s9Vj3Fu%2BIJ/qUBBz/D9cUNzzAaNRLryBJl/qr3gWe7np69usowtzi9Li4KKHsehC3cW4xAo8adUMDV0dFppQOQeUaT5ieAQIcbJwamiHIFOdCjL6/TFehKMV0ofUNYPbq60UjPWN/ZVyBGJxeFDEVL6gBDG8KZM6YltNfLvTRXy6NYqMwAodwjA30TBbl4ZKpOl1I%2BxMCZeNAtZok4hSh6BgowkE0gnsYpmPZe8LvXtHXY9Mo6RJSQBUOYIsJn56RZFYzzBonKuQckIJUXMU1EaDkaBrUUMwCoGnHPmB8MDI4lzYIQTmePeMWIc8QEyXERUU1Tx0w48iMJiOBKYYP0H6z3W1zYbvUMzBBrohnirKmOMuoD%2BUGTQVk1buwUcMKAxbvsCykdRsqz9o2HAPyNoCCMW42JnU1dnjvvAaP7dkm1yr/6O3zMmjoWvMF6gf4JA2ckJslX5gnOxVouM%2BzblNdaREvkO1sJEg7Jq5AQ8RcCFplDwBS/kw%2B%2BXbrdg0wzl67lRDgEON1FrOvo9qHOGs/nolmqTMZQkkdElyh0dBXfEHtAEz5SMYH7KEKuQiATZENaGb25osd%2BhUyjnp7we54hA36NKczxjvocGyXYgTwkGFKf2zcijIJTAItcly4DcD/TRI0pLyNb7Vd5GA1n/jDp51ypPwBuUm3IMfAFPIMMxfNFf8DqZnzV6H7zQgfxHJpKxMxfdgfmhbJD1xGRScfqQA1yDazEn6CgMZniCF0Z%2BAHTasuom/U2V6RbjC/09GVvKAekxQDdvx8AD%2BU8TPMqSoX8qiAiOAZbonFeaZw4icpXMG2tcCUYSELtOZSN2w2PKCwFtsJSFQALVSKwfjG4y9mYgi3nU0GzLBJINf0h5ANqDhuAL7CQqZ7BHIlHd0mH0TxD8z8oXD26qsvFwHPoM/UZ2FFnMGkTWsFMqyv1jF1L6TbXavprZsHwCO4pb4nforkf13AHvwbPoXfut2p9HFe%2BtL7g%2B646Nf8L6794NlWYXck5b69lHu9MRf3BnMQ6Boibqua2hzZRuZHSXKDHRLwzKB1QJk3EhIsW6D7IHrJtAwXEcETWO5dVX4RYApYwwwcmiNh0BjBEeuIWUZ2C8ktFUebsHOLNE9hgTa97WqfHJuIl%2B4/yy/iNk2OzfuABlrvN0bDh0G9XwJbMYnYHEeMAAIFLOnOAwB2C8la0dNvagWx4GAdH6Rr0vIoAHupG64%2BCB5gkoM7KB0DZGH/RJZJQSHNaK4ABuUQcJEKwg00BkFEXM76AB6I9ybbrukl3eX5DNhNfK1SCx0puojMnzOtbuMuIYmPAFhgnHwxc4dvAZgQ/uiXVYBwIMV9YZwnc4spFRZdbfYIyvU17GQcaAiQZ8jjETEf%2BR1rDxAbh3DCfHwALjcmN9s6yqCTUWgZbIdAMar5DF5hFSEcLv0Bc4Xshk1p4SLCHqT5MaDEGCgsjTA6E/rgl9cE2CDJFYW9tkxjHjjgwuIn8xmMmYYAxzfRrdwKs4bTS4gS8OBBisOA3g8e3VlqGPBE4mMoZlEASQuC7NOwIw3KWqY5ArAfhsZ1Or6TuSV5Gl8Y6BBc9xcUWjrdVmuQFOI3YQ1RY0v8F2sVJi1Q00fsKeIrPOmtwy1SE0zLMybuULKrwISu4v0D3wFsFxdBil2QEIAKI/6J0QyNcAVMngJF6teoxADqXRlDrDF4yNgM6B2igEFrk/eJDADhUvQWMrAjEEQp5UB5BALM3RaIIWCaoWCEqiVwKeZqkDwSJA0N3NqEMfLtniEESALJLa0GJrSiblpBrDIegwdhF%2Bts5EhQ7MDFhHiIJDeLx9XKFcOWO4raX65XHjrTEHTW/2R9mitIlacX2cqEhFGQDnkSxH9AJm7A26ZBHRoxSBNYBfnj1Kfnj0GGsgwvqTvpb%2B9YRNagh1NzaiXmQqiVxHDc9xCII1GWTaMIpn5WPMJVmjiZPVWeTxEnFdXxtSUtAfpag4kZTp0ADp0zNGyI%2BOHmtrWFmfQcR3f4FxCH9QghPwYSQwRGgkEw14KV35gjKoc0blyQcml9gGyvDqT44Zp0Zqxl5Bjf0FGRIi6z0Bft7XOhbHoQHk2lJ1cggIEBi8fHyRGWcXjS00g5LuvcuqGsK/DjW9IZBAsOWY4mzTF8jlq5X%2Bfnz0OHnPAcplKlBwSjHAeUWCJQkYyhimkfoCWxK%2BQIfhuFIa/eEpw6wZyG9OmCCfmzXCSp8PFFyRbM2upnbLwkQDfbEjoqOw49AF1RFkDqF31o8TxEAXIPMJqOBM0t02ALKcbCNdgo9XfUKzwO/MHyM/P3a8/PmUydZ8aX8BbVMKS2Mm49VubBXsl%2BimYfAFdhtZ8ck56bZsgKaEPzxqrPzuhInyp5MnS3FEtdSBgGwk2cDIxnGApRRUgfVUxup4a8CdxTgG2RAECOV1RMbouIXA4zMWIEfibWMK5e9nTpMnL5pthuf7JpVYqR4NClo6Os2hI4LVHyAKhdCNdsaIWF%2BjBshjbztC/nzqZPnk9OFWBsR9JKtB0aAOHE1BHI7egvV8/7HOpWLZRNbP0W2ULDXZMLLLbKsBoD%2B6iN573gy55bQpyhej5YpJxRa4yEtOtHVGkcbCwQYRWTIO0Thdefm3J06Qhy6YZc4hgR2aZhAIwmApb%2B4/XnUcfoBS6GZNdoLMA400MDPprI3TR1dbujQGwIH8qjqH9583U/50yiTrDHhp2LEk07xL%2Bae7tbYHC2QfQpnF8AcKxo1Bjw7jRRMlutvSsIdtn/g9jpzD0VuQ3UIfsLSFrCJlmvRtOEPlL8E9SvODyg8cMjLZfzh5ktx77gy5/sSJ8kmlv7NG5toyHNZw9zf9UTbKKxJkQQlq3nPODLn7nOlCAoAupEcWZxpv07MiOkjvcPQH3FmMY5A5JDpGCQPdpugiRxZldXWTdTYMQBSWRf%2BUSyDQqB3/2gvr5JNPrZbPPrPWSivIMOyPAYBiDxxNMoQY2dGgE2XI0A1/oCDzg5A7e1SeHfNqeYMtxv74k6vkyqfWyBeeW2vtmJvUCDgYoEypu7ERKaTMjn/VDncc4tjWEGpvTwSWDMS5Sl%2B0x8fIpeyZ6Ce0SskQ2wOQJZmUnWZGAQv/P/HkaqPBzz6z2soziabuL3BY4Q8UenflrPNVoUd3kySqzTYebPFCGSdrn2gIQEMcmj798GUa3DSbcXygIKtflNJz1Jk5OtDSPkd8ALqnFAx6ZE3flHCpGhKfz8kiByBYQdCOroes3fvF4i3y2afXGG98b8FGKx3tLuvWW2DwUo5JRUp361rJPJLVjFwaQZb90zOG27KCSnVWWTP2deWLDz2%2BUj73zBq5aeUOWy5woOCKOckJtm6NLGY00BOR69odhy7QA2S2aZiHfGa7igtHF5hegB8o6wyWDiCnPzdzpFWogMe21ViTpY88ga5YI19Re6W7tba9Bc4p6yIJBhLMie60C9AjwbrKAGzbgnNIwoCyzptW7JAvPbfOeJUGft9bsOGAdFgkcD4JzlCNEwkysywf6m7MjrcO3FKIY2D80lERpUo7cTppwbjU2PMKwLYVU/PSLLP3vCp/Onn9bfVOcxqfUIHIthMYhRzbV5A1pGwCgYQwwUBn64LgXAgWsjpEpSOVP5t%2B0xmOJiAY6v/ZVGHjwgi4e0O5rf2g3CFaMPUFlHKgCMjeYIQzB9ENHmizjpKg8QLrPHta5O04NMCzhp5YDwU9sk8W61/p0gltEWkFGKmhDcHTrPSNdasB/cEXj6sxAE0fCP2xjQVGB6Xi8Ge0Y3jKsFwr/44EvAMPYRSwRoVmHqwzZvN/mj7RnIAS1f3h1UgQ4IF3iZZTlRBphGAk46wS0MGIiS59chyaWFHVZMsT2Ej7PZOGWhMisirRThaBPZ4967IIoNy2aqd1q6Y78As7a81higz89RV0mKS8GQOT7TcigZymazE0GEnjrJOk%2BypO5uNba6y50l9Vh8GvZIBYF8xejQeC0PrOFjPY0U2RvQAADitjho/pG4COrWxx3jiUwTo7msDUtXfIUUXZVm1FkISmeegRnjPAUWNbMapRCNLfprqCtYvQH0FFSqcPZH0742CpDjYSe%2BzSDTvQPdAjegFZHR1cmZ6fLmx7xvILxkOjNppG0XSGLTLgl0EHqCtY1856XWy2c0rzrayVslnAeAm2nzAs25pdITO4D8dbD%2B4sxjFoXINBSWMZlCwvSuzW1DZZiUUAWzulzhDlCOyLhuwgEoUyZO8t9oBCAEQ6c70Fa09wyHipXWmRX7KYLP4nk3m8ChGcMTYrR%2BgF4G%2B7nv7PmBgbxj17vrH2hMg252J7kEiwzorSQO4Hg4IuldYwQQ3oaFCOS4dGHAe6RlKWyB5Gtu%2BjOolch7GepgYLc8jGyd2ta3QcWsBhpI04kWOitKw/YR3jvRsr92wlEawLJNNCRp3mLBjBBA%2BgXRyo6AYWfQVBGBoocd53jC8y2qNJjfGF8sSJyhvDohxI2ozDF6xdIQND8AJDgeYiHIMTR2ltdxm/dv0tLMaYCc5g1PbUUANjiGYizA3l30TMGRcvHGw2Jcc4oXSXTnaOQx%2BUob68q94MzvdNKrYAAduf0O0xEshm/iNQQFABI5EABp19ydKfpHTbXdatN6ArKWt1oSt48/2Th9oWMNAdzdkuGlNg283Al/y3B0rXOI%2Bv6%2BcEd%2BARdBh8cZw6dsj27vYs3aPz9D/WdNG4pCeeJrh4/6YK0wGcj7Wa3DO6gvFhKBOQpUIl6CAZqWcdhx541mwZQwCjVOUdW8sgd6lOiV6bB/1hwTQoT7QrXZFhhjaOUfqlSgr5DH3uD6gWgS8IzLF%2B/tQROXKsng976Ejli3eMLzTaTUvYm%2B%2BwgxgTvEr1AA4mW7tgO7F04RMq21mHGQlGiBOM3kM30aEXfdGTrkBGsCclzjCBnNNH5prtxL1zHcp2g4wjW1LR/dfx1oM7i3EONm4lWkxTG15EgaI3qGeDcVraU9pzxcRi%2Bfkx4%2BVnx46zxgA0zWCDb5QzBipKOIga9QUPbK60TpPjVdiy3oW1iL86foJcf%2BIkW1sSLUKDpjcIZAQQG%2B%2BzQf/Pjhknfzp5klw1f7QpaIxlRoNxjBFD5JcIOUMcn5Umn505wsowiIR3B8puyVTiWCPQqOn/xXETrDHCDadMtr0VEaZkR3%2B/dJvtVec4tEGJ3H1qAGAIUEoJrVCiE0mDdGijhJsyJJyiT6lS/Y3SK3xx7YkTrcFSsOcV9IeiDKnl3oNOdWx/QTdE1kF%2BdU6pbbpMQ6l/nj3dFG505nJTXasZLslDBlsg53c6FtZnGS%2BdNEm%2BrXwRjIIgUGT2hfvlbDh%2B/A4HNXKvyUiwVx3BEfiejZR/pPzwC%2BVZ%2BJbNlynH4syUwT6%2Bde%2BOlY5DEwQIWKPFGnWWLmAw8j46EwAdEXChHO%2B2M6YaTSCbWb8ILeJkIovhq75mLeCnv61h24Eayy6eorT6l9OnWvOcPyh9f015BCMUvRBpdwdZP7bGuHLmcPm9/hZehZf%2BduY0OVcduWBrm8CoB8h/mtUw1q%2BoXvqmyn%2ByR92BjcLZmiOUvR8s75k41NbSoyvQGd%2BeN9qOxUEkG%2BUZ98MD0OS62haT8ey5C72w7j1yDWJLZ6dsrGu2dexscI8c/7HSxU%2BVBqFf9uccnpFsQUdozXgjfGxvQEMngjkEOTkWJ49rwHfoo2/OHW16ivNHAlqEXwma//6kicYTjOu68N84mgA5H%2BJXMTsPO7G2tVMdyxS1CYfKd1Rf5PWgK9hWZ%2BGuOgsaYkP%2B6KixJhPgC/gWvUGigsQFwVGqFRxvPbizGOd4uqzWSg4QZrzYG4qIZySsxHPlDtunChE2uzCUpSAyu1Udy1v0O9r1YzgTST0zahPu3oC9c9iX68/Ly%2Bw8OHoYwzieL6oAoVSDLGSAdWq8/0l/y3gpcaAclVI9jGraqrNe8Z96zDJ1DDFOPji5REakJ6mT2anHNlsJUF7KEJlXGNosdlhG9%2BtI%2BP1NK8psrQ2bU2N/sA8QEXK2RrB95pZvkw8/sbLbZiOOQw/NygevVdTLyppGKwMlmELLb5RkADKMbM79w4Wb9LeNlm0hkwj90TGY/dloCgLdkoEk0JCd1LdsCmWo7Jl4%2BSPLzTFDWbP1BdmQ%2BvbdVuL3UhSvrq5t2lMOy6bJwUberDPDYKZclq0%2ByJRSLsjnAbgW9Mz4GS/ltz1lgLaqAXDLyjL59NNr9P4bpCA50e6fLUPIvCyrbpTPPbvW2rJjLDgOD7A2neg/nLBY/6X8LRo3K11cu3SbBRip2ggyf1RvsLfaOx9eZg4mmZWh%2B7l%2Bj711v6t0/FpVg%2BQoX6F32IoAQ5MycMrFI5s4sTfqR59YZdmXRLV4yYQwLrKJVIX8/NUt8sdl2y2AEuyzaMfp%2BdgPjwAS288cXZyt1%2BmagQxAhuZTT622%2B2dZAqWn6Ap0BvzLurTvqi75k/IFXWQdhz7IJAbNbKrbaA7YaEGUSMeMTDJrE5G/VDZRwn2CyuW5yhfQKeWfyGzsjVHpyUo3yXuCF70FzhZrH9kPm2oo5Dj6iOoPsv8sIYreI/QGtaH4/RrlR/TUUcpHZDrpAs49QKvYSvAuASKy4tD4I1uqjfcI9sBL6Jh9LbkgaEifi1tVN1F6TTk2fIF%2Bgrf%2Bvnanra1HXzremhj0%2Bv7m1Q8zwKTz7lgYl2UnCAEcLYDz2J0BwG%2BITCFEcOBU76kwDO0ZyNosWqUjnMwZU6HFXjoIqcAYRYkH5WhsqkwNP8IVAzUABrft3aXHka3jGuxHWKPKnH8pj0B5c64qvS4KnfKmYSpYQ2tURBDPRLy5B2wFzkl2CAeRcRHt4x4wuqmVR7xhIJMlYgNqHF3KK5apIcQxATiGcgs2Mg9KWxGaHIMxHL1uh3vAGGEsCOnoReJ8x/pHMpYYMPECntfCy%2Bbtd5lYX8EievZki0egaHlGZABWqMLkWUUKM/gAmmdfKkqgiSxDf2Q82EqFdSgoUPgGBQmtcy7ez1OlTKdenj3lPzSWIhPDlhYYFER7AQZmSkKC0HkSGoTPGAPHQp%2BUf0PfOH9sGo60ZSxEasmWwyPwEesLoUWuxX2gpDkP/BTQLs%2Be4%2BAZHONdOoZFes5SNfhxLMnSo/ThK0AVgZV%2B52dYJ0AaiwD4Aj7D2IDuI4121o0xZjLwRJEjwVxSKkXZIlksMkGRjVMGEmRlqXCIBWguNv/OheF38QUyAxh6PCsyadBOtE6D5ihhhp6CEjZkKvRDB1%2BWHGBcQutk7KETaJTtBIpUJvPcyZLwGWXXQRaTAExk4KEknbGkG73Ci2Q/cMDgMcYHyM5Da2QyuR6BFpxUtoKCV6FVeIkgS6ryGQ4hRj5b5KDLGD%2B8OVnpkmMIuGCUU5EzS/mV8W5XXUdmKXIekAnwEs3Zgkobyl/hPbasitSxyBiuy56ozCf3T%2BUM4Eh0CedjawTsCDKX8QAqep66eE74Xf/jikeXy%2B0RXXfjBTxe%2BGKa0gh6M1g%2BEA30w1wLpoXkOM%2BWMmfogsACdEJ2Ef6B/pDV0DH8hr2EE7pR%2BQW5y9IClhOw1hZ%2BCvZxJivO5/AfeoFzIn/ZfgmZnqGfURnAmvoABHRC69xD63zhC%2Bicsmr0CkmBHD0O3qOCiu9wZGfkZYTGq9dAv6Ab0ANUbLEVB8EiGg8GfRyYG8YGP%2BN8DtEJoIkbQVFsMGRBkG1nHKPU3oLumTdsRe4zEswN5d7MHx2Z4a14AR2gf3DU2PA7R2/gzmIY8ewsOhwB3Fl0OLrCnUWHY2%2B4s%2BhwdA93FvuOvuXRHQ6Hw%2BFwOBwOh8PxloA7iw6Hw%2BFwOBwOh8Ph6AJ3Fh0Oh8PhcDgcDofD0QXuLDocDofD4XA4HA6HowvcWXQ4HA6Hw%2BFwOBwORxe4s%2BhwOBwOh8PhcDgcji5wZ9HhcDgcDofD4XA4HF3gzqLD4XA4HA6Hw%2BFwOLrAN%2BUPY6A25c9PSZSvzy2VQv03YfAg%2B6yt83VZWdMo/1izSzbWt9hnvcGEnFS5eEyhFKYmyQ3Lt0t6YoKcPCzHrvGjVzZJY3tn%2BJf7xsemDZehaUnySnmd3LexMvxpbJE2ZLAcMzRbzi3Nl8e3Vcu/%2B3kc755QbPP02yVb7f0npg%2BX3coaL%2Bysk8UVDfZZgGOGZunvh8qfdI7X1jZLU0fv5vVg4K2yKf85o/LljJG5Uqx0GKC%2BrVOe3F4jf1vTt42fjyrOkjNH5kl5c7vctLJMSjNT5H%2BmlMjyqkZ5ZEu1lDW1hn/ZM6blpcv5o/OlSHnrt69t7RNfHmycpDx92fgi%2Bc%2BmSnl%2BR63UtHaEvzm4GJ6eLJ%2BbNVIW7KqTF3bUSVZygpyl8wjYfHtHU5ucPDxHPjL1jc3wWzt3K7/WyK2rdoQ/iQ3eCpvyF6h8ggaPLs6WzKQ3%2BB8N/s0X1/eZJsdlpcp3jxxjdPR0WY29v3B0gSyubFB5WyFVb0JXaCtk9Fmj8mRDXYvcrLw1UEgYNEi%2BcMRISUkYLM8qTzy2tTr8zcHH2Xq/yINtja3yf2vLTR/wTNbXNcvd68tliI7lPZOKZV5hpuTpM0OP1Krs%2BtHLm2S7HhMrvFU25R%2Bmcgq5dLrqi0gg3x9W%2BY786gveO2mojM9OlaV6/L0bKtQeGma2wbNltXq%2BqvCv9g3o48tHjNrDW8jKgcD8oizTo2lDEkyPV7a0h785uEA2wQdHKR/839pd8prKkBn5GXLljOEmm9C90fjA5BKZlJtmzynW%2BsI35e87PLM4QMhKGmJGLE7J7IIMadv9upQ1ttkL5%2BOd44vlPROHmtDpLQpSksx4w6AoTE2UESpET1TD8hxVbslhR7Q3OFrHhYCZktv7ax9sJA4ebAL7fHUWp/bzOCbodZinWfocApyhymdCdpoZv5HAUP%2BkPrOLxhTIsLRkHWfv59XRO1w8tlDeoc4Qzz/gCV6js1LkXROK7DWkD/M%2BKiNFTh%2BRK0erMsOoJDBzrjqjc9SYizS694VidRJPGZ6rvFUguXr8QGKsGvUXK/1Nzkkz47i/kJs8RK6YWGyBifbdu2W0Otln6byN1usn63XhmXeMK7JxYAzxjBjb28cVGn84Zxw8wAvv1mfxngnF0qnOR8ATLR27TY9g0J5QkiM5%2Bsx6Cwzgy/V8R6jcy1Z9NA55q7pjrvIFxuWbQVnJeJJAzLyizPCnAwPEAbL5NOXz8UqD/Qnm53id6xKV/1z3WHWYTxqWLWN0LphHxvBe1d0Y0DyjDtXtyB94iefoODhAlmOrvE%2Bdu%2BPVMWaeA75ALvKM%2BA45RfC5t0AvQNMz89NNz3Ae3k9UOddboC/eqTpshp4jU2lioDAyI1lOVdojwNGXOegrCIqcPiLPngNyhSAvzwa6T03YW5ZgM8Ez6IlLVdfDT474R8JVivDfb2nUtXXKH5dvl8YYZYlm5WdYBOuy8YVyx7pdcu3SbfYvkatXy%2BvlvNICMwIa23fLqpomaVajAOA8TlbnaawabmRIyAA2tHeaUzNSjWIUZm5yoty3sUJSVeHPVkZEgf1l1Q5p1t/AxBh3ZKgw/oIXWcgg%2BnPRmEJj%2BKrWdmnvfH3Pb5JUADOOdhXKATAOuZfgN2Q1MWbIYmJQMt7hKrAwMFHiJerANuk5AuNkStQ4MExxFBknQm52QaZUtLSbMcoM4EhjoDB%2BnLzguA69ZquOFeWN04zA5xzMES%2Bc86qWDukujU5kfE5hluzUazy6tdqE%2BwcmD5WXK%2Bots8p9TFehz/U%2BPHWYCrkimwe%2B29rY2sWh7E9ggHxUDUOeRSzwkNIj2dVYAAXPPH9r3mijx/9Tfvj5q5uNJ3gNUuOAyCVO24Jd9RathxZRVNAZzv2YMD2kqGKEbgDfHam8VNPWYZHeEqULnL4N9S12njr9HMNvqp3jDZrihbFAIIcs23GqCOGtFVVNSk8JRvvQNtdqUj6FBgFK%2BQilWwxDxjNKX3lqxO8MVy0QyBmjx3I81%2BBfHDM7PGx8BvfBa4ReI1u/h895f4bSKxHj7UqvRIqZh8F6IPw/U3kxOBZeJMvB/QGug8FTkp4kE3WuOC/3xnmjkaQ8OFbH/1Gl95/pM0AGcU/Il/t1DslifXjKMJmrTgKyhawJtML8HK%2B/Yc7/umZXt/zWH%2BB5xsph2a7GKFUFsQLZ7LepTP7glBKj6asWbJR/rA3pipdVV8ArBFeQqWSuqJIBo/T5TsxJ12cdojPkIAY2lSvwDTSNPHtuR61lE6EHjD2qJfisTumCrA3GXECLvNA5yUrjZLSnK72hpzjnurpm0y38BtmLLEavBuB6ZBKgveBcg3U8LSo/oX1onIAd48DIxflCD9Uq/RLwIXhJ9jM4lrFxP/AdY7xwTIHJ%2BUodF7wR6LPJes3pSo8cwxzAf%2BgKZDhzRwYEvQR/cH6%2BR%2B/ifHRHvzwL9NRj26plg94zQd3UxMGyuKLRsrGfnzXC7uO3S7ZZNme18g76BT1DtmpFdVP4TP0L7hWaiRX%2Bub5cllQ2ht/1L5DLBfqcPjtzhDkpK3WOr160ZQ9fbKxrMbl72ogco5Ony2qlPmyTIP%2BxDwI6glaTEgbtqdA4e1S%2B0SCZ4mf0OOwheIt7e2lXndHcTH2%2B8FUgw5lrzouNBV0h38kuL9PnjYwtUjoOfgMPB7qC8cAX0GdA28hn5C9yWQ8Nfa96EQd0gvIXfA3fcA5sQWT6WOWV4H4Y%2By69BsFsdNaRqiug1zXK1%2BWqgwL7iyBRpI7iNwHPME74j/srDf/LMVwzsEUjwb1hyzKHT22vkdyUIXKa6umspETLNDL3AdLVLv3y7JFytOo55hLd8uDm3mVsDxYIIGBfOnoPL0MNAwU7/46Fewy6/gSMTukpkUZKLL/y/DorsQuAAMEB%2B9mx44y5b1hRZgKQCM1fTp9iTIiwxCmrVAfoo0%2BstPI8DMhvzSs1QfWRx1eqgZpo5XYIl9PvXWQMS6QNZyMyW4fj8R91Lt/9yHJzfG48dYopdSJAM1SIYVggTP%2Bycqf8bslWMyxAqhoM3ztyjHxmxggTiGBzQ4tco0L7NjUgC1S43aznyk5OUAMjWR2dIbJNDa3PPr3GjAFK6Y5TgcF9AJwFDE4cNgTXF2eNNOXfsXu3LFSj6Hp1qO9VB%2B3rc0fJhaML7Rw6lZKsgvUbL643oYSTepoqjxtOmSSb9JliuGO0PLW9Vj7%2B5Ko9pbgIqWS9J54F5WsnDcuVZdWNcrUaxsz9d%2BaPln/pnFyrCh/jgTlhLqrVcGG4mTq%2Bjzy%2BSp7fWWvGTKxwOJehopxuOW2qOU44KN2VphCJ/MHRY2Wp0uC3Xtog61QBnleaL1cpHY7NTDV6gG7vXFcuH1G%2BQLFdps495TA4h598crVl8ilbxNiDpqpVwV2hiv3j04abYqZsDJpJUX67bulWuXHlDslVOvzKnFGm0FGV2Tr/ZCUxhinZ%2B7nSPEY2x5mjdMY0M6qT9APoe6kaGuf95zXjwUvHFliWaFY%2BGZzBxkfP76iTW1fvMIX%2B5EWzbQywFHQKT0JnX3p2nXxkaolVI2BMtClfPFVWI79avNWcBIJPn9B7UNPbnEeMd3iCuWzTc/zkmHHGc/VKr8gI/v3aC%2Bv3mmfmjmANYydD8sOjxsolDy6RRRUNFin/lF776y9usDHBNxg%2Bn3pqdfhoMQVMWSp0ijwNeLu/EdMyVJVFR%2Bq9xebORJ2RYnXMSyz4d/RdL4c/fQPQ/O/1/nHm791QbksOkKVfPGKk0T4GIWMlcHCd0vst%2BryXKP/MUwfm%2BUvnqpG92Rx%2BnL6vzh5lhhtBmnJ9tszpz48dvycgBn3gLN%2BzsVy%2B%2Btw6o6dPq/yHXjYrf2GEA5y1m1VvXaOyA9pDp8GDV6hjFWTX%2BOzXSrs3riyTrao35qn%2B%2BtPJkyxgAu0jY5eosf051Rc4PSwVIEgD0H8Ypjjt9yr//fmUKXZeZDo68aHNlfK%2B/65Q/krQuZlozjR6jHnAwcPhhm8xtv%2Bo10QPUImTo7oKh%2B5zT681PUdwUodj12O84JrjxkuGnvd3qhswdH9/8kT7/J71FbJWz/0HfRYEU5CbGM44K%2B9V%2BfLZmSPlq6rrY1VydziXoZJBx3aA7x9RmwEbiZL8SBDYIuN%2BstpLJ979ijl/paoj0CE/Uh0CXcI72H7/2lAh31Z9wme/OH6CZcUe3VolP3lls9ykuh9bg3uDfwjY/u3MqRZkSFWa4BzIaYLNn31mjQUkTyzJkYcumCXP6JhKM0IBCmQ6wb3z/7PE6AZaxLmDv5Hp2Ubbg2wp1D3qeCO3sUevVP6ipHWH/o3TxhivVTsMW4lSSnguqDAJ9MVx/3xFzi3Ns0AG2U1oGEf2vY%2BuMLo/QecGuudeOIbAyBOqD9%2Btz5DgzQcnl9g8EUA1HaW6kPlgngkWBoAn4I35RZk6p%2BNkpdpQNywvs/NSAUTA8mNPrJJNyg/oAq6FrEDHEWSFV7GBP6fzFkt4GWrf4ZnFMIji3L5mpxlzYb%2Bn3zA%2BO83KfYhqXr9suxp1IUYKgFAhq8d6qvs3VclqNSDz1fH7%2B5nTLAL6Bz3mi2rEs47vgtEFMk2FVq0qzkY1jPeVWbxImR%2BjmAzbOx5aZplUXhirJ6uzRHQJ45N1gjA/5yPjiUFJVoeadBgdYQgQNkStyAB9XI1wznWKGotWVqDSCcOXqBwRYQzzb6mRydgRGJ9UwwGh9OvXtpqjx7FcH0MbI%2BD3y7bZ9cluYEx87fn1ZoB8RY0ZznmXOgMYqawRGapzQrlDhd7XChVWCFSizGRccAY4/wNqANXonAazzFqba46bIF9TB%2BAMSk7VuMVJpCzr0nGFMh5jV%2BeAqP6zashT3vu9hRvlLr0ea0sZ51shs/iEGjtkL1Am/Q3mmhJfniFrjtYrX0Rjqyr2e1SxQ4MEHlC0H9NjiHldrE7N9Uu3m1JE2RNVvU9/R7ajp8wia614lgQ9CFL8WA1ty5IpvUAj45RXV6jxiFyABvgt48IIvlEVJ%2BbnpWqQU5FA9nuWGuw/VEOELMmVSp%2BcC8ORYAQRazKZRWo4sMYJen9YFTC0ffOqMsuefHveaBsXRgfGAms/yMpjSOOwoajL9DozC9LNcPnNa9vMCcD5PHNkvp0Ho4d1WxgoyIP69g6bN%2BbjKKVpjP2L7l9i43%2BlvGEv%2BmUdyTfnlcoXZo2UM0flWfSeyD1Bp3NULozUMeNEEjSBB%2BGHyMwkhsuRxZnmNN%2BiTnb/U00IscwsYsCxbpYAeyzuj9LfGXkZsr6%2B2bI33eFlpStkIUGHHJX/OHA4Zqwf/a7KLQzdFHWkCKQRMNtY32oBp31lFuHFC1XWYsy9Xx0vZHSD0jkVIWQtoB0cLJ45DhHZnUCvECw4UfkFHffizjr5ssptZOurFfV2rj8r7eWkDLEIf6KOi/VgnIM1k%2BiOv6/dZQYkjuCn9F7ItuPEYnjesGK70TOGNjxU0dxuvEIVCsbnX5QHyDKhX%2B44a5rRLLQOT92i3106tkim5KVZIIlgB7qCbOcdqse%2B%2B9JG01EEWuALni9yCT17%2BxlTTfYepbxIMPVc1eEfnDw0XEWQZs4qcvKbL62XfyufrqttMQeVMX5y%2BgjTxw/o54drZvF%2BddCpugiyZv0Jsm8EKrAtblY5g30RraPIkpFRvH31DgtkjFJHkedF0At74PPPKn1tqJQctY9OGZFjDhfP5iSl254yi8i6q48bb3OLbGc9Husa0S3IH7JmBB8ZyQd07otVfn5/4SaV8evkJeVRqmIoayVgAW%2BQMKBCA77md%2BgnnCj4gMw%2BsprMOAkCzvk%2BdfZ%2B/doWsxfRfccrX7xHHTzoHb57Tcd4idov8Oj9eo%2Bra5vMYSOoecUjy43PoPevzC4NjVF58Reqy2r1WvAiLwIc2EPoUJxybMkvPLfW7Dyc3CAxAL5/1Bj5iTqJ71LeJpBKlhP7EX7BSR2hOg3du0bHsVl5FgcbJ42/cTIJTKIfPbMY/4iNxXkIgMgL0UIMr/7GqMxkM%2BQQKq%2BYIb63swErosiIxrCIHqV%2Bujo0KOU7lWHvUCMNpiUC%2Bu9NFTJcBSfGJFmRfYGF3mT9iCRzfPDiOmQphul5iLACjGMau5CZYAEyCpSxTlPh/H4VuLwwYDAEblqxY8%2B5XlQjg4gwUesAGNJkVjBg%2BQ3Ne67RMRC9RjgGxyKcUa5Eq3BoibARRcMYsDHq3xgzZFEwnDkGR%2BbHKrRRDGeOzJXzMDbCYNz8DuOHYyJ1Cdf6o94TAnqzGk4ISKKIlA5x7y/rXBEhwyCuVicTp5bxE4WMzALHEghXHPv%2Bp9A3QKQ%2BiFr2N/RSkqcG5M6mUGlld%2BDZ8OzgC%2BiB0lMisjQyWKmKHpogMML7Y9Qhe9%2BkYilK63mNIcr/FaWRLz27VggsQAecAyWL04aBnaJyAUA%2BXPOWlaHoKor37vUVSqftltUncEIZ6EjlwxuVdnAMOdeTarAT2CH4QBMGjEf4jIw0mWtKm6Ar6OunyhPfW7BRjfU6OxYDBT5PJ7qr48AYxinFMILmoWvKssmQkJ25TZ0CeOJhVb4Y%2BYDsEuV8IJg/zk0wJzorTnT%2B98oXOORVakj/Vx1ojAmc5416HPf0vy%2Bss0g8hn904waMbsqlcMIj2K1fgXFFxitWINiVkTjEHOJYAHkKC%2B6rOQUBBIwunB/44TSlM3joRaWtF/SZQmMEFnYqzVAmfKzK0TcDBhzBE6pJoBder6qDCO%2BlJ5JtoGojNAlkFMjmB79D17So3CdwyE/gDbJ06BC%2BX6E6gKwiJZo4ggQ7AiCvWYoBfXJPlBZ%2BXx3eG9VJ5Fj4nGvxHbqR7CHGN%2BfnBV%2BwxpYqGpxWMk8498GxOMI4u1SKGJRQkQNckww%2B50X/BvqCYAt0/wPVFehE5AIVLj/QMTEO9ANO6A9VD6HPoH2cWdYx/vSYcfIzfXEtnHnGEAvAD/BFLAE9BNnX/gaZLuQoDj3B5e5KI6EFaBU6wsEhc07wHF6Bpgl2IHNZZsHx9EfIULreFzgWuwEZiA4I0Xq9/H3NLls/TKA%2BqPrp1GtC72Tx%2BR3XIYBGyShOE84YZfzwLHYGsvcJ1RWcFznNOkMymqC5s9NoHH7hfrjmX1btlM88s8ZsMM7PiyAH9hKltQRL%2BJt5YJ6Q%2ByyxoUSaDD0BlMCOonLt6e21xosEz0kKIOdY/4k%2B4HfQdGRgEOAoYxsybnQJCYFvvLDedA86DRuPpoFUABHEJ%2BCKo32bOvDw%2BUAAGsVJdfQNPmNhoPQwVGNhdCBMMMBhYAyASCemO7DeifpyjGkiyAgNQIQYRcq/RFkDwdITVtc0mwGA8UnWhOgzGQSEaKD0A%2BCoLVcDAAFDFAuBt0aPz1AhQ3SMF44EypffBUAgUr6E44rAYToR5rta2mztFOdCODIODHPmgjHwQqGm7UPZDFEG57wo66CDJUIe5c%2BcEMVingIQ6d7RQ6dLjNwHNleqkN9p94qBjZNBJJso4VK9J75HmXANHAPuYyBBua01sOh/Et0Dop8Yq7HAIP2P8kqUHMbem4GAC0oRBYbCDPgImlysjsxgJT4ya0SFewLXgnbIZKDYyRJAi5ThEBWOVCo4aBjbKGqcNJRjKGLaYrROyQ58SGT9ng3l5mwBAh3/3Vplype1KThT8AH0xDUDBczvcMIoKaeEjJJPyucwLvb10DkfJbEYuRgTGC1kv4iSYyyxroxSIoARwmc9gWMwVoioV6oRjNFP1Bv%2BILOPU/A3NYwwrKP5gbIlqg8wqDCMY7XCgcxArAIagIAiGX6CN7FAKIAZWjvbG2AMjc5MNRrdorTJM0dHUD6LMUlmO0RT%2BwbPHFqgeoIMFevDaHw2VOktEvDqKqUbytgCbKhXo1XpH9pk9PAHZXrQKIA0OD9VMxjYZNoD4GzBC4DfYUQzDoKOZIU%2Br/x5idIa2Z%2BeAM/D%2B8wYpayBYcoUonfgOdYb0%2ByK31DJg5MZbQwD5D98/ve1O%2B08BGWQNwRUyPxznxja6A6M6gBUxvAd90OFBmNi%2Bce%2Bxn2wgKO4L7nXHyCAQhAhFoD1CBIgk0P53zcHyxywpXBiXtXnhP2FTcKzhS7JpiW%2BiRxBpuNYEfwjuP7%2BSUOtVJv1qGlRepLA92PqRMGHgCVFZOkAYyErStYa/YEDCJ9Ch%2Bgv3uNcBQ2roEHGyDkB9gt0TLASp5kMJXrrsnGFFljqSTJhI1LBQsCbclR4CqzWaz67o8bkKMs0mCfmFvqFppmr7oC9R1aXOeS36L2/qk0FX6AHsKnQafxNppHgEeNGh5IsGQjgKKfFiE4PJ/iMhYGTCHNiHPc3Qsbw63ZNDPGeGJuSBhQ/L9ZxITA6I%2BQi56GlPUxHNDnIHvQEDBwyH3SCpKyMOnmMUQzYaGMLQzA6ks2YkVWMhRdjaY8yCCm92aWCASeQhhs4oYwzukSEcaDMMTC/NHuUdcCkhGFf888IURAodOYiEkTRMcAj74OoGgK4OzB%2BBBflCBgyOB5TctPkGHWcEeAYMAhzPosX4BDgjMQyt4gxhUMWC1DSCZ2wzq8nQ5zvoC2%2BJTuAQURzGRz%2ByHYULUqcGAKUg%2BKk7QvwGZ0koUcaG1F%2BR/c7yuL%2Bv737gJPrqu/%2B/9ve%2B662qHdZkmVZlnsBbJptwGBjTKgGU0NLIT3hISEPrycJJAQSAqEaEwcDf3oxNsa4d8mWJVm9a3e1q%2B296n8%2BZ%2BfI1zO7q7a72vJ9y8f3zp27M3dmbjm/e1r05lG40REu2GCW/ZEgl%2BOL/YrOPsggst%2BDfZBqN1Thy3fbTukwn7N74KX7Jn/PjRu246MuY041pqtnF/nS09E%2BAlWS2Sc4JqKHIwEC3ws3YMLNIEpG%2BV5eejS%2BiIwsXf5TzZWq73y/lJpyN5rjhOPiqqoCX%2BUoHKv8HhxHZOJpY0z1bDLSI73HWKNN9lB7n4lB5y4MUzRRASr7HUdH2ig7Ab8B%2ByD4rbk50N7Xn5DJI7PLPhvWHQ0ZWm4q0tnaX62b768VlDxwXESxv3ODMHoTgswo7x1Kt5hyPMQHYzxmvXAzA1x7wo0IPjHVOzkueH/ab1GNjkSmdyQc82S0OVYp4Qk4R3ATkY5o2FfDZ%2BHYGan6JMcX67GPUyWQDrXm52X445S2X5T4Uy2VbSQPEZAppmSFdoqf23jQ3wCiExCut%2BONNtZUEZ5InB/oXG4isFez3/A7828kXEf4nTkmOF5DfiSK/FON23fZB090VHDOI%2BihhhN5qNtWVvp9kTwV15EomhNRLTfsy9GAj/4kqOLP9nGeZt2A7at2ASTXnnDt4npBU6OwHvlGOmN7xexCHygy3Blt1qniynbQbGg47LsES13uWsS2hHwZN8gPu2sUtQE4bshzsp3x31U8Su75Ligp5bOE6xd5Oa4dlCKSz%2BL6QRDKeeo722rPWu0s0Jldqfv%2B5dSc%2BIoxQ3Bw0GCag3i8HWzr8YEMB1M4yIbDhYmDjZMG5wiO/%2BiazPMcJ0AO6egJZzjUxadx/reuOcdX2fzGC9X25t9sti9vrk4IDHnthK2KLeBtwlvFr0P%2Bie3hojzc9rA%2BF99/ddtBA3DuWFO19rU/f87fhRqt1AO8ot%2B2uDcOj0f/Bl5ECdKXrlpm97htIHClrdavXrfGvuO%2BGzLCNywo9dtI3f7JgiCW6lxhPM6JQEaNoHkicPOBKl0ETSOVZpa4zAhtS0OgQukVvz37XHRv5CHpBIeEvxgvc8fZPW84z379uvP8MUe1y7fcs8WX8MXfbHjxHV7EsrC/%2B/3TLeA4COsypVbA0HrD76P8pJTW3/Wa1W4/PM/3UEeVcdp3cad2tGP7eJAc3jCG7QjH4smixOa/Xr7c/vaCBT4jwHHBMUEtBPY9qkb9/Lo1PpPEjQvOP5zDfvuGtVbmMou0cWaM14lEZiQ%2BgBlP7JuMX0vGZyJw9588Gzc%2BhsPPTokVJWVcx8DuQoYxPiPNPoET7RHsN2RAf/Ta1f48SObuPb/b5ju8%2Bn114hiGcbuef%2B/o0rD7JmxPLI20PRyfnzhvjm%2Bv/%2BkLF/rmEi//yUafqPkxmpFek5JaviW26UTXTHDThE60fnrtucaQMOxvdIZ1xytX%2Bg6llhRkumB6nn39FSt8AMG5icAkZPT5/Z5vbPcdrnEOn4iSRWrZ0IZzIq10587QedF4I1AkuOF75uZb%2BK6j2IfJz13ofi9/Uz62TvyaPB46T54YeQM6fPrF9Wvs3csrjY7z2BfpvGy4vAvn9ej7sp1Mj%2BeP3H%2Bxp4/j4Ym2h2YHjCt6t8u/fHzNHF9b7MMPbPft5GnnGL1BEsX1kmfYpuh3xizvyUxss07KH7ljk%2B%2BDYTC4ZtGhIXkqauacV5pjP7vuXPvG1Sv8OWS9u35Q82W/C4TpMJG2ynw/XEMoBIhszrjiZg03wuTUKFiM4aRDaQJVdMYbVYGoJ061BTJn4QIfj2o/X7pyqW/ITbUDqnlEA0suSpTIcUeP4JOTxGi4yHH3jMGTX/nTZ33DcKoGDYdSrFAFIshM5s4cVSB6fGJb4u9Q065yQV6Gv1tHm8H4u7W85sfPne3vqtEo%2B7pfPOcbilMn/0SobkKpCFUk4oMJfrdwl%2B5kPHmk1ffKSqcmdOrzZ4/utpvu3mxfdRldqt6%2B53cv%2BIG36RxnsqAKLh0IpU7QSRWrirkjODGZDjIAVBmjQxfu/g2HEi86kPmhy8QSNFOiwZ1j2u1Fvxba%2BFEqSin3aFVa6bDm3cvL/X5zq/vN3%2BqCxJEG/efiyg2eaLtmllHaxrZTvanWHYOcS3xPp7F9NAztwb5PyeRwVd3m5FAtKdtd5Af9Rf%2BW32zxVe9OBtVOyXiwbdELbmH6UHftrZFhPU7ku9uP%2BB6BqXr6YE2L732T44AbOrRPYZ5ED6xUyeOu%2BjdcJpmq6B93GRXaNk40Snf4nBOFaq904MB0ItD9PudbMqrDcr85na1822XK6LCLjGJrX78v5SYjHcU5hPMkJQsjYR%2BipIySC9rocS68%2BTebj1chjcc1Jf59OEdTJZHSTbDPc85mm6JYj7/lvD4stqUg27d5%2BqvHdtt7XcB6MigV4drGsUhmNCBDzHirdJBS393r2yafCOux/9PDJJ3R/f1T%2B/wx8M7fbvXfyde21rjvZ4vvNIq2WmSW//6iBe6YP3u1UsgMn0y71LHEdWK8x0QODrrv/YubDvmaTZT4RpufBNz4%2B9DqKvvyy5b5m9IlmdSGOnb8RmNQ4fYFglz2wRFirOO4QcRNNNpy8/uzDSNhX6ODvjCWb647X3BjjZvl5NVoA9jjzvecp6OBG9tHyVwY4mY4/L4EjFSHvv4Xm%2Bwfntp7vJf60fC%2BVLfmWGRbQt6T6uJ8X3w%2Bel0d7fwQRft6ev2lWin5uJf/dKN9ibb/Lk/FcXHxD5%2BxG9xxQdtHbjbSodZTb17vE2P0co4h0PzRtasTqrePFzoi4ruTUzN8lDIDkfGkut0FbocerXrLWCBjyZ0gTnh0b0xQyF36gJIVukKn8fPWpk7f%2BJluizmZ0REAVebAgU2PWCCztq9t9AbDVE%2Bgsw4yxtSJpzSRzN7VcxIHc6YKKYEf1Wo4kdFD6JVVBb7zEbpPJlG9gg476Nk1oKpNjjs5kpGkV8r4W1S8Fr1zcbKiFIn1%2BFx0b02Db05aI%2BHkyZ0pTnIh48R2f8QFn2TGOXHSbuRkcLc3tCUgU7HLZRo4kXJuJtClx0vaDoR1zjZO5PRmyb4Z7lROBKqskHHkrmD0gjYeuIj904b9PvB514oK33tiFHf1P7J6jr87T5f5ZNRoC0Fm9bp5xce379p5JX6/puOZb/kbIiN3DEKnDFSdoioZwR7VR5mnd11urkSrFpHJpp3KDQtL/O9BxptjhyCP9sBPuwvm9ljmk2EC6HUVZKBucZ%2BF/YqOAHiPeGRoeV/aSJLJqHbBAccFvdq9fn6JXzYSjiHaKNLmlxteBDGUBnO8cjzdc7DBV2E6GbSZ4biluipVWDkGqD7FOYv34DGJ57jRRZsdltOjK%2B1XRsz0jwNuEtDDLMc%2B8xOF34heqS8qLzj%2BG48n9qvH3Hc7y70XpVnRwJgbhbRVeoM7B9OmkWOCcxtBHudIak1wXeOaQmdP7Iu0P6dnxtEQZJGRZJ/nvMsNNPYJqqFS1SyK8xHjnzJkSkC1M44PxrvjEkC7YNpWhSqY7M90QENPiZzT6ZhpOOz1IeCjqhxtGTnOP7l2ns98joT981vbavyNIgJ7qnODDM%2BrYr0gsm2Mt3giQ1XGh6p4d7prJz1rcgzQHpQYmc7b6IiKm8Ccw9ym%2Bt55Q5VUbmT6AfldBp%2BbxFsbx7eTG35z2oDy/U8krsUrXYBMzY/xxr7OuX9zY7vPN7zD7UtUlw/4Dv5g6Szfqyg3FJ89SqdfTX7/YR8Mw7BQI4Egl%2BsqPQn7PMsoqCHCNYN9kH4T2M8INOnhk2Msin2WTnNCfpL2ggzlQQDIfkCAtcedu/nO6JmTY5kgkFoLtJ%2Bks6X4DsQCjs2ha1OSPzb5LvzncvsZ%2BxvbORzOD1wv6OiGbeEGI7i%2BU3rO%2B9Eh1IkKHgLel3wgeUo%2BD/s2N4bofZVjns/JjZ7/3HzIPvbQDt/D8T8%2Bs8%2Bnp%2BtafR8AtKX%2B8vOHfVOi8UTBBh3scG7iBq6cGg2dEcOxxQmAi9hQQ/yhnjnHCxdeLn54tcvwkBHgYKeNCHfK37So1DeOppSDTBjrcrHirggnJ8ZJ484hddS5m08vbH3uPEd9cUo6aEhN5jM6dMYx9yk5YXFC4sTGe73KZS45mfHZuSDS%2B2kYlJaMAiUeZMZoL0ImhN7f6HaakwlVCZYX5vgTDZkmXo%2BBVmnczJhTlKLQ7TQnNk4kLCdjwcV0VVGur1bFe1/uPgPVYimJ4a4ameqHapr99lAfvshtKyde7grSyQaZoBJ3QlyYl%2BXvKvou2evbfccblGZykSZY4PtjGIbowNBRvO57V1T6BtxknqgKwXYwaO29h5r8hSAe3y13MQmkJ2roDC4KdEf95sWz/G83wnVgXHA8hAsPnZqQuRqvo4LjjbZ93NDg7i37HRdQ9isSQ79QrYh9lF7luECxafluP%2BI34QJIxoHfnuCBHuoYK5P9ZaShM9hvuIvK78nxxTHIMUSGmGOUEnC6I6dHOTIj5dkZfj8hWGC/J4AnQKJbcW5W0BaK44IqpezLZFrICHCnmFI73p/2sVSD5rjguCS/yvHHRZ7AnEwexynBKo/5/dmnfSdT7sJLR1wMS8B28AUQrPFd%2BMy4ez9KWXl/hgzhgs13RYaCY4X9h05sGA5mpN%2BRGz7nuO%2Bh2v0W9CrLzSnaMZKR4I46ePy%2BlZX%2Bu6IjD44VMl3ht%2BLOLcc/pZ4nkR8/ZbRd5TuhChbnuImqEgr3Ux2/VtA5BdXhxnO4ADJfBOy0%2B2PYEkriyJDzPVPT4JYl5bbfBT7s61Sd5iYgf8Nvzf7K3Xv2f4J7Ap0f7z3q91XOZcMNnUHmjmsONxpoD8pyvmvej%2BqNvt2T%2B763uHMrxyr7OFVg2S7aybHexS6RkfyBOy4IBjlDcq4nc06iIyQytgx/Qydt0aEzuEFKx09cB6i0x/7Mvj10XNCO0h137vVZn0COcwYdyLB/8zzHHZ//geoW32smbcxIPMf1iWObIRV%2BeYBr5jEfTII2hhxfw6EqKp37cHzxHVMqxPmJG1gMA8J3FjogIuDg2KUmENdqboTe4NYjo851kY59%2BD3HGudCjgOG8uEznajDu7E2dK0Y6ojp2YY2nx8Zv6NiCL8f53ryAtwg41zP/scwPwQF7EOcY%2BmRmv2RmwXsN5e7fYhjg3Mb53w682L8UYIbztfs85wz44fOoJSfcz%2B/K8eP7%2BfAvd%2B6slz/mGsBBQE0X%2BCGJ3kJrgdsG82AyOvQ%2BRd5KKpjshewX/J6HGPs1xeU5fvzKTdQeC1ujnO%2BZd%2Bkwz1ek/wQ%2BSeuW3TKs86dezkX%2BI4Q3XPk3Qj66D2bm0sM6cRvw01wbqKyn3C8cKywr9LbPte6O9x73r2/0d%2Bk5LgOecKRcD3mGsB1%2Btvu%2B%2BMaSxMeqphyjPF9guOF747vMCRutnEji%2BOB3ubHM7/N98FN5g%2BvnuO/4/gaDnJiChbjUP2Nk30otRgvHFQcPNxV4U4SJwpKK0icqLhYUsVhw9GhO8ActJy4CBI5QXGh4sJNAEabQ6ac0DgR0r6EQIkSGg4KxuthDCTW4YLO8Ber3cWT9yJzS0DFHTcusHe79ThRcDeOzhC4yJJx52Dj5BW9A8xBzt05TmRh2ynhvHNHnb94kxmmPRgnLO6Oc%2BeVEwKlMNyF5MRORo/Pzgnu1%2B4kxUmS7SCAI5NOlUSe50TMyZuqcJz02EYy73wf3Pmma3UyPdw98kGoO0GyDVQbIqM/HEqkyESFHmKpLsJ78T3x/Q8XZJL5ILNPRp0qrGzvyfZSeKq4%2BPO9czEjIOYixuOJlucyPWQcGXCagOFEd1/PFN38k/EgwKJRfNi3QIBOW77wlbPfEpTwe5MJZD32JW6gcAFiPYbOIIPLuuwjZOLIgHJj4WH3mMwppd38Lfs7mQEyGXe6YJMbLmQEGVeQ6kNU0SEjQKaY/ZcgjYskY6KyL5AodeCYYfs5TtlXyHyzj9KDKjeGOFbJ1N/tLuhsIxmAHjdDJyb0qhrOB1TfoadWMuOck8h4d7rvn9JQqtKxN9BDMsci7bkYsoa/5cK4y%2B37jK/HcUCQT4aESIcLNXfkR9prl7nzA8fRs%2B44oK0VmS4%2BO98TxwbIaPCdczyT%2BLzhdyJxjGw82uG/x7HOBLBv8NkJiD/qLv6cgyaytD3g/MtddfYrMmDjieOOcyuZPDKEZCD5ngnA2He%2B%2BPwhX3JCuymuFXTiwtdORpFMH1OO2i9tYniYoZ5u%2Bd54nYdruKnW6fd1OkmjKjg36zgPkxGP/q73HW6yTe7czDFErYuj3b3uPJ/im0GwjWQcWY9SeoZ44njl1%2BeYIKAm%2BCRzzf4Cho7hxiY9XfPZOJ8zJAvbE9oLE1BSRS3sY5S60LEbn5XrFzfsKBXi83Bd8f0OuP3hfretHO9kxvnO%2BFuOSa5FBBDcwOQ9uZaw7QwxwmsNpyDDfe8uIOhwERDrcW1g/yeDzfWO6yW4ocoxwve4rDDLHzt89wS1XM8537DNY429n6CBmwi0peOm2dm4VhS6757fgdIrvtPxujaCl%2BacRD6N8ynfddhPuRnOtf%2BH7tzJ%2BZMbur42ka%2BxMeC/H86TBNTsXz9y%2ByqdcrG9dCjGa5MvooSe12NfZJ5eVEPpJDdjeI6b3OxrnNvJ67BNXCfYXwmY2P8I6tgnyPN9dsPQ2LtsB%2Bd3/o5aIbwv%2BTfOJQSu3JijRgj5QW6EcrPwOXfu5rPwmbnW8Jn5O/KENI8gOCPfxI11bmoQYPJZ2BepvcZ1gnzkoY5uW%2B7O81SVJpDlnMqYrAyrxvmDayDHBsc4y0fCtYz9m8/A98d7cSOUwgTen/P/SLgR6n9D951wzR8vHAZ899xkomYS3zGfV05N0rGJ6t98CuEk94Pddfbnj%2B72GTB2aJGJxsWeNj1/Hustlovb2cIFigvPB36/3VcfIfOjw0ImGpd4bupQ3fZja%2BbYVZVDVfLPFu6YM14ZGUCuFbqaytlA6RU39P52/Xy7bn7phPeEGsUNNG6%2B0gacoJE2tDos5Gyh9JSbA%2BSjKEmOb18tJ0cli8OgjQR3H7jjwh1RTn7jWc1IJB7VTLhz/cm1c%2B31C8psrpunRPVs4b25a8%2BdVEpyaa/DHUiRicSFn/bT9DpJSfLZHlyZ0jiOTXoC5I4%2Bx4RuLspEIlBcUZhjf%2BYywwSKBI1n81pBKTE3OVcX5/oaVJSCjXdTDZHhEBjS1Ir2mS%2Bvogr/xPW6Ot2oZHEEVJ2iehvVEamzTccUtE8RGU/cqCBIpPMGOg%2BiWhdt7KjSOxlQHYcBuO891Oh7zNxQ3%2B4yxzqFyPihhJ0bFee4DPGr5xX5iz7Vp6iaPhmQEaadO1W8qLr8SM1Q%2ByiR8URGmNomVOulLSvtTQkUudF4tnFFoP0e1S5pn0fzkVCFXWQ8caOEm4pUzaVjK8aipGosTSPk9ClYPAnUvabu%2BWZ3sqPaEe0ayByoZOUEujvNNj5gxrAYKSku1%2BcO1pRUl9w0lalLYdnxafT5yJTnznIpwnggI0wHJkWZqVaeme672qZ9DW0paa9Hm6HJeCOMNhe02WQIEkoZOS7ofIh2rnICB7abHa0xS88wy8k3y3YpJ88sM2foOJjhuPNLm0naWzP8CW1%2BaN9CG2WGPaiaRDdP4tFWlk4oaO8UrhUM3zKebbemlEF3zeztNmtrGpq2t5h1uiAizR0LS851P3zmtDzPjwWuFbQjo7MdOgahrSXXCErYKcWjreBkRPvZ%2B921gjaiXCtoa8rNlDBY/bTDPj5Ie1t3zPe7zzjgUl%2BPS1wb3TIek3rdsv7YMvpV4Hjo7x36W7LlPe5xxVyX5pvlDfXgK4nIH9Fet8Dt/1wv6AiLds10QEf1U9pqqurpmVOweJIoUaEnORrf0/kK1VNDA3wZXveRatvwN7dZf5cLGtPShzICx5PLHPgUfTzyNCmNlOZizhQXP6b6qZ9nGnkc/3z841QXdCa7E0vyJMmQUIWIiz%2BlicvdxZ9ey%2BiIhJKUyY52i3SUQcN/jgnaqdD4X0bG6Xb39/7bqh%2B%2BdyhALKkwKyp303JLKSqzjLxCy87JtqysLMvOzvaJefbhmYKCEToFovOJJflZvpMgeqGc6N4dTxedPdDr4LP17lrR0O47nRiP3i8nq4GBAevtdQGySz09PdbZ2Wnt7e1%2B2uuuBYMEiEcOmnW0Dt00aa63zKISm//2j1rJvIWWmuHO%2B5KAGySUjtChCGNPnl%2BW63sBDeP4TWbcLKnv6vU9anNM0KkJHctMFpyX2W8HXNDW7wI5Px%2BXWGfQBYLs1zwOUwK7/sh6Ay4IHCQQJAhkSupqdxkietqNLSN47OoYuqHul7nvotMdDzwmgCRb7o6P0itfY5WveL0VrVjDZsowCBYJBgkSKWlfWZTt81FcL7jBImNDwaKMm71799oVV1xhTU1D3e2HXY1pdP5kpgR36enplpuba3l5eSedhls/JyfHv1boQZFpdP50piIno7%2B/3z75yU/aV7/6VZ/xYP9mSiIwnD17tq1YscKWL19%2BfLpo0SIrLCz0xwCJfS5Mtf/J2cK%2BG/bf6Hxra6vV1dVZTU2NHTp0yHbu3GnPPvusbdu2zWpra627u/sl%2By/zixcvtr/%2B67%2B26667zoqKVIoiJ8b%2BFoT54abx68WvA87LbW1t/qZGmMbPh5sfR44c8cvZxzs6OnyAGF2PxD4ehHP0cNMwj%2BEev%2BlNb7LbbrvNrr766thSkbNDwaKMG06s27dv9ydSTqqcRKPTaBrpufC4q6vL%2Bvr6fOlgyDTHp5Gei19OKU1mZqbPnBNMEjxGp9Hl0efiH4d1eX2Rk8HplmNi06ZNtmfPHp92797tp2Q%2ByLRkZGT4mxlMSexrBJFkqEkEj2FKqSP7tMhEIoPMeXnfvn3%2BpmB1dbUdPnzYtm7d6gNCbhCGczaZbDLPTMkAFxQU2MKFC23BggVWXl7u51euXOkTj9PSVBVbRsf%2BR0k1Ghsb/b5GfuPo0aMJyxoaGo4va2lpOZ634O%2BZso9y3iWvEW7cDZc4d5NYl8dhCqZsE1P2XxL7esgzcAMkPz//%2BDLyDfHLSkpK/E1Bzvmc08vKyvw6HBPFxcX%2BfUTOFgWLMq44gXIyDokTbPTxSMvil4e7elwAyHhEpyHxeLhl0SmJ1wJBYzixR9OpLCdTT%2BAZEpl30nDz0WXhcfx6XDhkemM/JmNCxoVSGKYkMjpkukMi882UzA77GZlsEhmMMF9VVeUTwWR0nv1J%2B5KcqVCCQvBHELh//36/n7L84MGDfv8lkelmnyZDTpaC/Y/9lAxwRUWFDwjD48rKSj9PItNMbQ8yyfwN51Ttt9NXuBaT2Gc474FzYvwyrtPsZ%2BxfPBeWMR%2BqNoPHIXDj9UdaxnuQovmKsA77XLiJHL0uRx%2BH6z/nXR4zJbDjZnFYh79hn2aKkE8INwBHW8bfh9cDr8N28Vg3pOVsU7AoUwIndE7s4aISptH5U3mOaQhCh0ujPRdSuNhwEeEkH0724cIRvyz%2BueHW5eIxWuKiMtzykMLzTJXpmlrI1BAYUs0pJDLoZNbr6%2Bt9Rjw%2BkfmeNWuWv/tMpjzMc1e6tLTU35EOibvUIRMjEnAuI/NNQMj%2BR4adfY5MOoEh%2ByEZeJZx84L9jn21ubnZl4iwn4VSEvY/9rWwjCCQZXPmzPFBIeuwTKaGcJ1jyrWO7CL7R7h%2BhmXsPyyLrh%2B9hoZ51gnXYPYh9jHwfFhGgAjWDzch%2BJuwLLxPyLqGmwwEVOyPoy3jGhu9hoZ1uIZTUyh6XQ7zIbEOiX2Yx%2BzPvAalgPHr87oi04mCRZlxCDy54HCBC4mL0nDzJ1rGxS1cNClFjaZTXcZ2cUHjjiIpVFcJ02iKLhvueRIXtnBXMlwQo4%2BHWzbcOlwMFXiePeyrlOJQqkOi6l%2BYsg%2BG/TDsl8wTLM6fP99X9YtOWc6%2BEd3PmCfTI9MXl3nOMewbpHA%2BY98iEORmRKhSSsBIVWluVJBx57xERpp9hHMN%2BwspBIJUiZ47d66/WUEbW25gsL4yzBOP3zj81iGQC4EVy7hWIbqMeaYk8HfhbwnUwr4S/oYbCCxnP2L/YRn7TFgWzkPhXBSdj15XuNaE8050Wdhvoo%2BZD8tI/B37GNgPw41W9r34ZaEKJ8sI9KLnPlK45rFcRIanYFHkDHCB5UJIporEBTPMc9d0uOXR56LLmSdjhuhhGeZPZUqAFzr04aIa5klcMIdbHk3RdbiIhot1NGgM86c6lTPHb8y%2Bt2PHDt9xCIkMPmnXrl0%2Bg8c67E9hym9IsBjfgc7SpUt91cBwUyB%2BKlMHv3VI/Obh9ydI4BxDZzOkLVu2HG9fSCJgZJ3ob0%2BilISS6rVr1/p9ZNWqVbZkyRIfGPKY80M4N8jp4zc62Wn8MoTl/Mb85lxfKClmGR0Nhd%2BfUmGW8ZsTELIeNwq49pA4p4T1EK5J0fMA8/HnheiykabsT%2Bwv4bpD8Ma%2BBYI2locbEGEZJdQkngvLwmuEZSIy/hQsipyhkCnjQhvmT2cZiUw%2BF2im0RRdxjwBavyyME/iLm7I8MUnMncnWhZ9zN1ZLtKUKnCBjs5Hp9EUXRad57V4TTlznLq528/dfVKYJ7NHSWS08xwSPVOy30SrS/F78LuQIYt2nMOUkkiCApkaqMZH1dBQbZmSQm4iEBzS/pX9gn2EFKriEzCwD1BllH2A35tSaIJC9gMy7hy/ofpe2GcojQklMnJm%2BC2i526O0ZHmSSHAYxnXDn7P8Dy/cbi%2BMEUoJQzXmOgy1mM%2BXIcQ/dtQCsc5gt%2BewC/sEyxjv2AZ%2Bw4BYDjPh7%2BJzjONv76wDyG6PCzjdcM%2BxnMI64UkIhNDwaLIJMGhyIWbu8CjpROtw/NkIAgYSaEtSPw0zMc/jn%2BO1%2BOCTWYxJC7i0cejpei6ZDjIYJAJITEf//hkn%2BN1w11reREZPW4eUFoUEhlMqopRohDfiQ7L2W9CpzkhcZefksj4DnRIlDyHTJ1MLI5JggV%2BSwJDAkSqj/KY35LfOwQVlC7xPMczvxfBIInMPb8tgSFTqurxm5KhZ8pj9gEeKyB8qXBuDedHcLyRossI3AjioutGU1gePW%2BPNh%2Bm4f2ZDwFfeJ7finMi50gCOnAjgOUsI2jjeY7tsIzfO5xrCcCi6xEAcnOAc3fYD9gnwvk8%2Bn5hWfR8Hz8vIlOTgkWRaYgMBCUHIWMSSp/C47AsOh1uPjwmgxKfQsblVJaTsSDjEDInIXgM05HmR1pGRiaaeO34ZSEN91xYFjJC0xm/AYEEQUW0Ax2mBB0EFqHjHBKZXX6v0HkO0zBP6QLtgwgqoolMozKFZ46An0AjBHu0E2Oe3zAEiSSWExRSksxvxnEfSnJCsB%2BCwPC7heCQRFtDgkKOp%2Bn6uxFE8X2GoIrvkGCLafiewXmO58N6LOf8xzrR9UIJXzg/gufC%2BtFlrBc9l5LiHxOgxZ%2BPhpuGec57/L7gN4uuR3DHuYzfk98c/L7RZRzT7BNhGUElr0liW6J/yzzri8jMpmBRREZFZitkkEghExT/OH55/HNk0MgckRELGbBoCpm6k11ORongJGSOmYYUHkeXMx/umsc/R0aJzBOJ1w3z8Wmk58LyqVg9iswrJVLxnecQgFCCxe8Wfs8wT2aTHi6pqho60CERjITghBR%2BHzKyU%2B17GW8hKOGY4Hvld6B0KnzHBPH8BqFKMb8JyykR5m/Dvsx3GzL8lBgSABLIM6U9KkEiAQOPpwo%2B33DHfHyKrkfwx2OyNNF1OPeEwDDswwTWTFnOvg%2BCbpaF9UJHLpx/ouuFgDIc%2ByBYi54bhls2UuJ3DMdJNEWXRef5rUNHLiGwiz6v40xExpqCRRGZMGSMyYBRahVS/GMybcMtj08hc4hwGjvRFPHLuHMeMtzcmSfYYUoKy0KKPkeKf57MO5n2cDf%2BVKcYbtnZwPdL5xh0mhM60Qkd6VBCSSY6ZM7DlGCFMfXoPCckOtGh6ioZWj4TmdnodCYI30/0uwoBCW1JaVdIx0SbNm3y85QaEuTw/US/KxL7IEH5smXL/ED2BIPr1q2zefPm%2BSCC7/ls4DNFpxhp2YnWZd/jGA/nAKYjzfM9MR86cgnnmOg6LA/CPhemCN9tMNzjgIAsHPsksN9TUhtdFkrew7Lo30TnCRhFRCYzBYsiMmE43ZBZ5u4902iKLot/frjHlMSQ4SazGDKN8elklvMaZCbJtJEpD2m0x/HPhWWUJITAk8CRFOZDqUD88vj5kMj0UzpxtvBbERCS%2BY5PBDi0k4vvRIfvku8iVGsLiYwzJY90mkIwGTrSoS0k39l0xj5GaSAlg9GSW5ZRWkVAQ3DE98o%2BzTzfPb9/KLGlB1JKCalCSnBIOzT2Gb5bvr%2Bwr4T9cqKxzeFYZMrxyTwleOE447iNLouuH9YJ87zecMd9SMMtZ1/leyNFn%2Bd4DCV84dhif%2BR4ZBlTlhHYhWU85rsmqAtVPHkM1uF7D8c9wutHl/E4LIs%2BF00sFxGZ7BQsisiURIaQIC8%2BkWkcbvloiUx6SKFKINMwf7LLQyaXDOaZpBB0Us0sJAICUpgfaflw65DInI4Vgm1KgEMHOiHRni50nhNNfD9kvKkOGU1k0KnOyjAMoRMd5plOFexvBDm0LaRdIaWCBNMkllGVlynfVzTxd/zWfC%2B0LyQIpHSQ7yP0OEkpIt8Tz5P4TQlW2DfO5PfkvdlXEfbjUAoXXRZuqpBN4DHbzbLwt3xejkMCXdYLxx7rh%2BOKKcvjl4Xlw83z2cJ%2BG78fj5T4fjhuSNHlfI/h%2B%2BI5vnOWM6XEMBxvvA/TsIzH4e8I6ngM1hnLY0lEZLJTsCgiM1rIxIZSs5BRDvPDLRvteRKZaRKvG%2Bajabjl8cvIhIdMMyUZIfMb5odbNtzjME/Gl0ww05HSyT4/EoIigqVo5zlMCZgoRaNUKaTQYUvoOCfakQ4BI4FkNFGaRuBEZv1sIJghIGK7SQSIVCMl8dlCZzNMqRJJIoDmJgIlVQQzbD/zBINUXQxVFVlOyVUImFnG7xbFfhr2jbCvnGgabmCEv40uZ/sR9lk%2BG2m0ZQSUPOZ1wOdjP%2BX1Q1aCz0dwxX7Cfhddxv4TXcZnjO5XTNnnmQ8lpyGFfTk%2BheV8h7wH%2B0f88yrBExE5fQoWRUTGCKWKZMZDdbroNKT45SM9HzL6ZMajiaAlftlIKawLMuBk0EO1u%2Bj0ZOajy8iQkwEfLYUSGBJBBiVtdNYSnRJMhsAlTAm62WaqX0Y7z2GeTlpCBzqhlC2UPPGeY4HvnG1ge0iUpLE9oeT0wIEDfvsJEKl6y2OCZL5rtiW6XZRSEfgRHBIIEgSz/bTjJFgMpV68J0LpWvj9QuIyzZTvMewnIYXvbbhlIahlv4w%2Bz%2BcLAV/09wqBeFhG8BZdxuMQ0IFt5zMSlPF5wediHb4DPivPExRHl4X1%2BPwsC4nXCO/Da4qIyNmnYFFEZBIiOCDTH6oHUmrF/OkkAtAQdCB%2BipN5LqDqZOikgykBQFgWEsvC89FlYZ7AgICFDnNCCoPIE4AN9/4EFASO8R3oEEhSskQQQ3ASn%2BLxevGJgIopJYIEgWwH27Nx40b//dPWkBLRIPq6BE0EgrQrpH0hg9ozz/bQPpMAjdegdI6AEwSb8ct4/bCc341pWIffj98xCO8f3Y74%2BehjhMchuOMxvxEplHLGL6NENywj6GWd8Ld85hDYhWBRRESmFwWLIiKTUDSIIVBgGtKpPiYoI/Ak2CARgAw3H00jLScRBFECxDSk6OPRnguPCTaYDyVa0cRyPkOoMkngRClZeG%2BCk2iiiifVN%2Bk0J3SeQ2IZQWQU3wWBN1VjQ4czBIY7duzwVWh5LlqlmHm%2BQ36LaOksgS/zLCcYpFSQbQnfP49Zzt%2BE34F1Q4le%2BI1YxrphGfNh/ejf8bkpQeU9o6W90Xmm0eXDdeQSEo/B65JYh%2B8%2BfhkpLON5HjMPHvM7ht9SRESmHwWLIiLTXAhICFTONEWraTKNzjMdaXl0GuZ5PQIOSqdC4jEpBErhffkb3nu4SxZ/R/BDYBjaBpJCSSbBEUEbrxFK6wg8CQxJoR1i2Kbh3iMES2E7eT2mrMvfEOghfM9sewi0KJUj0OPv2b7RloXl0RS%2BE54L70%2BKbk/8Y%2BYp7WMav%2B2ksG0iIiKjUbAoIiInjUtGCPiipW/RNNry%2BCmJIJAAi2n8/GjLossJ1uIvZwRJBHUEkgRIvFcICAnmhhOCQBIlhQSaiC4LpXI8JpgjKOPvQFDGPOsRqILX4DHPEcCOtiws53WZhnX4LMyLiIhMJAWLIiIjGBg8Zm09A9bU1W/tbsrjvoFj1u%2Bmgzp1njECNkr4qK7Z0UHnK7HOftx8R2x%2B5OWxZbHlPb09dmyg30WzLggcHBia8nigz88nuekxqkqmpLkoMjWW3OOkFJfc1M8n%2B0DQp6xsy83LtdKSUr%2BtGZlDQSKle1Tv9MsyMn0JJu36QgCZyXrub7P866gd35lJsvSUJMtMS7Ysl3IzUqw0J81SkpPcMyIiMhEULIqIxOGsOOD%2B19o9YI/ubbHf7miyZw61W5t7XNvWa80ueOzpH75kSs4SAsO%2BLrPezqHU51JHk1l73dDy1lqz9ByzvFlmOS4AzC4wS3MBXroL6NJcSnfzTGXSICCszE%2B3RaVZttSlC%2Bbm2s1ry6wgM9UFkXRmNLSeiIiMHwWLIiIRlBxWt/TYD547ar/b2WS7jnZZU2e/dfUN%2BgCy3z3PVGfOycb9IPwovmTRJabHBtw8pY1E/25KCWIKJYqUJpJctMEyn2LzMqmkpVC6mGzpqUm%2BdLEoK81ev6rE3nbBLDu3cqiKsIiIjB8FiyIiMQSFj%2B9vdYFivT25v80ONndbR89QkCgiZx8x/cLiLLtwXp5dd06xvWZFkZXnDbUXFRGRsadgUUTEoWrpfTub7fsb6%2Bw325us1T3WyVFkcspJT/EB41vWlvmqqYVZqZaarHqpIiJjTXVuRGRGIyCkw5oNh9rtf54%2BYj/b3GAtChRFJrWO3gF7ZG%2BLfeXRart3e9PQMauDVkRkzClYFJEZbdAFii3d/faNJ2rsoT0t1q2Oa0SmBNoXb6nttE/8eJdtq%2Buy3gEduyIiY03BoojMaM0uUPz%2BxnpfskhVVBGZOvzNHnfcfvvJWh84iojI2FKwKCIzFtVPj7T22V0b6%2Bxwc49/LCJTB0csJYz3bG%2B052s6fPVUEREZOwoWRWTGovfT56rbbeNhMpmqwiYyFREwHmjqsa21HXa4pXdooYiIjAkFiyIyY1W39vhOMjp7B2xQvWNMOgyJsKYqxy6en2/nz8617PSxv2TNLcyw9XPzbN2cXKvIT/fj%2BsnUtLO%2By/Yc7Yo9EhGRsaBgUURmrMaOftt6pNMFirEFMqncuKbUvvO2FfbAR8%2Bzu9690haXZMWeGTvvv7TSfvn%2Bc%2B0nt632wzAUZaXGnpGpZl9Ttx1s7ok9EhGRsaBgUURmLHpB3dfYbcc0UIbIlMexfEDBoojImFKwKCIzVu8APSkODDV6EpEpra17wNp71MGNiMhYUrAoIjNW38Cgb6%2BoWHHmemJ/m33ziRq7/cla21TdYV196uhoqqI34wHVKRcRGVMKFkVkxmL8fQ3CP7P9cmuD/dUv99rf/Xqf/X5Xs7WpZEpEROQ4BYsiIjKlJSWZpSQnWapL9GYaTSzjuZH6OE1xfxzWS%2BaFYpjj76LPucmI7xH9WxERkeki6ZgTmxcRmVFuf%2BqI3XrnttgjmWw%2BfHmVffDSSltRnu3H0bvpW1v8wOvxLpqXZ69cXmQXz8u3irx0y0h1waEL3mi/dqC52zYd7rBvPVVrta2JY/B95Ioqe/N5Zb796u1unbtfaLTGzn5bUpplb1hdYu%2B%2BsMJq23rtm4/XWmffgN28tswWFGVabmaKDzTpJGl7Xafdv6v5%2BN/K2fOxK2fbF29cEnskIiJnSiWLIiIyZb1zfbn90cvm%2BOlViwtsdWWOzS7IsKqCdFtZkW2vWlZk77yw3P7x2gX2Sjefm54S%2B8shjOW4sjzHznEBaWlOmi8lRGZaslXmD43zeK57zXe51yCw5DWWlGVZlXtuYUmmrZuTZ9evLLGPuyDlLWtn2YLiTP/3IiIi04GCRRERmXLyMlJ84EaQeO05xVbpgj4GZf/vx2rsvx6ttv96pNp%2BtOmo7W/qsaWlWXazC%2BTefsEsu2BuXuwVTl6ue6%2BL5%2BfZ/KJMe3x/q33ziVr7yqM1fvr4vlbLdoHlxfPz7S3nl9lSF0iKiIhMFwoWRURkypnlgsOPXzXbV0FNT0m2TTUd9vXHa%2ByPf7LLPvXrfT7933v323eerrVdR7vcOkm%2BBPB1q0qs4BQH3icw7Rs8Zo/sbbX/d98B%2B9tf7bVP373P/o97D95zS22nX%2B9CF4guLsmyzFRdWkVEZHrQFU1ERKYUKooSwF26IN9y3HRnfad9b0OdL1WM2tPQbXc%2BU2f/%2Bchhq2vvs7LcNFs3J8cumX/qpYv0lPo/zxyxJ/e3xZaYb6/45IFW%2B/mWo/4xJZCLSzNtblGGfywiIjLVKVgUEZEpZVFplq96WpiV6nssfdwFcFQPHU5TZ78LJOt96SJjKM4pzLTLFhTEnj15Tx1os6cOvhgoBodaev1YjQFtIEkiIiLTgYJFERGZUopckLikNNN3RtM/cMyXLBIMDoeB2o929Nnh5h7fOyp/S8c0p6K1e8Aa3Gvw9/F6%2BwdfMjYj1V1JIiIi04GCRRERmVJy0lOsLHeo9I4A7mhHvw/oRtPc3W/dLrBLT03yVVhPRUfvgB9aQ0REZKZRsCgiIhOuODvN1s7O9Z3CMAzFqXQKk5qSZFlpQ%2BsTAFJ6eCIDbp3THVW4zwWKJ/EWIiIi046CRRERmXDXLCu0n7x3lf3%2Bo2v9GIazC0fqFCaxSmdX71C1UFCtlMDxRBU/M1wwmsIVzwV9pxs0ioiIzDQKFkVEZMIxOH5lQYYf/L4wO3XYkkWqm%2BZkJC5vd8FibVsvcZ8PAhkI/0Q9kDLAPtVPaV9Y3dobWyoiIiKjUbAoIiITjqqj3X2DlpxktrI8xw9rEY%2BhKEis29jZd7y66cGmHntwd4t1uMDvmPu3bk6uS8MPh0GASCkmndpku%2BCzxgWKGw4l9moqIiIiiRQsiojIhKNjmprWHj%2B/qjzbLluQb0vLslzwONQBzSuXFdl5VTmWn5Hqg8rNtR1%2B6As0dfXb1tpO21Hf6Z9b49ZjKA0G6I8qzk61S9zrvvfiSptbmOnbHu6o77LH9g0/zIaIiIi8lIJFERGZcFQj3XCo3Xr6B317xRtWl9ot589ywV2evWJJkf3pK%2BbYVYsLLS0lyY64dX%2B7o9lau/tjfz00IP7PtzTY4ZZe3zMqweIHL6uy9XPzXJCZ69M1LuB8x/pyu/HcUstNT7btdZ0%2BUHzhSGfsVURERGQ0ChZFRGTCPV/TYd98otZXC8WF8/PsM9cusEc%2Bfr799H2r7LUrin3nNQ2dffbEgTb7/sY6a%2Bx8MVjk7z599367Z3ujHXHzcwoy7NaLKuyBj661n962yqev3rzM3umCRdo1Nri/5f3%2Bd0Nd7BVERETkRBQsiojIhOvsHfRtB9/3vR32o01Hbc/RruPVTFHrAsAHdjfb5%2B8/6ILCfSMOXfH/7jton3LP/%2BC5ejvQ1O2Xleen%2B85zaA%2B57Uin3fVsnb3rf7bZ91ygSImkiIiInJykY05sXkRkRrn9qSN2653bYo9kotE%2BMTs92c6tzPFjLdKZTZof32KoTSPDY%2Bxp6LbdDV1%2B2Uj42wUlmW6a4XtZ5SWS3Gv3Dgz6wfrr2nptU02Hf814F8zJsxXlWcaY%2B89Xd7j3Gwpai7JTfRvKNZW5/u%2BePNDqt2U4pTlp9sZzS/08f0%2BAqh5Xz46PXTnbvnjjktgjERE5UwoWRWTGUrAoMr0oWBQRGVuqhioiIiIiIiIJFCyKiIiIiIhIAgWLIiIiIiIikkDBooiIiIiIiCRQsCgiIiIiIiIJFCyKiIiIiIhIAgWLIiIiIiIikkDBooiIiIiIiCRQsCgiIiIiIiIJFCyKiIiIiIhIAgWLIiIiIiIikkDBooiIiIiIiCRQsCgiIiIiIiIJFCyKiIiIiIhIAgWLIiIiIiIikkDBooiIiIiIiCRQsCgiIiIiIiIJFCyKiIiIiIhIAgWLIiIiIiIikkDBooiIiIiIiCRQsCgiIiIiIiIJFCyKiIiIiIhIAgWLIiIiIiIikkDBooiIjIlZuWl245pS%2B4%2Bbltpnrl1gswsyYs9MXwWZqXbFogL7ys3L7PKFBf6xiIjIdKFgUURExsQ55Tn2%2BlUl9vYLZtkHLq2ySxbkW3H29A6estOTbcWsbPvAZZVummVZ7rGIiMh0kXTMic2LiMwotz91xG69c1vskZypT1w1x958XqkVuQBxcWmWfcd9v197rMaePtgWW2NIZX66leakWVpKkn/c3jNg6SnJ1tU/aDUtPdbZN%2Bhfo8ytk5uRYlykBgaPWe/AMeOK1dLVb00uUXKZ6mKzDPc/Xovn69p6rbGz38py017yHgODZoeae6yhs88/xhz394VZqZaemmTu5a3LvW%2B6W58p6zW51wHBIEFhFNvMNlTlZ/jP/Devnm%2Bf%2Bc1%2Bu3tbo22r67S27gEfKM/KS/eviX73JrzmEbeNbGtJdppVFaT7%2Bay0odevb%2B%2Bzw%2B47kNPzsStn2xdvXBJ7JCIiZ0rBoojMWAoWx9a/vGGRXbWowAdLlywosNbufvvsbw/Yjzcdja0x5NOvXWAfuqzSZuWm%2B8cP7mmxeUUZtqWmwz7163228XC73XJ%2BmX3UZfwvd69DkNXiXuugC/a6XUD5s80N9v2N9favNyy24pxUW1qWZeXutQ65IOtz9x%2B0722ss49eMds%2BcOmL78Hf/9GPd/nfPPic%2B/ub1pTa/KJM6%2BgdsOdrO2xeYYZtqu7wgS6vg6f%2BZJ1dMCfPzwcP7m6xHzxXZwuKM%2B1PXzHXCAe5mN6/s9kFyNX2u13N9tbzZ9kn3XMEpahr7/Xb/S9uG/kst15Ybp%2B5bqHf7tUVOX6drz5abZ/82R4/L6dOwaKIyNhSfRkRETlj77ig3NbOzrXnXcD35Ueq7bG9LVael%2B5Lz6L%2B7tXz7bpziu2e7U128Rc2%2BNQ/MOhLAUEJHq/z/kurbGDgmP3dr/fam7652e7Z1uhLGsPr%2BbI6978lpVm28VC7ffT/22mv/9pmu8sFY395zTy70gWt/7uh7vh7EGC%2Bc325/fHL5vjt%2BjMXxLEdv3PB3Zu/vcXefec2/5p5GSnHSwIJQu/98Brr6Bm0T7hAk9e5ya37kAtul83Kssr8DB80fvbeAz5Q/L/37HfB7l7b3dDt3%2BtPXz7H7nWf8w/u2Gpv%2B%2B4L9usXGu0tLggmEF5Ukum2P8mXihJMvu%2Bu7fay/3jW/v3Bw/69RUREJgMFiyIicsYuXZDvq1LucYHS5ppO%2B9rjNb6q5iXz8%2BzCeS%2BWyl22MN/aewfs97ua7akDbT497ALLmtahqpd0EPOmc0utIi/NlzgS/D3gArIvP1JjB5p6fCljvMf3tdpPXTBIiSbB3qXz84%2BXDob3%2BMqj1dbRO2hXLS6wa5YV2mtdoEh100fde9%2B3o9kHjf9830Gr7%2BizZBfEgeqs//NMnX3%2B9wftJ88fPf5aj%2B9vdeuYDyqbu/ptX2O3X58pgSLbcMVCSkTNfralwQeMBLsErJ1uG65ZVuSr6aLPBcqULD57uMM2uKCXEkcREZHJQsGiiIicNtoEVuSl24rybDvS1mdbj3T6Kp2P7G11wV23rarIsYtcsBjWo1SPoGpLbWfsFQj22o4HSTnpKb5jnJ7%2BYy7o7LBdR7t80PmIC%2BoONHdbl3vtKJ7b616Pdn657m8vd8FoeX66by%2B4fm6evf%2BSSp%2Bo5kn7xDmFGXbxvHxfIkkbRv6WKqoEfT92AWGd%2BwwDsdYZDS5w/PaTtfZzF/Cx3ZSevmVtma8ySxvLkRRlp/n32e9ee2tthw86SZS6bnGPFxZnHi9JpVrt7qPd7vO6yFJERGSSUbAoIiKnjeCO0kICKIJEmsGfPzvXJwKh0tw0W1aW7QNFqnVmpib7Tl4aIx3N1Lb1%2BqAPyclJvmSud2DQ%2BgZeWopISWBfXMlit1sWShupwnqBCxB5jysXF9iHr6h6ScrLTPGBKp3l5Lh123v6E4K0rr6B4%2B%2BbmZZsK10QfJ77LLddXGF/9cp5dutFFb7UMMdt40jS3GfIcMFkq/tMg5GX5zPx/ilJSZZK0aTDtjS6oJSpiIjIZKNgUURETltRVqovcSvJSfPTH713tT39pxf49IbVpb4Ej2qor1lRHPuL8Ufg%2Bem799v6zz%2BTkG65fat95%2BkXO7kZzeKSLP95HvvE%2BfamNaX2o0319uqvbLLP/f6g79FURERkulOwKCIip412iufNzvEd2nzw%2Bzvski9sOJ4u/rcN9l0XmJXkpNraObn2XHW7dfYO2Ky8NF/SGMwvHhrCApSwUfJI6WC6S1FUMx2t%2BidDUFCdlaEw6ECGoTUosItPlES2dg%2B490yz7LSXlhDmZqT6toilbpvPrczx20YPq2//7jb7yqM17u%2BHhu8YDSWIDANCVdjo5vKZzqnI9u/PtoqIiEx2ChZFZMYiFiEDL6eHgOz6lSU%2B8Pvx8w32i60NxzuBCYl2h/0DZivKsnyJH8Hc0tIsX100uGZpkXutoQ5fGG6DXkMJChmGg3aHlF7SVpAeSKk%2BOhL%2B9m73twebevzfvnp5UewZ86/DMBakVS5g2%2BQC1wVu%2B1e6%2BVm5aUM9pF491wWY6b6DmzT3/ozxyP5BVVXaH9IustgFmGwL2zQSSh131nf6ardXuO2oyk/34ylSNZa2k5vcd1LtXktERGSyS/m0E5sXEZlRttV1%2BSEcaPcmp46A710XlvsB9Bk7cGd9V%2ByZFzHUBT1/krYf6bK23gGbV5Tp2zjS3nHdnDy72gWLjHVIIPnbHc1%2BvEM6zKEKK%2BMYznfpjeeW2JzCTF9qR0%2BnW2s7fdVWgjZ6U/VB6eAxq2vv86Wd55Tn2EIXgBLw8R6vX13iX5OSze3ud6fX1jWVuVbstq/SBXOUjlLVtDIvww639vhSUILOdXNyfdtF2l4uLcu2NVW5PrCc67af3lm3u89MMHndymIXDA/6Usc2FxTToQ3bz/Aaswsz/DiNVywq9Nt45zN19syhNv%2BdMMTHrqPd9ui%2BVl/aKaePdqB0jvTaCazyLCIy3SlYFJEZi%2BCGIRzoDVNOHaWBBE8v1HXab7Y1WlNX4vfY6QLxjNQkF1xl%2Bmqk33qy1ly8Z%2Be5v7tqSYFvz1jb2us7taFE7pdbG22H%2B13oLIdqpOtcQErpH5U2e11Q1t13zF440mnPumDuHBf8UZ3zMRdo0Wtq8NTBNst27xV9D16LoSvuePqIC9Tafc%2BkVH2l852LXYBB4LbTvUalCwRrWnrda7bZkwfa/NiLKyty7HwXNPI6dJLz93fv98EtQeELRzpshws%2Bl83K9t9Htgss2Rb2qwYXMK53f0OpJoEqQSVDivzqhUard0HtPBdE0mvqFvd5GDaDzyynjxsDly1kaJQXS5RFROTMJB2j6zoRkRnoN9ua/KDvz7jggrZscmoYjjDZhVN8dbTlGwkdfya5lVmF9cLj4Oa1ZfbRK6psmwua3n/XDv96PJ3EvxdXs%2B%2B/e6UfmP%2BuZ%2BvtPx8%2BfHw8xGP8i3v7%2BPcAl7vo7%2BzXIRyMrcZrPPkn62xXfaf992M1fuzFhNc5/hmGPjfvzX9%2BW9x/vEbYHhbx/YTXZ73B2HMIz7O%2B9r8zR2n1H14%2B2/7imrmxJSIicqbUWEdEZqzyvDRf6hMfVMjJIehhTMLRAkUQCNFxDet96w%2BW2x1vP8duXFPql5Eunpfnq6RSdbOqIMP%2B5Q2L7N4Pn%2BernvI8Hc7Q2QzjEx7t7Ds%2BCD6vRxru7cN7RhPLQFvLH966ygWfq%2BzqZYX%2BOcY9/JOXz7by3DR7%2BmC7S21%2BXf7mJa/D%2B7nl4XPz3uHx0Hu8uD1Mw/Ljfxt7DuF595SMgeWzqLqcGXskIiJjQSWLIjJjURXw3h1Nduud2xLG9JPx8dnrF/pqglQppSoo1s7O9e0Of/hsvW2r67TLFhTYp1873w%2BUT9tCBvQnmKO9H9VU793e5DubOV30Ukp7x7%2B8Zp6v%2Bkq1UaowMjbkD9w2/PC5ett6pDO2tkwVf/ryOXbbJZW%2BerKIiIwNtVkUkRkrI5VKgEn2%2B10tvqdOghEZXwxU39M/VDqXkpzk03YXIBKg0UaQ5wje6A2Vqp2ZaSm%2B5JcB%2BX%2B%2BucH/VmcSKILXokOcstw0/9psA/cKDrf02hcfPPyS9o8y%2BVExgM6E3rpull0yP3/U4VVEROTUqGRRRGY0Spa%2B/Ei13fHUEd%2B7JdUCRWRqIFBkeJMPXVZl77m4wldXFhGRsaPbbyIyo1G98Y%2BumuN79czJ0ClRZCqhFJGebilVXFY2NFaniIiMHeWMRGRGS05OsvzMFPvYVVV2%2BcKC2FIRmexoy7q4JNM%2BcdVsW1CUoeqnIiLjQGdWEZnR6AeVNmvr5%2BTZey6qsJvOK/Pj76l/VJHJix5yqQ3wjvXl9rpVJVaYTfvT2JMiIjJm1GZRRCSGQeWfPtBmP3r%2BqN2/s9n2N3Vbd99g7FkROZsIBik9rMxPt5Xl2fbK5UV2/cpiW1am3k9FRMaLgkURkTh17X32lUeq/bAah5t7rL13wDp6BqxTgaPIhKJH3PTUJMtKTbbczBSryEv31cXfsKrE1s3NtYLM1NiaIiIyHhQsiogMg8HVq1t67amDbfbg7hZ7eE%2BLbTjUpgHURSZQXkaKLSjOtNWVOXbRvDy7anGhLZ%2BVZdl%2BSJXYSiIiMm4ULIqIjICB%2BilVbOvut7YeShcHbWiEQJkqtm/fbrfeeqtxqfvLv/xLe%2BMb3xh7RqYC2hMzHmp2WrLluMAx3yXG3nSLRURkAihYFBGRaWvDhg22fv16Hyx%2B5StfsQ9%2B8IOxZ0RERORE1BuqiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSIOmYE5sXEZGIQXd67OwdtPaeAevqG7QB97h/8JgNuKQz59TwwtbNdssbr/e/16c%2B81m7%2Ba1vjz0jU0FaSpKlpyRbemqSZaWlWEFmiqUkJ8WeFRGR8aZgUURkBASJj%2B1rtd/uaLJnDrZbW0%2B/1bb1WnNXv/X069Q5FRzraLTezb/x86kLL7SUWUv8vEx%2BhISV%2Bem2qDTLlrp0wdxcu/m8MsvNUMAoIjJRFCyKiET0DRzzAeFPNzfYA7uabWd9l9W391p774ANDJr1uv9Rsuj%2Bk6lgsN%2Bsq2VoPiPHRYyZQ/MyJaSnJFlmWrIvVcxJT7ZZeel27Ypie/PaMltZnh1bS0RExouCRRGRGEoMnznYZj9xgeKje1ttT0OXtfUQJOo0KTIZJCWZLS/LtssW5ttrXdD48iWFVpabFntWRETGmoJFERGntXvAHtrTbHdtrLdfbGnwgaNOjiKTE1VRL12Qb29bN8tuWF1q%2BWrLKCIyLtQbqojMeJQcPl/Tbrc/dcR%2B8Fy9NSlQFJnUaE9MNfEvPnTYHtzT7GsA6Na3iMjYU7AoIjMagSLtEb/%2BeK09uLvFuvsGY8%2BIyGTWO3DMNtd02Ed%2BuMt21HdZn6qLi4iMOQWLIjKjtXQP2A%2BfrbenD7RZY2dfbKmITAX9LmA82tFn33nqiL1wpCO2VERExoqCRRGZsShVrGvr9e0UDzb3%2BJ5QRWTq4Iilh%2BJfv9Bgz9d0WGfvwNATIiIyJhQsisiMRSc2z9d22JMH23ybJxGZemiruKeh27bWdlpNa29sqYiIjAUFiyIyY1W39NrDu1uswwWKg%2BodQ2RK21HXabuPdsUeiYjIWFCwKCIzVkNnn2090qkB9kWmgX1NPXawWSWLIiJjSeMsisiM9dPNDfbJn%2B623Q1d6nb/LPraLctsxazs2KNT97n7D7nf8mjs0cS6ZlmR/eHlVTYrN83%2Bd2Od3f1Co68SKROvJCfNPnJFlf39axfEloiIyJlSyaKIzFg9/YN%2BTEUNqnh2nVeVa1csKjjtVJmfHnuliVfmApQL5%2BbZ5W47FhRnWnZ6SuwZmWgt7lhu61bbYxGRsaRgUURmLHo/7egdUKw4SdS399lDe1rsjqeP%2BKEQTjbtqO%2BMvYLMZP2Dx3wSEZGxo2qoIjJj3e4CjVvv3BZ7JGfLk3%2B8zi6cl2eP7Wu1Lz102L63oW7KBPBvPX%2BW/fPrF9mcogz73P0HffDKQPFydnzsytn2xRuXxB6JiMiZUsmiiIiIiIiIJFCwKCIiIiIiIglUDVVEZixVQ50cxroa6nsvrrClZVlW29prz1V3WFZasi0pzbKK/HTLdvM0a%2BvuG7TDLT32%2BP4221bXaZ29iR2jZKQm2%2ByCdLtqcaEtKMq03IwUS0426x84ZvUdffb0wTZbVZFjf/6KuaqGOkmoGqqIyNhSyaKIiEwrr1tZYh%2B%2BvMpuu6TSbl5bZh%2B4tNI%2BdFmVvd89fp9LH7ys0j5%2B1Wz7yBWz7V3ry%2B2CObmxv3xRngsMV1dku%2Bcr/Gt9zK3/h1e417x46DUYLoP5yxfkW1pKUuyvREREphcFiyIiMi1VFaTb1UsL7TUrii3JxXNbj3T60svnqzt8L7iUPr7n4nLfSQ1j9AUpyUm2qCTTbnHL/%2BKaubamKscPs/KC%2B/vH3d8/V91uhIevW1lsr1tVYjkusBQREZmOFCyKiMi0VJydZnMKMuz5mg572x0v2LVf3WQ3fGOz3fKdrb7K6JG2XsvLTPVjNVLCGBS4ZS9bXOgCyQrLTEu2w8299tl7D9gbvr7Z3vjNzXbdfz9vn7p7n6/imp6S7KunioiITEcKFkVEZFKg/d%2BnXjPfHvjYWnvwJBNVQykFHA7tEJ852GYf/P4O21HXaV19gz7VtPbanRvqbJML9hjEvSQn1ZaUZcX%2ByuyieXl21eICHzTStvELDxyyX73Q6IPL8Bq/do%2B/9WStPbSn2ZcyioiITEcKFkVEZFLIz0yxFbOy7cpFBSedFhZnWk768CV7bT0Dtruh25473G6dLsAL%2BgaOWXVLrx1s7rF2t05WWoqVZKfGnjVfPXW5246BwWO2x/39Ewda3brdLxnwvb69z%2B7b0eSCxRYfUIqIiExHChZFRGRSONrRZw/vbbH/eeaIffck04ZD7dbc1R97hZdq7R6wQy4gHAntFvsGBi3FXQnp%2BTQoyU7zVVh73XMEic2d/T7AjEewSTBKwCkiIjIdKVgUEZFJYWd9l3354Wp753e3nXT63w11PmgbTlt3vx8e41SH4chITfKpf9Cs0QWK/aOMMMUwGlRLFRERmY4ULIqIyLTU6wI5qqKeqqJYySJtHp862GYdvQoGRURkZlKwKCIi09KpligGLV39vmorg/mvrcr1A/mLiIjMRLoCioiIRHT1D1q3S%2BmpyTanMMPSRxl0PzU5yQ%2BvISIiMh3pCiciIhJBqSIpzQWJDOyfn5nqg8J4FXnpNrcow3LSdSkVEZHpSVc4ERGRiJ31nbatrtNSkpJsQVGmranKdUHjS0sYGa6DwfxJ2SMM3SEiIjLVKVgUERGJ%2BM22Jvvp80etqavfVzH99Gvm2zsumGVzC18c/P%2BSBXn2zvXl9orFhbElIiIi00/SMSc2LyIyo9z%2B1BG79c5tsUdytjz5x%2Bvswnl5flxEhrqoa%2B9zS0/%2B0nT/rmb7%2BeYGP%2BYifvSeVXb1skLbUttp//HQYT%2B8xnA%2Bf8Niu2lNqRVmp9r9O5vtTd/cEnvGbO3sXHvPRRV260Xllp6SbEfaev0A/bVuSvXURSVZNqcgw7LTk43yxuyMFPvc/QftO26f2lzTMfQiMuE%2BduVs%2B%2BKNS2KPRETkTKlkUUREJoX8zBQ7pzzbXra4wKXCk04ry3OsyAV8Y2lvY7fd6YLMf7rvoB8%2BIyMt2da7gPbac4rttSuKbXZBuj1X3W6P7mu19l4Nyi8iItOTShZFZMZSyeLk8A/XLvAldafrif2t9rudzbaldqhE749eNsdWVWTboeYeu2d7kz3mArrhvP2Ccl%2BiSQc1W2s77d8eOBR7Zgid2hRmpdr1K0tspXu90pw0y0gdusdKSeOTB1otxa1z8bx8K3HP/XJrgz20p8W/r5wdKlkUERlbChZFZMZSsCgyvShYFBEZW6qGKiIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiIiIiIJFCwKCIiIiIiIgkULIqIiIiIiEgCBYsiIiIiIiKSQMGiiIiMidyMFFtTlWNvPq/MrltZbIVZqbFnpq/MtGRbVJJpb1lb5qc8FhERmS50VRMRkTGxqDjTbru40m5/2wr78k1LbVVFjqWnTu/LTJELiK9eWmTfe/dKe8WSwhkRIIuIyMyhYFFERMbERfPzbXVljjV09tmcwgx73apiW1KaGXtWREREppqkY05sXkRkRrn9qSN2653bYo/kTP3DtQvsVcuKrK69z9bNybVtdZ32%2Bd8fsrtfaIytMeTdF1bY610gWZid5h8/X91us3LTbW9jt337yVrbdbTLXr6k0N50bqmtcsHn4OAxa%2B8dsHr3ur0Dg/bo3lZ7YFezfejyKsvPTLHZBRlW5F6rvr3XfvBsvf1uZ7O90f3t61aWuPcYKunr6Bmw/3z4sN2zvck/xocuq7IrFxdYeV66dfcN%2BPeflZtme452%2B/Xud%2B%2BB/7p5qS0ty/bzwSa3zQ/sarF5RRn2h247VpRn2wtHOu2%2BHU32o01HbePhdrt6aaHdvLbMytxnQ7MLon/vXvMut418ltesKLL3XlRhdR19trB4KKj%2B%2BZYG%2B%2BqjNX5eTt3HrpxtX7xxSeyRiIicKZUsiojIGXuZC7pWuoDpYHOPfW9jnQ%2BWFrgAiOAriuDppvNKLS8j1e7Z1ujTeVW5PnA6ryrHSnPSfOD0jvWzbO3sXKtr67UnDrS69VPslcsKfRA53z2flUb7yFy73gWEBIu7XYD5wO4W29/UY289f5ZbXuze7djx98hOS/bLX7/KBZBZqXbD6hK75fwy/7rPHGxzwV%2BHrZ%2Bb596jyE8JICvz0%2B3/vGa%2BnesC1n0NXf51WLfSPUcgyvJ2F4Ruqe3wn40pr8Md2FctL7I/vKLKt%2BN8zn0Xm1zKSU%2Bx91xcYdcsLYq9foa9zH2e1ywvdn/bab/d0Ww76rr8a4mIiEwGChZFROSMvXZFsVW54GdHfaf9xgVVlK6lJCXZilnZvvQtIECrckHYfTub7J9/d9AnShJ7B4YquRBcXeMCtkvnF/jllAZ%2B4YHDduczddbdd8zSU5L8ekFBZqo9eaDNvvTQYfv6YzW2v7Hb3uyCUda6w/1NeI87nj5ii0ozfec7BKHvuKDcqlyQ%2BaALMP/tgUP2H%2B597nPBWm//MUtJHnqP9JRkt%2B2Z7vM0udev9q/z5Ueq7ZF9LVaUnepLNdnGu93zbD0lqD/f2mDd/YO%2Bg5%2B1Lpj91dZGt/2H7AsPHvLPzXevR/XcxSUvVs/tHzxmtz9V69cLpZkiIiKTgYJFERE5bcRV9ABKddGO3gHb09BtjZ39vjrp4ZYeu2JRge/4JaxHsLSjvsue2N8aewUXZLngcqcLuhBK/fpcAHXv9iZ7ZG%2BrNXT02bfc62090uHeY9CvF1Cyt/FQm22u6bDs9GRfKkiAR8kmyxaXZvn0yL5Wq27ptWWzslxgW2Tnz8mzPe49qU5a29rrn/vbX%2B31JaPHfOhntr%2Bp22773nb7zD373fJuX1Janpvm1%2B%2BLBbfDoUTyHKql1nX6oJjXPNDU44PRzbUddoF777luG9HVN2hbaztdIPzSzyUiIjIZKFgUEZHTRnXQ86tybWFJpm050mmPu6As2N3Q5Uv%2BCNb8erNzXUCXYrVtvS71xdZyQVljtzW7ABMp7qpUnJPq2xD2DCQGhj39L11GgBpKJfMzUn31T0r8/vpV8%2B3pP1n3knTTmlLLde9PtdUCt05TV791DhN8UroY78OXV9k9H1pj93x4jX3yFXN9NdKRUCKZlepev6PfBgZiCx0%2B0wsuMKTkMpSQEnTWtvaMGnyKiIicLQoWRUTktFESeMu6WVaWk2ZvOrfEvvbW5XavC6hIr1pW7McevGBunu/sJdUFSYRIdKs2GOlbjTgplOaZW4N//tGLq3hxD71oF21J7sXTXBBGAPgfDx22m7%2B99SXp%2Bq9ttvd/f4f94Ll6vy79u734vkOiSyhJ/PbbVtivPnCuCzTL7OE9Lfa%2B7%2B3wpZyUdo6Ez0iKLyvkdalyynMvOua%2Bi9isiIjIJKNgUURmLIYAzJjm4wCON9oYXrWowPY1dvv2eT97/qj9dnuTT1988JBtONRu8woz7JIF%2BX4dSgYLslJeMh5hWW6a7/wFBJGU7lE6lxrXPjHT/VYEnCMhEAtBXKObPnu43X67w21LJFHyubeh2zr7Bv17xv/%2BmakplubeIy8zxXfY84ZVJdbaPeADTILEB/c02/a6Tt8ucSRsB7225mYkW3Lk5flMDCky4D4j64iIiEx2yiWJyIyV6nLyOenJcSU9crJKctJ8D6aUHtLu8N8fPGz/9LuDL0n372zyVS5XVWT7Noz1Loij3SId3wS0M6SzGXT1Dvr2hvzNsrIs3zkOPZlShZXqo1lufiRUSaWzm9aeflvnXpPXDea616F0k0QVUgLXSvd6BG8EvPSKes2yQh%2B4Uk2U6qpsEx3Z0L6SDnYe2tPitiVWnXaU7Wju6reatl6bW5jpq%2BBSFZfE51lVkeO%2Bh15rilW7lbHD7xY6JxIRkbGhYFFEZiwCEgIFRYunh05cGI4iPTXZfrGl0Y%2BrGG9vY481uMBocUmWD9Ker%2B7ww2kw1AZBJukaF8DNj/WYSjvCHz5Xb209A37MRsZaXDsn14%2Bft8QFXqOVBFMC%2BOsXGu2xva0%2BiL1xTenx9%2BB1/uZV8%2B3jV832jxkPkWE6LpmfbxfNy7OL5ufZP1630Cry031F1IHBY9bpgs/W7n7fKQ6flb%2BjtPHVK4ot3wV/bAslnZSW0uaQ1%2BOzNXb2%2BQCz2AWalExeuiDfLluYb69fXeKDz9%2B5AHpPg4bIGGsE8JQ%2Bi4jI2En5tBObFxGZUejt8on9bb53S1UKPHWU3N10XpnVuO/vh88d9QPNx6M6JgPuX7W40Jey0ZaQDP2Na8rso1fMtg9cWmVHO/p94HWkrdfudUHccy6gJPha74K4t19Qbm9ZW2YLXbBZ39Hrq6huq%2BvyPYi%2BxgVtRVmp9vDeFt/zacBg%2BeUu6CNYDO/x6hVFPpBkGA4G7X9sX6stKM6yVy8vstsuqbDXrSq1FhcYEnDQM%2BrjLtgjFWWl%2BbEc37G%2B3N5/SaXvRfWPf7zLLnXBH9VVCZAZjP9aty2vdK%2B1tCzLDjb12K/ce7X3Drptn2XvvrDCD9lR4YJlhuj47tN1dqC5x48veeUihgjptkfd9hDsyunjZsSViwt8D7wiIjI2ko7Rwl9EZAbaeKjd/uvRavvmE7W%2BJElOzSyXOWe8QNrfEbxRDTQewRfVOanuSdXLnUc7fdDEY9rw4bzZOb6EkqDrIz/c6UsVyfjPdevQtpHLFO/xF9fM89VFKXn878dqbFlZtq%2BWSpVSeliNoiRzdmH68fcANwcOuyCN9oogsCOQ5TX4/dvd9n/zrct9m8SvP15rv9/VbPMZLiMv7fjr8BnZTkoaKVGkSinL1lTm%2BKFBqII6VM20zwesVLkNpV20c6TnV4JienBliI2F7vUb3Lr7G3tGbQcpJ0YV4w9eVuVvLoiIyNhQsCgiMxaZdgZcf99d2zV0wQT5yBVVvq3okwdafeke/u7V8/3YirQJZExDSvKo8slg91TnpJorA/nf8fYVvvSNAey//2y9/9vTQSD6zvXlfp6xHJ%2BrbvdB6VWLC%2Bxfb1hsd20cCkYJLmXq%2BMRVs%2B19l1Taahe4i4jI2FA1VBGZsSgJordKeu6kyqBKF8cfbQZfuazIlzZmpCT7Dl%2BuX1Vide19dt/OZj94/eWLCuzdF1X4UkR6LF07O9detrjQ5hVnugCzxR7Y3eLXP120JbzunGJfFZZeTxmf8aJ5%2BfaOC8rtUEuv7/mUTnZkaqDJMSW4f%2BB%2Bv8sW5PubCyIiMjZUsigiMxrVF2nH9r8bjtih5h6NeTfOrl9Z7NvwMfZiUNPaY/9030H7%2BZaG2BKzL795qe8UJs8FcsHXH6%2BxH2066quJjoWf3Lbazo0rhbrhG5tf0v5RJjfGy6SKMCWK77%2B0wrcDFRGRsaNgUURmtP6BY344hw/ctcMe2N3s28vJ%2BKEHWkp0o%2BMlEqAzkD6d4QT0UksHMgQDAW36evqHeiodC/RoGl8IRTVXjYE4ddAxEkOq3P625f4GxGhDq4iIyKlTsCgiMxonQIKPe7Y12RceOOR74xSRyS8tJckWlWTZ%2By6p8FWI6XBJwyyKiIwt3YITkRmNvCWlXIyF964Ly33nKpQ4icjkRaC4soJedMvshtWlfvxKBYoiImNPJYsiIjEMpv7o3lbf0%2BYje1v8EAgMjyAiZx9VkqmaXJab7oc9edXyInujCxRXVmTH1hARkbGmYFFEJA6d3vz7g4ftNy80WnVrr3X3DcbayylwFJlIlBZS8k8Pp4zZWeoCxSsXFdiNa0rtonl5fsgTEREZPwoWRUTi0L9JV9%2BAHWzqsWcOtdkje1vt8f2t9tzhdvWWKjKB6OhoXlGmrSzPtnVzcu0KFyiumJVtBS5IjO8ASURExp6CRRGREVCSSO%2BYVE9t7ur38zphikwcShXp4ZSgsSAzxYqy0ywnPdmSFSWKiEwIBYsiIiIiIiKSQL2hioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIpJAwaKIiIiIiIgkULAoIiIiIiIiCRQsioiIiIiISAIFiyIiIiIiIhLH7P8H8EcWx9Tag%2BYAAAAASUVORK5CYII%3D)

In [ ]:
class FederatedFlow(FLSpec):
    def __init__(self, model=None, optimizer=None, rounds=3, **kwargs):
        """
        Initialize the class with the given model, optimizer, and number of rounds.

        Parameters:
        model (torch.nn.Module, optional): The model to be used. If None, a ValueError is raised.
        optimizer (torch.optim.Optimizer, optional): The optimizer to be used.
        rounds (int, optional): The number of rounds for training or processing (default is 3).
        **kwargs: Additional keyword arguments to be passed to the superclass initializer.

        Raises:
        ValueError: If no model is provided.
        """
        super().__init__(**kwargs)
        if model is not None:
            self.model = model
            self.peft_params = get_peft_model_state_dict(self.model)
            self.optimizer = optimizer
        else:
            raise ValueError("No model inputted")

        self.rounds = rounds
        

    @aggregator
    def start(self):
        """
        Initialize the model and set up the collaborators for federated learning.

        This method performs the initial setup for the model, including setting the
        collaborators, initializing private variables, and starting the first round
        of the federated learning process.
        """
        print(f"Performing initialization for model")
        self.collaborators = self.runtime.collaborators
        self.private = 10
        self.current_round = 0
        self.next(
            self.aggregated_model_validation,
            foreach="collaborators",
            exclude=["model"],
        )

    def launch_gradio(self):
        """
        Launch Gradio interface to show federated learning statistics
        """
        interface = self.get_gradio_interface()
        interface.launch(share=True, inline=True, debug=True)  # Set share=True to create a public link

    
    @collaborator
    def aggregated_model_validation(self):
        """
        Perform aggregated model validation for a collaborator.

        This method loads the model, applies the PEFT configuration, and evaluates
        the model using the provided training and evaluation datasets. The validation
        score is then stored and the next step in the process is triggered.
        """
        print(f"Performing aggregated model validation for collaborator {self.input}")
        self.model = AutoModelForCausalLM.from_pretrained(
            checkpoint_path, return_dict=True, **model_kwargs
        )
        self.model = get_peft_model(self.model, peft_conf)
        set_peft_model_state_dict(self.model, self.peft_params)
        trainer = SFTTrainer(
            model=self.model,
            args=train_conf,
            peft_config=peft_conf,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            max_seq_length=sequence_max_length,
            dataset_text_field="text",
            tokenizer=tokenizer,
            packing=True,
            data_collator=transformers.DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
            ),
        )

        trainer.remove_callback(PrinterCallback)
        out = trainer.evaluate()
        self.agg_validation_score = out["eval_loss"]
        print(f"{self.input} value of {self.agg_validation_score}")
        self.next(self.train)

    @collaborator
    def train(self):
        """
        Train the model for a collaborator.

        This method trains the model using the provided training and evaluation datasets.
        The training loss is stored, the model is saved, and the next step in the process
        is triggered.
        """
        trainer = SFTTrainer(
            model=self.model,
            args=train_conf,
            peft_config=peft_conf,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            max_seq_length=sequence_max_length,
            dataset_text_field="text",
            tokenizer=tokenizer,
            packing=True,
            data_collator=transformers.DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
            ),
        )

        out = trainer.train()
        self.loss = out.training_loss
        trainer.save_model()
        self.training_completed = True
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        """
        Perform local model validation for a collaborator.

        This method evaluates the model using the provided training and evaluation datasets.
        The validation score is stored, the PEFT parameters are updated, and the next step
        in the process is triggered.
        """
        trainer = SFTTrainer(
            model=self.model,
            args=train_conf,
            peft_config=peft_conf,
            train_dataset=processed_train_dataset,
            eval_dataset=processed_test_dataset,
            max_seq_length=sequence_max_length,
            dataset_text_field="text",
            tokenizer=tokenizer,
            packing=True,
            data_collator=transformers.DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
            ),
        )
        out = trainer.evaluate()
        self.local_validation_score = out["eval_loss"]
        self.peft_params = get_peft_model_state_dict(self.model)
        print(f"Doing local model validation for collaborator {self.input}")
        self.next(self.join, exclude=["training_completed", "model"])

    @aggregator
    def join(self, inputs):
        """
        Aggregate the results from all collaborators and update the model.

        This method calculates the average loss, aggregated model accuracy, and local model
        accuracy from all collaborators. The model parameters are updated using Federated
        Averaging (FedAvg), and the next round of the process is triggered if applicable.
        """
        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(
            input.agg_validation_score for input in inputs
        ) / len(inputs)
        self.local_model_accuracy = sum(
            input.local_validation_score for input in inputs
        ) / len(inputs)
        print(
            f"Average aggregated model validation values = {self.aggregated_model_accuracy}"
        )
        print(f"Average training loss = {self.average_loss}")
        print(f"Average local model validation values = {self.local_model_accuracy}")

        self.model = FedAvg([input.peft_params for input in inputs], self.model)
        self.peft_params = get_peft_model_state_dict(self.model)

        self.model.save_pretrained("./aggregated/model")
        tokenizer.save_pretrained("./aggregated/tokenizer")
        self.current_round += 1
        if self.current_round < self.rounds:
            self.next(
                self.aggregated_model_validation,
                foreach="collaborators",
                exclude=["model"],
            )
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        """
        End the federated learning process.

        This method marks the end of the federated learning process and performs any
        necessary cleanup or finalization steps.
        """
        print(f"This is the end of the flow")


You'll notice in the `FederatedFlow` definition above that there were certain attributes that the flow was not initialized with, namely the `train_dataset` and `eval_dataset` for each of the collaborators. These are **private_attributes** that are exposed only throught the runtime. Each participant has it's own set of private attributes: a dictionary where the key is the attribute name, and the value is the object that will be made accessible through that participant's task.

Below, we segment shards of the MedQuAD dataset for **three collaborators**: Portland, Seattle, and Chandler. Each has their own slice of the dataset that's accessible via the `train_dataset` or `eval_dataset` attribute. Note that the private attributes are flexible, and you can choose to pass in a completely different type of object to any of the collaborators or aggregator (with an arbitrary name). These private attributes will always be filtered out of the current state when transfering from collaborator to aggregator, or vice versa.

In [ ]:
# Setup participants
_aggregator = Aggregator()
_aggregator.private_attributes = {}

# Setup collaborators with private attributes
collaborator_names = [
    "Portland",
    "Seattle",
]
_collaborators = [Collaborator(name=name) for name in collaborator_names]

for idx, current_collaborator in enumerate(_collaborators):
    # Set the private attributes of the Collaborator to include their specific training and testing data loaders
    current_collaborator.private_attributes = {
        "train_dataset": processed_train_dataset.shard(
            num_shards=len(_collaborators), index=idx
        ),
        "eval_dataset": processed_test_dataset.shard(
            num_shards=len(_collaborators), index=idx
        ),
    }

local_runtime = LocalRuntime(
    aggregator=_aggregator, collaborators=_collaborators, backend="single_process"
)
print(f"Local runtime collaborators = {local_runtime.collaborators}")

Now that we have our flow and runtime defined, let's run the experiment!

In [ ]:
flflow = FederatedFlow(model, rounds=2)
flflow.runtime = local_runtime
flflow.run()

Now that the flow has completed, let's get the final model accuracy:

In [ ]:
print(f'\nFinal aggregated model accuracy for {flflow.rounds} rounds of training: {flflow.aggregated_model_accuracy}')

## Run Experiment with Gradio interface 

In [ ]:
import gradio as gr

# Define the function to start the federated learning process with user-specified rounds and display the output
def start_federated_learning(rounds):
    """
    Start the federated learning process for the specified number of rounds.

    Parameters:
    rounds (int): The number of rounds for the federated learning process.

    Returns:
    tuple: A tuple containing the aggregated model accuracy, average loss, and local model accuracy.
    """
    flflow = FederatedFlow(model, rounds=rounds)
    flflow.runtime = local_runtime
    flflow.run()
    return (
        flflow.aggregated_model_accuracy,
        flflow.average_loss,
        flflow.local_model_accuracy
    )

# Create the Gradio interface with an input for the number of rounds and display the output
start_interface = gr.Interface(
    fn=start_federated_learning,
    inputs=gr.Number(label="Number of Rounds", value=2),
    outputs=[
        gr.Textbox(label="Final Aggregated Model Accuracy"),
        gr.Textbox(label="Average Training Loss"),
        gr.Textbox(label="Average Local Model Accuracy")
    ],
    title="Start Federated Learning",
    description="Enter the number of rounds and click the button to start the federated learning process and display the final results."
)

# Launch the interface inline in the Jupyter notebook
start_interface.launch(share=True, inline=True)

# Congratulations!
Now that you've completed this notebook, check out our [other tutorials](https://github.com/securefederatedai/openfl/tree/886704508b8b3b0638372003d72e0bcf7f2e7114/openfl-tutorials/experimental)

- Using the LocalRuntime Ray Backend for dedicated GPU access
- Vertical Federated Learning
- Model Watermarking
- Differential Privacy
- And More!
